# Project 3 — Truy vấn đa phương thức có giải thích bằng đồ thị

**Pipeline:** `truy vấn → CLIP → top-K ứng viên → điểm đồ thị (GNN + đường đi) → xếp hạng lại → giải thích → fidelity`

Điểm số cuối của một ứng viên:

$$\text{score} = \alpha \cdot \cos_{\text{CLIP}} + (1-\alpha)\big[\lambda \cdot s_{\text{GNN}} + (1-\lambda)\cdot s_{\text{path}}\big]$$

* `s_GNN` — GraphSAGE/GAT tự cài trên đồ thị dị thể (ảnh / lớp đối tượng / khái niệm).
  Mục 1.7.2 thay nhánh này bằng **SCENIR** (ICML 2025): Graph Autoencoder tự giám sát trên scene graph
  của từng ảnh — phương pháp mới cho cấp độ 2 (`CFG.gnn_type = "scenir"`).
* `s_path` — tổng **cộng tính** theo từng đường đi nối khái niệm truy vấn tới node ảnh.
  Vì cộng tính nên bỏ một đường đi làm điểm giảm đúng bằng phần đóng góp của nó ⇒ giải thích
  đến thẳng từ cơ chế xếp hạng, không phải hợp lý hoá sau khi có kết quả.
* `α`, `λ` chọn trên **validation**, không bao giờ trên test.

**Chạy trên Kaggle:** Settings → Accelerator → GPU P100 (hoặc T4 x2), Internet → On.
Notebook tự dò dữ liệu trong `/kaggle/input`; không thấy thì chạy chế độ `SYNTHETIC` để smoke-test toàn bộ mã.

## Cấu trúc notebook

| Phần | Nội dung | Vị trí |
|---|---|---|
| **1** | Nền tảng + Flickr8k: cấu hình, đồ thị, GNN cổ điển, huấn luyện, giải thích, fidelity, ablation | mục 1.1 – 1.17 |
| **2** | GNN của SCENIR thay GNN cổ điển (Flickr8k) | Phần 2 |
| **3** | Action Genome (dataset thứ hai): liên kết khái niệm, ablation, SCENIR | mục 3.1 – 3.5 |
| **4** | Cấp độ 4(b): truy vấn văn bản → video (MSVD + MSR-VTT) | mục 4.1 – 4.16 |


> **Bản v5 (chạy lại Phần 1–3).** Thay đổi so với bản trước:
> 1. Node **bộ ba cấp ảnh** + mẫu đường đi `P6` (mục 1.2b); nhãn đối tượng Flickr8k bằng linear probe ngoài-fold.
> 2. Đối chứng `no_triples`, `shuffled_triples`; **bộ đánh giá quan hệ khó** (cùng đối tượng, khác quan hệ) cho Flickr8k và AG.
> 3. Sửa đối chứng fidelity: giải thích luôn nhỏ hơn toàn bộ đường đi (bản trước in `sparsity 0.00` cho các ca ≤ 5 đường đi).
> 4. Dùng dataset Kaggle thay Drive; cache CLIP của lần chạy trước được dùng lại nếu output cũ được Add Input.
> 5. **Phần 4 (MSVD → MSR-VTT) không chạy lại** (các cell có `%%script true`); kết quả của nó nằm ở version trước.
>
> **Bản v6 (version 3).** (a) Flickr8k chạy với backbone YẾU hơn — CLIP **ViT-B/32** — để đo hiệu quả của đồ thị theo độ mạnh của backbone (kết quả ViT-L/14 nằm ở version 2). (b) AG thêm 2 đối chứng: `composite_swap` (gán nhầm quan hệ, giữ nguyên vật) và **truy vấn phản thực tế** (đổi vị từ trong truy vấn). (c) Không chạy lại oracle, SCENIR, liên kết ngữ nghĩa AG (đã có ở version 2) để tiết kiệm GPU.


---
# PHẦN 1 — Nền tảng và Flickr8k

_Mục 1.1 – 1.17: toàn bộ mã dùng chung (đồ thị, mô hình, truy vấn, đánh giá, giải thích) rồi chạy trên Flickr8k._


In [1]:
import os, sys, json, math, time, random, subprocess, hashlib, itertools, glob, re
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np

RUN_ON_KAGGLE = Path("/kaggle/input").exists()
print("Kaggle environment:", RUN_ON_KAGGLE)

Kaggle environment: True


In [2]:
# Cài đặt phụ thuộc. Trên Kaggle torch/numpy đã có sẵn; chỉ cần open_clip + nltk.
def pip_install(pkgs):
    for p in pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)

if RUN_ON_KAGGLE:
    pip_install(["open_clip_torch", "nltk"])   # không ghim phiên bản: Kaggle đổi torch thường xuyên

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

torch 2.11.0+cu130 | device: cuda
GPU: Tesla T4 | 15.6 GB


## 1.1 Cấu hình

Mọi siêu tham số nằm ở một chỗ. `CFG.n_images` là núm vặn chính cho thời gian chạy:
4000 ảnh chạy khoảng 12-15 phút trên P100 (gồm cả trích đặc trưng CLIP).

In [3]:
class CFG:
    # --- dữ liệu ---
    dataset      = "flickr8k"
    n_images     = 8000
    min_obj_freq = 5           # lớp đối tượng hiếm hơn ngưỡng này bị loại — ĐẾM TRÊN TRAIN
    max_concepts = 4000

    # --- nhãn ảnh Flickr8k ---
    # "tagger": mọi ảnh gắn nhãn bằng CLIP zero-shot trên từ vựng rút từ caption TRAIN
    #           (không ảnh val/test nào được dùng caption của chính nó).
    # "oracle": nhãn rút từ caption 1..4 của chính ảnh (cách cũ) — chỉ để so sánh, KHÔNG báo cáo là kết quả chính.
    flickr_annot = "probe"     # v5: linear probe khởi tạo từ văn bản CLIP, ngoài-fold (1.2b); "tagger" = zero-shot cũ
    tag_k_grid   = [3, 5, 8, 10, 12]          # số nhãn/ảnh — chọn trên VAL theo F1
    tag_prompts  = ["a photo of a {}.", "a photo of {}.", "a photo containing a {}."]

    # --- chia dữ liệu (chia theo ẢNH, không theo caption) ---
    split_seed   = 20240101
    val_ratio    = 0.15
    test_ratio   = 0.20

    # --- CLIP ---
    clip_model   = "ViT-L-14"                 # trước: ViT-B-32. Tagger + baseline mạnh hơn rõ rệt
    clip_weights = "laion2b_s32b_b82k"
    clip_bs      = 128                        # ViT-L tốn bộ nhớ hơn; L4 24GB chạy 128 ổn

    # --- đồ thị ---
    max_path_len      = 3
    scene_min_count   = 2      # cạnh scene phải xuất hiện ≥ ngưỡng này trong caption TRAIN (trước đây = 1)
    scene_top_k       = 10     # mỗi lớp đối tượng giữ tối đa k cạnh scene mạnh nhất
    cooc_min_count    = 10
    cooc_top_k        = 10
    use_wordnet       = True
    isa_min_depth     = 4      # bỏ khái niệm quá chung (entity, object, whole, ...)
    use_idf           = True   # trọng số đường đi = IDF của node đích (tính trên TRAIN)
    drop_redundant_indirect = True   # bỏ đường đi gián tiếp tới khái niệm ảnh đã chứa trực tiếp

    # --- v5: quan hệ CẤP ẢNH (node bộ ba, mẫu P6) — xem mục 1.2b ---
    use_triples      = True
    triple_min_count = 5                      # bộ ba phải xuất hiện ≥ ngưỡng này trong caption TRAIN
    triple_k_grid    = [1, 2, 3, 5, 8]        # số bộ ba/ảnh — chọn trên VAL theo F1
    triple_tau_grid  = [0.0, 0.05, 0.1, 0.2, 0.3]
    probe_folds      = 5                      # dự đoán ngoài-fold cho ảnh train
    probe_steps      = 300
    probe_lr         = 1e-3
    probe_wd_init    = 1e-3                   # phạt ‖W − text‖²
    rel_neg_min      = 3                      # bộ quan hệ khó: tối thiểu / tối đa số âm mỗi truy vấn
    rel_neg_max      = 30

    # --- sửa ablation âm ---
    # (1) IDF: "taxonomy" = đối tượng đếm df qua P0, khái niệm qua P2 (isa thật, trước khi xáo).
    #     "all" = cách cũ: đếm qua MỌI mẫu ⇒ thêm quan hệ làm df phình, IDF của khớp trực tiếp
    #     tụt về ≈0 — đây là lý do chính khiến 'none' thua 'no_relations'.
    idf_from        = "taxonomy"
    # (2) Cạnh related: xếp hạng theo NPMI thay vì đếm thô (đếm thô ⇒ mọi lớp nối vào các hub
    #     phổ biến như person/man/floor). Trọng số cạnh = P(đích | nguồn) trên TRAIN.
    cooc_score      = "npmi"         # "npmi" | "count" (cũ)
    cooc_min_npmi   = 0.10
    edge_weighting  = True           # trọng số đường đi = idf(c) × Π P(đích|nguồn); False = đếm (cũ)
    # (3) GNN tách theo loại cạnh + cổng học được, khởi tạo gần như đóng ⇒ mô hình bắt đầu
    #     từ 'no_relations' và chỉ mở quan hệ khi nó thực sự giảm loss.
    gnn_typed       = True
    gate_init       = -1.5           # sigmoid(-1.5) ≈ 0.18 cho cạnh scene/related/isa; contains mở sẵn
    path_init_indirect = -1.0        # path_w khởi tạo cho P1..P5 (softplus(-1) ≈ 0.31; P0 = 0.69)
    scalar_lr_mult  = 10.0           # lr của cổng + path_w = lr × hệ số này
    # (4) Huấn luyện giống lúc suy luận: mẫu âm khó lấy từ CLIP top-K trên gallery TRAIN,
    #     và có z(s_clip) trong logit ⇒ đồ thị học phần CLIP còn thiếu, không học lại CLIP.
    hard_neg_k      = 31             # 0 = tắt (chỉ dùng âm trong batch như cũ)
    train_with_clip = True
    # Hệ số của z(s_clip) trong logit HUẤN LUYỆN có học không. AG (CLIP yếu, MRR 0.12): học được
    # thì tốt hơn (clip_w → 0.07). Flickr (CLIP mạnh): học làm clip_w → 0.4 và MRR giảm.
    # CHỌN THEO VAL MRR cho từng bộ dữ liệu, không theo test.
    learn_clip_w    = True

    # --- GNN ---
    gnn_type     = "sage"
    gnn_dim      = 256
    gnn_layers   = 2
    gnn_dropout  = 0.1
    lr           = 1e-3
    weight_decay = 1e-4
    epochs       = 20
    train_bs     = 128
    temperature  = 0.07

    # --- truy vấn / xếp hạng ---
    rerank_k     = 100

    # --- thí nghiệm ---
    seeds        = [0, 1, 2]
    alpha_grid   = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    lambda_grid  = [0.0, 0.25, 0.5, 0.75]     # bỏ 1.0: λ=1 ⇒ s_path có trọng số 0 ⇒ giải thích không trung thực với điểm cuối
    ablations    = ("none", "random_edges", "no_relations", "no_scene", "no_related", "no_concept")

    adaptive_fusion = True
    fusion_steps    = 300
    fusion_tau      = 0.2
    fusion_wd       = 1e-2

    # --- xAI ---
    n_explain_queries = 20
    top_m_paths       = 5

    # --- đường dẫn ---
    out_dir = Path("/kaggle/working") if RUN_ON_KAGGLE else Path("./work")

# Flickr8k chạy thêm 2 đối chứng của node bộ ba; AG không có bộ ba dự đoán nên giữ CFG.ablations.
FLICKR_ABLATIONS = ("none", "random_edges", "no_relations", "no_triples", "shuffled_triples")   # v6: rút gọn

CFG.out_dir.mkdir(parents=True, exist_ok=True)
(CFG.out_dir / "cache").mkdir(exist_ok=True)
(CFG.out_dir / "results").mkdir(exist_ok=True)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 1.2 Nạp dữ liệu

Notebook dò tự động các tệp Visual Genome trong `/kaggle/input` theo tên
(`objects.json`, `relationships.json`, `image_data.json`, `region_descriptions.json`).
Không tìm thấy ⇒ tự sinh dữ liệu tổng hợp để toàn bộ mã vẫn chạy hết (smoke-test).

**Cấu trúc chuẩn hoá** dùng chung cho mọi nguồn:
```python
images = [{"image_id": int, "caption": str, "objects": [str], "relations": [(subj, pred, obj)]}]
```

In [4]:
# Lớp khái niệm dự phòng: dùng khi WordNet/nltk không tải được (Kaggle tắt Internet).
# Đây là một MMKG tí hon viết tay, đủ để tạo đường đi gián tiếp cho smoke-test và
# vẫn hoạt động trên Visual Genome vì các lớp đối tượng phổ biến đều có mặt.
FALLBACK_ISA = {
    "ball": ["sports equipment", "toy"], "field": ["open area", "sports venue"],
    "grass": ["vegetation", "ground"], "goal": ["sports equipment"],
    "shoe": ["clothing", "footwear"], "child": ["person"], "boy": ["person", "child"],
    "girl": ["person", "child"], "man": ["person"], "woman": ["person"],
    "person": ["human"], "keeper": ["person"],
    "pot": ["cookware", "container"], "stove": ["appliance", "cooking device"],
    "plate": ["tableware", "container"], "spoon": ["tableware", "utensil"],
    "table": ["furniture"], "bed": ["furniture"], "lamp": ["furniture", "light source"],
    "pillow": ["bedding"], "curtain": ["window covering"], "window": ["building part"],
    "book": ["reading material"], "sand": ["ground"], "umbrella": ["shade device"],
    "wave": ["water"], "towel": ["cloth"], "boat": ["vehicle", "watercraft"],
    "car": ["vehicle"], "truck": ["vehicle"], "bus": ["vehicle"], "bike": ["vehicle"],
    "sign": ["signage"], "road": ["open area", "transport way"],
    "building": ["structure"], "light": ["light source"], "elephant": ["animal"],
    "dog": ["animal"], "cat": ["animal"], "horse": ["animal"], "bird": ["animal"],
    "tree": ["vegetation", "plant"], "fence": ["barrier", "structure"],
    "water": ["liquid"], "rock": ["natural object"], "shirt": ["clothing"],
    "hat": ["clothing"], "chair": ["furniture"], "food": ["edible"],
    "pizza": ["food", "edible"], "plane": ["vehicle"], "train": ["vehicle"],
}


def find_vg_files():
    """Dò các tệp Visual Genome trong /kaggle/input theo tên tệp."""
    wanted = ["objects.json", "relationships.json", "image_data.json",
              "region_descriptions.json", "attributes.json"]
    found = {}
    if not Path("/kaggle/input").exists():
        return found
    for name in wanted:
        hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        if hits:
            found[name.replace(".json", "")] = sorted(hits, key=lambda p: -os.path.getsize(p))[0]
    return found


def _norm(s):
    """Chuẩn hoá chuỗi nhãn: chữ thường, bỏ ký tự lạ, gộp khoảng trắng."""
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def load_visual_genome(paths, n_images):
    """Nạp VG. Trả về danh sách dict đã chuẩn hoá."""
    print("Nạp Visual Genome từ:", {k: v for k, v in paths.items()})
    with open(paths["objects"]) as f:
        objects = json.load(f)
    obj_by_img, objid2name = {}, {}
    for rec in objects:
        iid = rec.get("image_id") or rec.get("id")
        names = []
        for o in rec.get("objects", []):
            nm = _norm((o.get("names") or [o.get("name", "")])[0])
            if nm:
                names.append(nm)
                objid2name[o.get("object_id")] = nm
        if names:
            obj_by_img[iid] = names
    del objects

    rel_by_img = defaultdict(list)
    if "relationships" in paths:
        with open(paths["relationships"]) as f:
            rels = json.load(f)
        for rec in rels:
            iid = rec.get("image_id") or rec.get("id")
            for r in rec.get("relationships", []):
                pred = _norm(r.get("predicate", ""))
                s = r.get("subject", {}); o = r.get("object", {})
                sn = _norm((s.get("names") or [s.get("name", "")])[0]) or objid2name.get(s.get("object_id"), "")
                on = _norm((o.get("names") or [o.get("name", "")])[0]) or objid2name.get(o.get("object_id"), "")
                if pred and sn and on:
                    rel_by_img[iid].append((sn, pred, on))
        del rels

    cap_by_img = {}
    if "region_descriptions" in paths:
        with open(paths["region_descriptions"]) as f:
            regions = json.load(f)
        for rec in regions:
            iid = rec.get("id") or rec.get("image_id")
            phrases = [r.get("phrase", "").strip() for r in rec.get("regions", [])]
            phrases = [p for p in phrases if 15 <= len(p) <= 160]
            if phrases:
                # ghép 3 vùng dài nhất thành một caption cấp ảnh
                phrases = sorted(phrases, key=len, reverse=True)[:3]
                cap_by_img[iid] = ". ".join(phrases)
        del regions

    url_by_img = {}
    if "image_data" in paths:
        with open(paths["image_data"]) as f:
            meta = json.load(f)
        for m in meta:
            url_by_img[m.get("image_id") or m.get("id")] = m.get("url", "")
        del meta

    ids = [i for i in obj_by_img if i in cap_by_img] or list(obj_by_img)
    ids = sorted(ids)[:n_images]
    out = []
    for iid in ids:
        out.append(dict(image_id=int(iid),
                        caption=cap_by_img.get(iid, " ".join(obj_by_img[iid][:8])),
                        objects=sorted(set(obj_by_img[iid])),
                        relations=list({tuple(r) for r in rel_by_img.get(iid, [])}),
                        url=url_by_img.get(iid, "")))
    print(f"  → {len(out)} ảnh, "
          f"{np.mean([len(r['objects']) for r in out]):.1f} đối tượng/ảnh, "
          f"{np.mean([len(r['relations']) for r in out]):.1f} quan hệ/ảnh")
    return out


def make_synthetic(n_images, seed=0):
    """Dữ liệu tổng hợp có cấu trúc quan hệ thật, dùng để smoke-test toàn bộ pipeline.

    Thiết kế có chủ ý: mỗi ảnh thuộc một 'chủ đề', chủ đề quyết định đối tượng và
    quan hệ, nên đồ thị MANG thông tin — nếu pipeline đúng thì điểm đồ thị phải
    cải thiện được kết quả so với đặc trưng nhiễu.
    """
    rng = random.Random(seed)
    themes = {
        "football":  (["ball", "field", "child", "goal", "grass", "shoe"],
                      [("child", "plays", "ball"), ("ball", "on", "grass"), ("child", "wears", "shoe")]),
        "kitchen":   (["pot", "stove", "woman", "plate", "spoon", "table"],
                      [("woman", "cooks with", "pot"), ("pot", "on", "stove"), ("plate", "on", "table")]),
        "beach":     (["sand", "umbrella", "wave", "man", "towel", "boat"],
                      [("man", "sits on", "towel"), ("umbrella", "on", "sand"), ("boat", "in", "wave")]),
        "street":    (["car", "sign", "road", "building", "light", "person"],
                      [("car", "on", "road"), ("sign", "near", "light"), ("person", "crosses", "road")]),
        "zoo":       (["elephant", "fence", "tree", "keeper", "water", "rock"],
                      [("elephant", "near", "fence"), ("tree", "behind", "fence"), ("keeper", "feeds", "elephant")]),
        "bedroom":   (["bed", "lamp", "pillow", "window", "curtain", "book"],
                      [("lamp", "next to", "bed"), ("pillow", "on", "bed"), ("curtain", "on", "window")]),
    }
    names = list(themes)
    out = []
    for i in range(n_images):
        th = names[i % len(names)]
        objs, rels = themes[th]
        k = rng.randint(3, len(objs))
        chosen = rng.sample(objs, k)
        r = [t for t in rels if t[0] in chosen and t[2] in chosen]
        # nhiễu: thỉnh thoảng mượn một đối tượng của chủ đề khác
        if rng.random() < 0.3:
            other = themes[rng.choice(names)][0]
            chosen.append(rng.choice(other))
        # 40% truy vấn nhắc tới KHÁI NIỆM (không có trong danh sách đối tượng) thay vì
        # gọi thẳng tên đối tượng ⇒ tạo ra nhóm truy vấn gián tiếp cần đường đi P2/P4/P5.
        if rng.random() < 0.4:
            concepts = sorted({c for o in chosen for c in FALLBACK_ISA.get(o, [])}
                              - set(chosen))
            if concepts:
                cap = (f"a photo showing {' and '.join(rng.sample(concepts, min(2, len(concepts))))}"
                       f" in a {th} scene")
            else:
                cap = f"a photo showing {' and '.join(chosen[:4])} in a {th} scene"
        else:
            cap = f"a photo showing {' and '.join(chosen[:4])} in a {th} scene"
        out.append(dict(image_id=i, caption=cap, objects=sorted(set(chosen)),
                        relations=r, url="", theme=th))
    return out


def load_dataset():
    mode = CFG.dataset
    if mode in ("auto", "vg"):
        paths = find_vg_files()
        if "objects" in paths:
            return load_visual_genome(paths, CFG.n_images), "visual_genome"
        if mode == "vg":
            raise FileNotFoundError("Không tìm thấy objects.json của Visual Genome trong /kaggle/input")
    print("⚠ Không tìm thấy dữ liệu công khai → chạy chế độ SYNTHETIC (smoke-test).")
    return make_synthetic(CFG.n_images), "synthetic"

## 1.3 Chia dữ liệu và phòng rò rỉ

Yêu cầu mục 6 của đề, cài đặt cụ thể:

| Quy định | Cách thực hiện trong mã |
|---|---|
| Test không dùng để huấn luyện / chọn siêu tham số | Chia **theo ảnh**; caption của test không vào tập huấn luyện |
| Cạnh suy ra từ thống kê phải xây từ train | `build_graph(..., stat_image_ids=train_ids)`; cạnh đồng xuất hiện chỉ đếm trên train |
| α và siêu tham số chọn trên validation | `tune_on_val()` quét lưới trên val, test chỉ chạy một lần |
| Nộp split cố định hoặc seed + mã tạo split | `split.json` được ghi ra kèm seed và hash |

Node ảnh của test **vẫn nằm trong đồ thị** (chúng là ứng viên cần xếp hạng), nhưng
cạnh của chúng chỉ đến từ annotation của chính bức ảnh, không từ thống kê toàn cục,
và không có gradient nào chảy qua chúng khi huấn luyện.

In [5]:
def make_split(records, seed=CFG.split_seed):
    ids = sorted(r["image_id"] for r in records)
    rng = random.Random(seed)
    rng.shuffle(ids)
    n = len(ids)
    n_test = int(n * CFG.test_ratio)
    n_val  = int(n * CFG.val_ratio)
    split = dict(test=sorted(ids[:n_test]),
                 val=sorted(ids[n_test:n_test + n_val]),
                 train=sorted(ids[n_test + n_val:]),
                 seed=seed)
    payload = json.dumps({k: split[k] for k in ("train", "val", "test")}, sort_keys=True)
    split["sha1"] = hashlib.sha1(payload.encode()).hexdigest()[:12]
    with open(CFG.out_dir / "split.json", "w") as f:
        json.dump(split, f)
    print(f"Split (seed={seed}, sha1={split['sha1']}): "
          f"train={len(split['train'])} val={len(split['val'])} test={len(split['test'])}")
    assert not (set(split["train"]) & set(split["test"])), "train/test chồng lấn!"
    assert not (set(split["val"]) & set(split["test"])), "val/test chồng lấn!"
    return split

## 1.4 Đặc trưng CLIP (đóng băng)

CLIP được **đóng băng hoàn toàn** và chỉ chạy một lần: đặc trưng lưu ra `.npy` rồi tái sử dụng
cho mọi thí nghiệm. Đây là baseline của Nhiệm vụ 1.

Thứ tự ưu tiên tìm ảnh: tệp ảnh trong `/kaggle/input` → tải theo URL → đặc trưng tổng hợp.

In [6]:
def index_local_images():
    """Lập chỉ mục mọi tệp ảnh trong /kaggle/input theo tên tệp (không đuôi)."""
    idx = {}
    roots = [Path("/kaggle/input")] if Path("/kaggle/input").exists() else []
    for root in roots:
        for p in root.rglob("*"):
            if p.suffix.lower() in (".jpg", ".jpeg", ".png"):
                idx.setdefault(p.stem, str(p))
    return idx


def encode_with_clip(records, tag):
    """Trả về (img_feat, txt_feat) đã chuẩn hoá L2, shape [N, D]."""
    cache = CFG.out_dir / "cache" / f"clip_{tag}_{len(records)}.npz"
    if cache.exists():
        z = np.load(cache)
        print(f"Dùng lại đặc trưng CLIP từ cache: {cache.name}")
        return z["img"], z["txt"]

    try:
        import open_clip
        from PIL import Image
    except ImportError:
        open_clip = None

    img_index = index_local_images()
    have_images = sum(1 for r in records if str(r["image_id"]) in img_index)
    print(f"Ảnh tìm thấy cục bộ: {have_images}/{len(records)}")

    if open_clip is None or have_images < 0.5 * len(records):
        print("⚠ Không đủ ảnh thật hoặc thiếu open_clip → sinh đặc trưng tổng hợp "
              "(vẫn chạy hết pipeline, nhưng con số không dùng để báo cáo).")
        return synthetic_features(records)

    model, _, preprocess = open_clip.create_model_and_transforms(
        CFG.clip_model, pretrained=CFG.clip_weights)
    tokenizer = open_clip.get_tokenizer(CFG.clip_model)
    model = model.to(DEVICE).eval()
    for p in model.parameters():          # đóng băng tuyệt đối
        p.requires_grad_(False)

    # --- ảnh ---
    feats = []
    from torch.utils.data import Dataset, DataLoader

    class ImgDS(Dataset):
        def __len__(self): return len(records)
        def __getitem__(self, i):
            p = img_index.get(str(records[i]["image_id"]))
            try:
                im = Image.open(p).convert("RGB")
            except Exception:
                im = Image.new("RGB", (224, 224), (128, 128, 128))
            return preprocess(im)

    dl = DataLoader(ImgDS(), batch_size=CFG.clip_bs, num_workers=2)
    with torch.no_grad():
        for batch in dl:
            f = model.encode_image(batch.to(DEVICE))
            feats.append(F.normalize(f, dim=-1).float().cpu().numpy())
    img_feat = np.concatenate(feats)

    # --- văn bản ---
    feats = []
    caps = [r["caption"] for r in records]
    with torch.no_grad():
        for i in range(0, len(caps), CFG.clip_bs):
            tok = tokenizer(caps[i:i + CFG.clip_bs]).to(DEVICE)
            f = model.encode_text(tok)
            feats.append(F.normalize(f, dim=-1).float().cpu().numpy())
    txt_feat = np.concatenate(feats)

    del model
    torch.cuda.empty_cache()
    np.savez_compressed(cache, img=img_feat, txt=txt_feat)
    return img_feat, txt_feat


def synthetic_features(records, dim=512, seed=0):
    """Đặc trưng giả có tín hiệu thật: ảnh cùng chủ đề/đối tượng thì gần nhau."""
    rng = np.random.RandomState(seed)
    vocab = sorted({o for r in records for o in r["objects"]})
    basis = rng.randn(len(vocab), dim).astype(np.float32)
    v2i = {v: i for i, v in enumerate(vocab)}
    img, txt = [], []
    for r in records:
        idx = [v2i[o] for o in r["objects"]]
        core = basis[idx].mean(0) if idx else rng.randn(dim).astype(np.float32)
        img.append(core + 0.9 * rng.randn(dim).astype(np.float32))
        txt.append(core + 0.9 * rng.randn(dim).astype(np.float32))
    f = lambda x: (lambda a: a / np.linalg.norm(a, axis=1, keepdims=True))(np.stack(x))
    return f(img), f(txt)

## 1.5 Xây đồ thị

**Đỉnh:** ảnh · lớp đối tượng · khái niệm.
Đối tượng là **lớp** (một node `dog` dùng chung mọi ảnh) chứ không phải từng thể hiện —
nhờ vậy đồ thị nối được các ảnh với nhau và message passing mới có tác dụng.

**Cạnh:**

| Loại | Nguồn | Có phải thống kê? |
|---|---|---|
| `contains` ảnh → lớp đối tượng | annotation của chính ảnh đó | không → an toàn cho test |
| `scene` đối tượng → đối tượng | scene graph, gộp trên **train** | có → chỉ train |
| `related` đối tượng → đối tượng | đồng xuất hiện trên **train** | có → chỉ train |
| `isa` đối tượng → khái niệm | WordNet hypernym | không (tri thức ngoài) |

**Mẫu đường đi** dùng cho `s_path` (từ node ảnh tới node khái niệm của truy vấn):

| Mẫu | Đường đi | Dài | Loại |
|---|---|---|---|
| P0 | `img –contains→ c` | 1 | trực tiếp |
| P1 | `img –contains→ o –scene→ c` | 2 | gián tiếp |
| P2 | `img –contains→ o –isa→ c` | 2 | gián tiếp |
| P3 | `img –contains→ o –related→ c` | 2 | gián tiếp |
| P4 | `img –contains→ o –scene→ o₂ –isa→ c` | 3 | gián tiếp |
| P5 | `img –contains→ o –related→ o₂ –isa→ c` | 3 | gián tiếp |

`child plays football` với ảnh chỉ nhận diện được `ball`, `field` được bắt qua P1/P2/P4.

In [7]:
import scipy.sparse as sp

PATTERNS = ["P0_contains", "P1_scene", "P2_isa", "P3_related", "P4_scene_isa", "P5_related_isa",
            "P6_triple"]                      # v5: ảnh –thể_hiện→ ⟨s p o⟩ (quan hệ CẤP ẢNH)
PATTERN_LEN = [1, 2, 2, 2, 3, 3, 1]
PIDX = {p: i for i, p in enumerate(PATTERNS)}
PATTERN_DESC = {
    "P0_contains":  "ảnh –chứa→ {c}",
    "P1_scene":     "ảnh –chứa→ {o} –{rel}→ {c}",
    "P2_isa":       "ảnh –chứa→ {o} –là_một→ {c}",
    "P3_related":   "ảnh –chứa→ {o} –liên_quan→ {c}",
    "P4_scene_isa": "ảnh –chứa→ {o} –{rel}→ {o2} –là_một→ {c}",
    "P5_related_isa": "ảnh –chứa→ {o} –liên_quan→ {o2} –là_một→ {c}",
    "P6_triple":    "ảnh –thể_hiện→ ⟨{c}⟩",
}

# ---------------------------------------------------------------------------------
# Lọc WordNet: chỉ giữ danh từ chỉ thứ NHÌN THẤY ĐƯỢC.
#   - nghĩa danh từ phổ biến nhất thuộc VISUAL_LEX (hoặc là môn thể thao/trò chơi),
#   - và từ đó được dùng như danh từ ít nhất ngang như động từ (loại stand, shake, jump, walk...).
# Loại được các node rác ở bản trước: pink, blue, beige, snowy, walking, shake, stand, bit...
# ---------------------------------------------------------------------------------
VISUAL_LEX = {"noun.person", "noun.animal", "noun.artifact", "noun.object", "noun.food",
              "noun.plant", "noun.substance", "noun.body", "noun.location", "noun.group",
              "noun.phenomenon"}
SPORT_ROOTS = {"sport.n.01", "game.n.01"}
GENERIC_CONCEPTS = {"entity", "physical entity", "abstraction", "object", "whole", "thing",
                    "matter", "artifact", "instrumentality", "organism", "living thing",
                    "causal agent", "location", "region", "group", "act", "event",
                    "unit", "part", "piece", "material", "substance", "device", "commodity",
                    "consumer goods", "covering", "creation", "structure", "area"}
# khái niệm "cấp cơ bản" — luôn thêm nếu nằm trên đường hypernym (dog→animal, car→vehicle...)
BASIC_CATEGORIES = {"person", "animal", "dog", "bird", "vehicle", "boat", "aircraft",
                    "clothing", "footwear", "headdress", "food", "plant", "tree", "flower",
                    "furniture", "building", "toy", "sport", "ball", "musical instrument",
                    "body of water", "geological formation", "precipitation", "child",
                    "game equipment", "sports equipment"}

_WN = None


def _wordnet():
    """Nạp WordNet một lần; không được thì trả None (pipeline rơi về FALLBACK_ISA)."""
    global _WN
    if _WN is None:
        try:
            from nltk.corpus import wordnet as wn
            wn.synsets("dog"); _WN = wn
        except Exception:
            try:
                import nltk
                nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
                from nltk.corpus import wordnet as wn
                wn.synsets("dog"); _WN = wn
            except Exception:
                _WN = False
    return _WN or None


def _main_noun_synset(w):
    wn = _wordnet()
    if wn is None:
        return None
    syns = wn.synsets(w.replace(" ", "_"), pos="n")
    return syns[0] if syns else None


_NOUN_CACHE = {}


def is_visual_noun(w):
    if w in _NOUN_CACHE:
        return _NOUN_CACHE[w]
    wn = _wordnet()
    if wn is None:
        _NOUN_CACHE[w] = True          # không có WordNet → không lọc được
        return True
    s, ok = _main_noun_synset(w), False
    if s is not None:
        lex = s.lexname()
        if lex in VISUAL_LEX:
            ok = True
        elif lex == "noun.act":       # football, soccer, tennis... vẫn giữ
            anc = {h.name() for p in s.hypernym_paths() for h in p[:-1]}
            ok = bool(anc & SPORT_ROOTS)
        if ok:
            key = w.replace(" ", "_")
            cnt = lambda pos: sum(l.count() for x in wn.synsets(key, pos=pos)
                                  for l in x.lemmas() if l.name().lower() == key)
            ok = cnt("n") >= cnt("v")
    _NOUN_CACHE[w] = ok
    return ok


def get_wordnet_hypernyms(words):
    """IsA: cha + ông (đủ sâu, không quá chung) + mọi tổ tiên thuộc BASIC_CATEGORIES.
    Chỉ dùng nghĩa phổ biến nhất ⇒ không còn 'bicycle là_một ride', 'beige là_một brown'."""
    if not CFG.use_wordnet:
        return {}
    wn = _wordnet()
    if wn is None:
        fb = {w: FALLBACK_ISA[w] for w in words if w in FALLBACK_ISA}
        print(f"⚠ WordNet không dùng được → FALLBACK_ISA ({len(fb)}/{len(words)} lớp có khái niệm cha).")
        return fb
    out = {}
    for w in words:
        s = _main_noun_synset(w)
        if s is None:
            continue
        hy = []
        for path in s.hypernym_paths()[:2]:
            for depth in range(len(path) - 2, -1, -1):        # path[-1] chính là w
                h = path[depth]
                name = h.lemmas()[0].name().replace("_", " ").lower()
                near = depth >= len(path) - 3                   # cha hoặc ông
                if name == w or name in GENERIC_CONCEPTS or name in hy:
                    continue
                if (near and depth >= CFG.isa_min_depth and
                        (h.lexname() in VISUAL_LEX or h.lexname() == "noun.Tops")) \
                        or name in BASIC_CATEGORIES:
                    hy.append(name)
        if hy:
            out[w] = hy[:5]
    for w in words:
        if w not in out and w in FALLBACK_ISA:
            out[w] = FALLBACK_ISA[w]
    print(f"WordNet(+fallback): {len(out)}/{len(words)} lớp đối tượng có khái niệm cha")
    return out


class MultimodalGraph:
    """Đồ thị dị thể ảnh–đối tượng–khái niệm, kèm ma trận đường đi (đã nhân IDF) theo mẫu."""

    def __init__(self, records, train_ids, ablation="none", seed=0):
        """ablation: none | no_relations | random_edges | no_scene | no_related | no_concept"""
        self.records = records
        self.ablation = ablation
        self.n_img = len(records)
        self.img_pos = {r["image_id"]: i for i, r in enumerate(records)}
        train_set = set(train_ids)
        rng = np.random.RandomState(seed)

        # ---------- từ vựng lớp đối tượng: tần suất CHỈ trên TRAIN ----------
        freq = Counter(o for r in records if r["image_id"] in train_set for o in r["objects"])
        objs = sorted([o for o, c in freq.items() if c >= CFG.min_obj_freq])
        self.obj2id = {o: self.n_img + i for i, o in enumerate(objs)}
        self.n_obj = len(objs)

        # ---------- khái niệm (WordNet) ----------
        hyper = {} if ablation in ("no_relations", "no_concept") else get_wordnet_hypernyms(objs)
        # Khái niệm trùng tên một lớp đối tượng (vd 'dog' là cha của 'terrier') được nối
        # THẲNG vào node đối tượng đó — bản trước tạo 2 node 'dog' và truy vấn 'dog' chỉ
        # khớp node khái niệm, làm mất đường đi P0.
        concepts = sorted({c for v in hyper.values() for c in v
                           if c not in self.obj2id})[:CFG.max_concepts]
        base = self.n_img + self.n_obj
        self.con2id = {c: base + i for i, c in enumerate(concepts)}
        self.n_con = len(concepts)
        # ---------- v5: node BỘ BA cấp ảnh (s, p, o) — bỏ khi no_relations / no_triples ----------
        self.tri_base = self.n_img + self.n_obj + self.n_con
        use_tri = (getattr(CFG, "use_triples", False) and ablation not in ("no_relations", "no_triples")
                   and any(r.get("img_triples") for r in records))
        tri_keys = sorted({k for r in records for k, _ in r.get("img_triples", [])
                           if k[0] in self.obj2id and k[2] in self.obj2id}) if use_tri else []
        self.tri2id = {k: self.tri_base + i for i, k in enumerate(tri_keys)}
        self.id2key = {v: k for k, v in self.tri2id.items()}
        self.n_tri = len(tri_keys)
        self.n_nodes = self.tri_base + self.n_tri

        self.id2name = {}
        for o, i in self.obj2id.items(): self.id2name[i] = o
        for c, i in self.con2id.items(): self.id2name[i] = c
        for k, i in self.tri2id.items(): self.id2name[i] = " ".join(k)

        # ---------- cạnh contains (nhãn riêng của mỗi ảnh) ----------
        rows, cols = [], []
        self.img_objects = []
        for i, r in enumerate(records):
            ids = sorted({self.obj2id[o] for o in r["objects"] if o in self.obj2id})
            self.img_objects.append(ids)
            rows += [i] * len(ids); cols += ids
        A_con = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)),
                              shape=(self.n_img, self.n_nodes))

        # ---------- cạnh scene: CHỈ train, có ngưỡng + top-k ----------
        scene = Counter(); pred_cnt = defaultdict(Counter)
        if ablation not in ("no_relations", "no_scene"):
            for r in records:
                if r["image_id"] not in train_set:
                    continue                      # ← chốt chặn rò rỉ
                for s, p, o in r["relations"]:
                    if s in self.obj2id and o in self.obj2id and s != o:
                        key = (self.obj2id[s], self.obj2id[o])
                        scene[key] += 1; pred_cnt[key][p] += 1
            by_src = defaultdict(list)
            for (a, b), v in scene.items():
                if v >= CFG.scene_min_count:
                    by_src[a].append((v, b))
            scene = {(a, b): v for a, lst in by_src.items()
                     for v, b in sorted(lst, reverse=True)[:CFG.scene_top_k]}
        # vị từ hiển thị = vị từ PHỔ BIẾN NHẤT của cặp (bản trước lấy vị từ gặp đầu tiên)
        self.scene_pred = {k: pred_cnt[k].most_common(1)[0][0] for k in scene}
        # số ảnh TRAIN chứa mỗi lớp — mẫu số cho P(đích | nguồn)
        n_tr = max(sum(1 for r in records if r["image_id"] in train_set), 1)
        obj_df = Counter(self.obj2id[o] for r in records if r["image_id"] in train_set
                         for o in set(r["objects"]) if o in self.obj2id)
        scene_w = {(a, b): min(1.0, v / max(obj_df[a], 1)) for (a, b), v in scene.items()}
        A_scene = self._mat(scene_w)

        # ---------- cạnh related: đồng xuất hiện, CHỈ train ----------
        cooc = Counter()
        if ablation not in ("no_relations", "no_related"):
            for r in records:
                if r["image_id"] not in train_set:
                    continue
                ids = sorted({self.obj2id[o] for o in r["objects"] if o in self.obj2id})
                for a, b in itertools.combinations(ids, 2):
                    cooc[(a, b)] += 1; cooc[(b, a)] += 1
            cooc = {k: v for k, v in cooc.items() if v >= CFG.cooc_min_count}
            by_src = defaultdict(list)
            for (a, b), v in cooc.items():
                if CFG.cooc_score == "npmi":
                    # NPMI ∈ [-1, 1]: 0 = độc lập. Đếm thô ưu ái lớp phổ biến (hub) ⇒ mọi lớp
                    # đều nối tới person/man/floor và P3/P5 thành nhiễu phủ khắp gallery.
                    p_ab = v / n_tr
                    pmi = math.log(p_ab / ((obj_df[a] / n_tr) * (obj_df[b] / n_tr)))
                    s = pmi / max(-math.log(p_ab), 1e-6)
                    if s < CFG.cooc_min_npmi:
                        continue
                else:
                    s = v
                by_src[a].append((s, b))
            cooc = {(a, b): cooc[(a, b)] for a, lst in by_src.items()
                    for s, b in sorted(lst, reverse=True)[:CFG.cooc_top_k]}
        cooc_w = {(a, b): v / max(obj_df[a], 1) for (a, b), v in cooc.items()}   # P(b | a)
        A_rel = self._mat(cooc_w)

        # ---------- cạnh isa (đích có thể là node khái niệm HOẶC node đối tượng) ----------
        isa = {}
        for o, cs in hyper.items():
            if o not in self.obj2id:
                continue
            for c in cs:
                t = self.obj2id.get(c, self.con2id.get(c))
                if t is not None and t != self.obj2id[o]:
                    isa[(self.obj2id[o], t)] = 1.0
        A_isa = self._mat(isa)
        A_isa_true = A_isa.copy()                 # cho IDF: không phụ thuộc ablation random_edges

        # ---------- ablation: cạnh ngẫu nhiên, GIỮ NGUYÊN số cạnh ----------
        if ablation == "random_edges":
            A_scene = self._randomize(A_scene, rng)
            A_rel   = self._randomize(A_rel, rng)
            A_isa   = self._randomize(A_isa, rng)
        if ablation == "no_relations":
            A_scene = A_rel = A_isa = self._mat({})

        # ---------- v5: cạnh ảnh –thể_hiện→ bộ ba, trọng số = xác suất probe của CHÍNH ảnh ----------
        # shuffled_triples / random_edges: ảnh i nhận bộ ba của ảnh π(i) ⇒ đối chứng gắn-với-ảnh.
        trows, tcols, tvals = [], [], []
        if self.n_tri:
            src_img = (rng.permutation(self.n_img) if ablation in ("shuffled_triples", "random_edges")
                       else np.arange(self.n_img))
            for i in range(self.n_img):
                for k, pr in records[int(src_img[i])].get("img_triples", []):
                    t = self.tri2id.get(k)
                    if t is not None:
                        trows.append(i); tcols.append(t); tvals.append(float(pr))
        A_tri = sp.csr_matrix((np.array(tvals, np.float32), (np.array(trows, int), np.array(tcols, int))),
                              shape=(self.n_img, self.n_nodes))
        A_tri.sum_duplicates()
        self.A_triple = A_tri

        self.scene_count = dict(scene)
        self.cooc_count = dict(cooc)
        for M in (A_scene, A_rel, A_isa):
            if CFG.edge_weighting:
                M.data[:] = np.clip(M.data, 0.0, 1.0)   # P(đích|nguồn); xáo cạnh có thể cộng dồn >1
            else:
                M.data[:] = 1.0                   # R_p[i,c] = SỐ đường đi thật sự (cách cũ)
        self.A_contains, self.A_scene, self.A_related, self.A_isa = A_con, A_scene, A_rel, A_isa

        # ---------- ma trận đếm đường đi theo mẫu ----------
        R = {
            "P0_contains":    A_con.tocsr(),
            "P1_scene":       (A_con @ A_scene).tocsr(),
            "P2_isa":         (A_con @ A_isa).tocsr(),
            "P3_related":     (A_con @ A_rel).tocsr(),
            "P4_scene_isa":   (A_con @ A_scene @ A_isa).tocsr(),
            "P5_related_isa": (A_con @ A_rel @ A_isa).tocsr(),
            "P6_triple":      A_tri.tocsr(),
        }
        # Bỏ đường đi gián tiếp tới khái niệm mà ảnh ĐÃ chứa trực tiếp: chúng chỉ đếm lại
        # tín hiệu P0 (vd 'dog –pushing→ terrier –là_một→ dog' ở bản trước).
        if CFG.drop_redundant_indirect:
            direct = (A_con > 0).astype(np.float32)
            for p in PATTERNS[1:]:
                R[p] = (R[p] - R[p].multiply(direct)).tocsr()
                R[p].eliminate_zeros()

        # IDF của node đích, tính CHỈ trên ảnh TRAIN: khái niệm mà hầu như ảnh nào cũng
        # với tới (person, man...) không phân biệt được ứng viên ⇒ trọng số ≈ 0.
        tr_rows = np.array(sorted(self.img_pos[i] for i in train_set if i in self.img_pos))
        if CFG.idf_from == "taxonomy":
            # df(đối tượng) = số ảnh train CHỨA nó; df(khái niệm) = số ảnh train chứa một hạ vị
            # của nó (isa thật). Không đếm scene/related ⇒ IDF của khớp trực tiếp giống hệt
            # nhau ở mọi ablation, và quan hệ không còn "pha loãng" P0.
            df_con = np.asarray((A_con[tr_rows] > 0).sum(0)).ravel()
            df_isa = np.asarray(((A_con @ A_isa_true)[tr_rows] > 0).sum(0)).ravel()
            df_tri = np.asarray((A_tri[tr_rows] > 0).sum(0)).ravel()      # v5: df của bộ ba = số ảnh train mang nó
            ar = np.arange(self.n_nodes)
            df = np.where(ar < self.n_img + self.n_obj, df_con, np.where(ar < self.tri_base, df_isa, df_tri))
        else:
            reach = None
            for p in PATTERNS:
                m = (R[p][tr_rows] > 0).astype(np.float32)
                reach = m if reach is None else reach + m
            df = np.asarray((reach > 0).sum(0)).ravel()
        idf = np.log((len(tr_rows) + 1.0) / (df + 1.0)).astype(np.float32)
        if not CFG.use_idf:
            idf = np.ones_like(idf)
        idf[:self.n_img] = 0.0
        self.idf = idf
        D = sp.diags(idf)
        # R[p][i, c] = Σ_{đường đi i→c theo mẫu p} idf(c). Liệt kê đường đi dùng ĐÚNG
        # trọng số này nên giải thích vẫn khớp điểm (self-test kiểm tra).
        self.R = {p: (R[p] @ D).tocsr() for p in PATTERNS}
        for p in PATTERNS:
            self.R[p].eliminate_zeros()

        self.name2id = dict(self.obj2id); self.name2id.update(self.con2id)
        self.edge_index, self.edge_type = self._build_edge_index()

    def _mat(self, d):
        if not d:
            return sp.csr_matrix((self.n_nodes, self.n_nodes), dtype=np.float32)
        keys = list(d.keys())
        r = np.fromiter((k[0] for k in keys), int, len(keys))
        c = np.fromiter((k[1] for k in keys), int, len(keys))
        v = np.fromiter(d.values(), np.float32, len(keys))
        return sp.csr_matrix((v, (r, c)), shape=(self.n_nodes, self.n_nodes))

    def _randomize(self, A, rng):
        """Giữ nguyên SỐ LƯỢNG cạnh, xáo trộn đích đến — đối chứng cho ablation."""
        A = A.tocoo()
        if A.nnz == 0:
            return A.tocsr()
        new_col = rng.randint(self.n_img, getattr(self, "tri_base", self.n_nodes), size=A.nnz)
        M = sp.csr_matrix((A.data, (A.row, new_col)), shape=A.shape)
        M.sum_duplicates()
        return M

    def _build_edge_index(self):
        parts, types = [], []
        for t, A in enumerate([self.A_contains.tocoo(), self.A_scene.tocoo(),
                               self.A_related.tocoo(), self.A_isa.tocoo()]):
            if A.nnz == 0:
                continue
            parts.append(np.stack([A.row, A.col])); types.append(np.full(A.nnz, t))
            parts.append(np.stack([A.col, A.row])); types.append(np.full(A.nnz, t + 4))
        if not parts:
            return (torch.zeros(2, 0, dtype=torch.long), torch.zeros(0, dtype=torch.long))
        ei = torch.tensor(np.concatenate(parts, 1), dtype=torch.long)
        et = torch.tensor(np.concatenate(types), dtype=torch.long)
        return ei, et

    def summary(self):
        return dict(nodes=self.n_nodes, images=self.n_img, object_classes=self.n_obj,
                    concepts=self.n_con, edges=int(self.edge_index.shape[1] // 2),
                    contains=int(self.A_contains.nnz), scene=int(self.A_scene.nnz),
                    related=int(self.A_related.nnz), isa=int(self.A_isa.nnz),
                    triple_nodes=int(getattr(self, "n_tri", 0)),
                    triple_edges=int(self.A_triple.nnz) if hasattr(self, "A_triple") else 0,
                    paths={p: int(self.R[p].nnz) for p in PATTERNS},
                    ablation=self.ablation)

## 1.6 Phân tích truy vấn → khái niệm trên đồ thị

Một truy vấn được quy về tập node (lớp đối tượng hoặc khái niệm) mà nó nhắc tới.
Đây là điểm neo để `s_path` tìm đường đi.

In [8]:
STOP = set("a an the of on in at to with and or is are was were this that these those "
           "there here it its his her their some any for from by as be being been very "
           "photo image picture showing shows scene view near next".split())


def parse_query(text, graph):
    """Trả về [(node_id, tên, n-gram khớp)] cho các khái niệm xuất hiện trong truy vấn."""
    toks = [t for t in _norm(text).split() if t not in STOP and len(t) > 1]
    cands = []
    for n in (2, 1):                                    # ưu tiên cụm 2 từ
        for i in range(len(toks) - n + 1):
            cands.append(" ".join(toks[i:i + n]))
    hits, seen = [], set()
    for g in cands:
        nid = graph.name2id.get(g)
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, g, g))
    return hits


def path_score_matrix(graph, concept_ids, cand_idx):
    """[len(cand), n_patterns] — số đường đi (đã log1p) theo từng mẫu, cho mỗi ứng viên."""
    out = np.zeros((len(cand_idx), len(PATTERNS)), np.float32)
    if not concept_ids:
        return out
    cols = np.array(sorted(set(concept_ids)))
    for p, name in enumerate(PATTERNS):
        sub = graph.R[name][cand_idx][:, cols]
        out[:, p] = np.asarray(sub.sum(1)).ravel()
    return np.log1p(out)

## 1.7 Mô hình

* **1.7.1 GNN cổ điển (cấp độ 1)**: GraphSAGE/GAT + PathScorer.
* **1.7.2 GNN của SCENIR (nhiệm vụ 2)**: thay nhánh `s_GNN` bằng encoder của SCENIR, bật bằng `CFG.gnn_type = "scenir"`.

### 1.7.1 GNN cổ điển (cấp độ 1)

GNN **tự cài bằng PyTorch thuần** (không cần PyTorch Geometric — tránh hẳn rủi ro cài đặt
trên Kaggle, và là thứ nhóm phải tự giải thích được khi vấn đáp).

* **GraphSAGE** — `h' = W₁h + W₂·mean(h_neighbours)`. Inductive: node ảnh test chưa từng
  tham gia huấn luyện vẫn có biểu diễn hợp lệ.
* **GAT** — hệ số chú ý từng cạnh, softmax theo node đích (dùng `index_reduce_` để ổn định số học).
* **PathScorer** — một trọng số học được cho mỗi mẫu đường đi, nhân hệ số suy giảm theo độ dài.
  Vì điểm là **tổng cộng tính** nên đóng góp của từng đường đi tách bạch được ⇒ fidelity đo được.

In [9]:
def scatter_mean(src, index, n):
    out = torch.zeros(n, src.size(1), device=src.device, dtype=src.dtype)
    out.index_add_(0, index, src)
    cnt = torch.zeros(n, 1, device=src.device, dtype=src.dtype)
    cnt.index_add_(0, index, torch.ones(src.size(0), 1, device=src.device, dtype=src.dtype))
    return out / cnt.clamp(min=1)


def scatter_softmax(logits, index, n):
    m = torch.full((n,), -1e30, device=logits.device, dtype=logits.dtype)
    m = m.index_reduce_(0, index, logits, "amax", include_self=True)
    e = (logits - m[index]).exp()
    s = torch.zeros(n, device=logits.device, dtype=logits.dtype).index_add_(0, index, e)
    return e / s[index].clamp(min=1e-16)


class SAGELayer(nn.Module):
    def __init__(self, din, dout):
        super().__init__()
        self.self_w = nn.Linear(din, dout)
        self.neigh_w = nn.Linear(din, dout)

    def forward(self, x, edge_index, edge_weight=None):
        src, dst = edge_index
        msg = x[src] if edge_weight is None else x[src] * edge_weight.unsqueeze(-1)
        agg = scatter_mean(msg, dst, x.size(0))
        return self.self_w(x) + self.neigh_w(agg)


class TypedSAGELayer(nn.Module):
    """h' = W₀h + Σ_t σ(g_t)·W_t·mean_{u∈N_t(v)} h_u — mỗi loại cạnh một ma trận và một cổng.
    SAGELayer cũ gộp contains/scene/related/isa vào CHUNG một trung bình ⇒ cạnh nhiễu pha
    loãng tín hiệu ảnh–đối tượng, và mô hình không có cách nào tắt riêng một loại cạnh.
    Cổng các loại quan hệ khởi tạo gần đóng ⇒ điểm xuất phát ≈ no_relations."""

    N_TYPES = 8                                   # 4 loại × 2 chiều (xem _build_edge_index)

    def __init__(self, din, dout, gate_init=-1.5):
        super().__init__()
        self.self_w = nn.Linear(din, dout)
        self.neigh_w = nn.ModuleList([nn.Linear(din, dout, bias=False) for _ in range(self.N_TYPES)])
        g = torch.full((self.N_TYPES,), float(gate_init))
        g[0] = g[4] = 3.0                         # contains (hai chiều) mở sẵn
        self.gate = nn.Parameter(g)

    def forward(self, x, edge_index, edge_type):
        src, dst = edge_index
        out = self.self_w(x)
        for t in range(self.N_TYPES):
            m = edge_type == t
            if not bool(m.any()):
                continue
            agg = scatter_mean(x[src[m]], dst[m], x.size(0))
            out = out + torch.sigmoid(self.gate[t]) * self.neigh_w[t](agg)
        return out


class GATLayer(nn.Module):
    def __init__(self, din, dout, heads=4):
        super().__init__()
        assert dout % heads == 0
        self.h, self.dk = heads, dout // heads
        self.lin = nn.Linear(din, dout)
        self.att_src = nn.Parameter(torch.empty(heads, self.dk))
        self.att_dst = nn.Parameter(torch.empty(heads, self.dk))
        nn.init.xavier_uniform_(self.att_src); nn.init.xavier_uniform_(self.att_dst)

    def forward(self, x, edge_index, edge_weight=None):
        N = x.size(0)
        src, dst = edge_index
        z = self.lin(x).view(N, self.h, self.dk)
        a = (z * self.att_src).sum(-1)[src] + (z * self.att_dst).sum(-1)[dst]   # [E,H]
        a = F.leaky_relu(a, 0.2)
        if edge_weight is not None:
            a = a + edge_weight.log().clamp(min=-5).unsqueeze(-1)
        out = torch.zeros(N, self.h, self.dk, device=x.device, dtype=z.dtype)
        for hh in range(self.h):                      # softmax riêng từng head
            al = scatter_softmax(a[:, hh], dst, N)
            out[:, hh].index_add_(0, dst, z[src, hh] * al.unsqueeze(-1))
        return out.reshape(N, -1)


class GraphScorer(nn.Module):
    """Sinh biểu diễn node bằng GNN, chiếu CLIP-text vào cùng không gian, cộng điểm đường đi."""

    def __init__(self, n_nodes, clip_dim, img_feat, cfg=CFG, edge_type=None):
        super().__init__()
        self.n_img = img_feat.shape[0]
        self.typed = cfg.gnn_typed and cfg.gnn_type == "sage"
        # loại cạnh của đồ thị ĐẦY ĐỦ — dùng khi encode_graph được gọi không kèm edge_type
        self.register_buffer("full_et", (edge_type if edge_type is not None
                                         else torch.zeros(0, dtype=torch.long)).clone())
        # Node ẢNH: chiếu từ đặc trưng CLIP đóng băng ⇒ inductive, ảnh test chưa từng
        # tham gia huấn luyện vẫn có biểu diễn hợp lệ (không phải vector ngẫu nhiên).
        self.register_buffer("img_feat", img_feat.clone())
        self.img_proj = nn.Linear(clip_dim, cfg.gnn_dim)
        # Node ĐỐI TƯỢNG / KHÁI NIỆM: embedding học được (chúng lặp lại giữa các ảnh).
        self.sym_emb = nn.Embedding(n_nodes - self.n_img, cfg.gnn_dim)
        nn.init.normal_(self.sym_emb.weight, std=0.02)
        if self.typed:
            self.layers = nn.ModuleList([TypedSAGELayer(cfg.gnn_dim, cfg.gnn_dim, cfg.gate_init)
                                         for _ in range(cfg.gnn_layers)])
        else:
            Layer = SAGELayer if cfg.gnn_type == "sage" else GATLayer
            self.layers = nn.ModuleList(
                [Layer(cfg.gnn_dim, cfg.gnn_dim) for _ in range(cfg.gnn_layers)])
        self.norms = nn.ModuleList(
            [nn.LayerNorm(cfg.gnn_dim) for _ in range(cfg.gnn_layers)])
        self.drop = nn.Dropout(cfg.gnn_dropout)
        self.q_proj = nn.Sequential(                     # CLIP-text → không gian đồ thị
            nn.Linear(clip_dim, cfg.gnn_dim), nn.GELU(),
            nn.Linear(cfg.gnn_dim, cfg.gnn_dim))
        # PathScorer: 1 trọng số / mẫu, softplus để luôn ≥0 (giải thích mới diễn giải được)
        # P0 khởi tạo 0 (softplus = 0.69), P1..P5 khởi tạo thấp: đường đi gián tiếp phải
        # "tự chứng minh" trên mẫu âm khó thì trọng số mới tăng.
        init = torch.full((len(PATTERNS),), float(cfg.path_init_indirect)); init[0] = 0.0
        self.path_w = nn.Parameter(init)
        self.decay = nn.Parameter(torch.tensor(0.5))
        # path_gain CỐ ĐỊNH (buffer, không học): sau znorm_per_row điểm bất biến theo tỉ lệ nên
        # gain không có tín hiệu thật — gradient chỉ đi qua eps=1e-6 của z-norm, rất nhỏ nhưng
        # cùng dấu, và Adam vẫn bước đủ lr ⇒ gain trôi về ~1e-7, nhánh path bị TẮT NGẦM và mọi
        # "đóng góp" trong giải thích hiện +0.0000 (đã gặp trên AG). Tắt nhánh path là việc của λ.
        self.register_buffer("path_gain", torch.tensor(1.0))
        # trọng số của z(s_clip) trong logit lúc HUẤN LUYỆN, học được (softplus, khởi tạo = 1):
        # trên AG CLIP rất yếu (MRR 0.12) nên mô hình cần hạ được trọng số này.
        self.clip_logit = nn.Parameter(torch.tensor(0.5413))
        # Hệ số của z(s_path) trong logit HUẤN LUYỆN (softplus, khởi tạo = 1). Thiếu nó, nhánh path
        # có hệ số cứng = 1, nên khi nó gây hại mô hình tự tắt bằng đường thoái hoá: đợt 2 kéo
        # path_gain → 1e-7, đợt 3 dồn trọng số chuẩn hoá sang mẫu không có dữ liệu (P0 → 0.000).
        # Lúc suy luận vẫn trộn bằng λ chọn trên val; hệ số này chỉ dùng khi huấn luyện.
        self.path_logit = nn.Parameter(torch.tensor(0.5413))

    def node_features(self):
        return torch.cat([self.img_proj(self.img_feat), self.sym_emb.weight], 0)

    def edge_gates(self):
        """σ(g_t) của từng lớp — in ra để biết mô hình có thực sự dùng quan hệ hay không."""
        return [torch.sigmoid(l.gate).detach().cpu().numpy() for l in self.layers] if self.typed else []

    def encode_graph(self, edge_index, edge_type=None):
        h = self.node_features()
        if self.typed and edge_type is None:
            assert self.full_et.numel() == edge_index.size(1), \
                "encode_graph trên đồ thị con cần truyền edge_type tương ứng"
            edge_type = self.full_et
        for i, (l, n) in enumerate(zip(self.layers, self.norms)):
            h = l(h, edge_index, edge_type) if self.typed else l(h, edge_index)
            h = n(h)
            if i < len(self.layers) - 1:
                h = self.drop(F.gelu(h))
        return F.normalize(h, dim=-1)

    def pattern_weights(self):
        """Trọng số hiệu dụng của mỗi mẫu: P0 NEO = 1, P_k = softplus(w_k) · decay^(len_k − 1).
        Neo P0 thay cho chuẩn hoá tổng = 1: tỉ lệ vẫn định danh được, nhưng trọng số không thể
        trôi sang mẫu vắng dữ liệu (no_relations seed 2 đợt 3: P0 = 0.000, nhánh path chết)."""
        d = torch.sigmoid(self.decay)
        lens = torch.tensor(PATTERN_LEN, device=self.path_w.device, dtype=torch.float32)
        w = F.softplus(self.path_w) * d.pow(lens - 1)
        return torch.cat([torch.ones(1, device=w.device, dtype=w.dtype), w[1:]])

    def gnn_score(self, h, q_clip, cand_idx):
        q = F.normalize(self.q_proj(q_clip), dim=-1)          # [B, d]
        return torch.einsum("bd,bkd->bk", q, h[cand_idx])      # [B, K]

    def path_score(self, path_mat):
        """path_mat: [B, K, n_patterns] → [B, K]. Cộng tính theo mẫu ⇒ tách được đóng góp."""
        return F.softplus(self.path_gain) * (path_mat * self.pattern_weights()).sum(-1)

### 1.7.2 GNN của SCENIR thay cho GNN cổ điển

SCENIR (ICML 2025, [arXiv 2505.15867](https://arxiv.org/html/2505.15867v1)). Chỉ thay nhánh `s_GNN`; split, ứng viên CLIP,
`s_path`, chọn α/λ trên val, ablation và fidelity giữ nguyên. Bật bằng `CFG.gnn_type = "scenir"`.

* **Đầu vào:** mỗi ảnh một scene graph. Đỉnh = lớp đối tượng của ảnh + khái niệm IsA của chúng; cạnh = cạnh
  `scene`/`related` (xây chỉ từ train) và `isa` của `MultimodalGraph` nằm trong tập đỉnh đó. Đặc trưng đỉnh =
  CLIP-text của tên đỉnh.
* **Encoder:** SCENIR GAE — 3 lớp GIN với hai nhánh `mu`/`logstd` tách rời, feature decoder và edge decoder MLP,
  discriminator đối kháng. Huấn luyện tự giám sát trên scene graph của ảnh **train** rồi đóng băng.
* **Điểm:** `z_I = L2norm(Σ_v mu_v)`, `s_GNN = cos(q_proj(CLIP_text(q)), z_I)`. `q_proj` và trọng số đường đi học
  bằng đúng vòng InfoNCE của GNN cổ điển trong `train_graph_scorer`.

In [10]:
from contextlib import contextmanager

# Siêu tham số SCENIR = config.json của repo gốc; chiều đặc trưng đỉnh lấy theo CLIP-text (512 thay vì 768).
for _k, _v in dict(scenir_latent=1000, scenir_layers=3, scenir_disc_hidden=32,
                   scenir_edge_hidden=256, scenir_edge_out=32, scenir_lr=1e-3,
                   scenir_lr_gamma=0.95, scenir_bs=64, scenir_disc_steps=5,
                   scenir_epochs=30, scenir_with_concepts=True,
                   scenir_proj_hidden=1024).items():
    setattr(CFG, _k, _v)


@contextmanager
def use_gnn(kind):
    """Tạm đổi CFG.gnn_type ('sage' | 'gat' | 'scenir'); khôi phục khi ra khỏi khối."""
    old = CFG.gnn_type
    CFG.gnn_type = kind
    try:
        yield
    finally:
        CFG.gnn_type = old


def image_scene_graphs(graph):
    """Hợp rời (disjoint union) scene graph của mọi ảnh, cache trên chính đồ thị.
    node_gid: id node toàn cục của từng đỉnh cục bộ; node_ptr/edge_ptr: đỉnh/cạnh của ảnh i nằm liền
    nhau trong [ptr[i], ptr[i+1]); edge_index vô hướng (mỗi cạnh lưu hai chiều)."""
    if hasattr(graph, "_scenir_sg"):
        return graph._scenir_sg
    A_obj = ((graph.A_scene + graph.A_related) > 0).tocsr()
    A_isa = graph.A_isa.tocsr()
    nbrs = lambda M, n: M.indices[M.indptr[n]:M.indptr[n + 1]]
    node_gid, node_ptr, edge_ptr, src, dst = [], [0], [0], [], []
    for i in range(graph.n_img):
        objs = [int(o) for o in graph.img_objects[i]]
        local = {o: k for k, o in enumerate(objs)}
        pairs = set()
        for o in objs:
            for t in nbrs(A_obj, o):
                t = int(t)
                if t in local and t != o:
                    pairs.add((min(local[o], local[t]), max(local[o], local[t])))
        if CFG.scenir_with_concepts:
            for o in objs:
                for c in nbrs(A_isa, o):
                    c = int(c)
                    if c == o:
                        continue
                    if c not in local:
                        local[c] = len(local)
                    pairs.add((min(local[o], local[c]), max(local[o], local[c])))
        base = len(node_gid)
        node_gid.extend(local)                 # dict giữ thứ tự chèn = thứ tự id cục bộ
        node_ptr.append(len(node_gid))
        for a, b in sorted(pairs):
            src += [base + a, base + b]; dst += [base + b, base + a]
        edge_ptr.append(len(src))
    sg = dict(node_gid=np.array(node_gid, np.int64), node_ptr=np.array(node_ptr, np.int64),
              edge_ptr=np.array(edge_ptr, np.int64),
              edge_index=np.array([src, dst], np.int64).reshape(2, -1))
    graph._scenir_sg = sg
    return sg


_SCENIR_NAME_EMB = {}


def _encode_node_names(names):
    """CLIP-text đóng băng (cùng encoder với truy vấn). Không có CLIP (synthetic/offline) thì dùng
    vector ngẫu nhiên cố định theo tên — chỉ để smoke-test."""
    try:
        return clip_text_features(names)
    except Exception as e:
        print(f"⚠ Không mã hoá được tên node bằng CLIP ({type(e).__name__}) → vector ngẫu nhiên theo tên.")
        out = []
        for n in names:
            v = np.random.RandomState(int(hashlib.md5(n.encode()).hexdigest()[:8], 16)).randn(512)
            out.append((v / np.linalg.norm(v)).astype(np.float32))
        return np.stack(out)


def node_name_features(graph):
    """[n_nodes, D] đặc trưng đỉnh; dòng của node ảnh để 0 (scene graph không chứa node ảnh)."""
    names = [graph.id2name[i] for i in range(graph.n_img, graph.n_nodes)]
    new = [n for n in dict.fromkeys(names) if n not in _SCENIR_NAME_EMB]
    if new:
        _SCENIR_NAME_EMB.update(zip(new, _encode_node_names(new)))
    dim = len(next(iter(_SCENIR_NAME_EMB.values()))) if _SCENIR_NAME_EMB else 512
    X = np.zeros((graph.n_nodes, dim), np.float32)
    if names:
        X[graph.n_img:] = np.stack([_SCENIR_NAME_EMB[n] for n in names])
    return torch.tensor(X)

In [11]:
def scatter_sum(src, index, n):
    return torch.zeros(n, src.size(1), device=src.device, dtype=src.dtype).index_add_(0, index, src)


class GINLayer(nn.Module):
    """GINConv(nn.Linear) của PyG với ε = 0: h'_v = W·(h_v + Σ_{u∈N(v)} h_u)."""

    def __init__(self, din, dout):
        super().__init__()
        self.lin = nn.Linear(din, dout)

    def forward(self, x, edge_index):
        src, dst = edge_index
        return self.lin(x + scatter_sum(x[src], dst, x.size(0)))


class ScenirEncoder(nn.Module):
    """MultiGINEncoder của SCENIR: hai nhánh GIN tách rời cho mu và logstd, không ReLU ở lớp cuối."""

    def __init__(self, din, dhid, dout, layers=3):
        super().__init__()
        dims = [din] + [dhid] * (layers - 1) + [dout]
        self.gnn_mu = nn.ModuleList([GINLayer(a, b) for a, b in zip(dims[:-1], dims[1:])])
        self.gnn_logstd = nn.ModuleList([GINLayer(a, b) for a, b in zip(dims[:-1], dims[1:])])

    @staticmethod
    def _run(layers, x, edge_index):
        for i, layer in enumerate(layers):
            x = layer(x, edge_index)
            if i < len(layers) - 1:
                x = F.relu(x)
        return x

    def forward(self, x, edge_index, mu_only=False):
        mu = self._run(self.gnn_mu, x, edge_index)
        return mu if mu_only else (mu, self._run(self.gnn_logstd, x, edge_index))


class ScenirMLP(nn.Module):
    """MLP 2 lớp — feature decoder và edge decoder của SCENIR."""

    def __init__(self, din, dhid, dout):
        super().__init__()
        self.layer_1, self.layer_2 = nn.Linear(din, dhid), nn.Linear(dhid, dout)

    def forward(self, x):
        return self.layer_2(F.relu(self.layer_1(x)))


class ScenirDiscriminator(nn.Module):
    def __init__(self, din, dhid):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(din, dhid), nn.ReLU(),
                                 nn.Linear(dhid, dhid), nn.ReLU(), nn.Linear(dhid, 1))

    def forward(self, z):
        return self.net(z)


SCENIR_EPS, SCENIR_MAX_LOGSTD = 1e-15, 10


def scenir_negative_edges(edge_index, batch, n_samples):
    """Cặp (u, v) cùng một đồ thị trong batch, không tự nối, không trùng cạnh thật."""
    N = batch.numel()
    cnt = torch.bincount(batch)
    ptr = torch.cat([cnt.new_zeros(1), cnt.cumsum(0)[:-1]])
    u = torch.randint(0, N, (2 * n_samples,), device=batch.device)
    b = batch[u]
    off = torch.minimum((torch.rand(u.numel(), device=batch.device) * cnt[b]).long(), cnt[b] - 1)
    v = ptr[b] + off
    keep = (u != v) & ~torch.isin(u * N + v, edge_index[0] * N + edge_index[1])
    return torch.stack([u[keep], v[keep]])[:, :n_samples]


class ScenirGAE(nn.Module):
    """SCENIR = ARGVA (VGAE + điều chuẩn đối kháng) + feature decoder MLP + edge decoder MLP."""

    def __init__(self, din, cfg=CFG):
        super().__init__()
        L = cfg.scenir_latent
        self.encoder = ScenirEncoder(din, L, L, cfg.scenir_layers)
        self.discriminator = ScenirDiscriminator(L, cfg.scenir_disc_hidden)
        self.feature_decoder = ScenirMLP(L, din, din)
        self.edge_decoder = ScenirMLP(L, cfg.scenir_edge_hidden, cfg.scenir_edge_out)

    def encode(self, x, edge_index):
        mu, logstd = self.encoder(x, edge_index)
        logstd = logstd.clamp(max=SCENIR_MAX_LOGSTD)
        z = mu + torch.randn_like(logstd) * logstd.exp() if self.training else mu
        return z, mu, logstd

    @staticmethod
    def _inner(z, edge_index):
        return torch.sigmoid((z[edge_index[0]] * z[edge_index[1]]).sum(-1))

    @staticmethod
    def kl_loss(mu, logstd):
        return -0.5 * torch.mean(torch.sum(1 + 2 * logstd - mu ** 2 - logstd.exp() ** 2, dim=1))

    def reg_loss(self, z):
        return -torch.log(torch.sigmoid(self.discriminator(z)) + SCENIR_EPS).mean()

    def discriminator_loss(self, z):
        real = torch.sigmoid(self.discriminator(torch.randn_like(z)))
        fake = torch.sigmoid(self.discriminator(z.detach()))
        return -torch.log(real + SCENIR_EPS).mean() - torch.log(1 - fake + SCENIR_EPS).mean()

    def recon_loss(self, x, edge_index, batch):
        """recon_loss_batched của repo: MSE tái tạo đặc trưng + BCE cạnh dương/âm trên đầu ra edge decoder.
        Encode lại như bản gốc; trả về (loss, mu, logstd) để kl_loss dùng đúng lần encode này."""
        z, mu, logstd = self.encode(x, edge_index)
        loss = F.mse_loss(self.feature_decoder(z), x)
        if edge_index.size(1) > 0:
            z_e = self.edge_decoder(z)
            loss = loss - torch.log(self._inner(z_e, edge_index) + SCENIR_EPS).mean()
            neg = scenir_negative_edges(edge_index, batch, edge_index.size(1))
            if neg.size(1) > 0:
                loss = loss - torch.log(1 - self._inner(z_e, neg) + SCENIR_EPS).mean()
        return loss, mu, logstd

In [12]:
def _sg_batch(sg, X, img_idx):
    """Gom scene graph của các ảnh img_idx thành một batch (x, edge_index, batch) trên DEVICE.
    X phải nằm sẵn trên DEVICE."""
    img_idx = np.asarray(img_idx)
    ranges = lambda p: np.concatenate([np.arange(p[i], p[i + 1]) for i in img_idx])
    nodes, edges = ranges(sg["node_ptr"]), ranges(sg["edge_ptr"])
    counts = sg["node_ptr"][img_idx + 1] - sg["node_ptr"][img_idx]
    remap = np.full(len(sg["node_gid"]), -1, np.int64)
    remap[nodes] = np.arange(len(nodes))
    ei = remap[sg["edge_index"][:, edges]]
    x = X[torch.from_numpy(sg["node_gid"][nodes]).to(X.device)]
    return (x, torch.from_numpy(ei).to(X.device),
            torch.from_numpy(np.repeat(np.arange(len(img_idx)), counts)).to(X.device))


def train_scenir_gae(sg, X, img_idx, seed=0, verbose=True):
    """Huấn luyện SCENIR GAE tự giám sát CHỈ trên scene graph của ảnh train (không truy vấn, không nhãn).
    loss = (1/6)·reg + 3·recon + (1/3)·(1/num_nodes)·kl; discriminator cập nhật disc_steps bước/lần."""
    set_seed(seed)
    # Giữ cả đồ thị không cạnh (recon_loss chịu được), nếu không ablation no_relations không còn gì để huấn luyện.
    n_nodes = np.diff(sg["node_ptr"])
    idx = np.array([i for i in img_idx if n_nodes[i] > 0], np.int64)
    assert len(idx) > 0, "Không ảnh train nào có scene graph — kiểm tra nhãn đối tượng."
    gae = ScenirGAE(X.shape[1]).to(DEVICE)
    disc_opt = torch.optim.AdamW(gae.discriminator.parameters(), lr=CFG.scenir_lr)
    enc_opt = torch.optim.AdamW(list(gae.encoder.parameters()) + list(gae.edge_decoder.parameters())
                                + list(gae.feature_decoder.parameters()), lr=CFG.scenir_lr)
    sched = torch.optim.lr_scheduler.ExponentialLR(enc_opt, gamma=CFG.scenir_lr_gamma)
    hist, t0 = [], time.time()
    for ep in range(CFG.scenir_epochs):
        gae.train()
        perm = np.random.permutation(idx)
        tot_e = tot_d = 0.0
        nb = 0
        for s in range(0, len(perm), CFG.scenir_bs):
            x, ei, b = _sg_batch(sg, X, perm[s:s + CFG.scenir_bs])
            enc_opt.zero_grad()
            z, _, _ = gae.encode(x, ei)
            d_acc = 0.0
            for _ in range(CFG.scenir_disc_steps):
                disc_opt.zero_grad()
                d_loss = gae.discriminator_loss(z)
                d_loss.backward()
                disc_opt.step()
                d_acc += d_loss.item()
            rec, mu, logstd = gae.recon_loss(x, ei, b)
            loss = ((1 / 6) * gae.reg_loss(z) + 3 * rec
                    + (1 / 3) * (1 / x.size(0)) * gae.kl_loss(mu, logstd))
            loss.backward()
            enc_opt.step()
            tot_e += loss.item(); tot_d += d_acc / CFG.scenir_disc_steps; nb += 1
        sched.step()
        hist.append((tot_e / max(nb, 1), tot_d / max(nb, 1)))
        if verbose and (ep % 5 == 0 or ep == CFG.scenir_epochs - 1):
            print(f"  [SCENIR GAE] epoch {ep:2d} | enc {hist[-1][0]:.4f} | disc {hist[-1][1]:.4f} "
                  f"| {len(idx)} đồ thị train | {time.time() - t0:.0f}s")
    gae.eval()
    for p in gae.parameters():
        p.requires_grad_(False)
    return gae, hist


@torch.no_grad()
def scenir_embed_images(gae, sg, X, chunk=1024):
    """z_I = Σ_v mu_v (eval ⇒ không lấy mẫu) cho MỌI ảnh; ảnh không có đỉnh nào → vector 0."""
    gae.eval()
    n_img = len(sg["node_ptr"]) - 1
    out = torch.zeros(n_img, CFG.scenir_latent)
    for s in range(0, n_img, chunk):
        idx = np.arange(s, min(s + chunk, n_img))
        x, ei, b = _sg_batch(sg, X, idx)
        if x.size(0) == 0:
            continue
        mu = gae.encoder(x, ei, mu_only=True)
        out[torch.from_numpy(idx)] = scatter_sum(mu, b, len(idx)).cpu()
    return out

In [13]:
class ScenirGraphScorer(nn.Module):
    """Cùng giao diện với GraphScorer (encode_graph / gnn_score / path_score / pattern_weights), nên
    score_queries, tune_on_val, fidelity và demo phía sau dùng lại nguyên vẹn."""

    def __init__(self, gae, z_img, clip_dim, cfg=CFG):
        super().__init__()
        self.gae = gae
        self.register_buffer("z_img", F.normalize(z_img, dim=-1))      # sum-pool ⇒ bắt buộc L2
        self.q_proj = nn.Sequential(nn.Linear(clip_dim, cfg.scenir_proj_hidden), nn.ReLU(),
                                    nn.Linear(cfg.scenir_proj_hidden, z_img.shape[1]))
        # Phần chấm đường đi khởi tạo giống GraphScorer (mục 7.1) để hai nhánh GNN so sánh công bằng.
        init = torch.full((len(PATTERNS),), float(cfg.path_init_indirect)); init[0] = 0.0
        self.path_w = nn.Parameter(init)
        self.decay = nn.Parameter(torch.tensor(0.5))
        self.register_buffer("path_gain", torch.tensor(1.0))
        self.clip_logit = nn.Parameter(torch.tensor(0.5413))
        self.path_logit = nn.Parameter(torch.tensor(0.5413))

    def encode_graph(self, edge_index=None, edge_type=None):
        """GAE đã đóng băng ⇒ nhúng ảnh tính một lần; edge_index/edge_type chỉ để khớp giao diện GraphScorer."""
        return self.z_img

    def edge_gates(self):
        return []                                # SCENIR không có cổng theo loại cạnh

    pattern_weights = GraphScorer.pattern_weights
    gnn_score = GraphScorer.gnn_score
    path_score = GraphScorer.path_score


def build_scenir_scorer(engine, train_img_idx, seed=0, verbose=True):
    """Dựng scene graph từng ảnh, huấn luyện GAE trên ảnh train, nhúng mọi ảnh."""
    g = engine.graph
    sg = image_scene_graphs(g)
    X = node_name_features(g).to(DEVICE)
    if verbose:
        nv, ne = np.diff(sg["node_ptr"]), np.diff(sg["edge_ptr"]) // 2
        print(f"  SCENIR scene graph/ảnh: {nv.mean():.1f} đỉnh, {ne.mean():.1f} cạnh | "
              f"ảnh không có cạnh: {(ne == 0).mean():.1%}")
    gae, _ = train_scenir_gae(sg, X, train_img_idx, seed=seed, verbose=verbose)
    return ScenirGraphScorer(gae, scenir_embed_images(gae, sg, X), engine.txt_feat.shape[1])

## 1.8 Bộ máy truy vấn: ứng viên, huấn luyện, xếp hạng lại

Để tính `s_path` cho mọi cặp (truy vấn, ảnh) mà không phải lặp Python, ta dựng
ma trận chỉ báo khái niệm `Q [n_query × n_node]` rồi nhân thưa một lần cho mỗi mẫu:
`S_p = R_p · Qᵀ` → `[n_img × n_query]`. Sau đó chỉ việc tra.

In [14]:
def znorm_per_row(x, eps=1e-6):
    """Chuẩn hoá z theo từng truy vấn. Dùng CHUNG cho train và eval để α, λ chọn
    trên val vẫn đúng khi áp lên test."""
    if isinstance(x, torch.Tensor):
        return (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + eps)
    return (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + eps)


minmax_per_row = znorm_per_row      # giữ tên cũ cho tương thích


class RetrievalEngine:
    def __init__(self, records, graph, img_feat, txt_feat, queries):
        """queries: list[dict(text, gold_image_id, group)] — group ∈ {direct, indirect}"""
        self.records, self.graph = records, graph
        self.img_feat = torch.tensor(img_feat)
        self.txt_feat = torch.tensor(txt_feat)
        self.queries = queries
        self.q_concepts = [parse_query(q["text"], graph) for q in queries]
        self._build_S()

    def _build_S(self):
        g = self.graph
        rows, cols = [], []
        for j, hits in enumerate(self.q_concepts):
            for nid, _, _ in hits:
                rows.append(nid); cols.append(j)
        Q = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)),
                          shape=(g.n_nodes, len(self.queries)))
        self.S = {p: (g.R[p] @ Q).tocsr() for p in PATTERNS}   # [n_img × n_query]

    def path_feats(self, img_idx, q_idx):
        """img_idx [B,K] hoặc [B]; q_idx [B] → [.., n_patterns]"""
        img_idx = np.asarray(img_idx); q_idx = np.asarray(q_idx)
        if img_idx.ndim == 1:
            img_idx = img_idx[:, None]
        B, K = img_idx.shape
        rr = img_idx.ravel(); cc = np.repeat(q_idx, K)
        out = np.zeros((B * K, len(PATTERNS)), np.float32)
        for p, name in enumerate(PATTERNS):
            out[:, p] = np.asarray(self.S[name][rr, cc]).ravel()
        return np.log1p(out).reshape(B, K, len(PATTERNS))   # log1p SAU khi cộng

    def clip_candidates(self, q_idx, gallery_idx, k):
        qf = self.txt_feat[q_idx]                       # [B, D]
        gf = self.img_feat[gallery_idx]                 # [G, D]
        sim = qf @ gf.T
        k = min(k, len(gallery_idx))
        top = sim.topk(k, dim=1)
        return gallery_idx[top.indices.numpy()], top.values.numpy()


def train_graph_scorer(engine, train_q_idx, train_img_idx, seed=0, verbose=True):
    """InfoNCE với mẫu âm trong batch. CLIP đóng băng; chỉ học GNN + trọng số đường đi."""
    set_seed(seed)
    g = engine.graph
    if CFG.gnn_type == "scenir":                 # mục 7.2: GAE tự giám sát trên ảnh train, rồi đóng băng
        model = build_scenir_scorer(engine, train_img_idx, seed=seed, verbose=verbose).to(DEVICE)
    else:
        model = GraphScorer(g.n_nodes, engine.txt_feat.shape[1],
                            engine.img_feat, edge_type=g.edge_type).to(DEVICE)
    # CFG.learn_clip_w = False ⇒ cố định hệ số z(s_clip) ở softplus(0.5413) = 1 (không vào optimizer).
    model.clip_logit.requires_grad_(bool(CFG.learn_clip_w))
    # Cổng cạnh + trọng số mẫu đường đi chỉ là vài vô hướng: cho lr riêng (×CFG.scalar_lr_mult),
    # không weight decay — nếu không chúng gần như đứng yên ở giá trị khởi tạo.
    # Tham số đã đóng băng (GAE của SCENIR) không đưa vào optimizer.
    params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    scalar = [p for n, p in params
              if n.endswith("gate") or n in ("path_w", "decay", "clip_logit")]
    sid = {id(p) for p in scalar}
    opt = torch.optim.AdamW(
        [dict(params=[p for _, p in params if id(p) not in sid]),
         dict(params=scalar, lr=CFG.lr * CFG.scalar_lr_mult, weight_decay=0.0)],
        lr=CFG.lr, weight_decay=CFG.weight_decay)
    # INDUCTIVE: khi huấn luyện bỏ mọi cạnh chạm node ảnh KHÔNG thuộc train (val/test).
    # Node ảnh val/test thành node cô lập ⇒ không gửi thông điệp, không nhận gradient,
    # không ảnh hưởng tới biểu diễn đối tượng/khái niệm học được. Lúc đánh giá
    # (score_queries) mới dùng đồ thị đầy đủ — GraphSAGE inductive nên hợp lệ.
    keep_img = np.zeros(g.n_img, bool)
    keep_img[np.asarray(train_img_idx)] = True
    src, dst = g.edge_index.numpy()
    is_bad = lambda v: (v < g.n_img) & ~keep_img[np.minimum(v, g.n_img - 1)]
    mask = ~(is_bad(src) | is_bad(dst))
    tmask = torch.from_numpy(mask)
    ei = g.edge_index[:, tmask].to(DEVICE)
    et = g.edge_type[tmask].to(DEVICE)
    if verbose:
        print(f"  cạnh dùng khi huấn luyện: {int(mask.sum())}/{len(mask)} "
              f"(đã bỏ {int((~mask).sum())} cạnh chạm ảnh val/test)")
    hist = []
    tr_q, tr_img = np.asarray(train_q_idx), np.asarray(train_img_idx)
    img_dev = engine.img_feat.to(DEVICE)

    # Mẫu âm KHÓ: CLIP top-K trong gallery TRAIN (bỏ ảnh đúng). Lúc test đồ thị chỉ xếp lại
    # top-100 của CLIP, nên phải học phân biệt ĐÚNG những ứng viên đó — với âm ngẫu nhiên
    # trong batch, đường đi gián tiếp "dễ thắng" và học trọng số không dùng được ở test.
    hard = None
    if CFG.hard_neg_k > 0:
        cand_hn, _ = engine.clip_candidates(tr_q, tr_img, CFG.hard_neg_k + 1)
        hard = np.stack([np.concatenate([row[row != gold], row[:1]])[:CFG.hard_neg_k]
                         for row, gold in zip(cand_hn, tr_img)])

    def logits_for(cand_np, qi, h, qf):
        cand = torch.tensor(cand_np, device=DEVICE)
        pm = torch.tensor(engine.path_feats(cand_np, qi)).to(DEVICE)
        z = znorm_per_row(model.gnn_score(h, qf, cand)) + znorm_per_row(model.path_score(pm))
        if CFG.train_with_clip:      # CLIP không có gradient; chỉ trọng số clip_w học được
            z = z + F.softplus(model.clip_logit) * znorm_per_row(torch.einsum("bd,bkd->bk", qf, img_dev[cand]))
        return z / CFG.temperature

    for ep in range(CFG.epochs):
        model.train()
        order = np.random.permutation(len(train_q_idx))
        tot, nb = 0.0, 0
        for s in range(0, len(order), CFG.train_bs):
            sel = order[s:s + CFG.train_bs]
            if len(sel) < 8:
                continue
            qi = tr_q[sel]
            gi = tr_img[sel]                              # ảnh đúng của từng truy vấn
            qf = engine.txt_feat[qi].to(DEVICE)
            h = model.encode_graph(ei, et)

            # (a) âm trong batch như cũ: [B, B], đáp án trên đường chéo
            loss = F.cross_entropy(logits_for(np.tile(gi, (len(gi), 1)), qi, h, qf),
                                   torch.arange(len(sel), device=DEVICE))
            # (b) âm khó từ CLIP: [B, 1+K], đáp án ở cột 0
            if hard is not None:
                cand_h = np.concatenate([gi[:, None], hard[sel]], 1)
                loss = loss + F.cross_entropy(logits_for(cand_h, qi, h, qf),
                                              torch.zeros(len(sel), dtype=torch.long, device=DEVICE))

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item(); nb += 1
        hist.append(tot / max(nb, 1))
        if verbose and (ep % 3 == 0 or ep == CFG.epochs - 1):
            w = model.pattern_weights().detach().cpu().numpy()
            gates = model.edge_gates()
            # Cổng của LỚP 0: ở lớp cuối chỉ node ảnh được dùng, mà ảnh chỉ nhận cạnh contains
            # ⇒ cổng scene/related/isa lớp cuối không có gradient (luôn đứng ở giá trị khởi tạo).
            gs = (" | gate L0[scene,rel,isa] " + " ".join(f"{x:.2f}" for x in gates[0][1:4])) if gates else ""
            print(f"  epoch {ep:2d} | loss {hist[-1]:.4f} | w_path " +
                  " ".join(f"{x:.3f}" for x in w) + gs +
                  f" | clip_w {F.softplus(model.clip_logit).item():.2f}")
    return model, hist


@torch.no_grad()
def score_queries(model, engine, q_idx, gallery_idx, k=None):
    """Trả về (cand_idx, s_clip, s_gnn, s_path) — mọi điểm đã chuẩn hoá z theo hàng. Dùng đồ thị ĐẦY ĐỦ."""
    k = k or CFG.rerank_k
    model.eval()
    ei = engine.graph.edge_index.to(DEVICE)
    h = model.encode_graph(ei)
    cand, s_clip = engine.clip_candidates(q_idx, gallery_idx, k)
    pm = torch.tensor(engine.path_feats(cand, q_idx)).to(DEVICE)
    qf = engine.txt_feat[q_idx].to(DEVICE)
    s_gnn = model.gnn_score(h, qf, torch.tensor(cand, device=DEVICE)).cpu().numpy()
    s_path = model.path_score(pm).cpu().numpy()
    return cand, znorm_per_row(s_clip), znorm_per_row(s_gnn), znorm_per_row(s_path)


def combine(s_clip, s_gnn, s_path, alpha, lam):
    return alpha * s_clip + (1 - alpha) * (lam * s_gnn + (1 - lam) * s_path)

## 1.9 Chỉ số đánh giá

In [15]:
def retrieval_metrics(cand, scores, gold_pos, ks=(1, 5, 10)):
    """gold_pos: chỉ số ảnh đúng (trong không gian toàn cục) cho từng truy vấn."""
    order = np.argsort(-scores, axis=1)
    ranked = np.take_along_axis(cand, order, axis=1)
    hit = ranked == np.asarray(gold_pos)[:, None]
    has = hit.any(1)
    rank = np.where(has, hit.argmax(1) + 1, 10**9)
    out = {f"R@{k}": float((rank <= k).mean()) for k in ks}
    out["MRR"] = float(np.where(has, 1.0 / rank, 0.0).mean())
    out["nDCG@10"] = float(np.where(rank <= 10, 1.0 / np.log2(rank + 1), 0.0).mean())
    return out


def fmt_metrics(runs):
    """runs: list[dict] từ nhiều seed → chuỗi 'mean ± std'."""
    keys = runs[0].keys()
    return {k: f"{np.mean([r[k] for r in runs]):.4f} ± {np.std([r[k] for r in runs]):.4f}"
            for k in keys}

## 1.10 Bộ truy vấn đánh giá: trực tiếp vs gián tiếp

Đây là bằng chứng chính cho câu hỏi "tri thức quan hệ giúp ở đâu".

* **Trực tiếp** — truy vấn nhắc tới lớp đối tượng có sẵn trong annotation của ảnh đúng
  (tồn tại đường đi P0). CLIP thường đã làm tốt nhóm này.
* **Gián tiếp** — truy vấn nhắc tới khái niệm **không** nằm trong danh sách đối tượng của ảnh,
  nhưng nối được qua quan hệ (P1–P5). Đây là nhóm đồ thị phải chứng minh giá trị.

Cách phân nhóm này tự động và kiểm tra được, tránh việc gán nhãn cảm tính.

In [16]:
def build_query_set(records, graph, img_ids, seed=0, verbose=True):
    pos = graph.img_pos
    queries = []
    for iid in img_ids:
        i = pos[iid]
        r = records[i]
        hits = parse_query(r["caption"], graph)
        # nhóm direct/indirect tính trên đối tượng/khái niệm như bản trước (không tính node bộ ba)
        hits = [h for h in hits if h[0] < getattr(graph, "tri_base", graph.n_nodes)]
        own = set(graph.img_objects[i])
        hit_ids = {n for n, _, _ in hits}
        direct = bool(hit_ids & own)
        indirect_ids = hit_ids - own
        reach = 0
        if indirect_ids:
            cols = np.array(sorted(indirect_ids))
            for p in PATTERNS[1:]:
                reach += int(graph.R[p][i][:, cols].nnz)
        group = "indirect" if (indirect_ids and reach > 0 and not direct) else \
                ("direct" if direct else "other")
        queries.append(dict(text=r["caption"], gold=iid, group=group,
                            n_concepts=len(hits), n_indirect=len(indirect_ids)))
    if verbose:
        print(f"Nhóm truy vấn: {dict(Counter(q['group'] for q in queries))}")
    return queries


def export_eval_queries(queries, graph, path, n=20, allowed_ids=None):
    """Xuất bộ ≥20 truy vấn đánh giá của cấp độ 2. allowed_ids = id ảnh TEST (bắt buộc)."""
    assert allowed_ids is not None, "Truyền allowed_ids=set(split['test']) — chỉ lấy truy vấn của tập test"
    allowed_ids = set(allowed_ids)
    pool = [q for q in queries if q["gold"] in allowed_ids]
    ind = [q for q in pool if q["group"] == "indirect"]
    dir_ = [q for q in pool if q["group"] == "direct"]
    rng = random.Random(0)
    sel = rng.sample(ind, min(n // 2, len(ind))) + rng.sample(dir_, min(n // 2, len(dir_)))
    payload = dict(
        n=len(sel),
        construction=("Truy vấn lấy từ caption của ảnh trong tập test. Nhóm 'indirect' là "
                      "truy vấn có khái niệm KHÔNG nằm trong annotation đối tượng của ảnh "
                      "đúng nhưng nối được qua đường đi P1-P5; nhóm 'direct' có ít nhất một "
                      "khái niệm trùng annotation (đường đi P0). Phân nhóm tự động, kiểm tra "
                      "lại được bằng hàm build_query_set."),
        usage="CHỈ dùng để đánh giá. Không tinh chỉnh mô hình trên bộ này.",
        queries=[{k: q[k] for k in ("text", "gold", "group", "n_concepts", "n_indirect")}
                 for q in sel])
    with open(path, "w") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Đã ghi {len(sel)} truy vấn đánh giá → {path}")
    return sel

## 1.11 Chọn siêu tham số trên VALIDATION

Quét lưới `(α, λ)` trên tập val. Test chưa hề được chạm vào ở bước này.

In [17]:
def tune_on_val(model, engine, val_q, val_gallery, verbose=True):
    cand, sc, sg, sp_ = score_queries(model, engine, val_q["q_idx"], val_gallery)
    best = (-1, None)
    grid = []
    for a in CFG.alpha_grid:
        for l in CFG.lambda_grid:
            m = retrieval_metrics(cand, combine(sc, sg, sp_, a, l), val_q["gold_pos"])
            grid.append(dict(alpha=a, lam=l, **m))
            if m["MRR"] > best[0]:
                best = (m["MRR"], (a, l))
    if verbose:
        print(f"  Tốt nhất trên val: α={best[1][0]}, λ={best[1][1]} (MRR={best[0]:.4f})")
    return best[1], grid

## 1.12 Giải thích và đo fidelity (Nhiệm vụ 3)

Giải thích = danh sách **đường đi cụ thể** đóng góp nhiều nhất vào `s_path` của ứng viên đó.
Phần đóng góp không phải suy đoán: nó được đo bằng cách **bỏ đúng đường đi đó rồi tính lại điểm**.

| Chỉ số | Định nghĩa | Kỳ vọng |
|---|---|---|
| fidelity− (cần thiết) | bỏ đường đi giải thích → Δ điểm, Δ thứ hạng | điểm giảm rõ, hạng tụt |
| fidelity+ (đủ) | **chỉ giữ** đường đi giải thích → Δ điểm | điểm gần như không đổi |
| sparsity | 1 − |đường đi giải thích| / |toàn bộ đường đi| | càng cao càng gọn |
| validity | đường đi có thật và nối đúng khái niệm truy vấn ↔ ảnh | 100% |
| **đối chứng** | bỏ **ngẫu nhiên** cùng số đường đi | phải ÍT ảnh hưởng hơn |

Dòng đối chứng là dòng quan trọng nhất: nếu bỏ đường đi giải thích không tệ hơn bỏ ngẫu nhiên
thì giải thích vô nghĩa. Đây gần như chắc chắn là câu hỏi vấn đáp.

In [18]:
def enumerate_paths(graph, img_pos_idx, concept_ids, cap=100000):
    """Liệt kê đường đi CỤ THỂ từ node ảnh tới các khái niệm truy vấn.
    Mỗi đường đi mang trọng số w = idf(node đích) × Π trọng số cạnh (P(đích|nguồn) nếu
    CFG.edge_weighting) — đúng trọng số trong graph.R, nên tổng theo mẫu khớp tuyệt đối
    với điểm tính bằng ma trận."""
    target = set(concept_ids)
    objs = graph.img_objects[img_pos_idx]
    own = set(objs)
    A_s, A_r, A_i = graph.A_scene, graph.A_related, graph.A_isa
    idf = graph.idf
    out = []

    def nbrs(M, n):
        sl = slice(M.indptr[n], M.indptr[n + 1])
        return zip(M.indices[sl], M.data[sl])

    def add(pattern, nodes, rel=None, ew=1.0):
        c = nodes[-1]
        if pattern != "P0_contains" and CFG.drop_redundant_indirect and c in own:
            return                                  # cùng quy tắc với ma trận R
        w = float(idf[c]) * float(ew)
        if w > 0:
            out.append(dict(pattern=pattern, nodes=nodes, rel=rel, w=w))

    for o in objs:
        if o in target:
            add("P0_contains", [o])
        for o2, w1 in nbrs(A_s, o):
            rel = graph.scene_pred.get((o, o2), "quan hệ")
            if o2 in target:
                add("P1_scene", [o, o2], rel, w1)
            for c, w2 in nbrs(A_i, o2):
                if c in target:
                    add("P4_scene_isa", [o, o2, c], rel, w1 * w2)
        for c, w1 in nbrs(A_i, o):
            if c in target:
                add("P2_isa", [o, c], None, w1)
        for o2, w1 in nbrs(A_r, o):
            if o2 in target:
                add("P3_related", [o, o2], None, w1)
            for c, w2 in nbrs(A_i, o2):
                if c in target:
                    add("P5_related_isa", [o, o2, c], None, w1 * w2)
        if len(out) > cap:
            break
    # v5: ảnh –thể_hiện→ bộ ba (trọng số = xác suất probe × idf, đúng như graph.R['P6_triple'])
    A_t = getattr(graph, "A_triple", None)
    if A_t is not None and A_t.nnz:
        sl = slice(A_t.indptr[img_pos_idx], A_t.indptr[img_pos_idx + 1])
        for t, v in zip(A_t.indices[sl], A_t.data[sl]):
            if int(t) in target:
                add("P6_triple", [int(t)], None, v)
    return out[:cap]


def paths_to_counts(paths):
    """Đặc trưng mẫu = log1p(Σ trọng số đường đi của mẫu) — giống hệt RetrievalEngine.path_feats."""
    v = np.zeros(len(PATTERNS), np.float64)
    for p in paths:
        v[PIDX[p["pattern"]]] += p["w"]
    return np.log1p(v).astype(np.float32)


def path_text(graph, p):
    n = [graph.id2name.get(x, f"#{x}") for x in p["nodes"]]
    t = PATTERN_DESC[p["pattern"]]
    d = dict(c=n[-1], o=n[0], rel=p.get("rel") or "quan_hệ")
    if len(n) >= 3:
        d["o2"] = n[1]
    return t.format(**d)


@torch.no_grad()
def explain_and_score(model, graph, img_idx, concept_ids, top_m=CFG.top_m_paths):
    """Trả về (giải thích, điểm đầy đủ, chi tiết fidelity) cho MỘT cặp (truy vấn, ảnh)."""
    paths = enumerate_paths(graph, img_idx, concept_ids)
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    score = lambda ps: float(gain * (paths_to_counts(ps) * w).sum())

    full = score(paths)
    # đóng góp biên của từng đường đi: bỏ đúng nó ra rồi tính lại
    contrib = []
    for i, p in enumerate(paths):
        rest = paths[:i] + paths[i + 1:]
        contrib.append(full - score(rest))
    order = np.argsort(-np.array(contrib)) if contrib else np.array([], int)
    # v5: giải thích phải NHỎ HƠN toàn bộ đường đi, nếu không đối chứng ngẫu nhiên trùng hệt giải thích
    # (bản trước in 'fidelity− = đối chứng, sparsity 0.00' cho mọi ca có ≤ top_m đường đi).
    m = min(top_m, len(paths) - 1) if len(paths) >= 2 else len(paths)
    expl_idx = list(order[:m])
    expl = [dict(text=path_text(graph, paths[i]), pattern=paths[i]["pattern"],
                 contribution=float(contrib[i])) for i in expl_idx]

    keep = [paths[i] for i in expl_idx]
    drop = [p for i, p in enumerate(paths) if i not in set(expl_idx)]
    # Đối chứng: bỏ NGẪU NHIÊN cùng số đường đi, lấy trung bình 10 lần bốc.
    # Bốc một lần thôi thì với truy vấn ít đường đi nó hay trùng đúng tập giải thích
    # và đối chứng mất ý nghĩa.
    rng = np.random.RandomState(0)
    rnd_scores = []
    for _ in range(10):
        ridx = set(rng.choice(len(paths), size=min(len(expl_idx), len(paths)),
                              replace=False).tolist()) if paths else set()
        rnd_scores.append(score([p for i, p in enumerate(paths) if i not in ridx]))
    rnd_score = float(np.mean(rnd_scores)) if (rnd_scores and len(paths) >= 2) else float("nan")

    detail = dict(
        n_paths=len(paths),
        score_full=full,
        score_without_explanation=score(drop),      # fidelity−
        score_only_explanation=score(keep),         # fidelity+
        score_without_random=rnd_score,             # ĐỐI CHỨNG (trung bình 10 lần)
        sparsity=1 - len(expl_idx) / max(len(paths), 1),
        validity=float(all(p["nodes"][-1] in set(concept_ids) for p in keep)) if keep else 0.0,
    )
    return expl, full, detail

## 1.13 Chạy thí nghiệm chính

Một hàm `run_experiment` cho cả 3 cấu hình ablation, mỗi cấu hình 3 seed.

In [19]:
_GROUP_CACHE = {}
PER_QUERY = {}          # (ablation, seed) → MRR từng truy vấn test; khoá "<ab>+adaptive" cho bản thích nghi
ART_BY_ABL = {}         # ablation → mô hình + đồ thị của seed đầu
FUSION_LOG = []         # nhật ký kết hợp thích nghi theo (ablation, seed)


def reciprocal_ranks(cand, scores, gold_pos):
    order = np.argsort(-scores, axis=1)
    hit = np.take_along_axis(cand, order, axis=1) == np.asarray(gold_pos)[:, None]
    return np.where(hit.any(1), 1.0 / (hit.argmax(1) + 1), 0.0)


def prepare(records, split, img_feat, txt_feat, ablation="none", seed=0):
    graph = MultimodalGraph(records, split["train"], ablation=ablation, seed=seed)
    # Nhóm trực tiếp/gián tiếp chốt MỘT LẦN trên đồ thị đầy đủ rồi dùng lại cho mọi ablation.
    if "groups" not in _GROUP_CACHE:
        base = graph if ablation == "none" else MultimodalGraph(records, split["train"])
        _GROUP_CACHE["groups"] = {q["gold"]: q["group"] for q in
                                  build_query_set(records, base, [r["image_id"] for r in records])}
    queries = build_query_set(records, graph, [r["image_id"] for r in records], verbose=False)
    for q in queries:
        q["group"] = _GROUP_CACHE["groups"].get(q["gold"], q["group"])
    eng = RetrievalEngine(records, graph, img_feat, txt_feat, queries)
    pos = graph.img_pos
    idx = lambda ids: np.array([pos[i] for i in ids])
    parts = {}
    for name in ("train", "val", "test"):
        ids = split[name]
        parts[name] = dict(q_idx=idx(ids), gold_pos=idx(ids), img_idx=idx(ids),
                           groups=[queries[pos[i]]["group"] for i in ids])
    return graph, eng, parts


def evaluate_split(model, engine, part, alpha, lam, by_group=True):
    gallery = part["img_idx"]
    cand, sc, sg, sp_ = score_queries(model, engine, part["q_idx"], gallery)
    res = {}
    full = combine(sc, sg, sp_, alpha, lam)
    res["CLIP"] = retrieval_metrics(cand, sc, part["gold_pos"])
    res["CLIP+Graph"] = retrieval_metrics(cand, full, part["gold_pos"])
    if by_group:
        g = np.array(part["groups"])
        for grp in ("direct", "indirect"):
            m = g == grp
            if m.sum() >= 5:
                res[f"CLIP [{grp}] (n={int(m.sum())})"] = retrieval_metrics(
                    cand[m], sc[m], np.array(part["gold_pos"])[m])
                res[f"CLIP+Graph [{grp}] (n={int(m.sum())})"] = retrieval_metrics(
                    cand[m], full[m], np.array(part["gold_pos"])[m])
    return res, (cand, sc, sg, sp_)


# ============================ KẾT HỢP ĐIỂM THÍCH NGHI ============================
FUSION_FEATS = ["has_concept", "log_n_concepts", "path_coverage", "clip_margin"]


def fusion_features(engine, q_idx, cand, sc):
    """Đặc trưng của TRUY VẤN, không dùng nhãn đúng: có khái niệm không, bao nhiêu khái niệm,
    tỉ lệ ứng viên top-K có ≥1 đường đi, độ chênh top1–top2 của CLIP (CLIP tự tin thì ít cần đồ thị)."""
    ncon = np.array([len(engine.q_concepts[q]) for q in q_idx], np.float32)
    cov = (engine.path_feats(cand, q_idx).sum(-1) > 0).mean(1)
    srt = -np.sort(-sc, axis=1)
    return np.stack([(ncon > 0).astype(np.float32), np.log1p(ncon), cov,
                     srt[:, 0] - srt[:, 1]], 1).astype(np.float32)


def _fuse(a, l, sc, sg, sp):
    return a[:, None] * sc + (1 - a[:, None]) * (l[:, None] * sg + (1 - l[:, None]) * sp)


class AdaptiveFusion(nn.Module):
    def __init__(self, d, a0, l0, mu, sd):
        super().__init__()
        self.wa, self.wl = nn.Linear(d, 1), nn.Linear(d, 1)
        for lin, v in ((self.wa, a0), (self.wl, l0)):
            v = min(max(float(v), 0.02), 0.98)
            nn.init.zeros_(lin.weight); nn.init.constant_(lin.bias, math.log(v / (1 - v)))
        self.register_buffer("mu", torch.tensor(mu, dtype=torch.float32))
        self.register_buffer("sd", torch.tensor(sd, dtype=torch.float32))

    def gates(self, Xn):
        return torch.sigmoid(self.wa(Xn)).squeeze(-1), torch.sigmoid(self.wl(Xn)).squeeze(-1)

    @torch.no_grad()
    def score(self, X, sc, sg, sp):
        a, l = self.gates((torch.tensor(X) - self.mu) / self.sd)
        a, l = a.numpy(), l.numpy()
        return _fuse(a, l, sc, sg, sp), a, l


def fit_fusion(X, sc, sg, sp, cand, gold, a0, l0):
    """Listwise CE trên top-K ứng viên của VAL. Khởi tạo w=0 ⇒ xuất phát đúng bằng (α, λ) toàn cục."""
    gold = np.asarray(gold); hit = cand == gold[:, None]; keep = hit.any(1)
    mu, sd = X.mean(0), X.std(0) + 1e-6
    m = AdaptiveFusion(X.shape[1], a0, l0, mu, sd)
    m.wa.bias.requires_grad_(False); m.wl.bias.requires_grad_(False)   # α(q) chỉ lệch khỏi α toàn cục theo đặc trưng
    if keep.sum() < 10:
        return m
    t = lambda a: torch.tensor(a[keep], dtype=torch.float32)
    Xn, SC, SG, SP = t((X - mu) / sd), t(sc), t(sg), t(sp)
    y = torch.tensor(hit[keep].argmax(1))
    opt = torch.optim.Adam([m.wa.weight, m.wl.weight], lr=0.05)
    for _ in range(CFG.fusion_steps):
        a, l = m.gates(Xn)
        loss = F.cross_entropy(_fuse(a, l, SC, SG, SP) / CFG.fusion_tau, y) \
            + CFG.fusion_wd * (m.wa.weight.pow(2).sum() + m.wl.weight.pow(2).sum())
        opt.zero_grad(); loss.backward(); opt.step()
    return m


def cv_fusion(X, sc, sg, sp, cand, gold, a0, l0, k=5, seed=0):
    """MRR ngoài-fold trên VAL: thích nghi vs (α, λ) toàn cục. Chống overfit khi chỉ có ~100–1000 truy vấn val."""
    gold = np.asarray(gold); n = len(X)
    idx = np.random.RandomState(seed).permutation(n)
    rr_a, rr_g = np.zeros(n), np.zeros(n)
    for f in np.array_split(idx, k):
        tr = np.setdiff1d(idx, f)
        m = fit_fusion(X[tr], sc[tr], sg[tr], sp[tr], cand[tr], gold[tr], a0, l0)
        rr_a[f] = reciprocal_ranks(cand[f], m.score(X[f], sc[f], sg[f], sp[f])[0], gold[f])
        rr_g[f] = reciprocal_ranks(cand[f], combine(sc[f], sg[f], sp[f], a0, l0), gold[f])
    return float(rr_a.mean()), float(rr_g.mean())


def adaptive_rows(fus, engine, part, raw):
    cand, sc, sg, sp_ = raw
    X = fusion_features(engine, part["q_idx"], cand, sc)
    s, a, l = fus.score(X, sc, sg, sp_)
    gold = np.asarray(part["gold_pos"])
    res = {"CLIP+Graph (adaptive)": retrieval_metrics(cand, s, gold)}
    g = np.array(part["groups"])
    for grp in ("direct", "indirect"):
        m = g == grp
        if m.sum() >= 5:
            res[f"CLIP+Graph (adaptive) [{grp}] (n={int(m.sum())})"] = retrieval_metrics(cand[m], s[m], gold[m])
    return res, s, a, l, X


def _adaptive_step(model, eng, parts, alpha, lam, test_raw):
    pv = parts["val"]
    vc, vsc, vsg, vsp = score_queries(model, eng, pv["q_idx"], pv["img_idx"])
    Xv = fusion_features(eng, pv["q_idx"], vc, vsc)
    cv_a, cv_g = cv_fusion(Xv, vsc, vsg, vsp, vc, pv["gold_pos"], alpha, lam)
    fus = fit_fusion(Xv, vsc, vsg, vsp, vc, pv["gold_pos"], alpha, lam)
    ad_res, s_ad, a_q, l_q, Xt = adaptive_rows(fus, eng, parts["test"], test_raw)
    has = Xt[:, 0] > 0
    flog = dict(alpha=alpha, lam=lam, cv_global=cv_g, cv_adaptive=cv_a,
                alpha_q_with_concept=float(a_q[has].mean()) if has.any() else float("nan"),
                alpha_q_no_concept=float(a_q[~has].mean()) if (~has).any() else float("nan"),
                w_alpha=dict(zip(FUSION_FEATS, fus.wa.weight.detach().numpy().ravel().round(3).tolist())))
    return fus, flog, ad_res, s_ad


def run_experiment(records, split, img_feat, txt_feat, ablations=("none",), seeds=None):
    seeds = seeds or CFG.seeds
    _GROUP_CACHE.clear(); PER_QUERY.clear(); ART_BY_ABL.clear(); FUSION_LOG.clear()
    all_res = defaultdict(list)
    artefacts = {}
    for ab in ablations:
        print(f"\n{'='*66}\nABLATION = {ab}\n{'='*66}")
        for si, seed in enumerate(seeds):
            graph, eng, parts = prepare(records, split, img_feat, txt_feat, ab, seed)
            if si == 0:
                print("Đồ thị:", graph.summary())
            model, hist = train_graph_scorer(eng, parts["train"]["q_idx"],
                                             parts["train"]["img_idx"], seed=seed,
                                             verbose=(si == 0))
            (alpha, lam), grid = tune_on_val(model, eng, parts["val"], parts["val"]["img_idx"],
                                             verbose=(si == 0))
            res, raw = evaluate_split(model, eng, parts["test"], alpha, lam)
            cand, sc, sg, sp_ = raw
            gold_t, groups_t = parts["test"]["gold_pos"], np.array(parts["test"]["groups"])
            rr_clip = reciprocal_ranks(cand, sc, gold_t)
            PER_QUERY[(ab, seed)] = dict(rr=reciprocal_ranks(cand, combine(sc, sg, sp_, alpha, lam), gold_t),
                                         rr_clip=rr_clip, groups=groups_t, lam=lam)
            if "relprobe_hook" in globals():          # v5: bộ quan hệ khó (α, λ giữ nguyên từ VAL)
                relprobe_hook(ab, seed, model, eng, parts, alpha, lam)
            fus, flog = None, None
            if CFG.adaptive_fusion:
                fus, flog, ad_res, s_ad = _adaptive_step(model, eng, parts, alpha, lam, raw)
                res.update(ad_res)
                FUSION_LOG.append(dict(ablation=ab, seed=seed, **flog))
                PER_QUERY[(f"{ab}+adaptive", seed)] = dict(rr=reciprocal_ranks(cand, s_ad, gold_t),
                                                           rr_clip=rr_clip, groups=groups_t, lam=lam)
            for k, v in res.items():
                all_res[f"{ab} | {k}"].append(v)
            if si == 0:
                ART_BY_ABL[ab] = dict(graph=graph, engine=eng, model=model, parts=parts, alpha=alpha,
                                      lam=lam, grid=grid, raw=raw, hist=hist, fusion=fus)
            if ab == "none" and si == 0:
                artefacts = ART_BY_ABL[ab]
            msg = (f"  seed {seed}: R@1={res['CLIP+Graph']['R@1']:.4f} "
                   f"MRR={res['CLIP+Graph']['MRR']:.4f} (α={alpha}, λ={lam})")
            if flog:
                msg += (f" | adaptive MRR={res['CLIP+Graph (adaptive)']['MRR']:.4f} "
                        f"[CV val {flog['cv_adaptive']:.4f} vs {flog['cv_global']:.4f}; "
                        f"α(q) có/không khái niệm {flog['alpha_q_with_concept']:.2f}/{flog['alpha_q_no_concept']:.2f}]")
            print(msg)
    return all_res, artefacts


def results_table(all_res):
    import pandas as pd
    rows = []
    for k, runs in all_res.items():
        ab, name = k.split(" | ", 1)
        rows.append(dict(ablation=ab, model=name, **fmt_metrics(runs)))
    return pd.DataFrame(rows)


def final_model_table(all_res, fusion_log):
    """MÔ HÌNH CUỐI = (cấu trúc đồ thị, global|adaptive) có MRR VAL cao nhất (adaptive tính ngoài-fold).
    Test chỉ được ĐỌC cho cấu hình đã chọn. Đây là con số chính đưa vào báo cáo."""
    import pandas as pd
    fl = pd.DataFrame(fusion_log)
    fl = fl[fl.ablation == "none"]          # ablation là phân tích, không phải ứng viên mô hình cuối
    sel = pd.concat([fl.groupby("ablation").cv_global.mean().rename("val_MRR").reset_index().assign(mode="global"),
                     fl.groupby("ablation").cv_adaptive.mean().rename("val_MRR").reset_index().assign(mode="adaptive")]
                    ).sort_values("val_MRR", ascending=False).reset_index(drop=True)
    best = sel.iloc[0]
    tab = results_table(all_res)
    pat = r"^CLIP\+Graph( \[|$)" if best["mode"] == "global" else r"^CLIP\+Graph \(adaptive\)"
    test = tab[(tab.ablation == best.ablation) & (tab.model.str.match(pat) | tab.model.str.match(r"^CLIP( \[|$)"))]
    print(f"→ Mô hình cuối (chọn trên VAL): đồ thị='{best.ablation}', kết hợp={best['mode']} "
          f"(VAL MRR {best.val_MRR:.4f})")
    return sel, test

## 1.14 Phân tích 5 thành công / 5 thất bại + fidelity trên ≥10 truy vấn

In [20]:
def analyse_cases(art, records, n_success=5, n_fail=5):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    part, (cand, sc, sg, sp_) = art["parts"]["test"], art["raw"]
    full = combine(sc, sg, sp_, art["alpha"], art["lam"])
    gold = np.array(part["gold_pos"])

    def rank_of(S):
        o = np.argsort(-S, 1); r = np.take_along_axis(cand, o, 1)
        hit = r == gold[:, None]
        return np.where(hit.any(1), hit.argmax(1) + 1, 10**9)

    r_clip, r_full = rank_of(sc), rank_of(full)
    delta = r_clip - r_full                                   # >0 = đồ thị giúp
    succ = [i for i in np.argsort(-delta) if r_full[i] <= 10][:n_success]
    fail = [i for i in np.argsort(delta) if r_clip[i] < r_full[i]][:n_fail]

    cases = []
    for kind, idxs in (("THÀNH CÔNG", succ), ("THẤT BẠI", fail)):
        for i in idxs:
            qi = part["q_idx"][i]
            cids = [n for n, _, _ in eng.q_concepts[qi]]
            expl, s, det = explain_and_score(model, graph, int(gold[i]), cids)
            cases.append(dict(kind=kind, query=eng.queries[qi]["text"][:110],
                              group=eng.queries[qi]["group"],
                              rank_clip=int(r_clip[i]), rank_graph=int(r_full[i]),
                              explanation=expl, detail=det))
    return cases


def fidelity_study(art, n_queries=CFG.n_explain_queries, top_m_paths_min=CFG.top_m_paths):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    part = art["parts"]["test"]
    rows = []
    for i in range(len(part["q_idx"])):
        qi = part["q_idx"][i]
        cids = [n for n, _, _ in eng.q_concepts[qi]]
        if not cids:
            continue
        expl, s, det = explain_and_score(model, graph, int(part["gold_pos"][i]), cids)
        if det["n_paths"] == 0:
            continue
        if det["n_paths"] <= top_m_paths_min:   # quá ít đường đi ⇒ giải thích = toàn bộ, đối chứng vô nghĩa
            continue
        rows.append(dict(group=eng.queries[qi]["group"], **det,
                         drop_explanation=det["score_full"] - det["score_without_explanation"],
                         drop_random=det["score_full"] - det["score_without_random"]))
        if len(rows) >= n_queries:
            break
    import pandas as pd
    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠ Không có truy vấn nào có đường đi — kiểm tra lại bước dựng đồ thị.")
        return df, {}
    summary = dict(
        n_queries=len(df),
        mean_paths=float(df.n_paths.mean()),
        fidelity_minus=float(df.drop_explanation.mean()),
        fidelity_minus_random_control=float(df.drop_random.mean()),
        fidelity_plus_gap=float((df.score_full - df.score_only_explanation).mean()),
        sparsity=float(df.sparsity.mean()),
        validity=float(df.validity.mean()),
    )
    summary["necessity_ratio"] = (summary["fidelity_minus"] /
                                  max(summary["fidelity_minus_random_control"], 1e-9))
    return df, summary

In [21]:
@torch.no_grad()
def final_rank_fidelity(art, n_queries=50, top_m=CFG.top_m_paths, n_rand=10):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    part, alpha, lam = art["parts"]["test"], art["alpha"], art["lam"]
    model.eval()
    h = model.encode_graph(graph.edge_index.to(DEVICE))
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    cand_all, clip_all = eng.clip_candidates(part["q_idx"], part["img_idx"], CFG.rerank_k)
    rng = np.random.RandomState(0)
    z = znorm_per_row
    sc = lambda ps: gain * float((paths_to_counts(ps) * w).sum())
    rows = []

    for i, qi in enumerate(part["q_idx"]):
        cids = [n for n, _, _ in eng.q_concepts[qi]]
        gold = int(part["gold_pos"][i])
        cand = cand_all[i]
        hit = np.where(cand == gold)[0]
        if not cids or len(hit) == 0:
            continue
        paths = enumerate_paths(graph, gold, cids)
        if len(paths) <= top_m:          # ít đường đi quá thì đối chứng ngẫu nhiên vô nghĩa
            continue
        j = int(hit[0])
        pm = eng.path_feats(cand[None], np.array([qi]))[0]
        sp_raw = gain * (pm * w).sum(-1)
        s_gnn = model.gnn_score(h, eng.txt_feat[[qi]].to(DEVICE),
                                torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
        s_clip = clip_all[i][None]

        def rank_with(kept):
            sp = sp_raw.copy()
            sp[j] = sc(kept)
            fin = combine(z(s_clip), z(s_gnn), z(sp[None]), alpha, lam)[0]
            return int((fin > fin[j]).sum()) + 1

        full = sc(paths)
        contrib = np.array([full - sc(paths[:k] + paths[k + 1:]) for k in range(len(paths))])
        top = set(np.argsort(-contrib)[:top_m].tolist())

        r_rand = np.mean([
            rank_with([p for k, p in enumerate(paths)
                       if k not in set(rng.choice(len(paths), top_m, replace=False).tolist())])
            for _ in range(n_rand)])
        rows.append(dict(group=eng.queries[qi]["group"], n_paths=len(paths),
                         rank_full=rank_with(paths),
                         rank_drop_expl=rank_with([p for k, p in enumerate(paths) if k not in top]),
                         rank_drop_rand=float(r_rand),
                         rank_only_expl=rank_with([paths[k] for k in top])))
        if len(rows) >= n_queries:
            break

    import pandas as pd
    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠ Không truy vấn nào có > top_m đường đi.")
        return df, {}
    summ = dict(n_queries=len(df),
                rank_full=df.rank_full.mean(),
                rank_after_drop_explanation=df.rank_drop_expl.mean(),
                rank_after_drop_random=df.rank_drop_rand.mean(),
                rank_with_only_explanation=df.rank_only_expl.mean(),
                frac_rank_worse_when_drop_expl=float((df.rank_drop_expl > df.rank_full).mean()),
                frac_rank_worse_when_drop_rand=float((df.rank_drop_rand > df.rank_full).mean()))
    return df, summ

## 1.15 Demo: nhập truy vấn → top-k + giải thích

In [22]:
@torch.no_grad()
def demo_query(text, art, k=5):
    eng, graph, model = art["engine"], art["graph"], art["model"]
    gallery = art["parts"]["test"]["img_idx"]
    hits = parse_query(text, graph)
    print(f"\nTruy vấn: {text!r}")
    print(f"Khái niệm nhận ra: {[h[1] for h in hits] or '(không có)'}")
    cids = [n for n, _, _ in hits]

    # mã hoá truy vấn tự do: dùng CLIP nếu có, nếu không lấy caption gần nhất làm proxy
    q_emb = encode_free_text(text, eng)
    sim = (q_emb @ eng.img_feat[gallery].T).numpy()[0]
    top = np.argsort(-sim)[:CFG.rerank_k]
    cand = gallery[top]
    pm = torch.tensor(np.stack([_path_feats_free(graph, cand, cids)])).to(DEVICE)
    h = model.encode_graph(graph.edge_index.to(DEVICE))
    s_gnn = model.gnn_score(h, q_emb.to(DEVICE), torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
    s_path = model.path_score(pm).cpu().numpy()
    s = combine(znorm_per_row(sim[top][None]), znorm_per_row(s_gnn),
                znorm_per_row(s_path), art["alpha"], art["lam"])[0]
    order = np.argsort(-s)[:k]
    for rank, j in enumerate(order, 1):
        img_i = int(cand[j])
        rec = eng.records[img_i]
        expl, _, det = explain_and_score(model, graph, img_i, cids)
        print(f"\n  #{rank}  image_id={rec['image_id']}  score={s[j]:.4f}")
        print(f"      caption: {rec['caption'][:90]}")
        print(f"      đối tượng: {rec['objects'][:8]}")
        if expl:
            print(f"      GIẢI THÍCH ({det['n_paths']} đường đi, hiển thị {len(expl)}):")
            for e in expl:
                print(f"        • {e['text']}   [đóng góp {e['contribution']:+.4f}]")
        else:
            print("      (không có đường đi đồ thị — xếp hạng chỉ dựa trên CLIP)")


def _proxy_text(text, eng):
    toks = set(_norm(text).split())
    best, bi = -1, 0
    for i, q in enumerate(eng.queries):
        ov = len(toks & set(_norm(q["text"]).split()))
        if ov > best:
            best, bi = ov, i
    return eng.txt_feat[bi:bi + 1]


def encode_free_text(text, eng):
    """CLIP thật chỉ dùng khi đặc trưng ảnh cũng là CLIP thật; synthetic thì dùng proxy."""
    if dataset_name == "synthetic":
        return _proxy_text(text, eng)
    try:
        import open_clip
        if not hasattr(encode_free_text, "_m"):
            m, _, _ = open_clip.create_model_and_transforms(CFG.clip_model,
                                                            pretrained=CFG.clip_weights)
            encode_free_text._m = m.to(DEVICE).eval()
            encode_free_text._t = open_clip.get_tokenizer(CFG.clip_model)
        with torch.no_grad():
            f = encode_free_text._m.encode_text(encode_free_text._t([text]).to(DEVICE))
        return F.normalize(f, dim=-1).float().cpu()
    except Exception:
        return _proxy_text(text, eng)

def _path_feats_free(graph, cand, cids):
    out = np.zeros((len(cand), len(PATTERNS)), np.float32)
    if not cids:
        return out
    cols = np.array(sorted(set(cids)))
    for p, name in enumerate(PATTERNS):
        out[:, p] = np.asarray(graph.R[name][cand][:, cols].sum(1)).ravel()
    return np.log1p(out)

## 1.16 Self-test — chạy trước khi tin bất kỳ con số nào

Ba kiểm tra này chính là ba câu hỏi vấn đáp dễ bị hỏi nhất. Chạy được và in ✓ thì
nhóm có bằng chứng ngay tại chỗ.

In [23]:
def self_test(records, split, img_feat, txt_feat):
    import copy
    ok = []

    # (1) RÒ RỈ — cấy một quan hệ chỉ tồn tại ở ảnh TEST, đồ thị train không được có
    rec2 = copy.deepcopy(records)
    pos = {r["image_id"]: i for i, r in enumerate(rec2)}
    A, B = "zzsecretobja", "zzsecretobjb"
    for iid in split["test"][:40]:
        r = rec2[pos[iid]]
        r["objects"] = sorted(set(r["objects"]) | {A, B})
        r["relations"] = list(r["relations"]) + [(A, "juggles", B)]
    gt = MultimodalGraph(rec2, split["train"])
    ga = MultimodalGraph(rec2, split["train"] + split["test"])
    e = lambda G, M: (M[G.obj2id[A], G.obj2id[B]] != 0) if A in G.obj2id else False
    c1 = (not e(gt, gt.A_scene)) and e(ga, ga.A_scene)
    c2 = (not e(gt, gt.A_related)) and e(ga, ga.A_related)
    ok.append(("Cạnh scene + đồng xuất hiện chỉ xây từ train", c1 and c2))

    # (2) TRUNG THỰC — điểm đường đi tính bằng ma trận phải khớp liệt kê từng đường đi
    graph = MultimodalGraph(records, split["train"])
    qs = build_query_set(records, graph, [r["image_id"] for r in records], verbose=False)
    eng = RetrievalEngine(records, graph, img_feat, txt_feat, qs)
    bad = n = 0
    for qi in range(min(300, len(qs))):
        cids = [x for x, _, _ in eng.q_concepts[qi]]
        if not cids:
            continue
        n += 1
        m = eng.path_feats(np.array([[qi]]), np.array([qi]))[0, 0]
        v = paths_to_counts(enumerate_paths(graph, qi, cids))
        bad += int(not np.allclose(m, v, atol=1e-5))
    ok.append((f"Điểm đường đi khớp giải thích liệt kê ({n - bad}/{n} cặp)", bad == 0))

    # (3) SPLIT — không chồng lấn và tái lập được
    s2 = make_split(records, seed=CFG.split_seed)
    ok.append(("Split tái lập được từ seed", s2["sha1"] == split["sha1"]))

    # (4) v5 — bộ ba cấp ảnh: nhãn chỉ từ caption TRAIN, ảnh train nhận dự đoán ngoài-fold
    if globals().get("TRIPLE_INFO") and any(r.get("img_triples") for r in records):
        ok.append(("Bộ ba cấp ảnh: nhãn huấn luyện chỉ từ ảnh TRAIN, ảnh train dùng dự đoán ngoài-fold",
                   bool(TRIPLE_INFO.get("train_only")) and bool(TRIPLE_INFO.get("oof"))))
        g3 = MultimodalGraph(records, split["train"])
        ok.append((f"Node bộ ba có trong đồ thị ({g3.n_tri} node, {g3.A_triple.nnz} cạnh ảnh–bộ ba)",
                   g3.n_tri > 0 and g3.A_triple.nnz > 0))

    print()
    for name, passed in ok:
        print(f"  {'✓' if passed else '✗ THẤT BẠI'}  {name}")
    assert all(p for _, p in ok), "Self-test không đạt — dừng lại và sửa trước khi chạy tiếp."
    return True

---
## 1.17 Thực thi (Flickr8k)

Từ đây trở xuống là phần chạy thật. Muốn chạy nhanh để thử, giảm `CFG.n_images`.

In [24]:
# ============ FLICKR8K: nạp dữ liệu, rút đối tượng/quan hệ, CLIP, split ============
import nltk
for _pkg in ("wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"):
    try: nltk.download(_pkg, quiet=True)
    except Exception: pass
from nltk.stem import WordNetLemmatizer

_LEM = WordNetLemmatizer(); _LEM_CACHE = {}
try:
    from nltk.tag import PerceptronTagger
    _TAGGER = PerceptronTagger()          # tạo MỘT lần (pos_tag() tạo lại mỗi lời gọi → rất chậm)
except Exception as e:
    print("⚠ Không tải được POS tagger → dùng cách rút danh từ thô:", type(e).__name__)
    _TAGGER = None

def _lemma(w):
    r = _LEM_CACHE.get(w)
    if r is None:
        try: r = _LEM.lemmatize(w, "n")
        except Exception: r = w[:-1] if (w.endswith("s") and len(w) > 3) else w
        _LEM_CACHE[w] = r
    return r

NOT_OBJECTS = set("front background foreground top bottom side end corner air group area region place space "
                  "picture photo image one other something someone anything thing stuff view middle edge "
                  "part way lot pair couple bunch number while each both several many few kind type".split())
_SKIP_P = set("is are was were be been being has have had while as that who which and or".split())


def extract_objects_relations(captions):
    """Danh từ NHÌN THẤY ĐƯỢC (lọc WordNet) → lớp đối tượng. Quan hệ = (danh từ, động từ/giới từ,
    danh từ kế tiếp). Heuristic — nên đọc thử ~30 mẫu để báo cáo độ chính xác."""
    objs, rels = set(), set()
    for cap in captions:
        toks = re.findall(r"[A-Za-z]+", cap)
        if not toks:
            continue
        if _TAGGER is not None:
            tags = _TAGGER.tag(toks)
        else:
            tags = [(t, "NN" if len(t) > 2 and t.lower() not in STOP else "DT") for t in toks]
        last_n, preds = None, []
        for w, t in tags:
            wl = w.lower()
            if t in ("NN", "NNS") and len(wl) > 2 and wl not in NOT_OBJECTS and wl not in STOP:
                n = _lemma(wl)
                if n in NOT_OBJECTS or not is_visual_noun(n):
                    continue                     # tính từ/động từ bị tag nhầm NN: pink, walking, stand...
                objs.add(n)
                if last_n is not None and 1 <= len(preds) <= 2 and last_n != n:
                    rels.add((last_n, " ".join(preds), n))
                last_n, preds = n, []
            elif (t.startswith("VB") or t in ("IN", "TO")) and wl not in _SKIP_P and last_n is not None:
                preds.append(wl)
    return objs, rels


def load_flickr8k():
    """Chỉ rút nhãn-từ-caption (cap_objects/cap_relations). objects/relations THẬT dùng cho
    đồ thị được gán sau bởi apply_flickr_annotation (tagger hoặc oracle)."""
    from datasets import load_dataset as hf_load
    dsd = hf_load("jxie/flickr8k")
    records, parts, iid = [], [], 0
    for hf_name, tag in (("train", "train"), ("validation", "val"), ("test", "test")):
        d = dsd[hf_name]; parts.append(d)
        for row in d.remove_columns("image"):
            allc = [str(row[f"caption_{k}"]).strip() for k in range(5)]
            objs, rels = extract_objects_relations(allc[1:])   # caption 1..4; caption_0 là truy vấn
            records.append(dict(image_id=iid, caption=allc[0], captions=allc,
                                cap_objects=sorted(objs), cap_relations=sorted(rels),
                                objects=[], relations=[], url="", split=tag))
            iid += 1
    print(f"Flickr8k: {len(records)} ảnh | {np.mean([len(r['cap_objects']) for r in records]):.1f} danh từ/ảnh | "
          f"{np.mean([len(r['cap_relations']) for r in records]):.1f} quan hệ/ảnh (rút từ caption)")
    for r in records[:3]:
        print("  •", r["caption"], "→", r["cap_objects"][:8], r["cap_relations"][:3])
    return records, parts


def encode_flickr8k(records, parts):
    cache = CFG.out_dir / "cache" / f"clip_flickr8k_{CFG.clip_model}_{CFG.clip_weights}_{len(records)}.npz"
    if cache.exists():
        z = np.load(cache); print("Dùng lại cache CLIP:", cache.name); return z["img"], z["txt"]
    import open_clip
    model, _, preprocess = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)
    tokenizer = open_clip.get_tokenizer(CFG.clip_model)
    model = model.to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    img, txt = [], []
    with torch.no_grad():
        for d in parts:                                  # đúng thứ tự train → val → test như records
            for s in range(0, len(d), 128):
                ims = d[s:s + 128]["image"]
                x = torch.stack([preprocess(im.convert("RGB")) for im in ims]).to(DEVICE)
                img.append(F.normalize(model.encode_image(x), dim=-1).float().cpu().numpy())
        caps = [r["caption"] for r in records]
        for s in range(0, len(caps), CFG.clip_bs):
            tok = tokenizer(caps[s:s + CFG.clip_bs]).to(DEVICE)
            txt.append(F.normalize(model.encode_text(tok), dim=-1).float().cpu().numpy())
    del model; torch.cuda.empty_cache()
    img, txt = np.concatenate(img), np.concatenate(txt)
    assert len(img) == len(records) == len(txt)
    np.savez_compressed(cache, img=img, txt=txt)
    return img, txt


# =====================================================================================
# NHÃN ẢNH BẰNG CLIP ZERO-SHOT (thay cho nhãn rút từ caption của chính ảnh)
#   - Từ vựng: danh từ nhìn thấy được trong caption TRAIN, tần suất ≥ min_obj_freq.
#   - Điểm: cos(ảnh, prompt-ensemble); chuẩn hoá theo từng từ bằng mean/std trên ảnh TRAIN.
#   - (cách chuẩn hoá, k) chọn trên VAL theo F1 so với danh từ trong caption val.
#   - Caption của ảnh val/test KHÔNG bao giờ vào đồ thị; caption test chỉ dùng để BÁO CÁO P/R/F1.
# =====================================================================================
@torch.no_grad()
def clip_text_features(texts, bs=512):
    import open_clip
    if not hasattr(clip_text_features, "_m"):
        m, _, _ = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)
        clip_text_features._m = m.to(DEVICE).eval()
        clip_text_features._t = open_clip.get_tokenizer(CFG.clip_model)
    m, tok = clip_text_features._m, clip_text_features._t
    out = [F.normalize(m.encode_text(tok(texts[s:s + bs]).to(DEVICE)), dim=-1).float().cpu().numpy()
           for s in range(0, len(texts), bs)]
    return np.concatenate(out)


def _prf(pred, truth):
    tp = sum(len(p & t) for p, t in zip(pred, truth))
    P = tp / max(sum(map(len, pred)), 1); R = tp / max(sum(map(len, truth)), 1)
    return P, R, 2 * P * R / max(P + R, 1e-9)


def clip_tagger(records, split, img_feat, text_encoder=None):
    text_encoder = text_encoder or clip_text_features
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    tr, va, te = (np.array([pos[i] for i in split[k]]) for k in ("train", "val", "test"))
    freq = Counter(o for i in tr for o in records[i]["cap_objects"])
    vocab = sorted(o for o, c in freq.items() if c >= CFG.min_obj_freq)
    vset = set(vocab)

    T = sum(text_encoder([tpl.format(w) for w in vocab]) for tpl in CFG.tag_prompts)
    T = T / np.linalg.norm(T, axis=1, keepdims=True)
    S = img_feat @ T.T                                          # [N, V]
    mu, sd = S[tr].mean(0), S[tr].std(0) + 1e-6                 # thống kê CHỈ trên train
    scores = {"raw": S, "z": (S - mu) / sd, "raw+z": S / S[tr].std() + (S - mu) / sd}

    def tags(Sc, idx, k):
        top = np.argpartition(-Sc[idx], k, axis=1)[:, :k]
        return [{vocab[j] for j in row} for row in top]
    truth = lambda idx: [set(records[i]["cap_objects"]) & vset for i in idx]

    grid, best = [], None
    for mode, Sc in scores.items():
        for k in CFG.tag_k_grid:
            P, R, F1 = _prf(tags(Sc, va, k), truth(va))
            grid.append(dict(norm=mode, k=k, val_P=P, val_R=R, val_F1=F1))
            if best is None or F1 > best[0]:
                best = (F1, mode, k)
    _, mode, k = best
    all_tags = tags(scores[mode], np.arange(len(records)), k)
    P, R, F1 = _prf([all_tags[i] for i in te], truth(te))
    info = dict(vocab_size=len(vocab), norm=mode, k=k, test_P=P, test_R=R, test_F1=F1, grid=grid)
    return all_tags, info


def apply_flickr_annotation(records, split, img_feat, mode=None, text_encoder=None):
    """Gán records[i]['objects'] / ['relations'] dùng cho đồ thị.
    relations chỉ giữ cho ảnh TRAIN (dùng để xây cạnh scene thống kê)."""
    mode = mode or CFG.flickr_annot
    tr = set(split["train"])
    info = dict(annotation=mode)
    if mode == "tagger":
        global TAG_INFO
        if not all("tags" in r for r in records) or "TAG_INFO" not in globals():
            tags, TAG_INFO = clip_tagger(records, split, img_feat, text_encoder)
            for r, t in zip(records, tags):
                r["tags"] = sorted(t)
        info.update(TAG_INFO)
    for r in records:
        r["objects"] = list(r["tags"]) if mode == "tagger" else list(r["cap_objects"])
        r["relations"] = list(r["cap_relations"]) if r["image_id"] in tr else []
    msg = f"Nhãn ảnh = {mode}"
    if mode == "tagger":
        msg += (f" | vocab={info['vocab_size']} | chuẩn hoá={info['norm']}, k={info['k']} (chọn trên val)"
                f" | TEST P={info['test_P']:.3f} R={info['test_R']:.3f} F1={info['test_F1']:.3f}")
    print(msg)
    return info


# Split chính thức của HF; bản ngẫu nhiên cũ giữ lại cho trường hợp synthetic.
if "_make_split_random" not in globals():
    _make_split_random = make_split

def make_split(records, seed=CFG.split_seed):
    if not all("split" in r for r in records):
        return _make_split_random(records, seed)
    split = {k: sorted(r["image_id"] for r in records if r["split"] == k) for k in ("train", "val", "test")}
    split["seed"] = "official-hf-split"
    payload = json.dumps({k: split[k] for k in ("train", "val", "test")}, sort_keys=True)
    split["sha1"] = hashlib.sha1(payload.encode()).hexdigest()[:12]
    with open(CFG.out_dir / "split.json", "w") as f:
        json.dump(split, f)
    print(f"Split chính thức HF (sha1={split['sha1']}): train={len(split['train'])} "
          f"val={len(split['val'])} test={len(split['test'])}")
    assert not (set(split["train"]) & set(split["test"])) and not (set(split["val"]) & set(split["test"]))
    return split


# Truy vấn Flickr có số nhiều ("dogs") → phải lemmatize như lúc rút đối tượng.
def parse_query(text, graph):
    toks = [_lemma(t) for t in _norm(text).split() if t not in STOP and len(t) > 1]
    cands = []
    for n in (2, 1):
        for i in range(len(toks) - n + 1):
            cands.append(" ".join(toks[i:i + n]))
    hits, seen = [], set()
    for g in cands:
        nid = graph.name2id.get(g)
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, g, g))
    # v5: bộ ba (s, p, o) tách từ truy vấn → node bộ ba cấp ảnh (mẫu P6)
    if getattr(graph, "n_tri", 0) and "query_triples" in globals():
        for key in query_triples(text):
            nid = graph.tri2id.get(key)
            if nid is not None and nid not in seen:
                seen.add(nid); hits.append((nid, " ".join(key), " ".join(key)))
    return hits

## 1.2b Quan hệ cấp ảnh (v5)

Lần chạy trước cho thấy `none ≈ no_relations ≈ random_edges` trên cả ba bộ dữ liệu. Lý do: mọi cạnh quan hệ (scene / related / isa) là **thống kê cấp lớp** gộp từ train, nên hai ảnh cùng chứa `dog` nhận đúng cùng một tập đường đi và quan hệ không phân biệt được ứng viên.

Bản này thêm **node bộ ba cấp ảnh** ⟨chủ thể, vị từ, đối tượng⟩: mỗi ảnh có tập bộ ba riêng, do một linear probe (khởi tạo từ văn bản CLIP, học trên caption **train**, ảnh train nhận dự đoán ngoài-fold) dự đoán từ đặc trưng CLIP của chính ảnh. Mẫu đường đi mới `P6: ảnh –thể_hiện→ ⟨s p o⟩` khớp với bộ ba tách từ truy vấn và vẫn cộng tính, nên giải thích và fidelity giữ nguyên cơ chế. Nhãn đối tượng cũng chuyển từ CLIP zero-shot sang probe cùng kiểu.

Đối chứng mới: `no_triples` (= `none` của bản trước), `shuffled_triples` (ảnh nhận bộ ba của ảnh khác). Đánh giá mới: **bộ quan hệ khó**, gallery gồm ảnh đúng và các ảnh test có cùng đối tượng nhưng không có quan hệ của truy vấn (theo caption test, chỉ dùng làm nhãn chấm).


In [25]:
# =====================================================================================
# 1.2b  QUAN HỆ CẤP ẢNH (bản v5) — vì sao cần
#   Lần chạy trước: 'none' ≈ 'no_relations' ≈ 'random_edges' trên cả Flickr8k, AG, MSR-VTT.
#   Nguyên nhân: mọi cạnh quan hệ (scene / related / isa) là thống kê CẤP LỚP gộp từ train —
#   hai ảnh cùng có 'dog' nhận ĐÚNG cùng tập đường đi 'dog → …', nên quan hệ không phân biệt
#   được ứng viên; nó chỉ là một kiểu mở rộng truy vấn mà CLIP đã làm tốt hơn.
#   Bản này thêm:
#     (1) Nhãn đối tượng bằng LINEAR PROBE khởi tạo từ văn bản CLIP (học trên caption TRAIN,
#         ảnh train dùng dự đoán NGOÀI-FOLD) thay cho zero-shot (F1 test cũ 0.25).
#     (2) Node BỘ BA (chủ thể, vị từ, đối tượng) CẤP ẢNH: mỗi ảnh có tập bộ ba riêng do probe
#         dự đoán từ đặc trưng CLIP của CHÍNH ẢNH (không dùng caption val/test).
#         Mẫu đường đi mới P6: ảnh –thể_hiện→ ⟨s p o⟩, khớp với bộ ba tách từ truy vấn.
#     (3) Đối chứng mới: no_triples (= 'none' của bản trước), shuffled_triples (bộ ba của
#         ảnh KHÁC ⇒ kiểm tra tín hiệu có thật sự gắn với ảnh không).
#     (4) Bộ đánh giá QUAN HỆ KHÓ: gallery = ảnh đúng + ảnh test có CÙNG đối tượng nhưng
#         KHÔNG có quan hệ của truy vấn (theo caption test — chỉ dùng làm NHÃN CHẤM).
# =====================================================================================
import shutil

# --- dùng lại cache CLIP của lần chạy trước nếu output cũ được Add Input ---
for _f in glob.glob("/kaggle/input/**/cache/clip_flickr8k_*.npz", recursive=True):
    _dst = CFG.out_dir / "cache" / Path(_f).name
    if not _dst.exists():
        shutil.copy2(_f, _dst)
        print("Dùng lại cache CLIP từ input:", _dst.name)

PREPS = set("in on at with by for from into onto over under near behind beside across through along "
            "around against toward towards up down out off past above below between inside outside "
            "beneath underneath atop upon".split())
TRIVIAL_PREDS = {"of", "for", "by"}          # 'body of water', 'pair of shoe' không phải quan hệ thị giác
_VLEM_CACHE = {}


def _verb_lemma(w):
    r = _VLEM_CACHE.get(w)
    if r is None:
        try: r = _LEM.lemmatize(w, "v")
        except Exception: r = w
        _VLEM_CACHE[w] = r
    return r


def canon_triple(s, p, o):
    """('dog', 'running after', 'ball') → ('dog', 'run', 'ball'); ('dog', 'in', 'snow') giữ giới từ."""
    toks = p.split()
    if not toks or s == o:
        return None
    t0 = toks[0]
    cp = t0 if t0 in PREPS else _verb_lemma(t0)
    if cp in TRIVIAL_PREDS:
        return None
    return (s, cp, o)


def caption_triples(caps):
    _, rels = extract_objects_relations(caps)
    return {t for t in (canon_triple(*r) for r in rels) if t is not None}


_QT_CACHE = {}


def query_triples(text):
    """Bộ ba chuẩn hoá tách từ VĂN BẢN TRUY VẤN — cùng bộ rút quan hệ dùng cho caption train."""
    if text not in _QT_CACHE:
        _QT_CACHE[text] = sorted(caption_triples([text]))
    return _QT_CACHE[text]


# ---------------------------------------------------------------------------------
# Linear probe khởi tạo từ văn bản CLIP: logit = e^s·cos(ảnh, w_v) + b_v, w_v khởi tạo = text_v,
# phạt ‖W − T‖² để từ hiếm vẫn bám prior văn bản. Ảnh TRAIN: dự đoán NGOÀI-FOLD (k-fold).
# ---------------------------------------------------------------------------------
def _fit_probe(X, Y, T, steps=None, lr=None, wd_init=None, seed=0):
    steps = steps or CFG.probe_steps; lr = lr or CFG.probe_lr
    wd_init = CFG.probe_wd_init if wd_init is None else wd_init
    torch.manual_seed(seed)
    X, Y, T = (torch.as_tensor(a, dtype=torch.float32, device=DEVICE) for a in (X, Y, T))
    W = nn.Parameter(T.clone())
    s = nn.Parameter(torch.tensor(math.log(30.0), device=DEVICE))
    prior = Y.mean(0).clamp(1e-4, 1 - 1e-4)
    with torch.no_grad():
        b0 = torch.log(prior / (1 - prior)) - (s.exp() * (X @ T.T)).mean(0)
    b = nn.Parameter(b0)
    opt = torch.optim.Adam([dict(params=[W], lr=lr), dict(params=[b, s], lr=lr * 10)])
    for _ in range(steps):
        logit = s.exp() * (X @ W.T) + b
        loss = F.binary_cross_entropy_with_logits(logit, Y) + wd_init * (W - T).pow(2).sum()
        opt.zero_grad(); loss.backward(); opt.step()
    W, b, s = W.detach(), b.detach(), s.detach()
    return lambda Xq: torch.sigmoid(s.exp() * (torch.as_tensor(Xq, dtype=torch.float32, device=DEVICE) @ W.T)
                                    + b).cpu().numpy()


def cross_fit_probe(X, Y_tr, T, tr, folds=None, seed=0):
    """X [N, D] mọi ảnh; Y_tr [len(tr), V] nhãn CHỈ của ảnh train. Trả P [N, V]:
    ảnh train ← mô hình KHÔNG thấy ảnh đó (ngoài-fold); ảnh val/test ← mô hình học trên toàn bộ train."""
    folds = folds or CFG.probe_folds
    tr = np.asarray(tr)
    P = np.zeros((len(X), T.shape[0]), np.float32)
    order = np.random.RandomState(seed).permutation(len(tr))
    for f in np.array_split(order, folds):
        keep = np.setdiff1d(order, f)
        pred = _fit_probe(X[tr[keep]], Y_tr[keep], T, seed=seed)
        P[tr[f]] = pred(X[tr[f]])
    rest = np.setdiff1d(np.arange(len(X)), tr)
    if len(rest):
        P[rest] = _fit_probe(X[tr], Y_tr, T, seed=seed)(X[rest])
    return P


def _multi_hot(sets, vocab_index, n_rows):
    Y = np.zeros((n_rows, len(vocab_index)), np.float32)
    for i, s in enumerate(sets):
        for t in s:
            j = vocab_index.get(t)
            if j is not None:
                Y[i, j] = 1.0
    return Y


def _topk_sets(P, idx, vocab, k, tau=0.0):
    out = []
    for i in idx:
        row = P[i]
        top = np.argpartition(-row, min(k, len(row) - 1))[:k]
        out.append({vocab[j] for j in top if row[j] >= tau})
    return out


def probe_tagger(records, split, img_feat, text_encoder=None):
    text_encoder = text_encoder or clip_text_features
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    tr, va, te = (np.array([pos[i] for i in split[k]]) for k in ("train", "val", "test"))
    freq = Counter(o for i in tr for o in records[i]["cap_objects"])
    vocab = sorted(o for o, c in freq.items() if c >= CFG.min_obj_freq)
    vix = {w: j for j, w in enumerate(vocab)}
    T = sum(text_encoder([tpl.format(w) for w in vocab]) for tpl in CFG.tag_prompts)
    T = T / np.linalg.norm(T, axis=1, keepdims=True)
    Y_tr = _multi_hot([records[i]["cap_objects"] for i in tr], vix, len(tr))
    P = cross_fit_probe(img_feat, Y_tr, T, tr)
    vset = set(vocab)
    truth = lambda idx: [set(records[i]["cap_objects"]) & vset for i in idx]
    grid, best = [], None
    for k in CFG.tag_k_grid:
        Pv, Rv, Fv = _prf(_topk_sets(P, va, vocab, k), truth(va))
        grid.append(dict(norm="probe", k=k, val_P=Pv, val_R=Rv, val_F1=Fv))
        if best is None or Fv > best[0]:
            best = (Fv, k)
    k = best[1]
    tags = _topk_sets(P, np.arange(len(records)), vocab, k)
    Pt, Rt, Ft = _prf([tags[i] for i in te], truth(te))
    info = dict(vocab_size=len(vocab), norm="probe(ngoài-fold)", k=k, test_P=Pt, test_R=Rt, test_F1=Ft,
                grid=grid, vocab=vocab)
    return tags, info


if "_apply_flickr_annotation_base" not in globals():
    _apply_flickr_annotation_base = apply_flickr_annotation


def apply_flickr_annotation(records, split, img_feat, mode=None, text_encoder=None):
    mode = mode or CFG.flickr_annot
    if mode != "probe":
        return _apply_flickr_annotation_base(records, split, img_feat, mode, text_encoder)
    global PROBE_INFO
    if not all("probe_tags" in r for r in records) or "PROBE_INFO" not in globals():
        tags, PROBE_INFO = probe_tagger(records, split, img_feat, text_encoder)
        for r, t in zip(records, tags):
            r["probe_tags"] = sorted(t)
    tr = set(split["train"])
    for r in records:
        r["objects"] = list(r["probe_tags"])
        r["relations"] = list(r["cap_relations"]) if r["image_id"] in tr else []
    info = dict(annotation="probe", **{k: v for k, v in PROBE_INFO.items() if k != "vocab"})
    print(f"Nhãn ảnh = probe | vocab={info['vocab_size']} | k={info['k']} (chọn trên val) | "
          f"TEST P={info['test_P']:.3f} R={info['test_R']:.3f} F1={info['test_F1']:.3f}")
    return info


def fit_image_triples(records, split, img_feat, text_encoder=None):
    """Gán records[i]['img_triples'] = [((s, p, o), xác_suất)] cho MỌI ảnh.
    Nhãn huấn luyện = bộ ba chuẩn hoá trong caption 1..4 của ảnh TRAIN; ảnh train nhận dự đoán
    ngoài-fold; (k, τ) chọn trên VAL theo F1; caption test chỉ để BÁO CÁO P/R/F1."""
    text_encoder = text_encoder or clip_text_features
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    tr, va, te = (np.array([pos[i] for i in split[k]]) for k in ("train", "val", "test"))
    labels = {}
    cnt, surface = Counter(), defaultdict(Counter)
    for i in range(len(records)):
        ts = set()
        for s_, p_, o_ in records[i]["cap_relations"]:
            t = canon_triple(s_, p_, o_)
            if t is None:
                continue
            ts.add(t)
        labels[i] = ts
    tr_set = set(tr.tolist())
    for i in tr:
        for s_, p_, o_ in records[i]["cap_relations"]:
            t = canon_triple(s_, p_, o_)
            if t is not None:
                cnt[t] += 1; surface[t][p_] += 1
    obj_vocab = set(PROBE_INFO["vocab"]) if "PROBE_INFO" in globals() else \
        {o for i in tr for o in records[i]["cap_objects"]}
    vocab = sorted(t for t, c in cnt.items()
                   if c >= CFG.triple_min_count and t[0] in obj_vocab and t[2] in obj_vocab)
    vix = {t: j for j, t in enumerate(vocab)}
    prompts = [f"a photo of a {s} {surface[(s, p, o)].most_common(1)[0][0]} the {o}." for s, p, o in vocab]
    T = text_encoder(prompts)
    T = T / np.linalg.norm(T, axis=1, keepdims=True)
    Y_tr = _multi_hot([labels[i] for i in tr], vix, len(tr))
    P = cross_fit_probe(img_feat, Y_tr, T, tr)
    vset = set(vocab)
    truth = lambda idx: [labels[i] & vset for i in idx]
    grid, best = [], None
    for k in CFG.triple_k_grid:
        for tau in CFG.triple_tau_grid:
            Pv, Rv, Fv = _prf(_topk_sets(P, va, vocab, k, tau), truth(va))
            grid.append(dict(k=k, tau=tau, val_P=Pv, val_R=Rv, val_F1=Fv))
            if best is None or Fv > best[0]:
                best = (Fv, k, tau)
    _, k, tau = best
    # zero-shot (chỉ văn bản, chưa học) để so sánh
    S0 = img_feat @ T.T
    zs = _topk_sets((S0 - S0[tr].mean(0)) / (S0[tr].std(0) + 1e-6), te, vocab, k)
    for i in range(len(records)):
        row = P[i]
        top = np.argpartition(-row, min(k, len(row) - 1))[:k]
        records[i]["img_triples"] = sorted(((vocab[j], float(row[j])) for j in top if row[j] >= tau),
                                           key=lambda x: -x[1])
    pred_te = [{t for t, _ in records[i]["img_triples"]} for i in te]
    Pt, Rt, Ft = _prf(pred_te, truth(te))
    _, _, Fz = _prf(zs, truth(te))
    qcov = np.mean([bool(set(query_triples(records[i]["caption"])) & vset) for i in te])
    info = dict(vocab_size=len(vocab), k=k, tau=tau, test_P=Pt, test_R=Rt, test_F1=Ft, zero_shot_test_F1=Fz,
                mean_triples_per_image=float(np.mean([len(r["img_triples"]) for r in records])),
                test_queries_with_vocab_triple=float(qcov), grid=grid,
                train_only=bool(Y_tr.shape[0] == len(tr_set)), oof=True,
                examples=[dict(caption=records[i]["caption"], triples=[" ".join(t) for t, _ in records[i]["img_triples"][:5]])
                          for i in te[:5]])
    print(f"Bộ ba cấp ảnh: vocab={len(vocab)} | k={k}, τ={tau} (chọn trên val) | "
          f"TEST P={Pt:.3f} R={Rt:.3f} F1={Ft:.3f} (zero-shot F1={Fz:.3f}) | "
          f"{info['mean_triples_per_image']:.2f} bộ ba/ảnh | truy vấn test có bộ ba trong vocab: {qcov:.1%}")
    return info


# ---------------------------------------------------------------------------------
# Bộ đánh giá QUAN HỆ KHÓ: cùng đối tượng, khác quan hệ.
# ---------------------------------------------------------------------------------
REL_GALLERIES, REL_PQ = {}, defaultdict(dict)
REL_GAL_KEY, REL_RENAME = None, {}


def _sample_negs(negs, rng):
    if len(negs) > CFG.rel_neg_max:
        negs = list(rng.choice(negs, CFG.rel_neg_max, replace=False))
    return negs


def build_rel_galleries_flickr(records, split, seed=0):
    """Truy vấn test có ≥1 bộ ba (s, p, o). Âm = ảnh test mà 5 caption nhắc ĐỦ mọi s, o của truy vấn
    nhưng KHÔNG có bộ ba nào của truy vấn. Caption test chỉ dùng làm NHÃN CHẤM."""
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    te = [pos[i] for i in split["test"]]
    nouns, tris = {}, {}
    for p in te:
        o, rels = extract_objects_relations(records[p]["captions"])
        nouns[p] = o
        tris[p] = {t for t in (canon_triple(*r) for r in rels) if t is not None}
    rng = np.random.RandomState(seed)
    gal, n_q = [], 0
    for p in te:
        qt = set(query_triples(records[p]["caption"]))
        if not qt:
            continue
        n_q += 1
        need = {t[0] for t in qt} | {t[2] for t in qt}
        negs = [p2 for p2 in te if p2 != p and need <= nouns[p2] and not (qt & tris[p2])]
        if len(negs) < CFG.rel_neg_min:
            continue
        gal.append((p, np.array([p] + _sample_negs(negs, rng), np.int64)))
    print(f"Bộ quan hệ khó (Flickr8k): {len(gal)}/{n_q} truy vấn test có bộ ba đủ ≥{CFG.rel_neg_min} âm "
          f"| trung bình {np.mean([len(g) - 1 for _, g in gal]):.1f} âm/truy vấn")
    return gal


def build_rel_galleries_ag(recs, split, seed=0):
    """Truy vấn test nhắc tới một quan hệ người–vật 'v o' có trong annotation của video đúng.
    Âm = video test có vật o nhưng KHÔNG có quan hệ 'v o' (theo annotation)."""
    pos = {r["image_id"]: i for i, r in enumerate(recs)}
    te = [pos[i] for i in split["test"]]
    objs = {p: set(recs[p]["objects"]) for p in te}
    rng = np.random.RandomState(seed)
    gal, n_q = [], 0
    for p in te:
        toks = [t for t in _norm(recs[p]["caption"]).split() if t not in STOP and len(t) > 1]
        verbs = set()
        for t in toks:
            v = _ag_lemma(t, "v"); verbs.add(AG_VERB_ALIAS.get(v, v))
        nouns = {AG_NOUN_ALIAS.get(_ag_lemma(t), _ag_lemma(t)) for t in toks}
        comps = {c for c in objs[p] if " " in c and c.split(" ", 1)[0] in verbs and c.split(" ", 1)[1] in nouns}
        if not comps:
            continue
        n_q += 1
        need = {c.split(" ", 1)[1] for c in comps}
        negs = [p2 for p2 in te if p2 != p and need <= objs[p2] and not (comps & objs[p2])]
        if len(negs) < CFG.rel_neg_min:
            continue
        gal.append((p, np.array([p] + _sample_negs(negs, rng), np.int64)))
    print(f"Bộ quan hệ khó (AG): {len(gal)}/{n_q} truy vấn test có quan hệ người–vật đủ ≥{CFG.rel_neg_min} âm "
          f"| trung bình {np.mean([len(g) - 1 for _, g in gal]):.1f} âm/truy vấn")
    return gal


@torch.no_grad()
def rel_probe_rr(model, eng, gal, alpha, lam):
    model.eval()
    h = model.encode_graph(eng.graph.edge_index.to(DEVICE))
    z = znorm_per_row
    rr, rr_clip, groups = [], [], []
    for q, cand in gal:
        cand = np.asarray(cand, np.int64)
        qf = eng.txt_feat[[int(q)]]
        s_clip = (qf @ eng.img_feat[torch.from_numpy(cand)].T).numpy()
        pm = torch.tensor(eng.path_feats(cand[None], np.array([q]))).to(DEVICE)
        s_gnn = model.gnn_score(h, qf.to(DEVICE), torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
        s_path = model.path_score(pm).cpu().numpy()
        fin = combine(z(s_clip), z(s_gnn), z(s_path), alpha, lam)[0]
        rr.append(1.0 / (1 + int((fin > fin[0]).sum())))
        rr_clip.append(1.0 / (1 + int((s_clip[0] > s_clip[0, 0]).sum())))
        groups.append(eng.queries[int(q)]["group"])
    return np.array(rr), np.array(rr_clip), np.array(groups)


def relprobe_hook(ab, seed, model, eng, parts, alpha, lam):
    """Gọi trong run_experiment sau khi đánh giá test (α, λ đã chọn trên VAL — không chọn lại)."""
    if REL_GAL_KEY is None or REL_GAL_KEY not in REL_GALLERIES:
        return
    rr, rr_clip, groups = rel_probe_rr(model, eng, REL_GALLERIES[REL_GAL_KEY], alpha, lam)
    name = REL_RENAME.get(ab, ab)
    store = REL_PQ[REL_GAL_KEY]
    store[(name, seed)] = dict(rr=rr, rr_clip=rr_clip, groups=groups, lam=lam)
    store.setdefault(("CLIP", seed), dict(rr=rr_clip, rr_clip=rr_clip, groups=groups, lam=None))
    print(f"    [quan hệ khó] {name} seed {seed}: MRR {rr.mean():.4f} (CLIP {rr_clip.mean():.4f}) "
          f"| R@1 {(rr == 1).mean():.4f} | n={len(rr)}")


def rel_probe_report(key, base="none", others=None, out_csv=None):
    pq = REL_PQ[key]
    names = list(dict.fromkeys(a for a, _ in pq))
    rows = []
    for a in names:
        runs = [v["rr"] for (b, s), v in pq.items() if b == a]
        rows.append(dict(model=a, n_queries=len(runs[0]), seeds=len(runs),
                         MRR=f"{np.mean([r.mean() for r in runs]):.4f} ± {np.std([r.mean() for r in runs]):.4f}",
                         R1=f"{np.mean([(r == 1).mean() for r in runs]):.4f}"))
    tab = pd.DataFrame(rows)
    others = others or [a for a in names if a != base]
    sig_ = pd.concat([paired_test(pq, base, b) for b in others if any(k[0] == b for k in pq)])
    sig_ = sig_[sig_.group == "all"] if len(sig_) else sig_
    if out_csv is not None:
        tab.to_csv(out_csv, index=False)
        sig_.to_csv(str(out_csv).replace(".csv", "_paired.csv"), index=False)
    return tab, sig_


Dùng lại cache CLIP từ input: clip_flickr8k_ViT-L-14_laion2b_s32b_b82k_8000.npz


In [26]:
# HF_TOKEN (Kaggle Secrets) — không bắt buộc; thiếu secret thì vẫn tải ẩn danh.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception as e:
    print("Không đọc được HF_TOKEN từ Kaggle Secrets → tải ẩn danh:", type(e).__name__)


In [27]:
# v6: Flickr8k dùng backbone YẾU hơn (ViT-B/32) — phân tích độ nhạy theo backbone. AG khôi phục ViT-L/14 ở Phần 3.
CFG.clip_model, CFG.clip_weights, CFG.clip_bs = "ViT-B-32", "laion2b_s34b_b79k", 256
for _a in ("_m", "_t"):
    clip_text_features.__dict__.pop(_a, None)
import pandas as pd
records, hf_parts = load_flickr8k()
dataset_name = "flickr8k"
split = make_split(records)
img_feat, txt_feat = encode_flickr8k(records, hf_parts)
annot_info = apply_flickr_annotation(records, split, img_feat, CFG.flickr_annot)
print(f"\nDữ liệu: {dataset_name} | {len(records)} ảnh | đặc trưng CLIP {img_feat.shape}")

TRIPLE_INFO = fit_image_triples(records, split, img_feat) if CFG.use_triples else {}
if CFG.flickr_annot in ("tagger", "probe"):
    print("\nChọn (chuẩn hoá, k) của tagger trên VALIDATION:")
    print(pd.DataFrame(annot_info["grid"]).pivot_table(index="k", columns="norm",
                                                         values="val_F1").round(3).to_string())
    for r in records[-3:]:                       # vài ảnh TEST: nhãn tagger vs danh từ trong caption
        print(f"  • {r['caption'][:70]}\n      nhãn   : {r['objects']}\n      caption: {r['cap_objects']}"
              f"\n      bộ ba  : {[' '.join(t) + f' ({p:.2f})' for t, p in r.get('img_triples', [])[:4]]}")
with open(CFG.out_dir / "results" / "annotation_info.json", "w") as f:
    json.dump({k: v for k, v in annot_info.items()}, f, indent=2)
with open(CFG.out_dir / "results" / "triple_info.json", "w") as f:
    json.dump({k: v for k, v in TRIPLE_INFO.items()}, f, indent=2, ensure_ascii=False)

README.md:   0%|          | 0.00/687 [00:00<?, ?B/s]

data/train-00000-of-00002-2f8f6bfa852eac(…): reconstructing file:   0%|          |  0.00B /  414MB            

data/train-00000-of-00002-2f8f6bfa852eac(…): downloading bytes:           |  0.00B            

data/train-00001-of-00002-2173151d8cd6c7(…): reconstructing file:   0%|          |  0.00B /  414MB            

data/train-00001-of-00002-2173151d8cd6c7(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-7025a2b59(…): reconstructing file:   0%|          |  0.00B /  138MB            

... (6 outputs left out)

Flickr8k: 8000 ảnh | 6.1 danh từ/ảnh | 4.8 quan hệ/ảnh (rút từ caption)
  • A black dog is running after a white dog in the snow . → ['body', 'dog', 'ground', 'snow', 'water'] [('body', 'of', 'water'), ('dog', 'chase across', 'ground'), ('dog', 'play in', 'snow')]
  • A little baby plays croquet . → ['ball', 'boy', 'car', 'child', 'croquet', 'croquette', 'girl', 'hammer'] [('ball', 'beside', 'car'), ('boy', 'playing with', 'croquet'), ('car', 'with', 'ball')]
  • A brown dog in the snow has something hot pink in its mouth . → ['dog', 'hat', 'mouth', 'shirt', 'snow'] [('dog', 'carrying in', 'mouth'), ('dog', 'holding', 'shirt'), ('dog', 'in holding', 'hat')]
Split chính thức HF (sha1=005202c69674): train=6000 val=1000 test=1000


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

Nhãn ảnh = probe | vocab=685 | k=5 (chọn trên val) | TEST P=0.515 R=0.449 F1=0.479

Dữ liệu: flickr8k | 8000 ảnh | đặc trưng CLIP (8000, 512)
Bộ ba cấp ảnh: vocab=523 | k=5, τ=0.1 (chọn trên val) | TEST P=0.206 R=0.167 F1=0.184 (zero-shot F1=0.100) | 0.73 bộ ba/ảnh | truy vấn test có bộ ba trong vocab: 24.7%

Chọn (chuẩn hoá, k) của tagger trên VALIDATION:
norm  probe
k          
3     0.445
5     0.479
8     0.460
10    0.432
12    0.406
  • A basketball player performing a lay-up .
      nhãn   : ['ball', 'basketball', 'man', 'men', 'player']
      caption: ['ball', 'basketball', 'block', 'boy', 'jersey', 'man', 'player', 'team']
      bộ ba  : []
  • A man who has a gray beard and gray hair laughs while wearing a purple
      nhãn   : ['beard', 'face', 'hair', 'man', 'shirt']
      caption: ['beard', 'crowd', 'hair', 'man', 'people', 'shirt']
      bộ ba  : ['man with hair (0.31)', 'man in shirt (0.24)', 'man with beard (0.13)', 'man wear shirt (0.12)']
  • A big dog stands on his h

In [28]:
self_test(records, split, img_feat, txt_feat)

WordNet(+fallback): 228/239 lớp đối tượng có khái niệm cha
WordNet(+fallback): 240/255 lớp đối tượng có khái niệm cha
WordNet(+fallback): 228/239 lớp đối tượng có khái niệm cha
Split chính thức HF (sha1=005202c69674): train=6000 val=1000 test=1000
WordNet(+fallback): 228/239 lớp đối tượng có khái niệm cha

  ✓  Cạnh scene + đồng xuất hiện chỉ xây từ train
  ✓  Điểm đường đi khớp giải thích liệt kê (295/295 cặp)
  ✓  Split tái lập được từ seed
  ✓  Bộ ba cấp ảnh: nhãn huấn luyện chỉ từ ảnh TRAIN, ảnh train dùng dự đoán ngoài-fold
  ✓  Node bộ ba có trong đồ thị (207 node, 5787 cạnh ảnh–bộ ba)


True

In [29]:
# Chọn CFG.learn_clip_w trên VALIDATION (1 seed, cấu hình 'none'). Không nhìn test.
def pick_learn_clip_w(records, split, img_feat, txt_feat, seed=0):
    rows = []
    for flag in (False, True):
        CFG.learn_clip_w = flag
        _GROUP_CACHE.clear()
        g, eng, parts = prepare(records, split, img_feat, txt_feat, "none", seed)
        model, _ = train_graph_scorer(eng, parts["train"]["q_idx"], parts["train"]["img_idx"],
                                      seed=seed, verbose=False)
        (a, l), grid = tune_on_val(model, eng, parts["val"], parts["val"]["img_idx"], verbose=False)
        rows.append(dict(learn_clip_w=flag, val_MRR=max(r["MRR"] for r in grid), alpha=a, lam=l,
                         clip_w=round(F.softplus(model.clip_logit).item(), 3),
                         path_w=round(F.softplus(model.path_logit).item(), 3)))
    df_ = pd.DataFrame(rows); display(df_)
    CFG.learn_clip_w = bool(df_.loc[df_.val_MRR.idxmax(), "learn_clip_w"])
    print("→ CFG.learn_clip_w =", CFG.learn_clip_w, "(chọn theo VAL MRR)")
    return df_

clipw_flickr = pick_learn_clip_w(records, split, img_feat, txt_feat)

WordNet(+fallback): 228/239 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 7272, 'indirect': 518, 'other': 210}
WordNet(+fallback): 228/239 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 7272, 'indirect': 518, 'other': 210}


,learn_clip_w,val_MRR,alpha,lam,clip_w,path_w
0,False,0.736586,0.7,0.75,1.000,1.0
1,True,0.733891,0.7,0.75,0.473,1.0


→ CFG.learn_clip_w = False (chọn theo VAL MRR)


In [30]:
# Thí nghiệm chính: 8 cấu hình ablation × 3 seed (6 cũ + no_triples + shuffled_triples), nhãn ảnh = CFG.flickr_annot
REL_GALLERIES["flickr"] = build_rel_galleries_flickr(records, split)
REL_GAL_KEY, REL_RENAME = "flickr", {}
all_res, art = run_experiment(records, split, img_feat, txt_feat, ablations=FLICKR_ABLATIONS)
REL_GAL_KEY = None
df = results_table(all_res)
df.to_csv(CFG.out_dir / "results" / f"main_results_{CFG.flickr_annot}.csv", index=False)
df.to_csv(CFG.out_dir / "results" / "main_results.csv", index=False)
display(df)

Bộ quan hệ khó (Flickr8k): 246/764 truy vấn test có bộ ba đủ ≥3 âm | trung bình 13.5 âm/truy vấn

ABLATION = none
WordNet(+fallback): 228/239 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 7272, 'indirect': 518, 'other': 210}
Đồ thị: {'nodes': 8655, 'images': 8000, 'object_classes': 239, 'concepts': 209, 'edges': 41797, 'contains': 39500, 'scene': 1113, 'related': 723, 'isa': 461, 'triple_nodes': 207, 'triple_edges': 5787, 'paths': {'P0_contains': 39500, 'P1_scene': 206047, 'P2_isa': 66139, 'P3_related': 196870, 'P4_scene_isa': 375717, 'P5_related_isa': 317603, 'P6_triple': 5787}, 'ablation': 'none'}
  cạnh dùng khi huấn luyện: 63860/83594 (đã bỏ 19734 cạnh chạm ảnh val/test)
  epoch  0 | loss 32.2945 | w_path 1.000 0.286 0.241 0.298 0.173 0.203 0.249 | gate L0[scene,rel,isa] 0.19 0.20 0.17 | clip_w 1.00
  epoch  3 | loss 26.0222 | w_path 1.000 0.344 0.262 0.479 0.280 0.411 0.173 | gate L0[scene,rel,isa] 0.20 0.21 0.16 | clip_w 1.00
  epoch  6 | loss 22.3192 | w_path 1.000 0.

,ablation,model,R@1,R@5,R@10,MRR,nDCG@10
0,none,CLIP,0.6120 ± 0.0000,0.8610 ± 0.0000,0.9290 ± 0.0000,0.7227 ± 0.0000,0.7697 ± 0.0000
1,none,CLIP+Graph,0.6153 ± 0.0049,0.8687 ± 0.0048,0.9337 ± 0.0050,0.7268 ± 0.0015,0.7742 ± 0.0017
2,none,CLIP [direct] (n=923),0.6241 ± 0.0000,0.8646 ± 0.0000,0.9285 ± 0.0000,0.7315 ± 0.0000,0.7763 ± 0.0000
3,none,CLIP+Graph [direct] (n=923),0.6277 ± 0.0036,0.8740 ± 0.0044,0.9332 ± 0.0050,0.7358 ± 0.0012,0.7810 ± 0.0016
4,none,CLIP [indirect] (n=52),0.4808 ± 0.0000,0.8462 ± 0.0000,0.9423 ± 0.0000,0.6345 ± 0.0000,0.7066 ± 0.0000
5,none,CLIP+Graph [indirect] (n=52),0.4872 ± 0.0240,0.8462 ± 0.0000,0.9487 ± 0.0091,0.6410 ± 0.0074,0.7136 ± 0.0048
6,none,CLIP+Graph (adaptive),0.6043 ± 0.0012,0.8637 ± 0.0037,0.9313 ± 0.0060,0.7189 ± 0.0011,0.7675 ± 0.0023
7,none,CLIP+Graph (adaptive) [direct] (n=923),0.6157 ± 0.0020,0.8682 ± 0.0042,0.9310 ± 0.0064,0.7277 ± 0.0006,0.7742 ± 0.0019
8,none,CLIP+Graph (adaptive) [indirect] (n=52),0.4808 ± 0.0157,0.8397 ± 0.0091,0.9423 ± 0.0157,0.6300 ± 0.0106,0.7028 ± 0.0121
9,random_edges,CLIP,0.6120 ± 0.0000,0.8610 ± 0.0000,0.9290 ± 0.0000,0.7227 ± 0.0000,0.7697 ± 0.0000


In [31]:
# =====================================================================================
# ABLATION THEO GIẢI THÍCH: giải thích có đúng với tri thức ĐỘC LẬP với đồ thị không?
#
# Với mỗi ablation (seed đầu), giải thích của ẢNH ĐÚNG trên mọi truy vấn test = top-m đường đi
# QUAN HỆ (P1..P5, bỏ P0) theo đóng góp biên — đúng định nghĩa explain_and_score. Mỗi CẠNH
# trong đường đi được chấm bằng nguồn KHÔNG dùng để dựng đồ thị:
#   • scene / related  : NPMI của hai đầu cạnh trên tập nhãn tham chiếu của ảnh VAL+TEST
#                        (Flickr: danh từ trong caption 1..4 — caption 0 là truy vấn nên bỏ;
#                         AG: annotation của video). Đồ thị chỉ thống kê trên TRAIN ⇒ độc lập.
#                        Cạnh "hợp lý" ⇔ NPMI > 0 (hai thứ hay đi cùng nhau hơn ngẫu nhiên).
#   • isa              : đúng ⇔ đích nằm trong bao đóng hypernym WordNet (hoặc FALLBACK_ISA).
# Chỉ số:
#   edge_ok    % cạnh hợp lý            path_ok  % đường đi có MỌI cạnh hợp lý
#   grounded   % đường đi mà mọi node ĐỐI TƯỢNG trung gian có trong nhãn tham chiếu của ảnh đó
#   rand_path_ok  path_ok khi chọn NGẪU NHIÊN cùng số đường đi quan hệ của chính ảnh đó
#                 ⇒ (path_ok − rand_path_ok) = cơ chế chọn giải thích có ưu tiên đường đi đúng không
# Không rò rỉ: nhãn tham chiếu chỉ dùng để CHẤM, không huấn luyện / chọn siêu tham số.
# Chạy NGAY sau cell thí nghiệm chính (cần ART_BY_ABL của cell 28 bản đợt 4).
# =====================================================================================
from functools import lru_cache


def reference_sets(records, ids):
    key = "cap_objects" if "cap_objects" in records[0] else "objects"
    pos = {r["image_id"]: i for i, r in enumerate(records)}
    return key, {i: set(records[pos[i]][key]) for i in ids}


def make_npmi(docs):
    n = len(docs)
    c1 = Counter(x for d in docs for x in d)
    c2 = Counter(p for d in docs for p in itertools.combinations(sorted(d), 2))
    def npmi(a, b):
        ab = c2.get(tuple(sorted((a, b))), 0)
        if ab == 0 or c1[a] == 0 or c1[b] == 0:
            return -1.0
        p_ab = ab / n
        return math.log(p_ab / ((c1[a] / n) * (c1[b] / n))) / max(-math.log(p_ab), 1e-9)
    return npmi


@lru_cache(maxsize=None)
def _isa_ok(a, b):
    if b in FALLBACK_ISA.get(a, []):
        return True
    wn = _wordnet()
    if wn is None:
        return False
    for s in wn.synsets(a.replace(" ", "_"), pos="n")[:3]:
        for h in s.closure(lambda x: x.hypernyms()):
            if b in {l.name().replace("_", " ").lower() for l in h.lemmas()}:
                return True
    return False


_EDGE_KIND = {"P1_scene": ["scene"], "P2_isa": ["isa"], "P3_related": ["related"],
              "P4_scene_isa": ["scene", "isa"], "P5_related_isa": ["related", "isa"]}


def _edges(p):
    nodes = p["nodes"]                      # [o, c] hoặc [o, o2, c]
    return list(zip(_EDGE_KIND[p["pattern"]], nodes[:-1], nodes[1:]))


def _rel_contrib(paths, model):
    """Đóng góp biên của từng đường đi = đúng công thức explain_and_score, nhưng O(P)."""
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    pidx = np.array([PIDX[p["pattern"]] for p in paths])
    pw = np.array([p["w"] for p in paths])
    tot = np.bincount(pidx, weights=pw, minlength=len(PATTERNS))
    return gain * w[pidx] * (np.log1p(tot[pidx]) - np.log1p(np.maximum(tot[pidx] - pw, 0)))


def explanation_grounding(art_by_abl, records, split, top_m=CFG.top_m_paths, n_rand=10, seed=0):
    key, ref = reference_sets(records, split["val"] + split["test"])
    npmi = make_npmi(list(ref.values()))
    rng = np.random.RandomState(seed)
    rows = []
    for ab, art in art_by_abl.items():
        g, eng, model, part = art["graph"], art["engine"], art["model"], art["parts"]["test"]
        name = lambda n: g.id2name.get(n, "")
        is_obj = lambda n: g.n_img <= n < g.n_img + g.n_obj
        edge_cache = {}

        def edge_ok(kind, a, b):
            k = (kind, a, b)
            if k not in edge_cache:
                edge_cache[k] = (_isa_ok(name(a), name(b)) if kind == "isa"
                                 else npmi(name(a), name(b)) > 0)
            return edge_cache[k]

        def path_ok(p):
            return all(edge_ok(*e) for e in _edges(p))

        for i, qi in enumerate(part["q_idx"]):
            cids = [n for n, _, _ in eng.q_concepts[qi]]
            if not cids:
                continue
            gold = int(part["gold_pos"][i])
            rel = [p for p in enumerate_paths(g, gold, cids) if p["pattern"] in _EDGE_KIND]   # P1..P5
            if not rel:
                continue
            top = [rel[j] for j in np.argsort(-_rel_contrib(rel, model))[:top_m]]
            truth = ref.get(records[gold]["image_id"], set())
            k = len(top)
            rand_ok = np.mean([np.mean([path_ok(rel[j]) for j in rng.choice(len(rel), k, replace=False)])
                               for _ in range(n_rand)])
            edges = [e for p in top for e in _edges(p)]
            rows.append(dict(
                ablation=ab, group=part["groups"][i], n_rel_paths=len(rel), n_expl=k,
                edge_ok=np.mean([edge_ok(*e) for e in edges]),
                path_ok=np.mean([path_ok(p) for p in top]),
                rand_path_ok=rand_ok,
                grounded=np.mean([all(name(n) in truth for n in p["nodes"][:-1] if is_obj(n))
                                  for p in top]),
                mean_npmi=np.mean([npmi(name(a), name(b)) for kd, a, b in edges if kd != "isa"] or [np.nan])))
    df = pd.DataFrame(rows)
    print(f"Nhãn tham chiếu: '{key}' của {len(ref)} ảnh VAL+TEST | top_m={top_m}")
    return df


def _boot_ci(x, n_boot=5000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    b = x[np.random.RandomState(seed).randint(0, len(x), (n_boot, len(x)))].mean(1)
    return f"{x.mean():.3f} [{np.percentile(b, 2.5):.3f}, {np.percentile(b, 97.5):.3f}]"


def grounding_table(df):
    out = []
    for grp in ("all", "direct", "indirect"):
        for ab in df.ablation.unique():
            d = df[(df.ablation == ab) & ((df.group == grp) | (grp == "all"))]
            if len(d) < 5:
                continue
            out.append(dict(ablation=ab, group=grp, n_queries=len(d),
                            rel_paths_per_q=round(d.n_rel_paths.mean(), 1),
                            edge_ok=_boot_ci(d.edge_ok), path_ok=_boot_ci(d.path_ok),
                            rand_path_ok=_boot_ci(d.rand_path_ok),
                            selection_lift=_boot_ci(d.path_ok - d.rand_path_ok),
                            grounded=_boot_ci(d.grounded), mean_npmi=round(d.mean_npmi.mean(), 3)))
    return pd.DataFrame(out)


# ------------------------------ Flickr8k ------------------------------
ART_FLICKR = dict(ART_BY_ABL)                     # giữ lại trước khi cell oracle / AG ghi đè
gdf = explanation_grounding(ART_FLICKR, records, split)
gtab = grounding_table(gdf)
gdf.to_csv(CFG.out_dir / "results" / "explanation_grounding_per_query.csv", index=False)
gtab.to_csv(CFG.out_dir / "results" / "explanation_grounding.csv", index=False)
display(gtab)

Nhãn tham chiếu: 'cap_objects' của 2000 ảnh VAL+TEST | top_m=5


,ablation,group,n_queries,rel_paths_per_q,edge_ok,path_ok,rand_path_ok,selection_lift,grounded,mean_npmi
0,none,all,431,4.1,"0.897 [0.874, 0.919]","0.889 [0.865, 0.912]","0.883 [0.859, 0.905]","0.006 [0.003, 0.010]","0.489 [0.452, 0.525]",0.181
1,random_edges,all,272,1.3,"0.285 [0.240, 0.329]","0.204 [0.159, 0.248]","0.204 [0.159, 0.248]","-0.000 [-0.000, -0.000]","0.260 [0.213, 0.308]",-0.326
2,no_triples,all,431,4.1,"0.897 [0.874, 0.919]","0.889 [0.865, 0.912]","0.884 [0.860, 0.906]","0.005 [0.002, 0.009]","0.488 [0.452, 0.525]",0.181
3,shuffled_triples,all,431,4.1,"0.897 [0.874, 0.919]","0.889 [0.865, 0.912]","0.882 [0.858, 0.905]","0.007 [0.004, 0.010]","0.488 [0.452, 0.525]",0.181
4,none,direct,379,4.0,"0.893 [0.868, 0.917]","0.884 [0.858, 0.908]","0.877 [0.851, 0.902]","0.006 [0.003, 0.010]","0.502 [0.462, 0.541]",0.176
5,random_edges,direct,239,1.3,"0.292 [0.245, 0.342]","0.213 [0.165, 0.264]","0.213 [0.165, 0.264]","-0.000 [-0.000, -0.000]","0.258 [0.209, 0.309]",-0.319
6,no_triples,direct,379,4.0,"0.893 [0.868, 0.917]","0.884 [0.858, 0.908]","0.878 [0.852, 0.903]","0.006 [0.002, 0.009]","0.502 [0.462, 0.540]",0.176
7,shuffled_triples,direct,379,4.0,"0.893 [0.868, 0.917]","0.884 [0.858, 0.908]","0.876 [0.850, 0.901]","0.007 [0.004, 0.011]","0.502 [0.462, 0.540]",0.176
8,none,indirect,52,4.8,"0.929 [0.865, 0.978]","0.928 [0.864, 0.978]","0.923 [0.860, 0.972]","0.005 [-0.003, 0.013]","0.392 [0.298, 0.488]",0.217
9,random_edges,indirect,28,1.6,"0.256 [0.143, 0.378]","0.158 [0.054, 0.286]","0.158 [0.054, 0.286]","-0.000 [-0.000, 0.000]","0.286 [0.155, 0.435]",-0.350


In [32]:
# Kiểm định CẶP giữa các ablation trên cùng truy vấn test (trung bình 3 seed / truy vấn).
# Chênh MRR 0.000x với std 0.002-0.003 thì KHÔNG kết luận được gì — cần CI cho hiệu số.
# Chạy NGAY sau cell thí nghiệm chính (cell oracle / AG sẽ ghi đè PER_QUERY).
def paired_test(pq, a="none", b="no_relations", n_boot=10000, seed=0):
    seeds = sorted({s for ab, s in pq if ab == a} & {s for ab, s in pq if ab == b})
    ra = np.mean([pq[(a, s)]["rr"] for s in seeds], 0)
    rb = np.mean([pq[(b, s)]["rr"] for s in seeds], 0)
    groups = pq[(a, seeds[0])]["groups"]
    rng = np.random.RandomState(seed)
    rows = []
    for grp in ("all", "direct", "indirect"):
        m = np.ones(len(ra), bool) if grp == "all" else groups == grp
        d = (ra - rb)[m]
        if len(d) < 5:
            continue
        boot = d[rng.randint(0, len(d), (n_boot, len(d)))].mean(1)
        lo, hi = np.percentile(boot, [2.5, 97.5])
        rows.append(dict(group=grp, n=int(m.sum()), MRR_a=ra[m].mean(), MRR_b=rb[m].mean(),
                         diff=d.mean(), ci_lo=lo, ci_hi=hi,
                         p_a_not_better=float((boot <= 0).mean()),
                         a_wins=int((d > 0).sum()), b_wins=int((d < 0).sum())))
    return pd.DataFrame(rows).assign(a=a, b=b)


PQ_MAIN = dict(PER_QUERY)            # giữ lại trước khi cell sau ghi đè
sig = pd.concat([paired_test(PQ_MAIN, "none", b)
                 for b in ("no_triples", "shuffled_triples", "no_relations", "random_edges",
                           "no_scene", "no_related", "no_concept")
                 if any(k[0] == b for k in PQ_MAIN)])
sig.to_csv(CFG.out_dir / "results" / "paired_ablation_test.csv", index=False)
print("λ chọn trên val cho 'none':", [v["lam"] for k, v in PQ_MAIN.items() if k[0] == "none"],
      "(λ=1.0 ⇒ s_path KHÔNG tham gia điểm cuối)")
display(sig.round(4))

λ chọn trên val cho 'none': [0.75, 0.75, 0.75] (λ=1.0 ⇒ s_path KHÔNG tham gia điểm cuối)


,group,n,MRR_a,MRR_b,diff,ci_lo,ci_hi,p_a_not_better,a_wins,b_wins,a,b
0,all,1000,0.7268,0.7280,-0.0012,-0.0052,0.0028,0.7188,155,164,none,no_triples
1,direct,923,0.7358,0.7371,-0.0013,-0.0055,0.0030,0.7169,138,149,none,no_triples
2,indirect,52,0.6410,0.6442,-0.0032,-0.0202,0.0160,0.6479,9,11,none,no_triples
0,all,1000,0.7268,0.7270,-0.0003,-0.0043,0.0037,0.5454,150,159,none,shuffled_triples
1,direct,923,0.7358,0.7363,-0.0005,-0.0046,0.0036,0.5958,136,142,none,shuffled_triples
2,indirect,52,0.6410,0.6440,-0.0029,-0.0242,0.0182,0.6106,9,11,none,shuffled_triples
0,all,1000,0.7268,0.7316,-0.0048,-0.0108,0.0012,0.9426,165,190,none,no_relations
1,direct,923,0.7358,0.7412,-0.0054,-0.0116,0.0009,0.9520,147,172,none,no_relations
2,indirect,52,0.6410,0.6310,0.0100,-0.0079,0.0303,0.1490,11,10,none,no_relations
0,all,1000,0.7268,0.7288,-0.0021,-0.0079,0.0039,0.7545,170,196,none,random_edges


In [33]:
# v5 — BỘ QUAN HỆ KHÓ trên Flickr8k: gallery = ảnh đúng + ảnh test cùng đối tượng, khác quan hệ.
# Đây là chỗ quan hệ CẤP ẢNH phải tạo khác biệt: nhãn đối tượng không còn phân biệt được ứng viên.
rel_tab, rel_sig = rel_probe_report("flickr", base="none",
                                    others=["CLIP", "no_triples", "shuffled_triples", "no_relations", "random_edges"],
                                    out_csv=CFG.out_dir / "results" / "relational_hard_negatives.csv")
display(rel_tab); display(rel_sig.round(4))
print("diff > 0 và ci_lo > 0 ⇒ 'none' (có bộ ba cấp ảnh) tốt hơn cấu hình so sánh một cách có ý nghĩa.")


,model,n_queries,seeds,MRR,R1
0,none,246,3,0.8179 ± 0.0102,0.7073
1,CLIP,246,3,0.8195 ± 0.0000,0.7195
2,random_edges,246,3,0.8261 ± 0.0037,0.7276
3,no_relations,246,3,0.8257 ± 0.0039,0.7182
4,no_triples,246,3,0.8267 ± 0.0050,0.7263
5,shuffled_triples,246,3,0.8288 ± 0.0102,0.7276


,group,n,MRR_a,MRR_b,diff,ci_lo,ci_hi,p_a_not_better,a_wins,b_wins,a,b
0,all,246,0.8179,0.8195,-0.0016,-0.0135,0.0102,0.5992,30,26,none,CLIP
0,all,246,0.8179,0.8267,-0.0089,-0.0192,0.0011,0.9598,32,37,none,no_triples
0,all,246,0.8179,0.8288,-0.0109,-0.0206,-0.0020,0.9914,24,30,none,shuffled_triples
0,all,246,0.8179,0.8257,-0.0078,-0.0225,0.0066,0.8518,32,38,none,no_relations
0,all,246,0.8179,0.8261,-0.0082,-0.0238,0.0067,0.8541,44,42,none,random_edges


diff > 0 và ci_lo > 0 ⇒ 'none' (có bộ ba cấp ảnh) tốt hơn cấu hình so sánh một cách có ý nghĩa.


In [34]:
# Mô hình cuối của Flickr8k chọn trên VAL + kiểm định cặp adaptive vs global
FUSION_FLICKR = list(FUSION_LOG)
sel_f, final_f = final_model_table(all_res, FUSION_FLICKR)
display(sel_f.round(4)); display(final_f)
display(pd.DataFrame(FUSION_FLICKR).round(3))
sig_ad = paired_test(dict(PER_QUERY), "none+adaptive", "none")
display(sig_ad.round(4))
for name, obj in (("final_model_val_selection.csv", sel_f), ("final_model_test.csv", final_f),
                  ("fusion_log.csv", pd.DataFrame(FUSION_FLICKR)), ("paired_adaptive_vs_global.csv", sig_ad)):
    obj.to_csv(CFG.out_dir / "results" / name, index=False)

→ Mô hình cuối (chọn trên VAL): đồ thị='none', kết hợp=global (VAL MRR 0.7356)


,ablation,val_MRR,mode
0,none,0.7356,global
1,none,0.7311,adaptive


,ablation,model,R@1,R@5,R@10,MRR,nDCG@10
0,none,CLIP,0.6120 ± 0.0000,0.8610 ± 0.0000,0.9290 ± 0.0000,0.7227 ± 0.0000,0.7697 ± 0.0000
1,none,CLIP+Graph,0.6153 ± 0.0049,0.8687 ± 0.0048,0.9337 ± 0.0050,0.7268 ± 0.0015,0.7742 ± 0.0017
2,none,CLIP [direct] (n=923),0.6241 ± 0.0000,0.8646 ± 0.0000,0.9285 ± 0.0000,0.7315 ± 0.0000,0.7763 ± 0.0000
3,none,CLIP+Graph [direct] (n=923),0.6277 ± 0.0036,0.8740 ± 0.0044,0.9332 ± 0.0050,0.7358 ± 0.0012,0.7810 ± 0.0016
4,none,CLIP [indirect] (n=52),0.4808 ± 0.0000,0.8462 ± 0.0000,0.9423 ± 0.0000,0.6345 ± 0.0000,0.7066 ± 0.0000
5,none,CLIP+Graph [indirect] (n=52),0.4872 ± 0.0240,0.8462 ± 0.0000,0.9487 ± 0.0091,0.6410 ± 0.0074,0.7136 ± 0.0048


,ablation,seed,alpha,lam,cv_global,cv_adaptive,alpha_q_with_concept,alpha_q_no_concept,w_alpha
0,none,0,0.7,0.75,0.736,0.731,0.648,1.0,"{'has_concept': -0.8420000076293945, 'log_n_co..."
1,none,1,0.8,0.75,0.740,0.735,0.700,1.0,"{'has_concept': -0.9110000133514404, 'log_n_co..."
2,none,2,0.9,0.75,0.730,0.728,0.770,1.0,"{'has_concept': -0.8880000114440918, 'log_n_co..."
3,random_edges,0,0.7,0.75,0.736,0.736,0.664,1.0,"{'has_concept': -1.1089999675750732, 'log_n_co..."
4,random_edges,1,0.7,0.75,0.732,0.728,0.653,1.0,"{'has_concept': -0.7620000243186951, 'log_n_co..."
5,random_edges,2,0.7,0.75,0.739,0.738,0.664,1.0,"{'has_concept': -1.093000054359436, 'log_n_con..."
6,no_relations,0,0.8,0.75,0.733,0.732,0.733,1.0,"{'has_concept': -1.1859999895095825, 'log_n_co..."
7,no_relations,1,0.7,0.75,0.735,0.731,0.660,1.0,"{'has_concept': -1.1490000486373901, 'log_n_co..."
8,no_relations,2,0.7,0.75,0.740,0.739,0.658,1.0,"{'has_concept': -1.2829999923706055, 'log_n_co..."
9,no_triples,0,0.8,0.75,0.735,0.725,0.703,1.0,"{'has_concept': -0.9120000004768372, 'log_n_co..."


,group,n,MRR_a,MRR_b,diff,ci_lo,ci_hi,p_a_not_better,a_wins,b_wins,a,b
0,all,1000,0.7189,0.7268,-0.0078,-0.0136,-0.0023,0.9976,154,183,none+adaptive,none
1,direct,923,0.7277,0.7358,-0.0081,-0.0141,-0.0022,0.9974,138,169,none+adaptive,none
2,indirect,52,0.6300,0.6410,-0.0111,-0.0326,0.0090,0.8577,9,11,none+adaptive,none


In [35]:
# So sánh với nhãn ORACLE (rút từ caption 1..4 của chính ảnh — cách của bản trước).
# Chỉ để định lượng "rò rỉ" của cách cũ: chênh lệch oracle − tagger chính là phần điểm
# đến từ việc ứng viên mang sẵn mô tả văn bản của chính nó. KHÔNG báo cáo là kết quả chính.
RUN_ORACLE = False      # v6: đã chạy ở version 2 (ViT-L/14)
if RUN_ORACLE:
    apply_flickr_annotation(records, split, img_feat, "oracle")
    oracle_res, _ = run_experiment(records, split, img_feat, txt_feat,
                                   ablations=("none", "no_relations"))
    df_oracle = results_table(oracle_res)
    df_oracle.to_csv(CFG.out_dir / "results" / "main_results_oracle.csv", index=False)
    apply_flickr_annotation(records, split, img_feat, CFG.flickr_annot)   # KHÔI PHỤC nhãn chính
    cmp_annot = pd.concat([df[df.ablation.isin(["none", "no_relations"])].assign(annotation=CFG.flickr_annot),
                           df_oracle.assign(annotation="oracle")])
    cmp_annot = cmp_annot[cmp_annot.model.isin(["CLIP", "CLIP+Graph"])]
    cmp_annot.to_csv(CFG.out_dir / "results" / "annotation_comparison.csv", index=False)
    display(cmp_annot[["annotation", "ablation", "model", "R@1", "R@5", "MRR"]])

In [36]:
# Fidelity đo trên THỨ HẠNG của điểm cuối (CLIP + GNN + path)
rdf, rsumm = final_rank_fidelity(art)
rdf.to_csv(CFG.out_dir / "results" / "fidelity_final_rank.csv", index=False)
for k, v in rsumm.items():
    print(f"  {k:36s} {v:.3f}" if isinstance(v, float) else f"  {k:36s} {v}")
with open(CFG.out_dir / "results" / "xai_final_rank_summary.json", "w") as f:
    json.dump(rsumm, f, indent=2)

  n_queries                            50
  rank_full                            2.400
  rank_after_drop_explanation          3.480
  rank_after_drop_random               2.958
  rank_with_only_explanation           2.500
  frac_rank_worse_when_drop_expl       0.420
  frac_rank_worse_when_drop_rand       0.400


**Cách đọc bảng.**

1. `none / CLIP` vs `none / CLIP+Graph` — đồ thị có giúp không (Nhiệm vụ 1).
2. `none` vs `random_edges` / `no_relations` — cải thiện có đến từ **cấu trúc quan hệ** không.
   Chỉ kết luận "quan hệ có ích" khi `none` > `random_edges` **và** `none` > `no_relations`
   vượt quá độ lệch chuẩn giữa các seed.
3. `no_scene` / `no_concept` — loại cạnh nào (scene graph rút từ caption hay IsA WordNet) đóng góp.
4. Dòng `[indirect]` — nhóm duy nhất mà đường đi P1–P5 có thể giúp; P0 không tồn tại cho nhóm này.

Nếu `no_relations` vẫn ≥ `none`, kết luận trung thực là: lợi ích đến từ nhãn đối tượng (P0)
và GNN, còn quan hệ trong đồ thị hiện tại chưa giúp — kèm phân tích lý do ở mục case study.

In [37]:
# Độ nhạy theo α (yêu cầu ở cột Mở rộng của Nhiệm vụ 2)
import pandas as pd
gdf = pd.DataFrame(art["grid"])
piv = gdf.pivot_table(index="alpha", columns="lam", values="MRR")
print("MRR trên VALIDATION theo (α, λ) — α=1.0 là CLIP thuần:")
print(piv.round(4).to_string())
piv.to_csv(CFG.out_dir / "results" / "alpha_sensitivity.csv")

MRR trên VALIDATION theo (α, λ) — α=1.0 là CLIP thuần:
lam      0.00    0.25    0.50    0.75
alpha                                
0.0    0.2622  0.1647  0.1351  0.0835
0.2    0.4415  0.4797  0.4323  0.3005
0.4    0.5792  0.6359  0.6727  0.6296
0.5    0.6378  0.6806  0.7108  0.7091
0.6    0.6802  0.7046  0.7278  0.7322
0.7    0.7017  0.7163  0.7283  0.7362
0.8    0.7150  0.7228  0.7280  0.7304
0.9    0.7213  0.7240  0.7278  0.7309
1.0    0.7241  0.7241  0.7241  0.7241


In [38]:
# Phân tích 5 thành công + 5 thất bại
cases = analyse_cases(art, records)
for c in cases:
    print(f"\n[{c['kind']}] nhóm={c['group']} | hạng CLIP {c['rank_clip']} → CLIP+Graph {c['rank_graph']}")
    print(f"  truy vấn: {c['query']}")
    for e in c["explanation"]:
        print(f"    • {e['text']}   [{e['contribution']:+.4f}]")
    d = c["detail"]
    ctrl = d['score_full'] - d['score_without_random']
    print(f"  fidelity−: {d['score_full'] - d['score_without_explanation']:+.4f} | "
          f"đối chứng ngẫu nhiên: {'n/a (1 đường đi)' if ctrl != ctrl else f'{ctrl:+.4f}'} | "
          f"sparsity {d['sparsity']:.2f} | {d['n_paths']} đường đi")
with open(CFG.out_dir / "results" / "case_analysis.json", "w") as f:
    json.dump(cases, f, ensure_ascii=False, indent=2)


[THÀNH CÔNG] nhóm=direct | hạng CLIP 23 → CLIP+Graph 10
  truy vấn: A large dog sniffing a human hand while a small dog is sniffing the large dog 's butt .
    • ảnh –chứa→ dog   [+1.1120]
  fidelity−: +1.1120 | đối chứng ngẫu nhiên: n/a (1 đường đi) | sparsity 0.00 | 1 đường đi

[THÀNH CÔNG] nhóm=direct | hạng CLIP 13 → CLIP+Graph 3
  truy vấn: A basketball player preparing to shoot the ball
    • ảnh –chứa→ basketball   [+0.7484]
    • ảnh –chứa→ player   [+0.4277]
  fidelity−: +1.6364 | đối chứng ngẫu nhiên: +1.2664 | sparsity 0.33 | 3 đường đi

[THÀNH CÔNG] nhóm=direct | hạng CLIP 15 → CLIP+Graph 8
  truy vấn: Two girls are fighting over a ball during a game .
    • ảnh –chứa→ ball   [+1.4991]
  fidelity−: +1.4991 | đối chứng ngẫu nhiên: +1.2921 | sparsity 0.50 | 2 đường đi

[THÀNH CÔNG] nhóm=direct | hạng CLIP 11 → CLIP+Graph 4
  truy vấn: A snowboarder in bright green performing a jump at a competition .
    • ảnh –chứa→ snowboarder   [+2.1408]
  fidelity−: +2.1408 | đối chứng n

In [39]:
# Fidelity / validity / sparsity trên ≥10 truy vấn (cấp độ 3)
fdf, summ = fidelity_study(art)
fdf.to_csv(CFG.out_dir / "results" / "fidelity_per_query.csv", index=False)
print("Tổng hợp xAI:")
for k, v in summ.items():
    print(f"  {k:34s} {v:.4f}" if isinstance(v, float) else f"  {k:34s} {v}")
print("\nnecessity_ratio > 1 nghĩa là bỏ đường đi giải thích gây hại NHIỀU HƠN bỏ ngẫu nhiên")
print("cùng số lượng ⇒ giải thích thực sự là thứ mô hình dùng, không phải trang trí.")
with open(CFG.out_dir / "results" / "xai_summary.json", "w") as f:
    json.dump(summ, f, indent=2)

Tổng hợp xAI:
  n_queries                          20
  mean_paths                         8.5000
  fidelity_minus                     2.5856
  fidelity_minus_random_control      1.5030
  fidelity_plus_gap                  0.1435
  sparsity                           0.3496
  validity                           1.0000
  necessity_ratio                    1.7203

necessity_ratio > 1 nghĩa là bỏ đường đi giải thích gây hại NHIỀU HƠN bỏ ngẫu nhiên
cùng số lượng ⇒ giải thích thực sự là thứ mô hình dùng, không phải trang trí.


In [40]:
# Bộ ≥20 truy vấn đánh giá (cấp độ 2) — CHỈ lấy truy vấn của ảnh test
eval_qs = export_eval_queries(art["engine"].queries, art["graph"],
                              CFG.out_dir / "results" / "eval_queries_20.json", n=20,
                              allowed_ids=split["test"])

Đã ghi 20 truy vấn đánh giá → /kaggle/working/results/eval_queries_20.json


In [41]:
# Demo
demo_query("a child playing football on the grass", art, k=3)
demo_query("a person cooking food in a kitchen", art, k=3)


Truy vấn: 'a child playing football on the grass'
Khái niệm nhận ra: ['child', 'football', 'grass']

  #1  image_id=7863  score=1.8589
      caption: A black and white dog is catching a Frisbee in the yard .
      đối tượng: ['ball', 'boy', 'dog', 'grass', 'lawn']
      GIẢI THÍCH (5 đường đi, hiển thị 4):
        • ảnh –chứa→ grass   [đóng góp +1.3423]
        • ảnh –chứa→ boy –liên_quan→ child   [đóng góp +0.4941]
        • ảnh –chứa→ ball –liên_quan→ football   [đóng góp +0.1039]
        • ảnh –chứa→ boy –liên_quan→ toddler –là_một→ child   [đóng góp +0.0841]

  #2  image_id=7036  score=1.8364
      caption: A girl is standing on one leg in the middle of the field .
      đối tượng: ['boy', 'child', 'field', 'girl', 'grass']
      GIẢI THÍCH (3 đường đi, hiển thị 2):
        • ảnh –chứa→ grass   [đóng góp +0.7053]
        • ảnh –chứa→ child   [đóng góp +0.5675]

  #3  image_id=7180  score=1.5938
      caption: A boy soccer player running down the field .
      đối tượng: ['ball', '

---
# PHẦN 2 — GNN của SCENIR thay GNN cổ điển (Flickr8k)

_Nhiệm vụ 2 · dùng lại toàn bộ mã của Phần 1._

Cùng `run_experiment`, cùng split, nhãn tagger và ứng viên CLIP; chỉ đổi `CFG.gnn_type = "scenir"` (mục 1.7.2).

In [42]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
with use_gnn("scenir"):
    sc_res, sc_art = run_experiment(records, split, img_feat, txt_feat,
                                    ablations=("none", "random_edges", "no_relations"))
sc_df = results_table(sc_res)
sc_df.to_csv(CFG.out_dir / "results" / f"main_results_scenir_{CFG.flickr_annot}.csv", index=False)
display(sc_df)

In [43]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
# GNN cổ điển vs SCENIR, tách nhóm trực tiếp / gián tiếp
cmp_gnn = pd.concat([df.assign(gnn=CFG.gnn_type), sc_df.assign(gnn="scenir")])
cmp_gnn = cmp_gnn[cmp_gnn.ablation.isin(["none", "random_edges", "no_relations"])]
cmp_gnn.to_csv(CFG.out_dir / "results" / "gnn_classic_vs_scenir.csv", index=False)
display(cmp_gnn[["gnn", "ablation", "model", "R@1", "R@5", "R@10", "MRR", "nDCG@10"]])

In [44]:
# Gói kết quả để nộp
print("\nTệp đã ghi:")
for p in sorted((CFG.out_dir / "results").glob("*")) + [CFG.out_dir / "split.json"]:
    print(f"  {p}  ({p.stat().st_size/1024:.1f} KB)")


Tệp đã ghi:
  /kaggle/working/results/alpha_sensitivity.csv  (0.7 KB)
  /kaggle/working/results/annotation_info.json  (0.9 KB)
  /kaggle/working/results/case_analysis.json  (8.4 KB)
  /kaggle/working/results/eval_queries_20.json  (4.0 KB)
  /kaggle/working/results/explanation_grounding.csv  (1.8 KB)
  /kaggle/working/results/explanation_grounding_per_query.csv  (94.6 KB)
  /kaggle/working/results/fidelity_final_rank.csv  (1.0 KB)
  /kaggle/working/results/fidelity_per_query.csv  (2.9 KB)
  /kaggle/working/results/final_model_test.csv  (0.7 KB)
  /kaggle/working/results/final_model_val_selection.csv  (0.1 KB)
  /kaggle/working/results/fusion_log.csv  (3.7 KB)
  /kaggle/working/results/main_results.csv  (5.4 KB)
  /kaggle/working/results/main_results_probe.csv  (5.4 KB)
  /kaggle/working/results/paired_ablation_test.csv  (1.8 KB)
  /kaggle/working/results/paired_adaptive_vs_global.csv  (0.5 KB)
  /kaggle/working/results/relational_hard_negatives.csv  (0.3 KB)
  /kaggle/working/results/r

---
# PHẦN 3 — Action Genome (dataset thứ hai)

_Cùng pipeline của Phần 1–2 trên video/frame Action Genome: liên kết khái niệm (dictionary vs semantic), 3 ablation × 3 seed, SCENIR._

In [45]:
import shutil, os, glob, json
from pathlib import Path

# Dataset hieutrinh123/action-genome được Kaggle mount sẵn (chỉ đọc, đĩa cục bộ)
# ⇒ không cần chép/giải nén. Dò thư mục chứa ag_records.json; frame nằm ở <thư mục>/ag_frames/.
_ag_found = sorted(glob.glob("/kaggle/input/**/ag_records.json", recursive=True))
assert _ag_found, ("Không thấy ag_records.json trong /kaggle/input — Add Input → "
                   "dataset 'hieutrinh123/action-genome'.")
DATA_PATH = str(Path(_ag_found[0]).parent)
AG_CACHE_DIR = Path("/kaggle/working/output/action_genome/cache")   # = AG_OUT / "cache"
AG_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Nếu có cache CLIP của lần chạy trước được Add Input thì dùng lại, khỏi mã hoá frame.
for _f in glob.glob("/kaggle/input/**/clip_ag_*.npz", recursive=True):
    _dst = AG_CACHE_DIR / Path(_f).name
    if not _dst.exists():
        shutil.copy2(_f, _dst)

_n_jpg = len(os.listdir(Path(DATA_PATH) / "ag_frames"))
print("AG root:", DATA_PATH, "|", _n_jpg, "frame")

# v6: AG giữ backbone ViT-L/14 (dùng lại cache CLIP) và chỉ 3 ablation chính; đối chứng quan hệ chạy riêng ở cuối Phần 3.
CFG.clip_model, CFG.clip_weights, CFG.clip_bs = "ViT-L-14", "laion2b_s32b_b82k", 128
for _f in (clip_text_features, encode_free_text):          # v6: bỏ model CLIP ViT-B/32 đã cache ở Phần 1
    for _a in ("_m", "_t"):
        _f.__dict__.pop(_a, None)
CFG.ablations = ("none", "random_edges", "no_relations")


AG root: /kaggle/input/datasets/hieutrinh123/action-genome/action-genome | 38238 frame


In [46]:
# =====================================================================================
# v6 — HAI ĐỐI CHỨNG KIỂM TRA "CHÍNH QUAN HỆ mang thông tin" trên AG
#  (1) composite_swap: mỗi node quan hệ 'v o' của MỌI video bị thay bằng 'v′ o' (v′ ≠ v, lấy theo
#      phân bố vị từ của vật o trên TRAIN). Giữ nguyên vật và số node ⇒ nếu hiệu quả trên bộ quan hệ
#      khó mất đi thì thông tin nằm ở QUAN HỆ ĐÚNG, không phải ở việc có thêm node.
#  (2) Truy vấn phản thực tế: đổi vị từ trong truy vấn ('drinking … cup' → 'holding … cup') sao cho
#      video đúng KHÔNG có quan hệ mới. Mô hình hiểu quan hệ phải hạ hạng video đúng; CLIP gần như không.
# =====================================================================================
import copy
AG_VO = defaultdict(Counter)                     # vật → phân bố vị từ (CHỈ train), dựng khi có ag_records


def _ag_vo():
    if not AG_VO:
        tr_ids = set(ag_split["train"])
        for r in ag_records:
            if r["image_id"] in tr_ids:
                for c in r["objects"]:
                    if " " in c:
                        v, o = c.split(" ", 1); AG_VO[o][v] += 1
    return AG_VO


# ---------------- truy vấn phản thực tế: đổi vị từ ----------------
VERB_ING = dict(look="looking", hold="holding", carry="carrying", drink="drinking", eat="eating",
                sit="sitting", stand="standing", lie="lying", lean="leaning", touch="touching",
                wear="wearing", wipe="wiping", write="writing", twist="twisting")


def build_ag_counterfactuals(recs, gal):
    out = []
    for q, cand in gal:
        text, objs = recs[q]["caption"], set(recs[q]["objects"])
        toks = re.findall(r"[A-Za-z]+|[^A-Za-z]+", text)
        nouns = {AG_NOUN_ALIAS.get(_ag_lemma(t.lower()), _ag_lemma(t.lower())) for t in toks if t.isalpha()}
        for k, t in enumerate(toks):
            if not t.isalpha():
                continue
            v = _ag_lemma(t.lower(), "v"); v = AG_VERB_ALIAS.get(v, v)
            hit = sorted(c for c in objs if " " in c and c.split(" ", 1)[0] == v and c.split(" ", 1)[1] in nouns)
            if not hit:
                continue
            o = hit[0].split(" ", 1)[1]
            alts = [x for x, _ in _ag_vo()[o].most_common() if x != v and f"{x} {o}" not in objs and x in VERB_ING]
            if not alts:
                continue
            new = list(toks); new[k] = VERB_ING[alts[0]]
            out.append(dict(q=int(q), cand=np.asarray(cand), text=text, cf="".join(new), rel=hit[0],
                            cf_rel=f"{alts[0]} {o}"))
            break
    return out


REL_CF, REL_CF_FEATS, REL_CF_RES = {}, {}, defaultdict(dict)


def _init_ag_cf():
    """Dựng bộ truy vấn phản thực tế từ bộ quan hệ khó của AG (gọi một lần, lúc hook chạy lần đầu)."""
    _ag_vo()
    REL_CF["ag"] = build_ag_counterfactuals(ag_records, REL_GALLERIES["ag"])
    texts = sorted({x["text"] for x in REL_CF["ag"]} | {x["cf"] for x in REL_CF["ag"]})
    REL_CF_FEATS.update(zip(texts, clip_text_features(texts)))        # CÙNG bộ mã hoá cho cả 2 phiên bản
    log(f"truy vấn phản thực tế: {len(REL_CF['ag'])}/{len(REL_GALLERIES['ag'])} truy vấn của bộ quan hệ khó")
    for x in REL_CF["ag"][:4]:
        print(f"  • [{x['rel']} → {x['cf_rel']}]\n      gốc : {x['text'][:110]}\n      đổi : {x['cf'][:110]}")


@torch.no_grad()
def cf_ranks(model, eng, cfs, alpha, lam):
    model.eval()
    h = model.encode_graph(eng.graph.edge_index.to(DEVICE))
    z = znorm_per_row
    out = []
    for x in cfs:
        cand = x["cand"]
        row = {}
        for tag in ("text", "cf"):
            qf = torch.tensor(REL_CF_FEATS[x[tag]][None])
            cids = [n for n, _, _ in parse_query(x[tag], eng.graph)]
            s_clip = (qf @ eng.img_feat[torch.from_numpy(cand)].T).numpy()
            pm = torch.tensor(path_score_matrix(eng.graph, cids, cand)[None]).to(DEVICE)
            s_gnn = model.gnn_score(h, qf.to(DEVICE), torch.tensor(cand[None], device=DEVICE)).cpu().numpy()
            s_path = model.path_score(pm).cpu().numpy()
            fin = combine(z(s_clip), z(s_gnn), z(s_path), alpha, lam)[0]
            row[f"rank_{tag}"] = 1 + int((fin > fin[0]).sum())
            row[f"clip_rank_{tag}"] = 1 + int((s_clip[0] > s_clip[0, 0]).sum())
        out.append(row)
    return pd.DataFrame(out)


if "_relprobe_hook_v5" not in globals():
    _relprobe_hook_v5 = relprobe_hook


def relprobe_hook(ab, seed, model, eng, parts, alpha, lam):
    _relprobe_hook_v5(ab, seed, model, eng, parts, alpha, lam)
    if REL_GAL_KEY == "ag" and "ag" not in REL_CF and "ag" in REL_GALLERIES:
        _init_ag_cf()
    if REL_GAL_KEY in REL_CF:
        d = cf_ranks(model, eng, REL_CF[REL_GAL_KEY], alpha, lam)
        name = REL_RENAME.get(ab, ab)
        REL_CF_RES[REL_GAL_KEY][(name, seed)] = d
        REL_CF_RES[REL_GAL_KEY].setdefault(("CLIP", seed), d.assign(rank_text=d.clip_rank_text,
                                                                    rank_cf=d.clip_rank_cf))
        print(f"    [phản thực tế] {name} seed {seed}: video đúng bị hạ hạng ở "
              f"{(d.rank_cf > d.rank_text).mean():.1%} truy vấn (CLIP {(d.clip_rank_cf > d.clip_rank_text).mean():.1%})")


def cf_report(key, base="none", n_boot=10000):
    res = REL_CF_RES[key]
    names = list(dict.fromkeys(a for a, _ in res))
    rows, per = [], {}
    for a in names:
        ds = [d for (b, s), d in res.items() if b == a]
        drr = np.mean([1 / d.rank_text - 1 / d.rank_cf for d in ds], 0)      # RR(gốc) − RR(đổi vị từ), TB theo seed
        per[a] = drr
        rows.append(dict(model=a, n=len(drr), seeds=len(ds),
                         demoted=np.mean([(d.rank_cf > d.rank_text).mean() for d in ds]),
                         promoted=np.mean([(d.rank_cf < d.rank_text).mean() for d in ds]),
                         delta_RR=drr.mean()))
    tab = pd.DataFrame(rows)
    rng = np.random.RandomState(0); sig_rows = []
    for b in names:
        if b == base:
            continue
        d = per[base] - per[b]
        bs = d[rng.randint(0, len(d), (n_boot, len(d)))].mean(1)
        sig_rows.append(dict(a=base, b=b, n=len(d), diff_delta_RR=d.mean(),
                             ci_lo=np.percentile(bs, 2.5), ci_hi=np.percentile(bs, 97.5)))
    return tab, pd.DataFrame(sig_rows)


In [47]:
# =====================================================================================
# CẤP ĐỘ 2 — DATASET THỨ HAI: ACTION GENOME (text-to-video)
# Chạy toàn bộ pipeline trên AG mà KHÔNG ghi đè kết quả Flickr8k.
# =====================================================================================
import os
import nltk
from contextlib import contextmanager
from nltk.stem import WordNetLemmatizer
import pandas as pd
for _p in ("wordnet", "omw-1.4"):
    try: nltk.download(_p, quiet=True)
    except Exception: pass

_AG_LEM = WordNetLemmatizer(); _AG_LEM_CACHE = {}

def _ag_lemma(w, pos="n"):
    k = (w, pos)
    if k not in _AG_LEM_CACHE:
        try: _AG_LEM_CACHE[k] = _AG_LEM.lemmatize(w, pos)
        except Exception: _AG_LEM_CACHE[k] = w
    return _AG_LEM_CACHE[k]

# ---- Từ vựng AG: phần lấy từ dữ liệu vs phần viết tay (ghi rõ trong báo cáo) ----
_ag_hits = glob.glob(f"{DATA_PATH}/**/ag_records.json", recursive=True)
assert _ag_hits, "Chưa Add Input output của notebook ag-download"
_ag_payload = json.load(open(_ag_hits[0]))

# (1) Quan hệ người–vật → động từ gốc. Ánh xạ thủ công 1-1 từ tên lớp AG; đã kiểm tra bên dưới.
#     Bỏ: quan hệ không gian, quan hệ phủ định/mơ hồ, covered_by (ngược chiều ngữ nghĩa).
AG_PRED2VERB = {"looking_at": "look", "holding": "hold", "carrying": "carry",
                "have_it_on_the_back": "carry", "drinking_from": "drink", "eating": "eat",
                "sitting_on": "sit", "standing_on": "stand", "lying_on": "lie", "leaning_on": "lean",
                "touching": "touch", "wearing": "wear", "wiping": "wipe", "writing_on": "write",
                "twisting": "twist"}
_pred_seen = Counter(p for r in _ag_payload["records"] for _, p, _ in r["triples"])
print("Predicate có trong dữ liệu nhưng KHÔNG dùng:",
      {p: n for p, n in _pred_seen.items() if p not in AG_PRED2VERB})
_bad = [p for p in AG_PRED2VERB if p not in _pred_seen]
assert not _bad, f"Khóa không tồn tại trong annotation AG: {_bad}"

# (2) Alias danh từ: SINH TỰ ĐỘNG từ tên lớp nhiều tên của AG (vd 'cup/glass/bottle')
_ag_classes = {c for r in _ag_payload["records"] for c in r["objects"]}
AG_NOUN_ALIAS = {_ag_lemma(alt.strip().lower()): _ag_lemma(c.split("/")[0].strip().lower())
                 for c in _ag_classes if "/" in c for alt in c.split("/")[1:]}
print("Alias danh từ sinh từ dữ liệu:", AG_NOUN_ALIAS)

# (3) Alias động từ: VIẾT TAY, căn cứ là tần suất động từ trong mô tả của tập TRAIN
AG_VERB_ALIAS = {"watch": "look", "grab": "hold"}
_AG_VERBS = set(AG_PRED2VERB.values())
_train_verbs = Counter(
    _ag_lemma(t, "v") for r in _ag_payload["records"] if r["split"] == "train"
    for t in _norm(r["caption"]).split()
    if t not in STOP and _ag_lemma(t, "v") != t)        # xấp xỉ: token có dạng biến tố động từ
print("Động từ phổ biến trên TRAIN chưa được ánh xạ:",
      [(v, n) for v, n in _train_verbs.most_common(40)
       if v not in _AG_VERBS and v not in AG_VERB_ALIAS][:20])


def ag_canon(cls):
    """'cup/glass/bottle' → 'cup'; 'groceries' → 'grocery' (khớp lemma phía truy vấn)."""
    return _ag_lemma(cls.split("/")[0].strip().lower().replace("_", " "))


def load_action_genome():
    root = Path(_ag_hits[0]).parent
    out = []
    for i, r in enumerate(_ag_payload["records"]):
        objs = {ag_canon(c) for c in r["objects"]} - {"person"}   # video nào cũng có người
        comps, rels = set(), set()
        for kind, pred, cls in r["triples"]:
            v, o = AG_PRED2VERB.get(pred), ag_canon(cls)
            if v is None or o == "person":
                continue
            c = f"{v} {o}"                       # node ghép, vd 'hold cup'
            comps.add(c)
            rels.add((o, "afford", c))           # cup –afford→ hold cup (gộp CHỈ trên train)
        out.append(dict(image_id=i, video_id=r["video_id"], split=r["split"],
                        caption=r["caption"], captions=r["descriptions"],
                        objects=sorted(objs | comps), relations=sorted(rels), url="",
                        frames=[str(root / "ag_frames" / f) for f in r["frames"]]))
    print(f"Action Genome: {len(out)} video | "
          f"{np.mean([len(r['objects']) for r in out]):.1f} node/video | "
          f"{np.mean([len(r['relations']) for r in out]):.1f} quan hệ/video | "
          f"{np.mean([len(r['frames']) for r in out]):.1f} frame/video")
    for r in out[:3]:
        print("  •", r["caption"][:80], "→", r["objects"][:8])
    return out


def encode_action_genome(recs):
    """CLIP từng frame → mean-pool theo thời gian → L2. Văn bản = mô tả Charades."""
    cache = CFG.out_dir / "cache" / f"clip_ag_{CFG.clip_model}_{CFG.clip_weights}_{len(recs)}.npz"
    if cache.exists():
        z = np.load(cache); print("Dùng lại cache CLIP:", cache.name); return z["img"], z["txt"]
    import open_clip
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    model, _, preprocess = open_clip.create_model_and_transforms(CFG.clip_model,
                                                                 pretrained=CFG.clip_weights)
    tokenizer = open_clip.get_tokenizer(CFG.clip_model)
    model = model.to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    items = [(i, p) for i, r in enumerate(recs) for p in r["frames"]]
    missing = [p for _, p in items if not os.path.exists(p)]
    if len(missing) > 0.01 * len(items) and "ag_ensure_frames" in globals():
        print(f"Thiếu {len(missing)}/{len(items)} frame (cache CLIP không khớp) → chép frame từ Drive…")
        _DONE.unlink(missing_ok=True)            # bỏ dấu 'chỉ annotation' để chép đủ
        ag_ensure_frames()
        missing = [p for _, p in items if not os.path.exists(p)]
    assert len(missing) <= 0.01 * len(items), \
        f"{len(missing)}/{len(items)} frame không tồn tại (vd {missing[:2]}) — kiểm tra DATA_PATH"
    class FrameDS(Dataset):
        def __len__(self): return len(items)
        def __getitem__(self, k):
            try: im = Image.open(items[k][1]).convert("RGB")
            except Exception: im = Image.new("RGB", (224, 224), (128, 128, 128))
            return preprocess(im)

    feats = []
    with torch.no_grad():
        from tqdm.auto import tqdm
        log(f"CLIP: mã hoá {len(items)} frame của {len(recs)} video (batch {CFG.clip_bs})")
        for x in tqdm(DataLoader(FrameDS(), batch_size=CFG.clip_bs, num_workers=2),
                      desc="CLIP frame", unit="batch"):
            feats.append(F.normalize(model.encode_image(x.to(DEVICE)), dim=-1).float().cpu().numpy())
        feats = np.concatenate(feats)
        img = np.zeros((len(recs), feats.shape[1]), np.float32)
        np.add.at(img, np.array([i for i, _ in items]), feats)
        img /= np.linalg.norm(img, axis=1, keepdims=True).clip(1e-8)
        caps = [r["caption"] for r in recs]
        # Mô tả Charades thường dài hơn 77 token ⇒ open_clip cắt cụt. Tách theo câu, mã hoá từng câu,
        # lấy trung bình. Giữ cả bản cắt cụt; chọn bản nào bằng MRR CLIP trên VAL (không nhìn test).
        pieces, owner = [], []
        for i, t in enumerate(caps):
            sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if len(s.strip().split()) >= 2] or [t]
            pieces += sents; owner += [i] * len(sents)
        enc = lambda xs: np.concatenate([F.normalize(model.encode_text(tokenizer(xs[s:s + CFG.clip_bs]).to(DEVICE)),
                                                     dim=-1).float().cpu().numpy()
                                         for s in range(0, len(xs), CFG.clip_bs)])
        txt_trunc = enc(caps)
        E = enc(pieces); txt_chunk = np.zeros_like(txt_trunc)
        np.add.at(txt_chunk, np.array(owner), E)
        txt_chunk /= np.linalg.norm(txt_chunk, axis=1, keepdims=True)
        va = np.array([i for i, r in enumerate(recs) if r["split"] == "val"])
        mrr = lambda T: reciprocal_ranks(np.tile(va, (len(va), 1)), T[va] @ img[va].T, va).mean()
        m_t, m_c = mrr(txt_trunc), mrr(txt_chunk)
        assert max(m_t, m_c) > 0.02, \
            f"CLIP VAL MRR {max(m_t, m_c):.4f} ≈ ngẫu nhiên — frame hỏng hoặc ảnh/văn bản lệch thứ tự"
        txt = txt_chunk if m_c > m_t else txt_trunc
        log(f"Văn bản AG — VAL MRR CLIP: cắt cụt {m_t:.4f} | theo câu {m_c:.4f} → dùng "
            f"{'theo câu' if m_c > m_t else 'cắt cụt'}")
    log(f"CLIP xong: ảnh {img.shape}, văn bản {txt.shape} → cache {cache.name}")
    del model; torch.cuda.empty_cache()
    np.savez_compressed(cache, img=img, txt=txt)
    return img, txt


def ag_make_split(recs, seed=None):
    """Split lấy từ ag_records.json: train/test gốc của AG, val tách bằng seed cố định."""
    sp_ = {k: sorted(r["image_id"] for r in recs if r["split"] == k) for k in ("train", "val", "test")}
    sp_["seed"] = "ag-official-train-test + val seed 20240101"
    payload = json.dumps({k: sp_[k] for k in ("train", "val", "test")}, sort_keys=True)
    sp_["sha1"] = hashlib.sha1(payload.encode()).hexdigest()[:12]
    with open(CFG.out_dir / "split.json", "w") as f:
        json.dump(sp_, f)
    print(f"Split AG (sha1={sp_['sha1']}): train={len(sp_['train'])} "
          f"val={len(sp_['val'])} test={len(sp_['test'])}")
    assert not (set(sp_["train"]) & set(sp_["test"])) and not (set(sp_["val"]) & set(sp_["test"]))
    return sp_


def ag_parse_query(text, graph):
    """'A person is drinking from a glass' → ['drink', 'cup'] → khớp 'drink cup' và 'cup'."""
    toks = []
    for t in _norm(text).split():
        if t in STOP or len(t) < 2:
            continue
        v = _ag_lemma(t, "v"); v = AG_VERB_ALIAS.get(v, v)
        if v in _AG_VERBS:
            toks.append(v); continue
        n = _ag_lemma(t, "n"); toks.append(AG_NOUN_ALIAS.get(n, n))
    cands = [" ".join(toks[i:i + n]) for n in (2, 1) for i in range(len(toks) - n + 1)]
    hits, seen = [], set()
    for g in cands:
        nid = graph.name2id.get(g)
        if nid is not None and nid not in seen:
            seen.add(nid); hits.append((nid, g, g))
    return hits


@contextmanager
def use_dataset(out_dir, **overrides):
    """Tạm thay hàm toàn cục + thư mục output; khôi phục nguyên trạng khi ra khỏi khối."""
    g = globals()
    saved = {k: g.get(k) for k in overrides}
    saved_out = CFG.out_dir
    g.update(overrides)
    CFG.out_dir = Path(out_dir)
    for sub in ("cache", "results"):
        (CFG.out_dir / sub).mkdir(parents=True, exist_ok=True)
    _GROUP_CACHE.clear()
    try:
        yield
    finally:
        for k, v in saved.items():
            if v is None: g.pop(k, None)
            else: g[k] = v
        CFG.out_dir = saved_out
        _GROUP_CACHE.clear()


# (4) IsA cho node ghép 'động từ vật' → khái niệm hành động 'động từ' (vd 'hold cup' –là_một→ 'hold').
#     WordNet chỉ phủ 31/202 lớp AG nên tầng khái niệm gần như rỗng; đây là tri thức NGOÀI
#     (cấu trúc tên lớp), không phải thống kê ⇒ an toàn cho test. Truy vấn "someone holding
#     something" giờ có đường đi P2/P4/P5 tới 'hold' thay vì không khớp node nào.
#     Ghi vào FALLBACK_ISA toàn cục để mọi cell AG sau (semantic linking) dùng cùng một đồ thị;
#     tên 'động từ vật' không bao giờ xuất hiện trong Flickr8k nên không ảnh hưởng phần trước.
AG_ACTION_ISA = {f"{v} {ag_canon(cls)}": [v]
                 for r in _ag_payload["records"] for _, pred, cls in r["triples"]
                 for v in [AG_PRED2VERB.get(pred)] if v is not None and ag_canon(cls) != "person"}
FALLBACK_ISA.update(AG_ACTION_ISA)
print(f"IsA hành động: {len(AG_ACTION_ISA)} node ghép → {len(set(AG_PRED2VERB.values()))} khái niệm động từ")

# ------------------------------------ LOG TIẾN ĐỘ ------------------------------------
# Mọi dòng có [mm:ss từ đầu cell | +ss so với dòng trước] và flush ngay (Colab hay giữ buffer).
# Các hàm nặng của pipeline được BỌC lại qua use_dataset(...) — chỉ có hiệu lực trong khối
# with, tự khôi phục khi ra khỏi khối, không sửa cell nào khác.
_T0 = _TL = time.time()


def log(msg):
    global _TL
    now = time.time()
    m, s_ = divmod(int(now - _T0), 60)
    print(f"[{m:02d}:{s_:02d} | +{now - _TL:5.1f}s] {msg}", flush=True)
    _TL = now


@contextmanager
def step(title):
    log(f"▶ {title}")
    t = time.time()
    yield
    log(f"✔ {title} ({time.time() - t:.1f}s)")


_N_ABL = len(CFG.ablations); _N_SEED = len(CFG.seeds)
_PROG = dict(abl=None, i_abl=0, i_seed=0)
_orig = dict(prepare=prepare, train_graph_scorer=train_graph_scorer, tune_on_val=tune_on_val,
             evaluate_split=evaluate_split, self_test=self_test)


def _prepare_logged(records, split, img_feat, txt_feat, ablation="none", seed=0):
    if ablation != _PROG["abl"]:
        _PROG.update(abl=ablation, i_abl=_PROG["i_abl"] + 1, i_seed=0)
    _PROG["i_seed"] += 1
    tag = f"[ablation {_PROG['i_abl']}/{_N_ABL} '{ablation}' | seed {_PROG['i_seed']}/{_N_SEED}]"
    with step(f"{tag} dựng đồ thị + bộ truy vấn"):
        out = _orig["prepare"](records, split, img_feat, txt_feat, ablation, seed)
    g = out[0]
    log(f"    nodes={g.n_nodes} | scene={g.A_scene.nnz} related={g.A_related.nnz} isa={g.A_isa.nnz}"
        f" | đường đi P1..P5={sum(g.R[p].nnz for p in PATTERNS[1:])}")
    return out


def _train_logged(engine, train_q_idx, train_img_idx, seed=0, verbose=True):
    with step(f"  huấn luyện GNN seed={seed} ({CFG.epochs} epoch, {len(train_q_idx)} truy vấn)"):
        model, hist = _orig["train_graph_scorer"](engine, train_q_idx, train_img_idx,
                                                  seed=seed, verbose=verbose)
    gates = model.edge_gates()
    log(f"    loss {hist[0]:.3f} → {hist[-1]:.3f} | clip_w {F.softplus(model.clip_logit).item():.2f}"
        f" | path_w {F.softplus(model.path_logit).item():.2f}"
        f" | w_path " +
        " ".join(f"{x:.3f}" for x in model.pattern_weights().detach().cpu().numpy()) +
        (" | gate L0[scene,rel,isa] " + " ".join(f"{x:.2f}" for x in gates[0][1:4]) if gates else ""))
    return model, hist


def _tune_logged(model, engine, val_q, val_gallery, verbose=True):
    with step(f"  quét (α, λ) trên VAL ({len(CFG.alpha_grid) * len(CFG.lambda_grid)} cặp)"):
        (a, l), grid = _orig["tune_on_val"](model, engine, val_q, val_gallery, verbose=False)
    best = max(grid, key=lambda r: r["MRR"])
    log(f"    chọn α={a}, λ={l} | VAL MRR={best['MRR']:.4f} (CLIP thuần: "
        f"{next(r['MRR'] for r in grid if r['alpha'] == 1.0):.4f})"
        + ("  ⚠ λ=1 ⇒ s_path không tham gia điểm cuối" if l == 1.0 else ""))
    return (a, l), grid


def _eval_logged(model, engine, part, alpha, lam, by_group=True):
    res, raw = _orig["evaluate_split"](model, engine, part, alpha, lam, by_group)
    log(f"    TEST: CLIP MRR={res['CLIP']['MRR']:.4f} → CLIP+Graph MRR={res['CLIP+Graph']['MRR']:.4f}")
    return res, raw


def _self_test_logged(*a, **k):
    with step("self-test (rò rỉ / khớp giải thích / split) — dựng đồ thị 3 lần, hơi lâu"):
        return _orig["self_test"](*a, **k)


# ------------------------------------ CHẠY ------------------------------------
AG_OUT = Path("/kaggle/working/output/action_genome")
log(f"Bắt đầu Action Genome | ablations={CFG.ablations} | seeds={CFG.seeds} | epochs={CFG.epochs}"
    f" | device={DEVICE}")
with use_dataset(AG_OUT, parse_query=ag_parse_query, make_split=ag_make_split,
                 prepare=_prepare_logged, train_graph_scorer=_train_logged,
                 tune_on_val=_tune_logged, evaluate_split=_eval_logged,
                 self_test=_self_test_logged):
    R = CFG.out_dir / "results"

    with step("1/9 nạp annotation AG"):
        ag_records = load_action_genome()
    with step("2/9 split"):
        ag_split = make_split(ag_records)
    with step("3/9 đặc trưng CLIP (dùng cache nếu có)"):
        ag_img, ag_txt = encode_action_genome(ag_records)
        if "ag_save_clip_cache_to_drive" in globals():
            ag_save_clip_cache_to_drive()
    with step("4/9 self-test"):
        self_test(ag_records, ag_split, ag_img, ag_txt)
    if "pick_learn_clip_w" in globals():
        with step("4b/9 chọn learn_clip_w trên VAL (2 lần huấn luyện 'none', seed 0)"):
            clipw_ag = pick_learn_clip_w(ag_records, ag_split, ag_img, ag_txt)

    with step(f"5/9 thí nghiệm chính: {_N_ABL} ablation × {_N_SEED} seed = {_N_ABL * _N_SEED} lần huấn luyện"):
        _PROG.update(abl=None, i_abl=0, i_seed=0)
        REL_GALLERIES["ag"] = build_rel_galleries_ag(ag_records, ag_split)      # v5: bộ quan hệ khó
        REL_GAL_KEY, REL_RENAME = "ag", {}
        ag_all_res, ag_art = run_experiment(ag_records, ag_split, ag_img, ag_txt,
                                            ablations=CFG.ablations)
        REL_GAL_KEY = None
    ag_df = results_table(ag_all_res)
    import pickle
    with open(R / "per_query_all_ablations.pkl", "wb") as f:      # cho kiểm định cặp về sau
        pickle.dump(dict(PER_QUERY), f)
    ag_df.to_csv(R / "main_results.csv", index=False)
    display(ag_df)
    PQ_AG = dict(PER_QUERY) if "PER_QUERY" in globals() else {}   # cho cell kiểm định cặp

    ag_piv = pd.DataFrame(ag_art["grid"]).pivot_table(index="alpha", columns="lam", values="MRR")
    ag_piv.to_csv(R / "alpha_sensitivity.csv")
    print("\nMRR trên VALIDATION theo (α, λ):\n", ag_piv.round(4).to_string())

    with step("6/9 phân tích 5 thành công / 5 thất bại"):
        ag_cases = analyse_cases(ag_art, ag_records)
    for c in ag_cases:
        print(f"\n[{c['kind']}] nhóm={c['group']} | hạng CLIP {c['rank_clip']} → "
              f"CLIP+Graph {c['rank_graph']}\n  truy vấn: {c['query']}")
        for e in c["explanation"]:
            print(f"    • {e['text']}   [{e['contribution']:+.4f}]")
    with open(R / "case_analysis.json", "w") as f:
        json.dump(ag_cases, f, ensure_ascii=False, indent=2)

    with step("7/9 fidelity trên điểm đường đi"):
        ag_fdf, ag_summ = fidelity_study(ag_art)
    ag_fdf.to_csv(R / "fidelity_per_query.csv", index=False)
    with open(R / "xai_summary.json", "w") as f:
        json.dump(ag_summ, f, indent=2)
    print("\nxAI (điểm đường đi):", {k: round(v, 4) if isinstance(v, float) else v
                                     for k, v in ag_summ.items()})

    if "final_rank_fidelity" in globals():
        with step("8/9 fidelity trên thứ hạng điểm cuối"):
            ag_rdf, ag_rsumm = final_rank_fidelity(ag_art)
        ag_rdf.to_csv(R / "fidelity_final_rank.csv", index=False)
        with open(R / "xai_final_rank_summary.json", "w") as f:
            json.dump(ag_rsumm, f, indent=2)
        print("xAI (thứ hạng điểm cuối):", {k: round(v, 3) if isinstance(v, float) else v
                                           for k, v in ag_rsumm.items()})

    with step("9/9 xuất bộ truy vấn đánh giá + demo"):
        export_eval_queries(ag_art["engine"].queries, ag_art["graph"],
                            R / "eval_queries_20.json", n=20, allowed_ids=ag_split["test"])
        demo_query("a person drinking from a glass while sitting on the couch", ag_art, k=3)
        demo_query("someone watching tv and holding a phone", ag_art, k=3)

# ------------------- Bảng tổng hợp cấp độ 2: hai bộ dữ liệu -------------------
if "df" in globals():
    both = pd.concat([df.assign(dataset="flickr8k"), ag_df.assign(dataset="action_genome")])
    both = both[both.ablation == "none"][["dataset", "model", "R@1", "R@5", "R@10", "MRR", "nDCG@10"]]
    both.to_csv(Path("/kaggle/working/results") / "level2_two_datasets.csv", index=False)
    display(both)
log("XONG toàn bộ cell Action Genome")

Predicate có trong dữ liệu nhưng KHÔNG dùng: {'behind': 9032, 'beneath': 7499, 'in_front_of': 28163, 'on_the_side_of': 19271, 'in': 3362, 'above': 2225, 'covered_by': 670}
Alias danh từ sinh từ dữ liệu: {'glass': 'cup', 'bottle': 'cup', 'couch': 'sofa', 'notebook': 'paper', 'camera': 'phone', 'cabinet': 'closet'}
Động từ phổ biến trên TRAIN chưa được ánh xạ: [('walk', 1962), ('take', 1224), ('put', 1073), ('pick', 828), ('open', 813), ('shoe', 467), ('clothe', 455), ('close', 409), ('get', 364), ('sneeze', 363), ('throw', 361), ('turn', 356), ('stave', 288), ('begin', 281), ('place', 257), ('start', 254), ('play', 253), ('leave', 249), ('laugh', 221), ('pour', 215)]
IsA hành động: 219 node ghép → 14 khái niệm động từ
[00:00 | +  0.0s] Bắt đầu Action Genome | ablations=('none', 'random_edges', 'no_relations') | seeds=[0, 1, 2] | epochs=20 | device=cuda
[00:00 | +  0.0s] ▶ 1/9 nạp annotation AG
Action Genome: 9587 video | 11.3 node/video | 7.9 quan hệ/video | 4.0 frame/video
  • A person

,learn_clip_w,val_MRR,alpha,lam,clip_w,path_w
0,False,0.416715,0.2,0.25,1.000,1.0
1,True,0.452643,0.0,0.50,0.068,1.0


→ CFG.learn_clip_w = True (chọn theo VAL MRR)
[14:16 | +  0.0s] ✔ 4b/9 chọn learn_clip_w trên VAL (2 lần huấn luyện 'none', seed 0) (842.9s)
[14:16 | +  0.0s] ▶ 5/9 thí nghiệm chính: 3 ablation × 3 seed = 9 lần huấn luyện
Bộ quan hệ khó (AG): 548/974 truy vấn test có quan hệ người–vật đủ ≥3 âm | trung bình 16.7 âm/truy vấn

ABLATION = none
[14:17 | +  0.3s] ▶ [ablation 1/3 'none' | seed 1/3] dựng đồ thị + bộ truy vấn
WordNet(+fallback): 198/202 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 8784, 'other': 346, 'indirect': 457}
[14:33 | + 16.5s] ✔ [ablation 1/3 'none' | seed 1/3] dựng đồ thị + bộ truy vấn (16.5s)
[14:33 | +  0.0s]     nodes=9847 | scene=167 related=1615 isa=221 | đường đi P1..P5=640036
Đồ thị: {'nodes': 9847, 'images': 9587, 'object_classes': 202, 'concepts': 58, 'edges': 110163, 'contains': 108160, 'scene': 167, 'related': 1615, 'isa': 221, 'triple_nodes': 0, 'triple_edges': 0, 'paths': {'P0_contains': 108160, 'P1_scene': 91495, 'P2_isa': 88992, 'P3_related':

open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.71GB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

[21:43 | + 23.4s] truy vấn phản thực tế: 545/548 truy vấn của bộ quan hệ khó
  • [look box → carry box]
      gốc : A person is vacuuming the floor around an open box, then the person looks into the box and closes the lid.
      đổi : A person is vacuuming the floor around an open box, then the person carrying into the box and closes the lid.
  • [sit sofa → lie sofa]
      gốc : A person is sitting on a sofa and putting shoes on, then starts to put on a coat, then starts sneezing
      đổi : A person is lying on a sofa and putting shoes on, then starts to put on a coat, then starts sneezing
  • [carry bag → hold bag]
      gốc : A person walks across a room and through a doorway while carrying a bag. The person then closes a door.
      đổi : A person walks across a room and through a doorway while holding a bag. The person then closes a door.
  • [look shelf → hold shelf]
      gốc : A person is placing things on a shelf, while another person watches.
      đổi : A person is placing 

,ablation,model,R@1,R@5,R@10,MRR,nDCG@10
0,none,CLIP,0.1044 ± 0.0000,0.2343 ± 0.0000,0.3127 ± 0.0000,0.1712 ± 0.0000,0.1954 ± 0.0000
1,none,CLIP+Graph,0.2759 ± 0.0144,0.4908 ± 0.0138,0.5661 ± 0.0104,0.3706 ± 0.0128,0.4134 ± 0.0126
2,none,CLIP [direct] (n=1626),0.1064 ± 0.0000,0.2380 ± 0.0000,0.3155 ± 0.0000,0.1730 ± 0.0000,0.1977 ± 0.0000
3,none,CLIP+Graph [direct] (n=1626),0.2925 ± 0.0145,0.5146 ± 0.0118,0.5865 ± 0.0080,0.3889 ± 0.0118,0.4330 ± 0.0111
4,none,CLIP [indirect] (n=100),0.0800 ± 0.0000,0.1900 ± 0.0000,0.2400 ± 0.0000,0.1433 ± 0.0000,0.1570 ± 0.0000
5,none,CLIP+Graph [indirect] (n=100),0.0967 ± 0.0094,0.2633 ± 0.0170,0.3633 ± 0.0340,0.1824 ± 0.0131,0.2157 ± 0.0180
6,none,CLIP+Graph (adaptive),0.2851 ± 0.0082,0.5013 ± 0.0025,0.5744 ± 0.0034,0.3803 ± 0.0056,0.4232 ± 0.0051
7,none,CLIP+Graph (adaptive) [direct] (n=1626),0.3003 ± 0.0093,0.5189 ± 0.0038,0.5888 ± 0.0036,0.3957 ± 0.0060,0.4389 ± 0.0054
8,none,CLIP+Graph (adaptive) [indirect] (n=100),0.1233 ± 0.0094,0.3100 ± 0.0141,0.3967 ± 0.0125,0.2179 ± 0.0036,0.2522 ± 0.0049
9,random_edges,CLIP,0.1044 ± 0.0000,0.2343 ± 0.0000,0.3127 ± 0.0000,0.1712 ± 0.0000,0.1954 ± 0.0000



MRR trên VALIDATION theo (α, λ):
 lam      0.00    0.25    0.50    0.75
alpha                                
0.0    0.2925  0.4232  0.4464  0.3561
0.2    0.3390  0.4116  0.4449  0.4221
0.4    0.2785  0.3114  0.3366  0.3402
0.5    0.2583  0.2757  0.2906  0.2963
0.6    0.2374  0.2460  0.2539  0.2563
0.7    0.2162  0.2222  0.2256  0.2271
0.8    0.2013  0.2040  0.2062  0.2072
0.9    0.1862  0.1875  0.1867  0.1874
1.0    0.1728  0.1728  0.1728  0.1728
[61:23 | +  0.0s] ▶ 6/9 phân tích 5 thành công / 5 thất bại
[61:23 | +  0.0s] ✔ 6/9 phân tích 5 thành công / 5 thất bại (0.0s)

[THÀNH CÔNG] nhóm=direct | hạng CLIP 100 → CLIP+Graph 1
  truy vấn: A person is putting some plates in a box. They pick up the box and put it on a bed. They then pick up a blanke
    • ảnh –chứa→ box   [+0.7729]

[THÀNH CÔNG] nhóm=indirect | hạng CLIP 100 → CLIP+Graph 1
  truy vấn: a person walks into a room and talks with another person while sitting
    • ảnh –chứa→ chair –afford→ sit chair –là_một→ sit   [+0.0814

,dataset,model,R@1,R@5,R@10,MRR,nDCG@10
0,flickr8k,CLIP,0.6120 ± 0.0000,0.8610 ± 0.0000,0.9290 ± 0.0000,0.7227 ± 0.0000,0.7697 ± 0.0000
1,flickr8k,CLIP+Graph,0.6153 ± 0.0049,0.8687 ± 0.0048,0.9337 ± 0.0050,0.7268 ± 0.0015,0.7742 ± 0.0017
2,flickr8k,CLIP [direct] (n=923),0.6241 ± 0.0000,0.8646 ± 0.0000,0.9285 ± 0.0000,0.7315 ± 0.0000,0.7763 ± 0.0000
3,flickr8k,CLIP+Graph [direct] (n=923),0.6277 ± 0.0036,0.8740 ± 0.0044,0.9332 ± 0.0050,0.7358 ± 0.0012,0.7810 ± 0.0016
4,flickr8k,CLIP [indirect] (n=52),0.4808 ± 0.0000,0.8462 ± 0.0000,0.9423 ± 0.0000,0.6345 ± 0.0000,0.7066 ± 0.0000
5,flickr8k,CLIP+Graph [indirect] (n=52),0.4872 ± 0.0240,0.8462 ± 0.0000,0.9487 ± 0.0091,0.6410 ± 0.0074,0.7136 ± 0.0048
6,flickr8k,CLIP+Graph (adaptive),0.6043 ± 0.0012,0.8637 ± 0.0037,0.9313 ± 0.0060,0.7189 ± 0.0011,0.7675 ± 0.0023
7,flickr8k,CLIP+Graph (adaptive) [direct] (n=923),0.6157 ± 0.0020,0.8682 ± 0.0042,0.9310 ± 0.0064,0.7277 ± 0.0006,0.7742 ± 0.0019
8,flickr8k,CLIP+Graph (adaptive) [indirect] (n=52),0.4808 ± 0.0157,0.8397 ± 0.0091,0.9423 ± 0.0157,0.6300 ± 0.0106,0.7028 ± 0.0121
0,action_genome,CLIP,0.1044 ± 0.0000,0.2343 ± 0.0000,0.3127 ± 0.0000,0.1712 ± 0.0000,0.1954 ± 0.0000


[61:29 | +  0.0s] XONG toàn bộ cell Action Genome


In [48]:
# Mô hình cuối của Action Genome chọn trên VAL + kiểm định cặp adaptive vs global
FUSION_AG = list(FUSION_LOG)
sel_ag, final_ag = final_model_table(ag_all_res, FUSION_AG)
display(sel_ag.round(4)); display(final_ag)
display(pd.DataFrame(FUSION_AG).round(3))
sig_ad_ag = paired_test(PQ_AG, "none+adaptive", "none")
display(sig_ad_ag.round(4))
R_AG = AG_OUT / "results"
for name, obj in (("final_model_val_selection.csv", sel_ag), ("final_model_test.csv", final_ag),
                  ("fusion_log.csv", pd.DataFrame(FUSION_AG)), ("paired_adaptive_vs_global.csv", sig_ad_ag)):
    obj.to_csv(R_AG / name, index=False)

→ Mô hình cuối (chọn trên VAL): đồ thị='none', kết hợp=adaptive (VAL MRR 0.4650)


,ablation,val_MRR,mode
0,none,0.4650,adaptive
1,none,0.4505,global


,ablation,model,R@1,R@5,R@10,MRR,nDCG@10
0,none,CLIP,0.1044 ± 0.0000,0.2343 ± 0.0000,0.3127 ± 0.0000,0.1712 ± 0.0000,0.1954 ± 0.0000
2,none,CLIP [direct] (n=1626),0.1064 ± 0.0000,0.2380 ± 0.0000,0.3155 ± 0.0000,0.1730 ± 0.0000,0.1977 ± 0.0000
4,none,CLIP [indirect] (n=100),0.0800 ± 0.0000,0.1900 ± 0.0000,0.2400 ± 0.0000,0.1433 ± 0.0000,0.1570 ± 0.0000
6,none,CLIP+Graph (adaptive),0.2851 ± 0.0082,0.5013 ± 0.0025,0.5744 ± 0.0034,0.3803 ± 0.0056,0.4232 ± 0.0051
7,none,CLIP+Graph (adaptive) [direct] (n=1626),0.3003 ± 0.0093,0.5189 ± 0.0038,0.5888 ± 0.0036,0.3957 ± 0.0060,0.4389 ± 0.0054
8,none,CLIP+Graph (adaptive) [indirect] (n=100),0.1233 ± 0.0094,0.3100 ± 0.0141,0.3967 ± 0.0125,0.2179 ± 0.0036,0.2522 ± 0.0049


,ablation,seed,alpha,lam,cv_global,cv_adaptive,alpha_q_with_concept,alpha_q_no_concept,w_alpha
0,none,0,0.0,0.5,0.446,0.469,0.033,0.110,"{'has_concept': 0.2290000021457672, 'log_n_con..."
1,none,1,0.2,0.5,0.455,0.465,0.200,0.233,"{'has_concept': 0.05000000074505806, 'log_n_co..."
2,none,2,0.2,0.5,0.450,0.461,0.200,0.222,"{'has_concept': 0.057999998331069946, 'log_n_c..."
3,random_edges,0,0.2,0.5,0.458,0.467,0.201,0.215,"{'has_concept': 0.06300000101327896, 'log_n_co..."
4,random_edges,1,0.2,0.5,0.448,0.460,0.199,0.228,"{'has_concept': 0.09300000220537186, 'log_n_co..."
5,random_edges,2,0.2,0.5,0.449,0.457,0.198,0.233,"{'has_concept': 0.07500000298023224, 'log_n_co..."
6,no_relations,0,0.2,0.5,0.453,0.458,0.201,0.214,"{'has_concept': 0.061000000685453415, 'log_n_c..."
7,no_relations,1,0.2,0.5,0.455,0.459,0.200,0.213,"{'has_concept': 0.057999998331069946, 'log_n_c..."
8,no_relations,2,0.2,0.5,0.459,0.465,0.203,0.217,"{'has_concept': 0.06499999761581421, 'log_n_co..."


,group,n,MRR_a,MRR_b,diff,ci_lo,ci_hi,p_a_not_better,a_wins,b_wins,a,b
0,all,1810,0.3803,0.3706,0.0097,0.0065,0.0130,0.0000,427,256,none+adaptive,none
1,direct,1626,0.3957,0.3889,0.0068,0.0038,0.0097,0.0000,355,222,none+adaptive,none
2,indirect,100,0.2179,0.1824,0.0355,0.0139,0.0595,0.0004,32,20,none+adaptive,none


In [49]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
# =====================================================================================
# LIÊN KẾT TRUY VẤN → NODE BẰNG NGỮ NGHĨA (CLIP-text), KHÔNG DÙNG TỪ ĐIỂN VIẾT TAY
# τ chọn trên VALIDATION. So sánh: dictionary (3 danh sách) | exact (khớp chuỗi) | semantic@τ
# =====================================================================================
import pandas as pd

class SemanticLinker:
    """CLIP-text đóng băng; cache embedding theo chuỗi để mọi lần dựng đồ thị dùng lại."""
    def __init__(self):
        import open_clip
        m, _, _ = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)
        self.m = m.to(DEVICE).eval()
        for p in self.m.parameters(): p.requires_grad_(False)
        self.tok = open_clip.get_tokenizer(CFG.clip_model)
        self.cache = {}
        self.dim = self.embed(["x"]).shape[1]

    @torch.no_grad()
    def embed(self, texts):
        new = [t for t in dict.fromkeys(texts) if t not in self.cache]
        for s in range(0, len(new), 1024):
            b = new[s:s + 1024]
            f = F.normalize(self.m.encode_text(self.tok(b).to(DEVICE)), dim=-1).float().cpu().numpy()
            self.cache.update(zip(b, f))
        if not texts:
            return np.zeros((0, self.dim), np.float32)
        return np.stack([self.cache[t] for t in texts])


if "LINKER" not in globals():
    LINKER = SemanticLinker()


def query_ngrams(text):
    toks = [t for t in _norm(text).split() if t not in STOP and len(t) > 1]
    return [" ".join(toks[i:i + n]) for n in (2, 1) for i in range(len(toks) - n + 1)]


def _node_index(graph):
    """Embedding tên node, gắn vào chính đồ thị (mỗi ablation có tập node khác nhau)."""
    if not hasattr(graph, "_sem"):
        names = sorted(graph.name2id)
        graph._sem = (names, np.array([graph.name2id[n] for n in names]), LINKER.embed(names))
    return graph._sem


def make_semantic_parse(tau):
    """Mỗi n-gram nối tới node gần nhất nếu cosine ≥ tau. Trả về (node_id, tên node, n-gram)."""
    def parse(text, graph):
        grams = query_ngrams(text)
        if not grams:
            return []
        names, ids, E = _node_index(graph)
        S = LINKER.embed(grams) @ E.T
        best = S.argmax(1)
        sc = S[np.arange(len(grams)), best]
        hits, seen = [], set()
        for g in np.argsort(-sc):
            if sc[g] < tau:
                break
            nid = int(ids[best[g]])
            if nid not in seen:
                seen.add(nid); hits.append((nid, names[best[g]], grams[g]))
        return hits
    parse.tau = tau
    return parse


# Mã hóa trước mọi n-gram của toàn bộ truy vấn (theo batch) để vòng thí nghiệm chạy nhanh
LINKER.embed(sorted({g for r in ag_records for g in query_ngrams(r["caption"])}))
_AG_COMPOSITES = {o for r in ag_records for o in r["objects"] if " " in o}   # node ghép 'động từ vật'


def val_score(parse_fn, tag, seed=0):
    """Huấn luyện 1 seed với cách liên kết cho trước, chỉ đo trên VAL (không chạm test)."""
    with use_dataset(AG_OUT / "semantic_tune" / tag, parse_query=parse_fn, make_split=ag_make_split):
        _GROUP_CACHE.clear()
        g, eng, parts = prepare(ag_records, ag_split, ag_img, ag_txt, "none", seed)
        model, _ = train_graph_scorer(eng, parts["train"]["q_idx"], parts["train"]["img_idx"],
                                      seed=seed, verbose=False)
        (a, l), _ = tune_on_val(model, eng, parts["val"], parts["val"]["img_idx"], verbose=False)
        cand, sc, sg, sp_ = score_queries(model, eng, parts["val"]["q_idx"], parts["val"]["img_idx"])
        m = retrieval_metrics(cand, combine(sc, sg, sp_, a, l), parts["val"]["gold_pos"])
        clip_m = retrieval_metrics(cand, sc, parts["val"]["gold_pos"])
        vq = parts["val"]["q_idx"]
        grp = Counter(parts["val"]["groups"])
    return dict(method=tag, val_MRR=m["MRR"], val_R1=m["R@1"], val_MRR_CLIP=clip_m["MRR"],
                alpha=a, lam=l,
                coverage=float(np.mean([len(eng.q_concepts[q]) > 0 for q in vq])),
                rel_coverage=float(np.mean([any(h[1] in _AG_COMPOSITES for h in eng.q_concepts[q])
                                            for q in vq])),
                concepts_per_q=float(np.mean([len(eng.q_concepts[q]) for q in vq])),
                n_direct=grp.get("direct", 0), n_indirect=grp.get("indirect", 0))

### 3.1 Chọn τ trên VALIDATION

### 3.2 Đánh giá (Evaluation)

In [50]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
TAU_GRID = [0.80, 0.84, 0.88, 0.92, 0.96]
rows = [val_score(ag_parse_query, "dictionary"),
        val_score(make_semantic_parse(0.999), "exact")]          # khớp chuỗi thuần, không danh sách
for tau in TAU_GRID:
    rows.append(val_score(make_semantic_parse(tau), f"semantic@{tau:.2f}"))
link_df = pd.DataFrame(rows)
(AG_OUT / "results").mkdir(parents=True, exist_ok=True)
link_df.to_csv(AG_OUT / "results" / "linking_tau_val.csv", index=False)
display(link_df.round(4))

sem_rows = link_df[link_df.method.str.startswith("semantic")]
BEST_TAU = float(sem_rows.loc[sem_rows.val_MRR.idxmax(), "method"].split("@")[1])
best_parse = make_semantic_parse(BEST_TAU)
print(f"τ tốt nhất trên val: {BEST_TAU}")

In [51]:
ART_AG = dict(ART_BY_ABL)
ag_gdf = explanation_grounding(ART_AG, ag_records, ag_split)
ag_gtab = grounding_table(ag_gdf)
ag_gdf.to_csv(AG_OUT / "results" / "explanation_grounding_per_query.csv", index=False)
ag_gtab.to_csv(AG_OUT / "results" / "explanation_grounding.csv", index=False)
display(ag_gtab)

Nhãn tham chiếu: 'objects' của 2976 ảnh VAL+TEST | top_m=5


,ablation,group,n_queries,rel_paths_per_q,edge_ok,path_ok,rand_path_ok,selection_lift,grounded,mean_npmi
0,none,all,1364,28.3,"0.999 [0.997, 1.000]","0.999 [0.997, 1.000]","0.989 [0.986, 0.992]","0.010 [0.008, 0.012]","0.845 [0.828, 0.861]",0.743
1,random_edges,all,1115,3.0,"0.154 [0.141, 0.169]","0.093 [0.081, 0.107]","0.091 [0.079, 0.105]","0.002 [-0.000, 0.004]","0.600 [0.579, 0.622]",-0.467
2,none,direct,1264,29.0,"0.999 [0.997, 1.000]","0.999 [0.997, 1.000]","0.988 [0.985, 0.991]","0.010 [0.008, 0.013]","0.851 [0.834, 0.868]",0.748
3,random_edges,direct,1033,3.1,"0.152 [0.137, 0.167]","0.091 [0.078, 0.105]","0.089 [0.076, 0.102]","0.002 [-0.000, 0.004]","0.599 [0.578, 0.621]",-0.475
4,none,indirect,100,19.8,"1.000 [1.000, 1.000]","1.000 [1.000, 1.000]","0.999 [0.997, 1.000]","0.001 [0.000, 0.003]","0.766 [0.686, 0.836]",0.683
5,random_edges,indirect,71,3.0,"0.200 [0.144, 0.259]","0.124 [0.072, 0.183]","0.128 [0.075, 0.186]","-0.003 [-0.009, -0.000]","0.625 [0.545, 0.699]",-0.408


In [52]:
# =====================================================================================
# ABLATION ĐÚNG NGHĨA CHO AG: bỏ NODE QUAN HỆ người–vật ('hold cup', 'sit sofa'...).
# Trên AG, 'no_relations' KHÔNG bỏ quan hệ: các bộ ba (người, holding, cup) đã được "biên dịch"
# thành node ghép và nối vào video bằng cạnh contains (P0). 'no_relations' chỉ bỏ cạnh
# vật–vật (afford/related/isa). Muốn hỏi "tri thức quan hệ có giúp không" thì phải bỏ chính
# các node ghép này: đồ thị chỉ còn video –chứa→ lớp đồ vật.
# Chạy NGAY sau cell 54 (cần ag_records, ag_split, ag_img, ag_txt, PQ_AG).
# =====================================================================================
import copy
ag_records_nc = copy.deepcopy(ag_records)
for r in ag_records_nc:
    r["objects"] = [o for o in r["objects"] if " " not in o]     # bỏ node ghép 'động từ vật'
    r["relations"] = []
log(f"no_composite: {np.mean([len(r['objects']) for r in ag_records_nc]):.1f} node/video "
    f"(đầy đủ: {np.mean([len(r['objects']) for r in ag_records]):.1f})")

with use_dataset(AG_OUT / "no_composite", parse_query=ag_parse_query, make_split=ag_make_split,
                 prepare=_prepare_logged, train_graph_scorer=_train_logged,
                 tune_on_val=_tune_logged, evaluate_split=_eval_logged):
    _N_ABL = 1; _PROG.update(abl=None, i_abl=0, i_seed=0)
    REL_GAL_KEY, REL_RENAME = "ag", {"none": "no_composite"}      # cùng gallery quan hệ khó, bỏ node quan hệ
    nc_res, _ = run_experiment(ag_records_nc, ag_split, ag_img, ag_txt, ablations=("none",))
    REL_GAL_KEY, REL_RENAME = None, {}
    nc_df = results_table(nc_res).assign(ablation="no_composite")
    PQ_NC = {("no_composite", s): v for (ab, s), v in PER_QUERY.items() if ab == "none"}
_N_ABL = len(CFG.ablations)

R_AG = AG_OUT / "results"
ag_full = pd.concat([ag_df, nc_df])
ag_full.to_csv(R_AG / "main_results_with_no_composite.csv", index=False)
display(ag_full[ag_full.model.str.startswith("CLIP+Graph")][["ablation", "model", "R@1", "R@5", "MRR"]])

PQ_ALL = {**PQ_AG, **PQ_NC}           # nhóm direct/indirect lấy theo đồ thị ĐẦY ĐỦ ('none')
sig_ag = pd.concat([paired_test(PQ_ALL, "none", b)
                    for b in ("no_composite", "no_relations", "random_edges", "no_scene",
                              "no_related", "no_concept") if any(k[0] == b for k in PQ_ALL)])
sig_ag.to_csv(R_AG / "paired_ablation_test.csv", index=False)
display(sig_ag.round(4))

[61:37 | +  8.6s] no_composite: 3.4 node/video (đầy đủ: 11.3)

ABLATION = none
[61:37 | +  0.0s] ▶ [ablation 1/1 'none' | seed 1/3] dựng đồ thị + bộ truy vấn
WordNet(+fallback): 31/35 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 8784, 'other': 639, 'indirect': 164}
[61:45 | +  7.8s] ✔ [ablation 1/1 'none' | seed 1/3] dựng đồ thị + bộ truy vấn (7.8s)
[61:45 | +  0.0s]     nodes=9666 | scene=0 related=104 isa=54 | đường đi P1..P5=234426
Đồ thị: {'nodes': 9666, 'images': 9587, 'object_classes': 35, 'concepts': 44, 'edges': 32973, 'contains': 32815, 'scene': 0, 'related': 104, 'isa': 54, 'triple_nodes': 0, 'triple_edges': 0, 'paths': {'P0_contains': 32815, 'P1_scene': 0, 'P2_isa': 50771, 'P3_related': 66878, 'P4_scene_isa': 0, 'P5_related_isa': 116777, 'P6_triple': 0}, 'ablation': 'none'}
[61:45 | +  0.0s] ▶   huấn luyện GNN seed=0 (20 epoch, 6611 truy vấn)
  cạnh dùng khi huấn luyện: 43198/65946 (đã bỏ 22748 cạnh chạm ảnh val/test)
  epoch  0 | loss 44.1197 | w_path 1.000 0.16

,ablation,model,R@1,R@5,MRR
1,none,CLIP+Graph,0.2759 ± 0.0144,0.4908 ± 0.0138,0.3706 ± 0.0128
3,none,CLIP+Graph [direct] (n=1626),0.2925 ± 0.0145,0.5146 ± 0.0118,0.3889 ± 0.0118
5,none,CLIP+Graph [indirect] (n=100),0.0967 ± 0.0094,0.2633 ± 0.0170,0.1824 ± 0.0131
6,none,CLIP+Graph (adaptive),0.2851 ± 0.0082,0.5013 ± 0.0025,0.3803 ± 0.0056
7,none,CLIP+Graph (adaptive) [direct] (n=1626),0.3003 ± 0.0093,0.5189 ± 0.0038,0.3957 ± 0.0060
8,none,CLIP+Graph (adaptive) [indirect] (n=100),0.1233 ± 0.0094,0.3100 ± 0.0141,0.2179 ± 0.0036
10,random_edges,CLIP+Graph,0.2856 ± 0.0021,0.4930 ± 0.0017,0.3782 ± 0.0012
12,random_edges,CLIP+Graph [direct] (n=1626),0.3044 ± 0.0039,0.5172 ± 0.0027,0.3982 ± 0.0024
14,random_edges,CLIP+Graph [indirect] (n=100),0.0700 ± 0.0163,0.2133 ± 0.0047,0.1484 ± 0.0088
15,random_edges,CLIP+Graph (adaptive),0.2910 ± 0.0043,0.4996 ± 0.0011,0.3845 ± 0.0028


,group,n,MRR_a,MRR_b,diff,ci_lo,ci_hi,p_a_not_better,a_wins,b_wins,a,b
0,all,1810,0.3706,0.3659,0.0047,-0.0031,0.0124,0.1226,514,484,none,no_composite
1,direct,1626,0.3889,0.3849,0.0040,-0.0044,0.0123,0.1826,440,444,none,no_composite
2,indirect,100,0.1824,0.1780,0.0044,-0.0293,0.0353,0.3806,38,20,none,no_composite
0,all,1810,0.3706,0.3815,-0.0109,-0.0181,-0.0036,0.9990,441,514,none,no_relations
1,direct,1626,0.3889,0.3989,-0.0100,-0.0177,-0.0022,0.9944,380,461,none,no_relations
2,indirect,100,0.1824,0.2157,-0.0333,-0.0764,0.0082,0.9394,32,26,none,no_relations
0,all,1810,0.3706,0.3782,-0.0076,-0.0145,-0.0006,0.9837,454,497,none,random_edges
1,direct,1626,0.3889,0.3982,-0.0093,-0.0167,-0.0018,0.9939,390,447,none,random_edges
2,indirect,100,0.1824,0.1484,0.0340,0.0049,0.0660,0.0112,43,18,none,random_edges


In [53]:
# v5 — BỘ QUAN HỆ KHÓ trên AG: cùng vật, khác quan hệ người–vật (gallery cố định, α/λ từ VAL)
rel_tab_ag, rel_sig_ag = rel_probe_report("ag", base="none",
                                          others=["CLIP", "no_composite", "no_relations", "random_edges"],
                                          out_csv=R_AG / "relational_hard_negatives.csv")
display(rel_tab_ag); display(rel_sig_ag.round(4))


,model,n_queries,seeds,MRR,R1
0,none,548,3,0.8644 ± 0.0230,0.7908
1,CLIP,548,3,0.6824 ± 0.0000,0.5511
2,random_edges,548,3,0.8621 ± 0.0059,0.7950
3,no_relations,548,3,0.8871 ± 0.0099,0.8266
4,no_composite,548,3,0.7739 ± 0.0424,0.6624


,group,n,MRR_a,MRR_b,diff,ci_lo,ci_hi,p_a_not_better,a_wins,b_wins,a,b
0,all,548,0.8644,0.6824,0.1820,0.1472,0.2170,0.0000,222,106,none,CLIP
0,all,548,0.8644,0.7739,0.0906,0.0711,0.1105,0.0000,208,71,none,no_composite
0,all,548,0.8644,0.8871,-0.0226,-0.0384,-0.0072,0.9980,70,121,none,no_relations
0,all,548,0.8644,0.8621,0.0023,-0.0136,0.0180,0.3844,87,105,none,random_edges


In [54]:
# v6 — ĐỐI CHỨNG composite_swap: gán nhầm quan hệ, giữ nguyên vật (xem mô tả ở cell định nghĩa đầu Phần 3)
_ag_vo()
_rng_sw = np.random.RandomState(0)
ag_records_sw = copy.deepcopy(ag_records)
_n_sw = _n_drop = 0
for r in ag_records_sw:
    comps = [c for c in r["objects"] if " " in c]
    plain = [c for c in r["objects"] if " " not in c]
    new = set()
    for c in comps:
        v, o = c.split(" ", 1)
        alts = sorted(x for x in _ag_vo()[o] if x != v and f"{x} {o}" not in comps and f"{x} {o}" not in new)
        if alts:
            new.add(f"{alts[_rng_sw.randint(len(alts))]} {o}"); _n_sw += 1
        else:
            _n_drop += 1                         # không có vị từ thay thế ⇒ bỏ (không giữ quan hệ thật)
    r["objects"] = sorted(set(plain) | new)
    r["relations"] = sorted((c.split(" ", 1)[1], "afford", c) for c in new)
log(f"composite_swap: đổi {_n_sw} node quan hệ, bỏ {_n_drop} (vật không có vị từ thay thế) | "
    f"{np.mean([len(r['objects']) for r in ag_records_sw]):.1f} node/video (gốc "
    f"{np.mean([len(r['objects']) for r in ag_records]):.1f})")


# v6 — chạy đối chứng composite_swap (3 seed) + đánh giá phản thực tế cho mọi cấu hình đã có
with use_dataset(AG_OUT / "composite_swap", parse_query=ag_parse_query, make_split=ag_make_split,
                 prepare=_prepare_logged, train_graph_scorer=_train_logged,
                 tune_on_val=_tune_logged, evaluate_split=_eval_logged):
    _N_ABL = 1; _PROG.update(abl=None, i_abl=0, i_seed=0)
    REL_GAL_KEY, REL_RENAME = "ag", {"none": "composite_swap"}
    sw_res, _ = run_experiment(ag_records_sw, ag_split, ag_img, ag_txt, ablations=("none",))
    REL_GAL_KEY, REL_RENAME = None, {}
    sw_df = results_table(sw_res).assign(ablation="composite_swap")
    PQ_SW = {("composite_swap", s): v for (ab, s), v in PER_QUERY.items() if ab == "none"}
_N_ABL = len(CFG.ablations)

R_AG = AG_OUT / "results"
pd.concat([ag_full, sw_df]).to_csv(R_AG / "main_results_with_controls.csv", index=False)
sig_sw = paired_test({**PQ_AG, **PQ_SW}, "none", "composite_swap")
sig_sw.to_csv(R_AG / "paired_none_vs_composite_swap.csv", index=False)
display(sig_sw.round(4))

rel_tab_ag, rel_sig_ag = rel_probe_report("ag", base="none",
                                          others=["CLIP", "no_composite", "composite_swap", "no_relations", "random_edges"],
                                          out_csv=R_AG / "relational_hard_negatives.csv")
print("Bộ quan hệ khó (AG):"); display(rel_tab_ag); display(rel_sig_ag.round(4))

cf_tab, cf_sig = cf_report("ag")
cf_tab.to_csv(R_AG / "counterfactual_verb_swap.csv", index=False)
cf_sig.to_csv(R_AG / "counterfactual_verb_swap_paired.csv", index=False)
print("Truy vấn phản thực tế (đổi vị từ): demoted = % truy vấn mà video đúng bị hạ hạng; "
      "delta_RR = RR(gốc) − RR(đổi vị từ), càng lớn càng nhạy với quan hệ")
display(cf_tab.round(4)); display(cf_sig.round(4))


[69:52 | + 13.2s] composite_swap: đổi 65476 node quan hệ, bỏ 10020 (vật không có vị từ thay thế) | 10.3 node/video (gốc 11.3)

ABLATION = none
[69:52 | +  0.0s] ▶ [ablation 1/1 'none' | seed 1/3] dựng đồ thị + bộ truy vấn
WordNet(+fallback): 224/228 lớp đối tượng có khái niệm cha
Nhóm truy vấn: {'direct': 8784, 'other': 317, 'indirect': 486}
[70:09 | + 17.1s] ✔ [ablation 1/1 'none' | seed 1/3] dựng đồ thị + bộ truy vấn (17.1s)
[70:09 | +  0.0s]     nodes=9873 | scene=193 related=1760 isa=247 | đường đi P1..P5=742365
Đồ thị: {'nodes': 9873, 'images': 9587, 'object_classes': 228, 'concepts': 58, 'edges': 100442, 'contains': 98242, 'scene': 193, 'related': 1760, 'isa': 247, 'triple_nodes': 0, 'triple_edges': 0, 'paths': {'P0_contains': 98242, 'P1_scene': 122811, 'P2_isa': 96562, 'P3_related': 253225, 'P4_scene_isa': 83396, 'P5_related_isa': 186371, 'P6_triple': 0}, 'ablation': 'none'}
[70:09 | +  0.0s] ▶   huấn luyện GNN seed=0 (20 epoch, 6611 truy vấn)
  cạnh dùng khi huấn luyện: 132808/

,group,n,MRR_a,MRR_b,diff,ci_lo,ci_hi,p_a_not_better,a_wins,b_wins,a,b
0,all,1810,0.3706,0.3687,0.0019,-0.0062,0.0101,0.3163,481,489,none,composite_swap
1,direct,1626,0.3889,0.3876,0.0013,-0.0073,0.0101,0.3845,417,436,none,composite_swap
2,indirect,100,0.1824,0.1791,0.0033,-0.0352,0.0454,0.4450,36,25,none,composite_swap


Bộ quan hệ khó (AG):


,model,n_queries,seeds,MRR,R1
0,none,548,3,0.8644 ± 0.0230,0.7908
1,CLIP,548,3,0.6824 ± 0.0000,0.5511
2,random_edges,548,3,0.8621 ± 0.0059,0.7950
3,no_relations,548,3,0.8871 ± 0.0099,0.8266
4,no_composite,548,3,0.7739 ± 0.0424,0.6624
5,composite_swap,548,3,0.7634 ± 0.0053,0.6521


,group,n,MRR_a,MRR_b,diff,ci_lo,ci_hi,p_a_not_better,a_wins,b_wins,a,b
0,all,548,0.8644,0.6824,0.1820,0.1472,0.2170,0.0000,222,106,none,CLIP
0,all,548,0.8644,0.7739,0.0906,0.0711,0.1105,0.0000,208,71,none,no_composite
0,all,548,0.8644,0.7634,0.1010,0.0791,0.1239,0.0000,185,77,none,composite_swap
0,all,548,0.8644,0.8871,-0.0226,-0.0384,-0.0072,0.9980,70,121,none,no_relations
0,all,548,0.8644,0.8621,0.0023,-0.0136,0.0180,0.3844,87,105,none,random_edges


Truy vấn phản thực tế (đổi vị từ): demoted = % truy vấn mà video đúng bị hạ hạng; delta_RR = RR(gốc) − RR(đổi vị từ), càng lớn càng nhạy với quan hệ


,model,n,seeds,demoted,promoted,delta_RR
0,none,545,3,0.3737,0.0422,0.1879
1,CLIP,545,3,0.2477,0.0881,0.0524
2,random_edges,545,3,0.3988,0.0569,0.2157
3,no_relations,545,3,0.3835,0.0318,0.2198
4,no_composite,545,3,0.1382,0.0648,0.0267
5,composite_swap,545,3,0.2061,0.1645,0.0150


,a,b,n,diff_delta_RR,ci_lo,ci_hi
0,none,CLIP,545,0.1356,0.1096,0.1625
1,none,random_edges,545,-0.0277,-0.0510,-0.0045
2,none,no_relations,545,-0.0319,-0.0539,-0.0097
3,none,no_composite,545,0.1613,0.1398,0.1832
4,none,composite_swap,545,0.1729,0.1426,0.2041


### 3.3 Ví dụ liên kết (đọc tay để kiểm tra chất lượng)

In [55]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
_g = MultimodalGraph(ag_records, ag_split["train"])
names, ids, E = _node_index(_g)
for iid in ag_split["val"][:8]:
    text = ag_records[iid]["caption"]
    print(f"\n{text[:90]}")
    for nid, node, gram in best_parse(text, _g):
        cos = float(LINKER.embed([gram])[0] @ E[names.index(node)])
        print(f"   '{gram}' → {node}   (cos {cos:.3f})")

### 3.4 Chạy đầy đủ với τ tốt nhất: 3 ablation × 3 seed

In [56]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
with use_dataset(AG_OUT / "semantic", parse_query=best_parse, make_split=ag_make_split):
    ag_sem_all_res, ag_sem_art = run_experiment(ag_records, ag_split, ag_img, ag_txt,
                                                ablations=("none", "random_edges", "no_relations"))
    ag_sem_df = results_table(ag_sem_all_res)
    ag_sem_df.to_csv(CFG.out_dir / "results" / "main_results.csv", index=False)
    ag_sem_fdf, ag_sem_summ = fidelity_study(ag_sem_art)
    with open(CFG.out_dir / "results" / "xai_summary.json", "w") as f:
        json.dump(ag_sem_summ, f, indent=2)
    if "final_rank_fidelity" in globals():
        ag_sem_rdf, ag_sem_rsumm = final_rank_fidelity(ag_sem_art)
        ag_sem_rdf.to_csv(CFG.out_dir / "results" / "fidelity_final_rank.csv", index=False)
    demo_query("a person drinking from a glass while sitting on the couch", ag_sem_art, k=3)

### 3.5 So sánh dictionary vs semantic trên TEST (cùng split)

In [57]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
cmp_ = pd.concat([ag_df.assign(linking="dictionary"),
                  ag_sem_df.assign(linking=f"semantic@{BEST_TAU}")])
cmp_.to_csv(AG_OUT / "results" / "linking_comparison_test.csv", index=False)
display(cmp_[cmp_.ablation == "none"][["linking", "model", "R@1", "R@5", "MRR", "nDCG@10"]])

In [58]:
%%script true
# v6: không chạy lại ở bản này — kết quả xem ở version 2 của notebook.
# Nhiệm vụ 2 — GNN của SCENIR trên dataset thứ hai (Action Genome)
with use_dataset(AG_OUT / "scenir", parse_query=ag_parse_query, make_split=ag_make_split), use_gnn("scenir"):
    ag_sc_res, ag_sc_art = run_experiment(ag_records, ag_split, ag_img, ag_txt,
                                          ablations=("none", "random_edges", "no_relations"))
    ag_sc_df = results_table(ag_sc_res)
    ag_sc_df.to_csv(CFG.out_dir / "results" / "main_results.csv", index=False)

ag_cmp_gnn = pd.concat([ag_df.assign(gnn=CFG.gnn_type), ag_sc_df.assign(gnn="scenir")])
ag_cmp_gnn.to_csv(AG_OUT / "results" / "gnn_classic_vs_scenir.csv", index=False)
display(ag_cmp_gnn[["gnn", "ablation", "model", "R@1", "R@5", "R@10", "MRR", "nDCG@10"]])

---
# PHẦN 4 — Cấp độ 4(b): truy vấn văn bản → video (MSVD + MSR-VTT)

> **Gộp từ `project03_level4b.ipynb`.** Phần này là một pipeline độc lập với Phần 1–3: nó **định nghĩa lại** `CFG`, `make_split`, `scatter_sum`, `znorm_per_row`… nên **chỉ chạy sau khi Phần 1–3 đã chạy xong** (hoặc khởi động lại runtime rồi chạy riêng từ cell `import` đầu tiên của Phần 4). Không chạy lại các cell của Phần 1–3 sau khi đã chạy Phần 4.


> **v5:** phần này không chạy lại trong bản này (mọi cell code có `%%script true`). Xoá dòng đó ở đầu cell để chạy.

### Cấp độ 4(b) — Truy vấn văn bản → video có xếp hạng lại và giải thích bằng đồ thị

**Phạm vi.** Notebook này chỉ làm **Level 4(b): mở rộng text-to-video** của hệ thống truy vấn có đồ thị. Không đề xuất thuật toán mới (Level 4(a)), không khảo sát người dùng (Level 4(c)), không làm image-to-text. Phần nền tảng text→image nằm ở `image/` (notebook của nhóm, Flickr8k + Action Genome).

**Pipeline** (mục 3.3 của master instruction):
1. truy vấn văn bản;
2. CLIP text encoder đóng băng;
3. lấy mẫu 12 khung hình / video;
4. CLIP image encoder đóng băng;
5. gộp khung hình thành embedding video (cách gộp chọn trên VAL);
6. top-K = 50 video ứng viên theo cosine;
7. xếp hạng lại bằng điểm đồ thị;
8. trả top-K cùng giải thích là các số hạng đồ thị đã thực sự cộng vào điểm.

**Mô hình đồ thị được tái sử dụng, không phải đóng góp mới:**
* **QC-RGAT** (v4) là mô hình chính vì điểm của nó tách chính xác thành các số hạng.
* **GraphSAGE có kiểu cạnh** (mô hình cuối của v3) là baseline GNN đã học.
* Cả hai được đánh giá, **không mặc định** cái nào hơn CLIP-only hay chỉ-contains.

**Dữ liệu và trạng thái từng benchmark:**

| Tập | Vai trò | Đã nhìn trước | Trạng thái kết quả |
|---|---|---|---|
| MSVD train (1200 video) | dựng đồ thị, huấn luyện GNN | — | — |
| MSVD val (100 video) | chọn cấu hình | — | phát triển |
| MSVD test (670 video) | đánh giá | 4 lần (v1–v4) | **EXPLORATORY** |
| MSR-VTT 1k-A | — | 1 lần (v3) | không dùng |
| MSR-VTT fresh1k (seed 20261025) | — | 1 lần (v4) | không dùng (đặt `MSVD_V4_CONFIRM_SET=fresh1k` thì chạy lại, ghi EXPLORATORY) |
| **MSR-VTT fresh1k-B** (seed 20261026) | **xác nhận**, đánh giá **một lần** sau `freeze()` | 0 | **CONFIRMATORY** |

fresh1k-B gồm 1000 video train-9k, rời 1k-A, rời 200 video dev và rời fresh1k. Nó cùng miền với fresh1k, mà số liệu fresh1k đã ảnh hưởng tới thiết kế notebook này; điều đó được khai báo trong báo cáo.

**Phần thêm so với v4** (các ô "Level 4(b)"):
* Ablation **huấn luyện lại** theo họ quan hệ:
  - G7: bỏ cạnh vai `agent_of`/`object_of`;
  - G8: bỏ cạnh `svo`/`prep`.
* Bộ truy vấn **trực tiếp / gián tiếp ≥ 20 / nhóm**: chọn theo luật đăng ký trước, dựng **trước** khi chấm mô hình, báo metric riêng.
* xAI theo **thứ hạng**: giải thích là các số hạng đồ thị ủng hộ (contains và đường đi quan hệ). Xoá chúng rồi đo Δ điểm, Δ hạng, top-1 và top-10, so với xoá ngẫu nhiên.
* Phân tích **5 thành công + 5 thất bại**: chọn theo luật, phân tầng nguyên nhân.
* **Demo**: nhập văn bản tự do, nhận top-K với điểm CLIP, điểm đồ thị, đường đi giải thích và thumbnail.
* `data_inventory.json`, `data_leakage_audit.md`, `audit_report.md`, `level4b_conclusions.md` và `artifacts_manifest.json`, **tự sinh** từ chính lần chạy.

**Chạy trên Kaggle:**
1. Settings: GPU T4, **Internet On**.
2. *(Khuyến nghị)* Add Input output của lần chạy v4 để dùng lại cache đặc trưng MSVD.
3. Save & Run All.
4. Tải `/kaggle/working/project03_level4b_review.zip`.

**Biến môi trường:**
* `MSVD_V4_STAGES` (mặc định `V1,V7`);
* `MSVD_L4_SEEDS` (mặc định `0,1,2,3,4`);
* `MSVD_L4_EPOCHS`;
* `MSVD_V4_CONFIRM_SET`: `fresh1kB` (mặc định), `fresh1k` hoặc `1kA`. Hai giá trị sau ⇒ EXPLORATORY.

**Nguyên tắc.**
* Test và tập ngoài chỉ được mở sau `freeze()`.
* Caption của test và tập ngoài không bao giờ dùng để dựng từ vựng hay quan hệ, huấn luyện hoặc chọn cấu hình.
* PASS của kiểm thử phần mềm **không** phải kết luận khoa học.
* Thí nghiệm không chạy được thì ghi `NOT RUN` kèm lý do.


In [59]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
import os, sys, json, math, time, random, subprocess, hashlib, itertools, glob, re, io, socket
import shutil, zipfile, tempfile, platform, copy, threading, contextlib
from pathlib import Path
from collections import defaultdict, Counter
from contextlib import contextmanager

import numpy as np

RUN_ON_KAGGLE = Path("/kaggle/input").exists()
NB_PID = os.getpid()
try:
    _EXEC_COUNT_START = get_ipython().execution_count      # = 1 khi chạy từ kernel mới (Restart & Run All)
except NameError:
    _EXEC_COUNT_START = None
STAGES = []


def mark_stage(name, requires=()):
    """Ghi thứ tự chạy các cell chính. Cell chạy lệch thứ tự (hoặc dựa vào biến sót từ phiên trước)
    dừng ngay với thông báo rõ ràng thay vì cho ra số liệu sai."""
    done = {s["name"] for s in STAGES}
    missing = [r for r in requires if r not in done]
    assert not missing, f"Cell '{name}' phải chạy sau {missing} — hãy Restart & Run All."
    assert os.getpid() == NB_PID, "PID kernel đã đổi giữa chừng — hãy Restart & Run All."
    STAGES.append(dict(name=name, t=time.time(), pid=os.getpid()))


def has_internet(host="huggingface.co", port=443, timeout=5.0):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


INTERNET = has_internet()
mark_stage("env")
print(f"Kaggle: {RUN_ON_KAGGLE} | Internet: {INTERNET} | Python {platform.python_version()} | "
      f"execution_count đầu = {_EXEC_COUNT_START}")
if not INTERNET:
    print("⚠ Không có Internet: chỉ chạy được nếu open_clip, trọng số CLIP, dữ liệu NLTK và MSVD đã có sẵn "
          "(Add Input). Bật: Kaggle → Settings → Internet → On.")

In [60]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# Phụ thuộc tối thiểu. Kaggle đã có torch/torchvision/opencv/nltk/pandas/scipy; chỉ cài open_clip khi thiếu.
# Cài thất bại thì DỪNG kèm log (không bỏ qua lỗi cài đặt).
def ensure_package(module, pip_spec):
    import importlib
    try:
        return importlib.import_module(module)
    except ImportError:
        pass
    if not INTERNET:
        raise RuntimeError(f"Thiếu gói '{pip_spec}' và không có Internet — bật Internet hoặc Add Input wheel.")
    print(f"→ pip install {pip_spec}")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_spec], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"pip install {pip_spec} thất bại (mã {r.returncode}) — xem log phía trên.")
    importlib.invalidate_caches()
    return importlib.import_module(module)


cv2 = ensure_package("cv2", "opencv-python-headless")
open_clip = ensure_package("open_clip", "open_clip_torch==3.2.0")
nltk = ensure_package("nltk", "nltk")

import scipy
import scipy.sparse as sp
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENV_INFO = dict(python=platform.python_version(), platform=platform.platform(), torch=torch.__version__,
                cuda=torch.version.cuda, device=DEVICE, open_clip=getattr(open_clip, "__version__", "?"),
                opencv=cv2.__version__, numpy=np.__version__, scipy=scipy.__version__, pandas=pd.__version__,
                nltk=nltk.__version__, kaggle=RUN_ON_KAGGLE, internet=INTERNET)
if DEVICE == "cuda":
    ENV_INFO["gpu"] = torch.cuda.get_device_name(0)
    print(f"GPU: {ENV_INFO['gpu']} | {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ KHÔNG có GPU: trích đặc trưng ViT-L-14 cho ~23,6k frame + 80,8k caption trên CPU rất chậm (hàng giờ).\n"
          "  Kaggle: Settings → Accelerator → GPU T4/P100. Chạy thử trên CPU: MSVD_L4_CLIP=ViT-B-32, "
          "MSVD_L4_CLIP_W=laion2b_s34b_b79k.")
print(json.dumps(ENV_INFO, indent=1))
mark_stage("deps", requires=("env",))

## 4.1 Cấu hình

Mọi siêu tham số nằm ở cell dưới. Phần dữ liệu, CLIP, lấy mẫu frame và tagger giữ nguyên như v1/v2, nên cache đặc trưng MSVD của v2 dùng lại được. Các khối mới của v3 gồm:
* vị từ: họ có vị từ tự do, prompt, ngưỡng support;
* người chấm: mẫu, ngưỡng precision, file nhãn;
* benchmark xác nhận: MSR-VTT ở commit ghim;
* thống kê: Holm.

In [61]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
_env = os.environ.get


class CFG:
    # ---------------- chế độ chạy ----------------
    # "msvd": dữ liệu thật (mặc định). "synthetic": smoke-test mã bằng video tổng hợp — phải chọn TƯỜNG MINH.
    # Notebook KHÔNG bao giờ tự rơi về synthetic khi thiếu MSVD: chỉ có caption thì không phải text-to-video.
    dataset  = _env("MSVD_L4_DATASET", "msvd")
    # dev_mode: chạy đủ pipeline trên dữ liệu thật nhưng KHOÁ test; phân tích cuối chạy trên val, gắn nhãn DEV.
    dev_mode = _env("MSVD_L4_DEV", "0") == "1"

    # ---------------- dữ liệu MSVD ----------------
    hf_repo      = "friedrichor/MSVD"
    hf_revision  = "9aa95a3039c106ee9e0969d82d7ce5a9efab1196"     # ghim commit ⇒ tái lập được
    msvd_root    = _env("MSVD_ROOT", "")             # thư mục chứa msvd_*.json + YouTubeClips/ hoặc MSVD_Videos.zip
    video_source = _env("MSVD_VIDEO_SOURCE", "auto")  # auto | dir | zip
    expected     = dict(train=(1200, 48774), val=(100, 4290), test=(670, 27763))   # (video, caption) — dataset card
    max_bad_video_frac = 0.0                         # MSVD thật: không chấp nhận video nào giải mã lỗi

    # ---------------- lấy mẫu khung hình ----------------
    n_frames_max   = 12
    frame_policy   = "uniform-segment-midpoint"      # frame k = floor((k + 0.5) / F · n_frame_giải_mã_được)
    n_frames_grid  = [4, 8, 12]                       # chọn trên VAL; F=1 chỉ in ra để tham chiếu
    pool_grid      = ["mean", "centrality"]           # chọn trên VAL
    centrality_tau = 0.1
    decode_workers = 4
    thumb_width    = 160

    # ---------------- CLIP (đóng băng hoàn toàn) ----------------
    clip_model   = _env("MSVD_L4_CLIP", "ViT-L-14")
    clip_weights = _env("MSVD_L4_CLIP_W", "laion2b_s32b_b82k")
    clip_bs      = 128
    text_bs      = 512

    # ---------------- từ vựng (TRAIN) + tagger (chọn trên VAL) ----------------
    vocab_min_caps  = 2        # một từ được tính cho video nếu xuất hiện trong ≥ 2 caption của video đó ...
    min_term_df     = 3        # ... và ở ≥ 3 video TRAIN
    truth_min_caps  = 3        # nhãn tham chiếu (chỉ để chấm F1 của tagger trên VAL)
    tag_norms       = ["raw", "z", "raw+z"]
    tag_pools       = ["video", "frame_max"]
    tag_k_noun_grid = [3, 5, 8, 10, 12]
    tag_k_verb_grid = [1, 2, 3, 5]
    noun_prompts = ["a photo of a {}.", "a video frame showing a {}.", "a photo containing a {}."]
    verb_prompts = ["a photo of someone {}.", "a video frame of a person {}.", "someone is {}."]
    min_tag_freq = 2           # lớp phải được gắn cho ≥ 2 video TRAIN mới thành node
    max_concepts = 4000

    # ---------------- đồ thị (cùng ý nghĩa với notebook gốc) ----------------
    max_path_len = 3
    scene_min_count, scene_top_k = 2, 10
    cooc_min_count, cooc_top_k = 5, 10
    cooc_score, cooc_min_npmi = "npmi", 0.10
    use_wordnet, isa_min_depth = True, 4
    use_idf, idf_from = True, "taxonomy"
    drop_redundant_indirect = True
    edge_weighting = True

    # ---------------- GNN ----------------
    gnn_type, gnn_typed = "sage", True
    gate_init, path_init_indirect, scalar_lr_mult = -1.5, -1.0, 10.0
    hard_neg_k = 31
    train_with_clip = True
    learn_clip_w = False       # chọn lại trên VAL
    gnn_dim, gnn_layers, gnn_dropout = 256, 2, 0.1
    lr, weight_decay, temperature = 1e-3, 1e-4, 0.07
    epochs = int(_env("MSVD_L4_EPOCHS", "15"))
    caption_passes = 4         # mỗi epoch: 4 lượt × (1 caption ngẫu nhiên / video TRAIN), batch không trùng video
    train_bs = 128

    # ---------------- xếp hạng lại ----------------
    rerank_k_grid = [10, 20, 50]                       # chọn trên VAL (cấu hình 'none', seed đầu)
    rerank_k      = None
    alpha_grid    = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    lambda_grid   = [0.0, 0.25, 0.5, 0.75]             # bỏ λ=1: khi α<1, s_path luôn có trọng số > 0
    eval_chunk    = 2048

    # ---------------- thí nghiệm (v2: theo stage, mỗi stage một câu hỏi) ----------------
    seeds  = [int(s) for s in _env("MSVD_L4_SEEDS", "0,1,2,3,4").split(",")]
    stages = [s for s in _env("MSVD_V4_STAGES", "V1,V7").split(",") if s]
    # Luật CHẤP NHẬN một thay đổi (đăng ký trước, chỉ dùng VAL): Δ MRR val trung bình seed ≥ adopt_min_delta
    # VÀ bootstrap theo cụm video trên val cho P(Δ ≤ 0) < adopt_max_p. Không đạt ⇒ giữ thiết kế đang dùng.
    adopt_min_delta, adopt_max_p = 0.002, 0.10
    save_checkpoints = _env("MSVD_G2_CKPT", "1") == "1"

    # ---------------- đồ thị v2 ----------------
    reliable_min_support = 3          # quan hệ 'đáng tin' = được ≥ 3 video TRAIN hỗ trợ (không có nhãn người ⇒ chỉ là lọc theo tần suất)
    tax_hub_frac = 0.15               # taxonomy 'lọc hub': bỏ khái niệm mà > 15% video TRAIN với tới
    tax_min_depth_filtered = 6        # ... và khái niệm danh từ quá nông (độ sâu WordNet < 6)
    sense_max_candidates, sense_margin = 6, 1.0   # chọn sense kiểu Lesk trên ngữ cảnh caption TRAIN
    conf_modes = ["sigmoid", "minmax"]            # chuẩn hoá trọng số contains — chọn trên VAL theo MRR
    conf_floor = 0.05
    rewire_swaps_per_edge = 10        # số lần thử swap / cạnh trong constrained rewiring
    rewire_min_changed = 0.5          # đối chứng hợp lệ ⇔ ≥ 50% cạnh của mỗi loại (≥ 10 cạnh) bị đổi
    n_audit_captions = 200            # mẫu caption TRAIN cho audit quan hệ (để người chấm)

    # ---------------- v3: vị từ (predicate) là thành phần của cạnh ----------------
    # Họ có vị từ TỰ DO: svo (vị từ = cụm động từ), prep (vị từ = giới từ). Ở agent_of / object_of vị từ chính là node
    # động từ đích ⇒ không mang thông tin riêng của cạnh; vẫn ghi vào schema nhưng không đưa embedding vị từ vào message.
    pred_families = ("svo", "prep")
    pred_min_support = 5              # chính sách 'support': (họ, vị từ) xuất hiện ở ≥ 5 video TRAIN
    pred_prompts = {"svo": ["someone {} something.", "a video of a person {} something."],
                    "prep": ["something {} something else.", "a photo of an object {} another object."]}
    pred_adapt_hidden = 128           # phương án B: MLP dư DÙNG CHUNG cho mọi vị từ (không có ID riêng cho từng vị từ)
    # ---------------- chất lượng quan hệ: kiểm chứng TỰ ĐỘNG (mặc định) + nhãn người chấm (TUỲ CHỌN) ----------------
    dep_model = "en_core_web_sm"      # parser phụ thuộc cú pháp độc lập (spaCy) để đối chiếu với parser heuristic
    agree_min = 0.5                   # chính sách 'agreement': giữ (họ, nguồn, vị từ, đích) nếu ≥ 50% caption TRAIN của nó được spaCy xác nhận
    annot_n_per_cell, annot_n_double_per_cell, annot_seed = 30, 10, 20261024
    annot_strata = ((2, 2, "low"), (3, 5, "mid"), (6, 10 ** 9, "high"))   # support = số video TRAIN có quan hệ
    annot_file = _env("MSVD_G3_LABELS", "")                               # trống ⇒ tự tìm relation_annotation*.csv trong /kaggle/input
    prec_min_lower = 0.60             # chính sách 'precision': giữ ô (họ × mức phổ biến) có cận dưới Wilson 95% ≥ 0.60
    # ---------------- benchmark xác nhận ĐỘC LẬP (chưa dùng ở v1/v2) ----------------
    confirm = _env("MSVD_G3_CONFIRM", "msrvtt")                           # "msrvtt" | "none"
    msrvtt_repo = "friedrichor/MSR-VTT"
    msrvtt_revision = "c1af215a96934854f42683c19c51391aaee6f962"
    msrvtt_dev_n = 200                # dev_mode: 200 video TRAIN-9k (KHÔNG thuộc test 1k-A) chỉ để thử đường ống
    msrvtt_root = _env("MSRVTT_ROOT", "")  # thư mục có sẵn msrvtt_*.json / MSRVTT_Videos.zip (trống ⇒ tải từ HF)
    confirm_set = _env("MSVD_V4_CONFIRM_SET", "fresh1kB")  # "fresh1kB" (mặc định, CHƯA nhìn) | "fresh1k" (đã nhìn ở v4) | "1kA" (đã nhìn ở v3)
    fresh_n, fresh_seed = 1000, 20261025  # fresh1k: 1000 video train-9k, ngoài 1k-A và ngoài 200 video dev — chọn ngẫu nhiên cố định
    fresh_b_seed = 20261026           # fresh1k-B (Level 4(b)): 1000 video train-9k KHÁC — rời 1k-A, rời dev, rời fresh1k
    # ---------------- Level 4(b) ----------------
    n_eval_queries = 20               # bộ truy vấn đăng ký trước: số truy vấn / nhóm (trực tiếp, gián tiếp) / tập đánh giá
    eval_query_seed = 20261026
    n_cases = 5                       # 5 thành công + 5 thất bại
    demo_topk = 5
    demo_queries = ["a man is playing a guitar", "a child is playing football on the grass", "a woman is slicing a tomato",
                    "a dog is running on the beach", "two people are dancing on a stage"]   # cố định trước, không chỉnh theo kết quả
    # ---------------- v4: QC-RGAT ----------------
    v4_dim = 64                       # chiều embedding của QC-RGAT
    v4_lr = float(_env("MSVD_V4_LR", "2e-3"))
    v4_score_bs = 256                 # số truy vấn / lô khi chấm điểm
    v4_rerank_k = 50                  # k chọn trên VAL ở v3 (Kaggle), giữ cố định
    # ---------------- thống kê ----------------
    holm_alpha = 0.05                 # Holm–Bonferroni cho họ so sánh chính đăng ký trước

    # ---------------- xAI ----------------
    n_explain_queries, top_m_paths = 20, 5
    n_rank_fidelity, n_edge_deletion = 50, 15
    n_seed_sensitivity = 20

    # ---------------- đường dẫn ----------------
    _base      = Path("/kaggle/working") if RUN_ON_KAGGLE else Path(".")
    out_dir    = Path(_env("MSVD_L4_OUT", str(_base / ("project03_level4b" if dataset == "msvd" else "project03_level4b_synthetic"))))
    data_cache = Path(_env("MSVD_L4_DATA_CACHE", "/tmp/msvd_hf" if RUN_ON_KAGGLE else str(_base / "msvd_hf")))
    extra_cache_dirs = [d for d in _env("MSVD_L4_CACHE_DIRS", "").split(":") if d]


assert CFG.dataset in ("msvd", "synthetic"), f"CFG.dataset không hợp lệ: {CFG.dataset}"
RESULTS = CFG.out_dir / "results"
for _d in (CFG.out_dir / "cache", RESULTS, CFG.out_dir / "thumbs", CFG.out_dir / "checkpoints"):
    _d.mkdir(parents=True, exist_ok=True)
RES_TAG = "DEV_val" if CFG.dev_mode else ("test" if CFG.dataset == "msvd" else "SYNTHETIC_test")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (set, frozenset)): return sorted(o)
    return str(o)


def save_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=_jsonable)
    return path


def sha1_of(obj):
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=_jsonable).encode()).hexdigest()[:12]


def cfg_snapshot():
    return {k: v for k, v in vars(CFG).items() if not k.startswith("_") and not callable(v)}


CHECKS = []


def check(cid, desc, passed, evidence="", hard=True, status=None):
    """Ghi một kiểm tra vào sổ audit. hard=True ⇒ FAIL dừng notebook ngay (không chạy tiếp trên dữ liệu sai)."""
    st = status or ("PASS" if passed else "FAIL")
    CHECKS.append(dict(id=cid, check=desc, status=st, evidence=str(evidence)[:900]))
    print(f"  [{st}] {cid} — {desc}: {evidence}")
    if hard and st == "FAIL":
        raise AssertionError(f"Kiểm tra {cid} THẤT BẠI: {desc} — {evidence}")
    return st == "PASS"


save_json(dict(cfg=cfg_snapshot(), env=ENV_INFO), RESULTS / "config_initial.json")
print(f"dataset={CFG.dataset} | dev_mode={CFG.dev_mode} | CLIP={CFG.clip_model}/{CFG.clip_weights} | "
      f"seeds={CFG.seeds} | epochs={CFG.epochs} | out={CFG.out_dir.resolve()}")
if CFG.dataset == "synthetic":
    print("⚠⚠ CHẾ ĐỘ SYNTHETIC: chỉ smoke-test mã. Mọi con số dưới đây KHÔNG phải kết quả MSVD.")
if CFG.dev_mode:
    print("⚠ dev_mode: test bị KHOÁ; mọi phân tích cuối chạy trên VAL, gắn nhãn DEV — không dùng để báo cáo.")
mark_stage("config", requires=("deps",))

In [62]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# --- Gói kết quả để TẢI VỀ xem xét: MỘT file zip nhẹ <out_dir>_review.zip, nằm cạnh thư mục output. ---
# Gồm: results/** (mọi CSV/JSON/MD, log truy cập, pip freeze), notebook đã chạy (nếu tìm được), pack_manifest.json.
# KHÔNG gồm dữ liệu và các file nặng suy ra từ dữ liệu: video MSVD (CFG.data_cache), cache/ (đặc trưng CLIP),
# thumbs/ (khung hình video), checkpoints/, synthetic_videos/. Manifest vẫn ghi số file và dung lượng của chúng.
# Cell cuối gọi pack_review(); nếu một cell bị LỖI, hook bên dưới gói ngay những gì đã có, kèm traceback.
import traceback

REVIEW_ZIP = CFG.out_dir.parent / f"{CFG.out_dir.name}_review.zip"
PACK_MAX_FILE_MB = 50


def _find_executed_notebook():
    """Kaggle chạy bằng papermill (input = output = __notebook__.ipynb) và lưu notebook sau mỗi cell, nên bản này
    có output của mọi cell đã chạy xong. Chỉ nhận file được ghi SAU khi kernel này khởi động (tránh bản cũ)."""
    t0 = STAGES[0]["t"] if STAGES else 0.0
    cands = [Path.cwd() / "__notebook__.ipynb", *sorted(Path("/kaggle").glob("*/__notebook__.ipynb")),
             *sorted(Path("/kaggle/src").glob("*.ipynb"))]
    for p in cands:
        try:
            if p.is_file() and p.stat().st_mtime >= t0 and json.loads(p.read_text()).get("cells"):
                return p
        except Exception:
            continue
    return None


def _dir_size(d):
    fs = [f for f in d.rglob("*") if f.is_file()] if d.is_dir() else [d]
    return dict(n_files=len(fs), mb=round(sum(f.stat().st_size for f in fs) / 2**20, 1))


def pack_review(reason, error=None):
    """Ghi lại REVIEW_ZIP từ đầu (gọi nhiều lần được). Không bao giờ ném lỗi ra ngoài: trả về None nếu thất bại."""
    try:
        root, inc, too_large = REVIEW_ZIP.stem, [], []
        nb_path = _find_executed_notebook()
        tmp = REVIEW_ZIP.with_name(REVIEW_ZIP.name + ".tmp")
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
            for p in sorted(p for p in RESULTS.rglob("*") if p.is_file()):
                rel, size = p.relative_to(CFG.out_dir).as_posix(), p.stat().st_size
                if size > PACK_MAX_FILE_MB * 2**20:
                    too_large.append(dict(path=rel, mb=round(size / 2**20, 1)))
                    continue
                z.write(p, f"{root}/{rel}")
                inc.append(dict(path=rel, bytes=size))
            if nb_path is not None:
                z.write(nb_path, f"{root}/executed_notebook.ipynb")
            excluded = {d.name + ("/" if d.is_dir() else ""): _dir_size(d) for d in sorted(CFG.out_dir.iterdir()) if d != RESULTS}
            excluded[f"video MSVD ({CFG.data_cache})"] = "không gói"
            manifest = dict(created=time.strftime("%Y-%m-%d %H:%M:%S"), reason=reason, status="error" if error else "ok",
                            error_traceback=error, dataset=CFG.dataset, dev_mode=CFG.dev_mode, res_tag=RES_TAG, kaggle=RUN_ON_KAGGLE,
                            stages_done=[s["name"] for s in STAGES], checks=dict(Counter(c["status"] for c in CHECKS)),
                            checks_fail=[c["id"] for c in CHECKS if c["status"] == "FAIL"],
                            executed_notebook=str(nb_path) if nb_path else None, included=inc, skipped_too_large=too_large,
                            excluded=excluded)
            z.writestr(f"{root}/pack_manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2, default=_jsonable))
        tmp.replace(REVIEW_ZIP)
        return REVIEW_ZIP
    except Exception as e:
        print(f"⚠ Không đóng gói được {REVIEW_ZIP}: {type(e).__name__}: {e}")
        return None


def _pack_on_error(result):
    err = result.error_in_exec or result.error_before_exec
    if err is None:
        return
    tb = "".join(traceback.format_exception(type(err), err, err.__traceback__))[-8000:]
    z = pack_review(f"cell lỗi (execution_count={result.execution_count})", error=tb)
    if z is not None:
        print(f"⚠ Notebook dừng vì lỗi — đã gói những gì đã có (kèm traceback) → {z} ({z.stat().st_size / 2**20:.1f} MB)")


try:
    _IP = get_ipython()
    for _cb in [c for c in _IP.events.callbacks["post_run_cell"] if getattr(c, "__name__", "") == "_pack_on_error"]:
        _IP.events.unregister("post_run_cell", _cb)          # chạy lại cell này thì không đăng ký trùng
    _IP.events.register("post_run_cell", _pack_on_error)
except NameError:
    pass
print(f"Gói xem xét: {REVIEW_ZIP} (tạo ở cell cuối; tự tạo nếu một cell bị lỗi)")

## 4.2 Giao thức đăng ký trước (viết TRƯỚC khi chạm MSR-VTT fresh1k-B)

**Đã biết trước khi viết notebook này** (v4, Kaggle, fresh1k):
* v4 hơn CLIP-only: ΔMRR +0,0109, CI [+0,0072; +0,0145].
* v4 **không** hơn chỉ-contains, không vị từ, vị từ hoán vị, rewired, attention tĩnh hay v3.
* Bỏ mọi cạnh quan hệ lúc suy luận làm MRR **tăng** nhẹ.
* Xoá giải thích (chỉ đường đi quan hệ) làm điểm giảm nhưng **không** đổi thứ hạng nào.

Vì vậy notebook này **không kỳ vọng** quan hệ giúp. Mục tiêu là đo lại một cách trung thực trên tập chưa nhìn, kèm các đối chứng còn thiếu.

**Họ so sánh chính** (một phía, `a > b`, MRR trên mọi caption, bootstrap theo video, Holm–Bonferroni α = 0,05). Chỉ **fresh1k-B** mới là xác nhận; test MSVD là thăm dò.

| | So sánh | Câu hỏi |
|---|---|---|
| H1 | G3 vs G0 | đồ thị (QC-RGAT) có hơn CLIP-only? |
| H2 | G3 vs G1 | cạnh quan hệ có đóng góp ngoài contains? |
| H3 | G3 vs G2 | khớp vị từ có ích? |
| H4 | G3 vs G4 | ghép **đúng** vị từ vào **đúng** cạnh có ích? |
| H5 | G3 vs G5 | topology thật có hơn rewiring có ràng buộc? |
| H6 | G3 vs G6 | attention theo truy vấn có hơn attention tĩnh? |
| H7 | G3 vs R3 | QC-RGAT có hơn GraphSAGE có kiểu cạnh (v3)? |
| H8 | G3 vs G7 | họ cạnh **vai** (`agent_of`, `object_of`) có giá trị? (G7 huấn luyện lại không có họ này) |
| H9 | G3 vs G8 | họ cạnh **`svo`/`prep`** có giá trị? (G8 huấn luyện lại không có họ này) |

**Luật kết luận:**
* "Quan hệ có giá trị thật" chỉ được kết luận khi **H2, H4 và H5 cùng được ủng hộ** và đối chứng đạt audit.
* Một họ cạnh "có giá trị" khi H8 hoặc H9 tương ứng được ủng hộ.

**So sánh phụ** (không vào họ Holm, chỉ báo cáo kèm CI):
* G3 vs G0 theo nhóm truy vấn trực tiếp / gián tiếp;
* trên bộ truy vấn đăng ký trước (≥ 20 / nhóm).

**Chọn cấu hình** (chỉ VAL):
* Incumbent là T = 1 (đơn giản hơn).
* T = 2 chỉ được nhận khi Δ MRR val ≥ 0,002 **và** p Holm < 0,10.
* Cố định trước: d = 64, lr = 2e-3, k = 50.
* α (trọng số CLIP trong điểm cuối) chọn trên VAL trong lưới có α = 1 (CLIP-only). Nếu α = 1 thắng trên VAL thì đồ thị không được dùng ở điểm cuối, và kết quả được báo đúng như vậy.

**Ngân sách:** 5 seed × 15 epoch cho mọi thí nghiệm. Trên cùng một tập đánh giá, mọi thí nghiệm dùng chung tập ứng viên CLIP top-k.

**Bộ truy vấn đăng ký trước** (mục 7.3). Lấy từ caption của tập đánh giá, chỉ dựa vào hai thứ: nhãn CLIP zero-shot của video đúng và cạnh đồ thị dựng từ train. **Không** dựa vào điểm của mô hình.
* **Trực tiếp:** mọi khái niệm khớp được của truy vấn đều là nhãn của video đúng.
* **Gián tiếp:** truy vấn nêu một quan hệ (bộ ba chủ thể–vị từ–đối tượng); nhãn video chỉ phủ **một phần** khái niệm của truy vấn; có ít nhất một cạnh quan hệ nối khái niệm còn thiếu với một nhãn của video. Đó là các quan hệ dự kiến phải khai thác.
* Mỗi nhóm lấy 20 truy vấn (mỗi video tối đa 1) bằng seed cố định.


## 4.3 Hàng rào chống rò rỉ

* **`CaptionStore`**: caption không được lưu trong `RECORDS`. Mọi lần đọc caption, hoặc dùng đặc trưng của caption, đều phải nêu **mục đích**; một ma trận quyết định split nào được dùng cho mục đích nào.
  * `vocab`, `relations`, `annotation` và `gnn_train` chỉ dùng train; `tune` và `tagger_select` chỉ dùng val.
  * Caption test MSVD và tập ngoài (`ext`, MSR-VTT) chỉ dùng để `encode`, `query_input` và (sau đóng băng) `eval`, `report`, `display`.
  * Truy cập sai sẽ ném `PermissionError` ngay.
* **`TestGate`**: test MSVD và tập ngoài chỉ mở sau `freeze()`, và mọi lần chạm đều được ghi log kèm tên split. Ở `dev_mode`, test MSVD bị khoá; tập ngoài khi đó là video **train-9k** của MSR-VTT, rời hẳn 1k-A.
* **`provenance()`**: mỗi artifact được ghi lại nguồn, split caption và split video đã dùng; khai báo sai loại thì `assert` thất bại.

In [63]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
class TestGate:
    """Test chỉ mở SAU khi cấu hình đã đóng băng trên validation. Mọi lần chạm test được ghi log."""

    def __init__(self):
        self.frozen, self.frozen_sha, self.frozen_at, self.log = None, None, None, []

    def freeze(self, config):
        assert self.frozen is None, "Cấu hình đã đóng băng — không được chọn lại sau khi mở test."
        payload = json.dumps(config, sort_keys=True, default=_jsonable)
        self.frozen, self.frozen_at = json.loads(payload), time.time()
        self.frozen_sha = hashlib.sha1(payload.encode()).hexdigest()[:12]
        save_json(dict(config=self.frozen, sha1=self.frozen_sha, frozen_at=self.frozen_at,
                       frozen_at_str=time.strftime("%Y-%m-%d %H:%M:%S")), RESULTS / "frozen_config.json")
        print(f"🔒 Đã đóng băng cấu hình (sha1={self.frozen_sha}) → {RESULTS / 'frozen_config.json'}")

    def check(self, what, split="test"):
        """split 'test' = test MSVD (đã nhìn ở v1/v2 ⇒ ở v3 chỉ là THĂM DÒ); 'ext' = benchmark xác nhận độc lập (MSR-VTT).
        Cả hai chỉ mở sau khi đóng băng. dev_mode: test MSVD bị khoá; 'ext' khi đó là video TRAIN-9k (không phải tập xác nhận)."""
        if self.frozen is None:
            raise RuntimeError(f"Truy cập {split.upper()} ('{what}') trước khi đóng băng cấu hình — vi phạm giao thức.")
        if CFG.dev_mode and split == "test":
            raise RuntimeError(f"dev_mode: test bị khoá ('{what}')")
        self.log.append(dict(what=what, split=split, t=time.time(), frozen_sha=self.frozen_sha, dev=bool(CFG.dev_mode)))


TEST_GATE = TestGate()


class CaptionStore:
    """Nơi DUY NHẤT giữ caption. Mỗi lần đọc phải nêu mục đích; split không được phép ⇒ PermissionError."""
    ALLOWED = {
        "vocab":         {"train"},                  # từ vựng khái niệm
        "relations":     {"train"},                  # cạnh quan hệ thống kê từ caption
        "gnn_train":     {"train"},                  # truy vấn huấn luyện GNN
        "tagger_select": {"val"},                    # chọn (pool, norm, k) của tagger theo F1
        "tune":          {"val"},                    # chọn n_frames/pooling/learn_clip_w/k/α/λ
        "encode":        {"train", "val", "test", "ext"},   # CLIP-text đóng băng (không học) — chỉ để cache đặc trưng
        "query_input":   {"train", "val", "test", "ext"},   # đọc văn bản của CHÍNH truy vấn đang được trả lời
        "eval":          {"val", "test", "ext"},
        "display":       {"train", "val", "test", "ext"},   # chỉ hiển thị SAU khi đã xếp hạng
        "report":        {"val", "test", "ext"},            # báo cáo P/R/F1 của tagger sau khi đóng băng
    }
    GATED = {"eval", "report", "display"}            # với split test / ext: chỉ sau khi đóng băng

    def __init__(self, caps_by_vid, split_by_vid):
        self._caps = {int(v): list(c) for v, c in caps_by_vid.items()}
        self.split_of = {int(v): s for v, s in split_by_vid.items()}
        vid_of_cap, self._text = [], []
        for v in sorted(self._caps):
            for c in self._caps[v]:
                vid_of_cap.append(v); self._text.append(c)
        self.vid_of_cap = np.array(vid_of_cap, np.int64)
        self.split_of_cap = np.array([self.split_of[v] for v in vid_of_cap])
        self.log = Counter()

    def _authorize(self, splits, purpose):
        assert purpose in self.ALLOWED, f"mục đích lạ: {purpose}"
        for s in splits:
            if s not in self.ALLOWED[purpose]:
                raise PermissionError(f"RÒ RỈ bị chặn: caption split '{s}' không được dùng cho mục đích '{purpose}'")
            if s in ("test", "ext") and purpose in self.GATED:
                TEST_GATE.check(f"captions:{purpose}", s)
        for s, n in splits.items():
            self.log[(purpose, s)] += int(n)

    def by_video(self, vids, purpose):
        vids = [int(v) for v in np.atleast_1d(vids)]
        self._authorize(Counter(self.split_of[v] for v in vids), purpose)
        return {v: list(self._caps[v]) for v in vids}

    def texts(self, cap_idx, purpose):
        cap_idx = np.atleast_1d(np.asarray(cap_idx, np.int64))
        self._authorize(Counter(self.split_of_cap[cap_idx].tolist()), purpose)
        return [self._text[i] for i in cap_idx]

    def use(self, cap_idx, purpose):
        """Khai báo dùng ĐẶC TRƯNG của caption (không lấy văn bản) — cùng luật cho phép."""
        cap_idx = np.atleast_1d(np.asarray(cap_idx, np.int64))
        self._authorize(Counter(self.split_of_cap[cap_idx].tolist()), purpose)

    def replaced(self, new_caps, new_split=None):
        """Bản sao với caption (và tuỳ chọn split) của một số video bị thay — CHỈ dùng trong self-test."""
        caps = {v: list(c) for v, c in self._caps.items()}
        caps.update({int(v): list(c) for v, c in new_caps.items()})
        return CaptionStore(caps, {**self.split_of, **(new_split or {})})

    def access_table(self):
        return pd.DataFrame([dict(purpose=p, split=s, n=n) for (p, s), n in sorted(self.log.items())])


PROVENANCE = []
_PROV_RULES = {            # loại artifact → split CAPTION được phép dùng
    "learned":  {"train"},             # thống kê / trọng số học được
    "selected": {"val"},               # siêu tham số chọn trên validation
    "frames":   set(),                 # chỉ từ frame (không caption)
    "external": set(),                 # tri thức ngoài / mô hình tiền huấn luyện
    "encoded":  {"train", "val", "test", "ext"},   # mã hoá bằng encoder đóng băng, không học gì
    "evaluated": {"val", "test", "ext"},
}


def provenance(artifact, kind, source, caption_splits=(), video_splits=(), note=""):
    cs = set(caption_splits)
    bad = cs - _PROV_RULES[kind]
    assert not bad, f"RÒ RỈ: '{artifact}' (loại {kind}) dùng caption của {sorted(bad)}"
    PROVENANCE.append(dict(artifact=artifact, kind=kind, source=source,
                           caption_splits=",".join(sorted(cs)) or "—",
                           video_splits=",".join(sorted(set(video_splits))) or "—",
                           note=note, stage=STAGES[-1]["name"], time=time.strftime("%H:%M:%S")))

## 4.4 Dữ liệu: MSVD (phát triển) + MSR-VTT 1k-A (xác nhận) — Cổng A

**MSVD.** Như v1/v2. Video thật nằm trong `MSVD_Videos.zip`. Notebook dò `MSVD_ROOT`, rồi `/kaggle/input`, rồi cache, rồi tải từ HF ở commit ghim. Không có video thì **dừng**.

**MSR-VTT** (`friedrichor/MSR-VTT`, commit ghim). Dùng `msrvtt_test_1k.json` (1000 video, caption chuẩn JSFusion), `raw_data/MSRVTT_data.json` (20 caption/video) và `MSRVTT_Videos.zip` (2,2 GB, mp4).
* Video tập ngoài được nối **sau** MSVD trong `RECORDS` với split `ext`. Vì vậy cache MSVD giữ nguyên tên và thứ tự.
* Truy vấn chính là mọi caption của 1000 video; truy vấn phụ là caption chuẩn 1k-A. Nếu caption chuẩn không nằm trong 20 caption thì nó được thêm vào và đếm riêng.
* Không nạp được MSR-VTT thì tập xác nhận ghi **NOT RUN** kèm lý do; notebook không thay bằng tập khác.

In [64]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
HF_JSON = {"train": "msvd_train.json", "val": "msvd_val.json", "test": "msvd_test.json"}
HF_ZIP = "MSVD_Videos.zip"


class DirVideoSource:
    kind = "dir"

    def __init__(self, root):
        self.root = Path(root)
        self.files = {p.stem: p for p in self.root.rglob("*") if p.suffix.lower() == ".avi"}
        self.label = str(self.root)

    def has(self, vid): return vid in self.files
    def size(self, vid): return self.files[vid].stat().st_size

    @contextmanager
    def open(self, vid):
        yield self.files[vid]


class ZipVideoSource:
    """Đọc video thẳng từ zip cục bộ (không giải nén toàn bộ): mỗi video được ghi ra file tạm, giải mã, rồi xoá."""
    kind = "zip"

    def __init__(self, zip_file, label, exts=(".avi",)):
        self.z = zipfile.ZipFile(zip_file)
        self.members = {Path(i.filename).stem: i for i in self.z.infolist()
                        if i.filename.lower().endswith(exts) and not Path(i.filename).name.startswith("._")}
        self.lock = threading.Lock()
        self.tmp = Path(tempfile.mkdtemp(prefix="msvd_zip_"))
        self.label = label

    def has(self, vid): return vid in self.members
    def size(self, vid): return self.members[vid].file_size

    @contextmanager
    def open(self, vid):
        with self.lock:                                   # ZipFile không an toàn khi đọc song song
            data = self.z.read(self.members[vid])
        p = self.tmp / f"{vid}{Path(self.members[vid].filename).suffix.lower()}"
        p.write_bytes(data)
        try:
            yield p
        finally:
            p.unlink(missing_ok=True)


def _first(root, name):
    hits = sorted(root.rglob(name), key=lambda p: len(str(p)))
    return hits[0] if hits else None


def locate_msvd():
    roots = [Path(CFG.msvd_root)] if CFG.msvd_root else []
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    roots.append(CFG.data_cache)
    found = dict(json={}, video_dir=None, zip=None, roots=[str(r) for r in roots])
    for root in roots:
        if not root.exists():
            continue
        for s, name in HF_JSON.items():
            if s not in found["json"]:
                p = _first(root, name)
                if p is not None:
                    found["json"][s] = p
        if found["video_dir"] is None:
            for d in root.rglob("YouTubeClips"):
                if d.is_dir() and any(d.glob("*.avi")):
                    found["video_dir"] = d
                    break
        if found["zip"] is None:
            found["zip"] = _first(root, HF_ZIP)
    return found


def hf_download(filename):
    from huggingface_hub import hf_hub_download
    CFG.data_cache.mkdir(parents=True, exist_ok=True)
    print(f"  tải {CFG.hf_repo}@{CFG.hf_revision[:8]}/{filename} → {CFG.data_cache}")
    return Path(hf_hub_download(CFG.hf_repo, filename, repo_type="dataset", revision=CFG.hf_revision,
                                local_dir=str(CFG.data_cache)))


def open_video_source(found):
    mode = CFG.video_source
    assert mode in ("auto", "dir", "zip"), mode
    if mode in ("auto", "dir") and found["video_dir"] is not None:
        return DirVideoSource(found["video_dir"])
    if mode in ("auto", "zip") and found["zip"] is not None:
        return ZipVideoSource(found["zip"], str(found["zip"]))
    if mode in ("auto", "zip") and INTERNET:
        return ZipVideoSource(hf_download(HF_ZIP), "hf_hub_download")
    raise FileNotFoundError(
        "KHÔNG tìm thấy file video MSVD (YouTubeClips/*.avi hoặc MSVD_Videos.zip) và không tải được. DỪNG: chỉ có "
        "caption thì KHÔNG được chạy/gọi là thí nghiệm text-to-video. Khắc phục: bật Internet, hoặc Add Input "
        f"dataset chứa video, hoặc đặt MSVD_ROOT. Đã dò: {found['roots']}")


def load_msvd():
    found = locate_msvd()
    for s, name in HF_JSON.items():
        if s not in found["json"]:
            if not INTERNET:
                raise FileNotFoundError(f"Thiếu {name} trong {found['roots']} và không có Internet để tải.")
            found["json"][s] = hf_download(name)
    source = open_video_source(found)
    records, caps = [], {}
    report = dict(source=source.kind, source_label=source.label, n_video_files=len(
        source.files if source.kind == "dir" else source.members),
        json={k: str(v) for k, v in found["json"].items()}, splits={})
    for s in ("train", "val", "test"):
        rows = json.load(open(found["json"][s]))
        fields = sorted({k for r in rows for k in r})
        bad = [r.get("video_id") for r in rows if not r.get("video_id") or not isinstance(r.get("caption"), list)]
        assert not bad, f"Schema {s} sai (thiếu video_id hoặc caption không phải list): {bad[:5]}"
        n_cap, missing, ncaps = 0, [], []
        for r in rows:
            vid = len(records)
            c = [str(x).strip() for x in r["caption"] if str(x).strip()]
            records.append(dict(vid=vid, video_id=r["video_id"], file=r.get("video") or f"{r['video_id']}.avi",
                                split=s, n_captions=len(c)))
            caps[vid] = c; n_cap += len(c); ncaps.append(len(c))
            if not source.has(r["video_id"]):
                missing.append(r["video_id"])
        report["splits"][s] = dict(videos=len(rows), captions=n_cap, fields=fields, missing_files=len(missing),
                                   missing_examples=missing[:5],
                                   captions_per_video=[int(min(ncaps)), round(float(np.mean(ncaps)), 1), int(max(ncaps))],
                                   video_field_is_filename=all(str(r.get("video", "")).endswith(".avi") for r in rows))
    ids = [r["video_id"] for r in records]
    assert len(set(ids)) == len(ids), "video_id trùng lặp giữa các bản ghi"
    return records, caps, source, report


def probe_video(path):
    """Codec / fps / kích thước / số frame giải mã thật sự (đếm bằng grab, không tin FRAME_COUNT)."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return dict(ok=False, error="cv2 không mở được")
    fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))
    info = dict(codec="".join(chr((fourcc >> 8 * i) & 0xFF) for i in range(4)), fps=round(cap.get(cv2.CAP_PROP_FPS), 3),
                width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                frames_meta=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), backend=cap.getBackendName())
    n = 0
    while cap.grab():
        n += 1
    cap.release()
    info.update(ok=n > 0, frames_decoded=n, duration_s=round(n / info["fps"], 2) if info["fps"] > 0 else None)
    return info

In [65]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# --- Level 4(b): benchmark xác nhận ĐỘC LẬP = MSR-VTT fresh1k-B (CHƯA dùng ở v1–v4); fresh1k (v4) và 1k-A (v3) đã nhìn ⇒ chỉ THĂM DÒ.
# (Ghi chú lịch sử của v4:) Benchmark xác nhận cho v4: MSR-VTT fresh1k (CHƯA dùng ở v1–v3). Video nối vào đồ thị theo giao thức inductive. ---
# 1k-A (JSFusion) đã được nhìn MỘT lần ở v3 ⇒ v4 dùng 1000 video KHÁC: lấy ngẫu nhiên (seed cố định) trong train-9k, loại 1k-A và
# loại 200 video dev (đã chạy ở máy local). Truy vấn CHÍNH = mọi caption (MSRVTT_data.json); truy vấn PHỤ = caption đầu tiên.
# CFG.confirm_set = "1kA" chạy lại 1k-A (lần nhìn thứ 2 — phải khai báo). Chỉ đánh giá MỘT lần, sau khi đóng băng.
# dev_mode: msrvtt_dev_n video TRAIN-9k đầu theo id — chỉ để thử đường ống, không phải kết quả xác nhận.
MSRVTT_FILES = dict(test="msrvtt_test_1k.json", train="msrvtt_train_9k.json", all_caps="raw_data/MSRVTT_data.json",
                    zip="MSRVTT_Videos.zip")


def msrvtt_file(key):
    name = MSRVTT_FILES[key]
    roots = ([Path(CFG.msrvtt_root)] if CFG.msrvtt_root else []) + ([Path("/kaggle/input")] if RUN_ON_KAGGLE else []) \
        + [CFG.data_cache / "msrvtt"]
    for root in roots:
        if root.exists():
            p = _first(root, Path(name).name)
            if p is not None:
                return p
    if not INTERNET:
        raise FileNotFoundError(f"Thiếu {name} (đã dò {roots}) và không có Internet.")
    from huggingface_hub import hf_hub_download
    d = CFG.data_cache / "msrvtt"
    d.mkdir(parents=True, exist_ok=True)
    print(f"  tải {CFG.msrvtt_repo}@{CFG.msrvtt_revision[:8]}/{name} → {d}")
    return Path(hf_hub_download(CFG.msrvtt_repo, name, repo_type="dataset", revision=CFG.msrvtt_revision, local_dir=str(d)))


def load_msrvtt(vid_offset):
    test = json.load(open(msrvtt_file("test")))
    test_ids = sorted({r["video_id"] for r in test}, key=lambda x: int(x[5:]))
    tr_all = sorted([r for r in json.load(open(msrvtt_file("train"))) if r["video_id"] not in set(test_ids)],
                    key=lambda r: int(r["video_id"][5:]))
    dev_ids = {r["video_id"] for r in tr_all[:CFG.msrvtt_dev_n]}
    fresh1k_ids = set()
    set_name = "dev" if CFG.dev_mode else CFG.confirm_set
    if CFG.dev_mode:
        rows = tr_all[:CFG.msrvtt_dev_n]
        caps_of = {r["video_id"]: [str(c).strip() for c in r["caption"] if str(c).strip()] for r in rows}
        std = {v: c[0] for v, c in caps_of.items()}
        label = f"MSR-VTT train-9k, {len(rows)} video đầu theo id (DEV — không phải tập xác nhận)"
    else:
        assert CFG.confirm_set in ("fresh1kB", "fresh1k", "1kA"), CFG.confirm_set
        pool = [r for r in tr_all if r["video_id"] not in dev_ids]
        pick = np.random.RandomState(CFG.fresh_seed).choice(len(pool), CFG.fresh_n, replace=False)
        fresh1k_ids = {pool[i]["video_id"] for i in pick}          # tập xác nhận của v4 — ĐÃ nhìn một lần
        if CFG.confirm_set == "fresh1kB":
            pool_b = [r for r in pool if r["video_id"] not in fresh1k_ids]
            pick_b = np.random.RandomState(CFG.fresh_b_seed).choice(len(pool_b), CFG.fresh_n, replace=False)
            rows = sorted([pool_b[i] for i in pick_b], key=lambda r: int(r["video_id"][5:]))
            label = (f"MSR-VTT fresh1k-B: {len(rows)} video train-9k (ngoài 1k-A, ngoài {CFG.msrvtt_dev_n} video dev, ngoài fresh1k của v4; "
                     f"seed {CFG.fresh_b_seed}) — CHƯA dùng ở v1–v4")
        elif CFG.confirm_set == "fresh1k":
            rows = sorted([pool[i] for i in pick], key=lambda r: int(r["video_id"][5:]))
            label = (f"MSR-VTT fresh1k: {len(rows)} video train-9k (seed {CFG.fresh_seed}) — ĐÃ nhìn ở v4 ⇒ chỉ THĂM DÒ")
        else:
            rows = sorted(test, key=lambda r: int(r["video_id"][5:]))
            label = "MSR-VTT test 1k-A (JSFusion) — LẦN NHÌN THỨ 2 (đã dùng ở v3)"
        data = json.load(open(msrvtt_file("all_caps")))
        allc = defaultdict(list)
        for x in sorted(data["sentences"], key=lambda x: int(x["sen_id"])):
            allc[x["video_id"]].append(str(x["caption"]).strip())
        caps_of = {r["video_id"]: [c for c in allc[r["video_id"]] if c] for r in rows}
        std = ({r["video_id"]: str(r["caption"]).strip() for r in rows} if CFG.confirm_set == "1kA"
               else {v: c[0] for v, c in caps_of.items()})
    n_std_added = 0
    for v, c in caps_of.items():                       # caption chuẩn phải nằm trong tập truy vấn chính
        if std[v] not in c:
            c.append(std[v]); n_std_added += 1
    source = ZipVideoSource(msrvtt_file("zip"), f"{CFG.msrvtt_repo}@{CFG.msrvtt_revision[:8]}", exts=(".mp4",))
    records, caps, missing = [], {}, []
    for j, r in enumerate(rows):
        vid, c = vid_offset + j, caps_of[r["video_id"]]
        records.append(dict(vid=vid, video_id=r["video_id"], file=r.get("video") or f"{r['video_id']}.mp4", split="ext",
                            n_captions=len(c), std_caption=c.index(std[r["video_id"]]), dataset="msrvtt"))
        caps[vid] = c
        if not source.has(r["video_id"]):
            missing.append(r["video_id"])
    ids = {r["video_id"] for r in records}
    report = dict(label=label, set=set_name, dev=bool(CFG.dev_mode), videos=len(records), captions=sum(len(c) for c in caps.values()),
                  captions_per_video=[min(map(len, caps.values())), round(float(np.mean(list(map(len, caps.values())))), 1),
                                      max(map(len, caps.values()))],
                  std_captions_added=n_std_added, missing_files=len(missing), missing_examples=missing[:5],
                  n_test_1kA_ids=len(test_ids), overlap_with_test_1kA=len(ids & set(test_ids)),
                  overlap_with_dev=len(ids & dev_ids) if not CFG.dev_mode else None,
                  overlap_with_fresh1k=len(ids & fresh1k_ids) if not CFG.dev_mode else None, zip_members=len(source.members))
    if set_name in ("fresh1k", "fresh1kB"):
        assert report["overlap_with_test_1kA"] == 0 and report["overlap_with_dev"] == 0, "fresh1k / fresh1k-B phải rời 1k-A và tập dev"
    if set_name == "fresh1kB":
        assert report["overlap_with_fresh1k"] == 0 and report["videos"] == CFG.fresh_n, "fresh1k-B phải rời fresh1k của v4"
    return records, caps, source, report


def make_synthetic_ext(vid_offset, n_videos=30, seed=7):
    """Tập 'ext' tổng hợp cho smoke-test: cùng thế giới đối tượng, câu chữ khác (kiểm tra đường ống, không phải kết quả)."""
    recs, caps, source, _ = make_synthetic_msvd(n_videos=n_videos, seed=seed, prefix="ext", all_split="ext")
    out_r, out_c = [], {}
    for j, r in enumerate(recs):
        out_r.append(dict(r, vid=vid_offset + j, split="ext", std_caption=0, dataset="synthetic_ext"))
        out_c[vid_offset + j] = caps[r["vid"]]
    report = dict(label="synthetic ext", dev=True, videos=len(out_r), captions=sum(len(c) for c in out_c.values()),
                  missing_files=0, overlap_with_test_1kA=0, std_captions_added=0)
    return out_r, out_c, source, report

In [66]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# Video TỔNG HỢP — chỉ dùng khi CFG.dataset == "synthetic" (smoke-test mã, không phải kết quả).
# Mỗi video có 3 cảnh (chủ thể → hành động → đồ vật), chữ in trên frame để CLIP đọc được; được ghi
# thành AVI (MJPG) thật để kiểm tra luôn đường ống đọc video bằng OpenCV.
SYN_SUBJ = {"man": "person", "woman": "person", "child": "person", "dog": "animal", "cat": "animal"}
SYN_ACT = {"play": (["guitar", "piano", "ball"], "playing"), "ride": (["horse", "bicycle"], "riding"),
           "eat": (["apple", "pizza"], "eating"), "cut": (["onion", "bread"], "cutting")}
SYN_COLORS = [(40, 40, 160), (40, 140, 40), (160, 60, 40), (120, 40, 120), (30, 120, 150), (150, 150, 30)]


def make_synthetic_msvd(n_videos=150, seed=0, prefix="syn", all_split=None):
    rng = random.Random(seed)
    vdir = CFG.out_dir / "synthetic_videos"
    vdir.mkdir(parents=True, exist_ok=True)
    records, caps = [], {}
    n_tr, n_va = int(n_videos * 0.6), int(n_videos * 0.15)
    fourcc = cv2.VideoWriter_fourcc(*"MJPG")
    for i in range(n_videos):
        s = rng.choice(sorted(SYN_SUBJ)); a = rng.choice(sorted(SYN_ACT))
        objs, ing = SYN_ACT[a]; o = rng.choice(objs)
        split_name = all_split or ("train" if i < n_tr else ("val" if i < n_tr + n_va else "test"))
        name = f"{prefix}{i:04d}_{s}_{a}_{o}"
        path = vdir / f"{name}.avi"
        if not path.exists():
            w = cv2.VideoWriter(str(path), fourcc, 12.0, (224, 224))
            assert w.isOpened(), "cv2.VideoWriter không mở được (MJPG/AVI)"
            for k in range(36):
                word = (s, ing, o)[k // 12]
                fr = np.full((224, 224, 3), SYN_COLORS[(i + k // 12) % len(SYN_COLORS)], np.uint8)
                cv2.putText(fr, word, (8 + (k % 12) * 2, 122), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (255, 255, 255), 3)
                w.write(fr)
            w.release()
        c = [f"a {s} is {ing} a {o}", f"a {s} {a}s the {o}", f"someone is {ing} a {o}", f"a {s} with a {o}",
             f"the {s} is {ing}", f"a {s} is {ing} a {o} slowly", f"a {SYN_SUBJ[s]} is {ing}", f"a {o} and a {s}"]
        if all_split == "ext":                         # tập ngoài: câu chữ khác với tập huấn luyện
            c = [f"there is a {s} {ing} the {o}", f"{s} {ing} {o}", f"video of a {o} being used by a {s}",
                 f"the {SYN_SUBJ[s]} keeps {ing} a {o}"]
        rng.shuffle(c)
        records.append(dict(vid=i, video_id=name, file=path.name, split=split_name, n_captions=len(c)))
        caps[i] = c
    source = DirVideoSource(vdir)
    report = dict(source="synthetic-dir", source_label=str(vdir), n_video_files=len(source.files), json={}, splits={
        s: dict(videos=sum(r["split"] == s for r in records),
                captions=sum(len(caps[r["vid"]]) for r in records if r["split"] == s),
                fields=["caption", "video", "video_id"], missing_files=sum(
                    not source.has(r["video_id"]) for r in records if r["split"] == s))
        for s in ("train", "val", "test")})
    return records, caps, source, report

## 4.5 Split chính thức và manifest

Split MSVD giữ đúng `msvd_{train,val,test}.json` (1200 / 100 / 670 video), chia **theo video**. sha1 chỉ tính trên ba split MSVD, nên **trùng với v2**; kiểm tra C7 xác nhận điều này. Tập ngoài `ext` có sha1 riêng và không được trùng video nào với MSVD.

In [67]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
SPLITS_MSVD = ("train", "val", "test")


def make_split(records, write=True):
    """Split MSVD giữ NGUYÊN như v1/v2 (sha1 chỉ tính trên train/val/test MSVD ⇒ so sánh được với v2);
    'ext' = benchmark xác nhận độc lập (MSR-VTT), có sha1 riêng."""
    keys = SPLITS_MSVD + ("ext",)
    split = {k: [r["vid"] for r in records if r["split"] == k] for k in keys}
    ids = {k: [records[v]["video_id"] for v in split[k]] for k in keys}
    for a, b in itertools.combinations(keys, 2):
        assert not set(split[a]) & set(split[b]) and not set(ids[a]) & set(ids[b]), f"{a}/{b} chồng lấn video!"
    payload = {k: sorted(ids[k]) for k in SPLITS_MSVD}
    split["sha1"] = sha1_of(payload)
    split["ext_sha1"] = sha1_of(sorted(ids["ext"])) if ids["ext"] else None
    if write:
        manifest = dict(
            dataset=CFG.dataset, hf_repo=CFG.hf_repo, hf_revision=CFG.hf_revision, sha1=split["sha1"],
            policy=("split chính thức của MSVD theo msvd_{train,val,test}.json; chia THEO VIDEO (không theo caption); "
                    "1 video = 1 ứng viên trong gallery của split đó; mỗi caption = 1 truy vấn có đúng 1 video dương"),
            counts={k: dict(videos=len(split[k]), captions=int(sum(records[v]["n_captions"] for v in split[k]))) for k in keys},
            video_ids=payload, ext=dict(sha1=split["ext_sha1"], source=EXT_REPORT.get("label") if EXT_REPORT else None,
                                        video_ids=sorted(ids["ext"])))
        save_json(manifest, RESULTS / "split_manifest.json")
        print(f"Split (sha1={split['sha1']}): " + " | ".join(
            f"{k}={manifest['counts'][k]['videos']} video / {manifest['counts'][k]['captions']} caption" for k in keys))
    return split

## 4.6 Lấy mẫu khung hình và đặc trưng CLIP đóng băng (Cổng B)

* **Chính sách lấy mẫu `uniform-segment-midpoint`:** video được chia thành `F_max = 12` đoạn bằng nhau, mỗi đoạn lấy frame ở giữa: `idx_k = floor((k + 0.5)/12 · n)`, với `n` là số frame **giải mã được thực sự**. Số này được đếm bằng `grab()`, vì `CAP_PROP_FRAME_COUNT` của AVI có thể sai. Ở lượt đọc thứ hai, notebook chỉ `retrieve()` đúng 12 frame cần lấy, nên bộ nhớ chỉ chứa 12 frame/video.
* Mọi frame được mã hoá **một lần** bằng CLIP-image đóng băng, chuẩn hoá L2, và lưu cache `[N_video, 12, D]` (float16) cùng chỉ số frame, timestamp, fps và thời lượng. Cache được dò ở `msvd_l4/cache` và ở `/kaggle/input/**/cache`.
* Mọi caption được mã hoá **một lần** bằng CLIP-text của cùng mô hình. Bước này không học gì, chỉ để cache đặc trưng; việc dùng đặc trưng caption về sau vẫn phải theo luật của `CaptionStore`.
* Thumbnail của frame giữa video (rộng 160 px) được lưu để demo hiển thị kết quả.

In [68]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
_CLIP = {}


def get_clip():
    """CLIP/OpenCLIP tiền huấn luyện, đóng băng tuyệt đối (eval + requires_grad=False). Nạp một lần."""
    if "m" not in _CLIP:
        m, _, pre = open_clip.create_model_and_transforms(CFG.clip_model, pretrained=CFG.clip_weights)
        m = m.to(DEVICE).eval()
        for p in m.parameters():
            p.requires_grad_(False)
        _CLIP.update(m=m, pre=pre, tok=open_clip.get_tokenizer(CFG.clip_model))
    return _CLIP["m"], _CLIP["pre"], _CLIP["tok"]


def _amp():
    return torch.autocast("cuda", dtype=torch.float16) if DEVICE == "cuda" else contextlib.nullcontext()


@torch.no_grad()
def clip_encode_images(x):
    m, _, _ = get_clip()
    with _amp():
        f = m.encode_image(x.to(DEVICE))
    return F.normalize(f.float(), dim=-1).cpu().numpy()


@torch.no_grad()
def clip_encode_texts(texts, bs=None):
    m, _, tok = get_clip()
    bs = bs or CFG.text_bs
    out = []
    for s in range(0, len(texts), bs):
        with _amp():
            f = m.encode_text(tok(texts[s:s + bs]).to(DEVICE))
        out.append(F.normalize(f.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)


def segment_midpoints(n, n_frames):
    """Chỉ số frame tại điểm giữa của n_frames đoạn đều nhau trên n frame giải mã được."""
    return np.minimum(np.floor((np.arange(n_frames) + 0.5) / n_frames * n).astype(int), n - 1)


def read_sampled_frames(path, n_frames):
    """Lượt 1: đếm frame GIẢI MÃ ĐƯỢC. Lượt 2: chỉ retrieve các frame cần lấy (RGB). Trả về (frames, meta)."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise IOError("cv2 không mở được video")
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    n_meta = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    n = 0
    while cap.grab():
        n += 1
    cap.release()
    if n == 0:
        raise IOError("0 frame giải mã được")
    idx = segment_midpoints(n, n_frames)
    want, got = set(idx.tolist()), {}
    cap = cv2.VideoCapture(str(path))
    i = 0
    while i <= idx[-1] and cap.grab():
        if i in want:
            ok, fr = cap.retrieve()
            if ok:
                got[i] = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        i += 1
    cap.release()
    lost = sorted(j for j in want if j not in got)
    if lost:
        raise IOError(f"không đọc được frame {lost[:5]}")
    fps_eff = fps if fps > 0 else 30.0
    meta = dict(n_decoded=n, n_meta=n_meta, fps=round(fps, 3), fps_assumed=fps <= 0,
                duration_s=round(n / fps_eff, 3), frame_idx=idx.tolist(),
                t_sec=[round(float(j) / fps_eff, 3) for j in idx])
    return [got[int(j)] for j in idx], meta


def _cache_name(prefix, n, data_tag=None):
    data_tag = data_tag or f"{CFG.dataset}"
    rev = CFG.hf_revision[:8] if data_tag == CFG.dataset else ""
    tag = f"{data_tag}_{CFG.clip_model}_{CFG.clip_weights}_F{CFG.n_frames_max}_{CFG.frame_policy}_{rev}_{n}".replace("__", "_")
    return f"{prefix}_" + re.sub(r"[^A-Za-z0-9_-]", "-", tag)


def find_cache(name):
    """Cache của v2, cache của v1 (MSVD_L4_CACHE_DIRS, ví dụ output v1 đã Add Input) và /kaggle/input/**/cache."""
    dirs = [CFG.out_dir / "cache"] + [Path(d) for d in CFG.extra_cache_dirs]
    if RUN_ON_KAGGLE:
        dirs += [p for p in Path("/kaggle/input").glob("*/**/cache") if p.is_dir()]
    for d in dirs:
        if (d / name).exists():
            return d / name
    return None


def extract_frame_features(records, source, allow_encode=True, data_tag=None):
    """[N_video, F_max, D] float16 (mỗi frame đã chuẩn hoá L2) + meta lấy mẫu. Dùng cache nếu có.
    Hàng thứ j ứng với records[j] (không giả định vid = j: tập ngoài nối sau MSVD)."""
    name = _cache_name("frames", len(records), data_tag) + ".npz"
    vids = [r["video_id"] for r in records]
    p = find_cache(name)
    if p is not None:
        z = np.load(p, allow_pickle=False)
        assert z["video_ids"].tolist() == vids, "cache đặc trưng frame không khớp thứ tự video"
        meta = json.load(open(p.with_suffix(".json")))
        print(f"Dùng lại cache đặc trưng frame: {p} {z['feat'].shape}")
        return z["feat"].astype(np.float32), meta, p
    if not allow_encode:
        raise FileNotFoundError(name)
    _, pre, _ = get_clip()
    Fm, N = CFG.n_frames_max, len(records)
    feats, metas, bad = None, [None] * N, {}
    thumbs = CFG.out_dir / "thumbs"

    def work(j):
        r = records[j]
        try:
            with source.open(r["video_id"]) as path:
                frames, meta = read_sampled_frames(path, Fm)
            mid = frames[Fm // 2]
            h, w = mid.shape[:2]
            th = cv2.resize(mid, (CFG.thumb_width, max(1, int(h * CFG.thumb_width / w))), interpolation=cv2.INTER_AREA)
            cv2.imwrite(str(thumbs / f"{r['video_id']}.jpg"), cv2.cvtColor(th, cv2.COLOR_RGB2BGR))
            return j, torch.stack([pre(Image.fromarray(f)) for f in frames]), meta
        except Exception as e:                          # ghi lại, quyết định dừng hay không ở dưới
            return j, None, f"{type(e).__name__}: {e}"

    from concurrent.futures import ThreadPoolExecutor
    t0, chunk = time.time(), max(8, 4 * CFG.clip_bs // Fm)
    with ThreadPoolExecutor(CFG.decode_workers) as ex:
        for s in tqdm(range(0, N, chunk), desc=f"giải mã + CLIP ({N} video)", unit="lô"):
            out = list(ex.map(work, range(s, min(s + chunk, N))))     # theo lô ⇒ bộ nhớ có chặn trên
            ok = [(v, x, m) for v, x, m in out if x is not None]
            bad.update({v: m for v, x, m in out if x is None})
            if not ok:
                continue
            X = torch.cat([x for _, x, _ in ok])
            E = np.concatenate([clip_encode_images(X[i:i + CFG.clip_bs]) for i in range(0, len(X), CFG.clip_bs)])
            if feats is None:
                feats = np.zeros((N, Fm, E.shape[1]), np.float16)
            for j, (v, _, m) in enumerate(ok):
                feats[v] = E[j * Fm:(j + 1) * Fm]
                metas[v] = m
    if bad:
        print(f"⚠ {len(bad)} video giải mã lỗi:", list(bad.items())[:5])
    assert feats is not None, "Không giải mã được video nào"
    assert len(bad) / N <= CFG.max_bad_video_frac, (
        f"{len(bad)}/{N} video lỗi > ngưỡng {CFG.max_bad_video_frac:.0%} — dừng, không chạy thí nghiệm trên dữ liệu hỏng")
    meta = dict(policy=CFG.frame_policy, n_frames_max=Fm, clip=f"{CFG.clip_model}/{CFG.clip_weights}",
                source=source.kind, seconds=round(time.time() - t0, 1),
                per_video={records[v]["video_id"]: m for v, m in enumerate(metas)},
                bad={records[v]["video_id"]: e for v, e in bad.items()})
    p = CFG.out_dir / "cache" / name
    np.savez(p, feat=feats, video_ids=np.array(vids))
    save_json(meta, p.with_suffix(".json"))
    print(f"Đặc trưng frame {feats.shape} sau {meta['seconds']}s → {p}")
    return feats.astype(np.float32), meta, p


def extract_text_features(store, cap_idx=None, allow_encode=True, data_tag=None):
    """[N_caption, D]: CLIP-text đóng băng của các caption cap_idx (mặc định: mọi caption; mục đích 'encode' — không học gì)."""
    cap_idx = np.arange(len(store.vid_of_cap)) if cap_idx is None else np.asarray(cap_idx)
    texts = store.texts(cap_idx, "encode")
    digest = hashlib.sha1("\n".join(texts).encode()).hexdigest()[:10]
    name = _cache_name("text", len(texts), data_tag) + f"_{digest}.npz"
    p = find_cache(name)
    if p is not None:
        feat = np.load(p)["feat"].astype(np.float32)
        assert feat.shape[0] == len(texts)
        print(f"Dùng lại cache đặc trưng caption: {p} {feat.shape}")
        return feat, p
    if not allow_encode:
        raise FileNotFoundError(name)
    t0 = time.time()
    feat = clip_encode_texts(texts).astype(np.float16)
    p = CFG.out_dir / "cache" / name
    np.savez(p, feat=feat)
    print(f"Đặc trưng caption {feat.shape} sau {time.time() - t0:.0f}s → {p}")
    return feat.astype(np.float32), p


def ext_data_tag():
    if not EXT_RECORDS:
        return None
    if CFG.dataset != "msvd":
        return "synthetic_ext"
    return f"msrvtt_{CFG.msrvtt_revision[:8]}_{'dev' + str(len(EXT_RECORDS)) if EXT_REPORT['dev'] else EXT_REPORT.get('set', '1kA')}"


def all_features(allow_encode=True):
    """MSVD (cache v1/v2 dùng lại được: cùng tên, cùng thứ tự) + tập ngoài (cache riêng), nối theo thứ tự RECORDS / caption."""
    fm, mm, pm = extract_frame_features(RECORDS[:N_MSVD], SOURCE, allow_encode)
    in_msvd = CAP_VID < N_MSVD
    assert in_msvd[:in_msvd.sum()].all(), "caption MSVD phải đứng trước caption tập ngoài"
    tm, ptm = extract_text_features(CAPS, np.where(in_msvd)[0], allow_encode)
    caches = [pm, ptm]
    if EXT_RECORDS:
        fe, me, pe = extract_frame_features(RECORDS[N_MSVD:], EXT_SOURCE, allow_encode, data_tag=ext_data_tag())
        te, pte = extract_text_features(CAPS, np.where(~in_msvd)[0], allow_encode, data_tag=ext_data_tag())
        fm, tm = np.concatenate([fm, fe]), np.concatenate([tm, te])
        mm = dict(mm, per_video={**mm["per_video"], **me["per_video"]}, bad={**mm["bad"], **me["bad"]},
                  seconds=round(mm["seconds"] + me["seconds"], 1) if "seconds" in mm and "seconds" in me else None,
                  source=f"{mm['source']}+{me['source']}")
        caches += [pe, pte]
    return fm, mm, tm, caches

### 4.6b Gộp theo thời gian (chọn trên VAL)

Notebook dùng một embedding duy nhất cho mỗi video, gộp từ `F ∈ {4, 8, 12}` frame con, cách đều nhau trong 12 frame đã lấy mẫu. Có hai cách gộp:

* `mean`: chuẩn hoá L2 từng frame, lấy trung bình, rồi chuẩn hoá lại.
* `centrality`: trọng số `softmax(cos(frame, trung bình)/τ)`. Cách này giảm trọng số các frame lạc chủ đề, chẳng hạn frame chuyển cảnh hoặc frame tối; trọng số không phụ thuộc truy vấn, nên mỗi video vẫn chỉ có một vector.

Cặp `(F, cách gộp)` được chọn theo **MRR text→video của CLIP thuần trên VAL**, xếp hạng trong toàn bộ gallery val. Dòng `F=1` (frame giữa) chỉ được in ra để tham chiếu và không được chọn, để bảo đảm embedding thật sự gộp từ nhiều frame.

In [69]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def frame_subset(n_frames):
    """F frame con cách đều trong F_max frame đã lấy mẫu (F=4 ⇒ đúng điểm giữa của 4 đoạn: 1, 4, 7, 10)."""
    n_frames = min(int(n_frames), CFG.n_frames_max)
    pos = np.round((np.arange(n_frames) + 0.5) * CFG.n_frames_max / n_frames - 0.5)
    return np.unique(np.clip(pos, 0, CFG.n_frames_max - 1).astype(int))


def pool_frames(frame_feat, n_frames, mode):
    X = frame_feat[:, frame_subset(n_frames)].astype(np.float32)
    X /= np.linalg.norm(X, axis=-1, keepdims=True).clip(1e-8)
    if mode == "mean" or X.shape[1] == 1:
        v = X.mean(1)
    elif mode == "centrality":
        c = X.mean(1, keepdims=True)
        c /= np.linalg.norm(c, axis=-1, keepdims=True).clip(1e-8)
        a = (X * c).sum(-1) / CFG.centrality_tau
        w = np.exp(a - a.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)
        v = (w[..., None] * X).sum(1)
    else:
        raise ValueError(mode)
    return (v / np.linalg.norm(v, axis=-1, keepdims=True).clip(1e-8)).astype(np.float32)


def full_gallery_ranks(txt, vid, q_idx, gallery, gold):
    """Thứ hạng (1 = tốt nhất) của video đúng trong TOÀN BỘ gallery theo cosine CLIP."""
    G = torch.from_numpy(np.ascontiguousarray(vid[np.asarray(gallery)]))
    pos = {int(v): i for i, v in enumerate(gallery)}
    gi = torch.tensor([pos[int(v)] for v in gold], dtype=torch.long)
    out = np.empty(len(q_idx), np.int64)
    for s in range(0, len(q_idx), CFG.eval_chunk):
        S = torch.from_numpy(np.ascontiguousarray(txt[np.asarray(q_idx[s:s + CFG.eval_chunk])])) @ G.T
        g = S[torch.arange(S.shape[0]), gi[s:s + CFG.eval_chunk]]
        out[s:s + CFG.eval_chunk] = (S > g[:, None]).sum(1).numpy() + 1
    return out


def select_pooling_on_val(frame_feat, txt, split):
    va = np.where(CAP_SPLIT == "val")[0]
    CAPS.use(va, "tune")
    gal, gold = np.array(split["val"]), CAP_VID[va]
    rows = []
    for nf in sorted(set(CFG.n_frames_grid) | {1}):
        for mode in CFG.pool_grid:
            if nf == 1 and mode != "mean":
                continue
            m = metrics_from_ranks(full_gallery_ranks(txt, pool_frames(frame_feat, nf, mode), va, gal, gold), gold)
            rows.append(dict(n_frames=nf, pooling=mode, frames=frame_subset(nf).tolist(),
                             selectable=nf in CFG.n_frames_grid, **{k: m[k] for k in ("R@1", "R@5", "R@10", "MRR")}))
    df = pd.DataFrame(rows)
    best = df[df.selectable].sort_values(["MRR", "n_frames"], ascending=[False, True]).iloc[0]
    return dict(n_frames=int(best.n_frames), pooling=str(best.pooling), val_MRR=float(best.MRR)), df


def gate_b_checks(frame_feat, frame_meta, vid_feat, txt, split):
    N, Fm, D = frame_feat.shape
    ff = frame_feat.reshape(-1, D)
    fn = np.linalg.norm(ff, axis=1)
    m, _, _ = get_clip()
    check("B0", "CLIP đóng băng (eval, không tham số nào requires_grad)",
          not m.training and not any(p.requires_grad for p in m.parameters()), f"{CFG.clip_model}/{CFG.clip_weights}")
    check("B1", "đặc trưng frame có shape [N_video, F_max, D]", (N, Fm) == (len(RECORDS), CFG.n_frames_max),
          f"{frame_feat.shape}")
    check("B2", "đặc trưng frame hữu hạn, chuẩn L2 ≈ 1", bool(np.isfinite(ff).all() and np.abs(fn - 1).max() < 2e-2),
          f"max|‖f‖−1| = {np.abs(fn - 1).max():.2e}")
    vn = np.linalg.norm(vid_feat, axis=1)
    check("B3", "đúng 1 embedding / video, hữu hạn, chuẩn 1, video_id duy nhất",
          vid_feat.shape == (N, D) and bool(np.isfinite(vid_feat).all()) and np.abs(vn - 1).max() < 1e-4
          and len({r["video_id"] for r in RECORDS}) == N, f"{vid_feat.shape}")
    check("B4", "đúng 1 embedding / caption, hữu hạn, cùng chiều với video",
          txt.shape == (len(CAP_VID), D) and bool(np.isfinite(txt).all()), f"{txt.shape}")
    qs = np.random.RandomState(0).choice(len(txt), min(3000, len(txt)), replace=False)
    S = txt[qs] @ vid_feat.T
    check("B5", "cosine văn bản–video nằm trong [−1, 1]", S.min() >= -1 - 1e-3 and S.max() <= 1 + 1e-3,
          f"[{S.min():.3f}, {S.max():.3f}]")
    idx = frame_subset(POOL_SEL["n_frames"])
    Xn = frame_feat[:, idx] / np.linalg.norm(frame_feat[:, idx], axis=-1, keepdims=True).clip(1e-8)
    cos_fv = (Xn * vid_feat[:, None]).sum(-1)
    frac_single = float((cos_fv.max(1) > 0.9999).mean())
    pair = np.einsum("nfd,ngd->nfg", Xn, Xn)
    off = pair[:, ~np.eye(len(idx), dtype=bool)].mean() if len(idx) > 1 else 1.0
    check("B6", "embedding video thật sự gộp từ NHIỀU frame (không vô tình dùng 1 frame)",
          len(idx) > 1 and frac_single < 0.05,
          f"F={len(idx)} frame {idx.tolist()} ({POOL_SEL['pooling']}); tỉ lệ video ≡ 1 frame = {frac_single:.3f}; "
          f"cos TB giữa các frame = {off:.3f}")
    bad_ts = []
    for name, mt in frame_meta["per_video"].items():
        if mt is None:
            bad_ts.append(name); continue
        t = np.array(mt["t_sec"])
        ok = (len(t) == Fm and (t >= 0).all() and (np.diff(t) >= 0).all() and t.max() < mt["duration_s"] + 1e-6
              and max(mt["frame_idx"]) < mt["n_decoded"])
        if not ok:
            bad_ts.append(name)
    check("B7", "lấy mẫu ổn định: đủ F_max frame / video, timestamp tăng dần trong [0, thời lượng)", not bad_ts,
          f"{len(frame_meta['per_video'])} video; lỗi {len(bad_ts)} {bad_ts[:3]}")
    cnt = np.bincount(CAP_VID, minlength=N)
    vsplit = np.array([r["split"] for r in RECORDS])
    probe = np.random.RandomState(1).choice(len(CAP_VID), min(500, len(CAP_VID)), replace=False)
    ok_text = all(CAPS._text[q] in CAPS._caps[int(CAP_VID[q])] for q in probe)        # kiểm tra chỉ số nội bộ
    galleries_ok = all(len(set(split[k])) == len(split[k]) and set(CAP_VID[CAP_SPLIT == k]) == set(split[k])
                       for k in ("train", "val", "test", "ext"))
    check("B8", "mọi caption-truy vấn ánh xạ đúng 1 video dương cùng split; gallery mỗi split không nhân bản video",
          all(cnt[r["vid"]] == r["n_captions"] for r in RECORDS) and bool((vsplit[CAP_VID] == CAP_SPLIT).all())
          and ok_text and galleries_ok,
          f"{len(CAP_VID)} truy vấn → {N} video; caption/video TB {cnt.mean():.1f}; gallery = "
          + ", ".join(f"{k}:{len(split[k])}" for k in ("train", "val", "test", "ext")))
    f2, _, t2, paths = all_features(allow_encode=False)
    check("B9", "cache đặc trưng nạp lại được mà KHÔNG chạy encoder, giống hệt bản trong bộ nhớ",
          np.array_equal(f2, frame_feat) and np.array_equal(t2, txt), " | ".join(p.name for p in paths))

## 4.7 Từ vựng (chỉ TRAIN), quan hệ, tagger và mẫu cho người chấm

Parser quan hệ của v2 được **giữ nguyên**, để đo đúng chất lượng của nó. Mỗi quan hệ là `(nguồn, họ, đích, vị từ)`:
* `agent_of`: chủ thể → động từ;
* `object_of`: đồ vật → động từ. Ở hai họ này vị từ chính là động từ đích;
* `svo`: danh từ –cụm động từ→ danh từ, ví dụ `man –ride→ horse`;
* `prep`: danh từ –giới từ→ danh từ, ví dụ `cat –on→ table`.

**Kiểm chứng tự động (Nhiệm vụ B), không cần người chấm.**
* spaCy `en_core_web_sm` là parser phụ thuộc cú pháp, độc lập với heuristic POS. Nó trích lại quan hệ cùng schema trên **chính caption train**:
  * chủ ngữ (`nsubj`, kể cả kế thừa qua `conj`, `xcomp`, `acl`) → `agent_of`;
  * tân ngữ (`dobj`, `nsubjpass`) → `object_of`;
  * chủ–động–tân → `svo`;
  * giới từ gắn vào **danh từ** → `prep`.
* Một quan hệ heuristic được xác nhận khi spaCy trích cùng (nguồn, họ, đích), cùng động từ đầu (svo) hoặc cùng giới từ (prep). Tỉ lệ xác nhận được tính cho từng quan hệ, từng họ và từng ô (họ × mức phổ biến), kèm khoảng Wilson.
* Độ phủ ngược (spaCy tìm được mà heuristic cũng tìm được) chỉ là so sánh tương đối, **không** phải recall thật.
* Đây là nhãn **silver**: hai parser có thể cùng sai, nên báo cáo không gọi nó là precision.

**Mẫu phân tầng** (họ × mức phổ biến; 360 dòng) vẫn được xuất kèm cột `auto_parser_agrees`. Nếu có file nhãn người chấm (`relation_annotation*`, **tuỳ chọn**), notebook tính thêm precision, Wilson và Cohen's kappa.

In [70]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# --- NLTK: WordNet + POS tagger. Không tải được thì cảnh báo rõ và dùng phương án dự phòng (ghi vào ENV_INFO). ---
NLTK_STATUS = {}
for _pkg in ("wordnet", "omw-1.4", "averaged_perceptron_tagger_eng", "averaged_perceptron_tagger"):
    try:
        NLTK_STATUS[_pkg] = bool(nltk.download(_pkg, quiet=True, raise_on_error=False))
    except Exception as e:
        NLTK_STATUS[_pkg] = f"lỗi {type(e).__name__}"
from nltk.stem import WordNetLemmatizer

_LEM, _LEM_CACHE = WordNetLemmatizer(), {}


def _verb_lemma(w):
    """WordNetLemmatizer trả về ứng viên NGẮN NHẤT ('riding' → 'rid', 'shining' → 'shin'); ở đây chọn ứng viên
    có tần suất WordNet cao nhất ('riding' → 'ride')."""
    wn = _wordnet()
    cands = wn._morphy(w, "v") if wn is not None and hasattr(wn, "_morphy") else []
    if len(cands) > 1:
        cnt = lambda c: sum(l.count() for s in wn.synsets(c, pos="v") for l in s.lemmas() if l.name().lower() == c)
        return max(sorted(cands), key=lambda c: (cnt(c), len(c)))      # sorted: không phụ thuộc PYTHONHASHSEED
    return sorted(cands)[0] if cands else _LEM.lemmatize(w, "v")


def lemma(w, pos="n"):
    k = (w, pos)
    r = _LEM_CACHE.get(k)
    if r is None:
        try:
            if pos == "v":
                r = _verb_lemma(w)
            else:                                   # như NLTK (ứng viên ngắn nhất) nhưng hoà thì theo thứ tự chữ
                wn_ = _wordnet()
                cands = wn_._morphy(w, pos) if wn_ is not None and hasattr(wn_, "_morphy") else []
                r = min(sorted(cands), key=len) if cands else _LEM.lemmatize(w, pos)
        except LookupError:
            r = w[:-1] if (pos == "n" and w.endswith("s") and len(w) > 3) else w
        _LEM_CACHE[k] = r
    return r


try:
    from nltk.tag import PerceptronTagger
    _TAGGER = PerceptronTagger()                    # tạo MỘT lần (pos_tag() tạo lại mỗi lời gọi → rất chậm)
    _TAGGER.tag(["a", "man", "is", "playing"])
except Exception as e:
    _TAGGER = None
    print("⚠ Không nạp được POS tagger NLTK → phân loại từ theo WordNet (kém chính xác hơn):", type(e).__name__)

# --- Lớp khái niệm dự phòng (notebook gốc) — dùng khi WordNet không có ---
FALLBACK_ISA = {
    "ball": ["sports equipment", "toy"], "field": ["open area", "sports venue"],
    "grass": ["vegetation", "ground"], "goal": ["sports equipment"],
    "shoe": ["clothing", "footwear"], "child": ["person"], "boy": ["person", "child"],
    "girl": ["person", "child"], "man": ["person"], "woman": ["person"],
    "person": ["human"], "keeper": ["person"],
    "pot": ["cookware", "container"], "stove": ["appliance", "cooking device"],
    "plate": ["tableware", "container"], "spoon": ["tableware", "utensil"],
    "table": ["furniture"], "bed": ["furniture"], "lamp": ["furniture", "light source"],
    "pillow": ["bedding"], "curtain": ["window covering"], "window": ["building part"],
    "book": ["reading material"], "sand": ["ground"], "umbrella": ["shade device"],
    "wave": ["water"], "towel": ["cloth"], "boat": ["vehicle", "watercraft"],
    "car": ["vehicle"], "truck": ["vehicle"], "bus": ["vehicle"], "bike": ["vehicle"],
    "sign": ["signage"], "road": ["open area", "transport way"],
    "building": ["structure"], "light": ["light source"], "elephant": ["animal"],
    "dog": ["animal"], "cat": ["animal"], "horse": ["animal"], "bird": ["animal"],
    "tree": ["vegetation", "plant"], "fence": ["barrier", "structure"],
    "water": ["liquid"], "rock": ["natural object"], "shirt": ["clothing"],
    "hat": ["clothing"], "chair": ["furniture"], "food": ["edible"],
    "pizza": ["food", "edible"], "plane": ["vehicle"], "train": ["vehicle"],
    "guitar": ["musical instrument"], "piano": ["musical instrument"], "onion": ["vegetable"],
    "apple": ["fruit"], "bread": ["food"], "bicycle": ["vehicle"],
}
FALLBACK_VERB_ISA = {"slice": ["cut"], "chop": ["cut"], "dice": ["cut"], "strum": ["play"], "gallop": ["ride"],
                     "jog": ["run"], "sprint": ["run"], "stir": ["mix"], "whisk": ["mix"], "fry": ["cook"],
                     "bake": ["cook"], "boil": ["cook"], "grill": ["cook"], "sip": ["drink"], "munch": ["eat"]}

# --- Lọc WordNet (notebook gốc): chỉ giữ danh từ chỉ thứ NHÌN THẤY ĐƯỢC ---
VISUAL_LEX = {"noun.person", "noun.animal", "noun.artifact", "noun.object", "noun.food",
              "noun.plant", "noun.substance", "noun.body", "noun.location", "noun.group",
              "noun.phenomenon"}
SPORT_ROOTS = {"sport.n.01", "game.n.01"}
GENERIC_CONCEPTS = {"entity", "physical entity", "abstraction", "object", "whole", "thing",
                    "matter", "artifact", "instrumentality", "organism", "living thing",
                    "causal agent", "location", "region", "group", "act", "event",
                    "unit", "part", "piece", "material", "substance", "device", "commodity",
                    "consumer goods", "covering", "creation", "structure", "area"}
BASIC_CATEGORIES = {"person", "animal", "dog", "bird", "vehicle", "boat", "aircraft",
                    "clothing", "footwear", "headdress", "food", "plant", "tree", "flower",
                    "furniture", "building", "toy", "sport", "ball", "musical instrument",
                    "body of water", "geological formation", "precipitation", "child",
                    "game equipment", "sports equipment", "vegetable", "fruit"}
GENERIC_VERBS = {"act", "move", "change", "make", "be", "have", "do", "get", "go", "cause", "interact",
                 "transfer", "use", "take", "give", "put", "perform", "modify", "remove", "create", "travel",
                 "come", "displace", "connect", "communicate", "express", "refer", "think", "consume"}

_WN = None

_syn_key = lambda x: x.name()
_path_key = lambda path: [x.name() for x in path]      # NLTK lưu con trỏ WordNet trong set ⇒ thứ tự phụ thuộc PYTHONHASHSEED


def _wordnet():
    """Nạp WordNet một lần; không được thì trả None (pipeline rơi về FALLBACK_ISA)."""
    global _WN
    if _WN is None:
        try:
            from nltk.corpus import wordnet as wn
            wn.synsets("dog"); _WN = wn
        except Exception:
            _WN = False
    return _WN or None


def _main_noun_synset(w):
    wn = _wordnet()
    if wn is None:
        return None
    syns = wn.synsets(w.replace(" ", "_"), pos="n")
    return syns[0] if syns else None


_NOUN_CACHE, _VERB_CACHE = {}, {}


def is_visual_noun(w):
    if w in _NOUN_CACHE:
        return _NOUN_CACHE[w]
    wn = _wordnet()
    if wn is None:
        _NOUN_CACHE[w] = True
        return True
    s, ok = _main_noun_synset(w), False
    if s is not None:
        lex = s.lexname()
        if lex in VISUAL_LEX:
            ok = True
        elif lex == "noun.act":
            anc = {h.name() for p in s.hypernym_paths() for h in p[:-1]}
            ok = bool(anc & SPORT_ROOTS)
        if ok:
            key = w.replace(" ", "_")
            cnt = lambda pos: sum(l.count() for x in wn.synsets(key, pos=pos)
                                  for l in x.lemmas() if l.name().lower() == key)
            ok = cnt("n") >= cnt("v")
    _NOUN_CACHE[w] = ok
    return ok


def is_action_verb(v):
    if v not in _VERB_CACHE:
        wn = _wordnet()
        _VERB_CACHE[v] = True if wn is None else bool(wn.synsets(v, pos="v"))
    return _VERB_CACHE[v]


def get_wordnet_hypernyms(words):
    """IsA danh từ (notebook gốc): cha + ông (đủ sâu, không quá chung) + tổ tiên thuộc BASIC_CATEGORIES."""
    if not CFG.use_wordnet:
        return {}
    wn = _wordnet()
    if wn is None:
        return {w: FALLBACK_ISA[w] for w in words if w in FALLBACK_ISA}
    out = {}
    for w in words:
        s = _main_noun_synset(w)
        if s is None:
            continue
        hy = []
        for path in sorted(s.hypernym_paths(), key=_path_key)[:2]:
            for depth in range(len(path) - 2, -1, -1):
                h = path[depth]
                name = h.lemmas()[0].name().replace("_", " ").lower()
                near = depth >= len(path) - 3
                if name == w or name in GENERIC_CONCEPTS or name in hy:
                    continue
                if (near and depth >= CFG.isa_min_depth and
                        (h.lexname() in VISUAL_LEX or h.lexname() == "noun.Tops")) or name in BASIC_CATEGORIES:
                    hy.append(name)
        if hy:
            out[w] = hy[:5]
    for w in words:
        if w not in out and w in FALLBACK_ISA:
            out[w] = FALLBACK_ISA[w]
    return out


def get_verb_hypernyms(verbs):
    """IsA động từ: cha/ông của nghĩa động từ phổ biến nhất (slice → cut, gallop → ride), bỏ động từ quá chung."""
    if not CFG.use_wordnet:
        return {}
    wn, out = _wordnet(), {}
    for v in verbs:
        hy = []
        syns = wn.synsets(v, pos="v") if wn is not None else []
        if syns:
            for path in sorted(syns[0].hypernym_paths(), key=_path_key)[:2]:
                for depth in (len(path) - 2, len(path) - 3):
                    if depth < 0:
                        continue
                    name = path[depth].lemmas()[0].name().replace("_", " ").lower()
                    if name != v and name not in GENERIC_VERBS and " " not in name and name not in hy:
                        hy.append(name)
        for h in FALLBACK_VERB_ISA.get(v, []):
            if h != v and h not in hy:
                hy.append(h)
        if hy:
            out[v] = hy[:3]
    return out


def get_hypernyms(classes, types):
    nouns = [c for c in classes if types.get(c, "noun") == "noun"]
    verbs = [c for c in classes if types.get(c) == "verb"]
    out = get_wordnet_hypernyms(nouns)
    out.update(get_verb_hypernyms(verbs))
    return out


ENV_INFO.update(nltk_status=NLTK_STATUS, wordnet=_wordnet() is not None, pos_tagger=_TAGGER is not None)
print("NLTK:", NLTK_STATUS, "| WordNet:", _wordnet() is not None, "| POS tagger:", _TAGGER is not None)

In [71]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
STOP = set("a an the of on in at to with and or is are was were this that these those "
           "there here it its his her their some any for from by as be being been very "
           "photo image picture showing shows scene view near next video clip".split())
NOT_OBJECTS = set("front background foreground top bottom side end corner air group area region place space "
                  "picture photo image one other something someone somebody anything thing stuff view middle edge "
                  "part way lot pair couple bunch number while each both several many few kind type video clip "
                  "footage scene time minute second".split())
_SKIP_P = set("is are was were be been being has have had while as that who which and or".split())
AUX_VERBS = set("be have do can will would shall should may might must seem appear become begin start try "
                "keep continue let get go come".split())


def _norm(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def _fallback_pos(toks):
    wn, out = _wordnet(), []
    for t in toks:
        if wn is None:
            out.append((t, "NN" if len(t) > 2 and t not in STOP else "DT"))
        elif t.endswith("ing") and wn.synsets(lemma(t, "v"), pos="v"):
            out.append((t, "VBG"))
        elif wn.synsets(t, pos="n"):
            out.append((t, "NN"))
        elif wn.synsets(lemma(t, "v"), pos="v"):
            out.append((t, "VB"))
        else:
            out.append((t, "DT"))
    return out


_PARSE_CACHE = {}


def parse_caption(cap):
    """(danh từ nhìn thấy được, {động từ: dạng V-ing}, bộ ba quan hệ) của MỘT caption — heuristic POS.
    Bộ ba gồm (danh từ, vị từ, danh từ) như notebook gốc + (chủ thể, agent_of, động từ), (đồ vật, object_of,
    động từ) và chiều ngược (done_by, acts_on). Nên đọc thử ~30 mẫu để báo cáo độ chính xác."""
    if cap in _PARSE_CACHE:
        return _PARSE_CACHE[cap]
    toks = re.findall(r"[a-z]+", str(cap).lower())
    tags = (_TAGGER.tag(toks) if _TAGGER is not None else _fallback_pos(toks)) if toks else []
    nouns, verbs, triples = set(), {}, set()
    subj, last_n, preds, cur_v = None, None, [], None
    for w, t in tags:
        if t in ("NN", "NNS") and len(w) > 2 and w not in NOT_OBJECTS and w not in STOP:
            n = lemma(w, "n")
            if n in NOT_OBJECTS or not is_visual_noun(n):
                continue
            nouns.add(n)
            if last_n is not None and 1 <= len(preds) <= 2 and last_n != n:
                triples.add((last_n, " ".join(preds), n))
            if cur_v is not None and n != subj:
                triples.add((n, "object_of", cur_v)); triples.add((cur_v, "acts_on", n))
                cur_v = None
            subj = subj or n
            last_n, preds = n, []
        elif t.startswith("VB"):
            v = lemma(w, "v")
            if v in AUX_VERBS or len(v) < 3 or not is_action_verb(v):
                continue
            if t == "VBG":
                verbs[v] = w
            else:
                verbs.setdefault(v, None)
            if subj is not None and v != subj:
                triples.add((subj, "agent_of", v)); triples.add((v, "done_by", subj))
            if last_n is not None:
                preds.append(v)
            cur_v = v
        elif t in ("IN", "TO", "RP") and w not in _SKIP_P and last_n is not None:
            preds.append(w)
    res = (frozenset(nouns), dict(verbs), frozenset(triples))
    _PARSE_CACHE[cap] = res
    return res


def naive_gerund(v):
    if v.endswith("ie"):
        return v[:-2] + "ying"
    if v.endswith("e") and not v.endswith(("ee", "ye", "oe")):
        return v[:-1] + "ing"
    return v + "ing"


def build_vocab(store, split):
    """Từ vựng khái niệm từ caption TRAIN (mục đích 'vocab' — CaptionStore chặn mọi split khác)."""
    tr = split["train"]
    caps = store.by_video(tr, "vocab")
    df, pos_cnt, ger = Counter(), defaultdict(Counter), defaultdict(Counter)
    for v in tr:
        cnt = Counter()
        for c in caps[v]:
            nouns, verbs, _ = parse_caption(c)
            for n in nouns:
                cnt[n] += 1; pos_cnt[n]["noun"] += 1
            for vb, surf in verbs.items():
                cnt[vb] += 1; pos_cnt[vb]["verb"] += 1
                if surf:
                    ger[vb][surf] += 1
        df.update(t for t, k in cnt.items() if k >= CFG.vocab_min_caps)
    terms = sorted(t for t, k in df.items() if k >= CFG.min_term_df)
    types = {t: ("verb" if pos_cnt[t]["verb"] > pos_cnt[t]["noun"] else "noun") for t in terms}
    gerund = {t: (ger[t].most_common(1)[0][0] if ger[t] else naive_gerund(t)) for t in terms if types[t] == "verb"}
    return dict(terms=terms, types=types, gerund=gerund, df={t: int(df[t]) for t in terms},
                built_from="train", n_train_videos=len(tr), sha1=sha1_of([terms, types]))


def build_relations(store, split, vocab):
    """Bộ ba quan hệ của từng video TRAIN (mục đích 'relations'). Video val/test KHÔNG có quan hệ từ caption."""
    vset = set(vocab["terms"])
    caps = store.by_video(split["train"], "relations")
    rel = {}
    for v in split["train"]:
        trip = set()
        for c in caps[v]:
            trip |= {t for t in parse_caption(c)[2] if t[0] in vset and t[2] in vset and t[0] != t[2]}
        rel[v] = sorted(trip)
    return rel


def caption_truth(store, vids, vocab, purpose):
    """Nhãn tham chiếu: từ của từ vựng xuất hiện ≥ truth_min_caps caption của video (chỉ để CHẤM tagger)."""
    vset, caps, out = set(vocab["terms"]), store.by_video(vids, purpose), {}
    for v in vids:
        cnt = Counter()
        for c in caps[v]:
            n, vb, _ = parse_caption(c)
            cnt.update(set(n) | set(vb))
        out[v] = {t for t, k in cnt.items() if k >= CFG.truth_min_caps and t in vset}
    return out


_TERM_FEAT = {}


def term_text_features(vocab):
    """CLIP-text của prompt cho từng từ (ensemble 3 prompt; danh từ / động từ dạng V-ing)."""
    key = vocab["sha1"] + sha1_of(vocab["gerund"])
    if key not in _TERM_FEAT:
        terms, T = vocab["terms"], None
        for typ, prompts in (("noun", CFG.noun_prompts), ("verb", CFG.verb_prompts)):
            idx = [i for i, t in enumerate(terms) if vocab["types"][t] == typ]
            if not idx:
                continue
            words = [terms[i] if typ == "noun" else vocab["gerund"][terms[i]] for i in idx]
            E = sum(clip_encode_texts([p.format(w) for w in words]) for p in prompts)
            if T is None:
                T = np.zeros((len(terms), E.shape[1]), np.float32)
            T[idx] = E / np.linalg.norm(E, axis=1, keepdims=True)
        _TERM_FEAT[key] = T
    return _TERM_FEAT[key]


def _prf(pred, truth):
    tp = sum(len(p & t) for p, t in zip(pred, truth))
    P = tp / max(sum(map(len, pred)), 1)
    R = tp / max(sum(map(len, truth)), 1)
    return P, R, 2 * P * R / max(P + R, 1e-9)


def run_tagger(frame_feat, vid_feat, n_frames, vocab, split, store, select=None):
    """Gắn nhãn MỌI video bằng CLIP zero-shot trên frame với từ vựng TRAIN — không đọc caption của video được gắn.
    select=None ⇒ chọn (pool, norm, k_noun, k_verb) theo F1 trên VAL. Trả về (tags theo vid, lựa chọn, lưới)."""
    terms = vocab["terms"]
    T = term_text_features(vocab)
    tr = np.array(split["train"])
    N, D = vid_feat.shape
    X = frame_feat[:, frame_subset(n_frames)].astype(np.float32)
    X /= np.linalg.norm(X, axis=-1, keepdims=True).clip(1e-8)
    S_raw = {"video": vid_feat @ T.T, "frame_max": (X.reshape(-1, D) @ T.T).reshape(N, X.shape[1], -1).max(1)}
    cols = {t: np.array([i for i, w in enumerate(terms) if vocab["types"][w] == t], int) for t in ("noun", "verb")}

    def normed(S, mode):
        mu, sd = S[tr].mean(0), S[tr].std(0) + 1e-6                 # thống kê CHỈ trên video TRAIN
        return S if mode == "raw" else ((S - mu) / sd if mode == "z" else S / S[tr].std() + (S - mu) / sd)

    def topk(Sc, rows, kn, kv):
        out = [set() for _ in rows]
        for typ, k in (("noun", kn), ("verb", kv)):
            c = cols[typ]
            if k <= 0 or len(c) == 0:
                continue
            k = min(k, len(c))
            top = np.argpartition(-Sc[np.ix_(rows, c)], k - 1, axis=1)[:, :k]
            for j, row in enumerate(top):
                out[j].update(terms[c[i]] for i in row)
        return out

    grid = []
    if select is None:
        va = np.array(split["val"])
        truth = caption_truth(store, va, vocab, "tagger_select")
        tv = [truth[v] for v in va]
        for pool in CFG.tag_pools:
            for mode in CFG.tag_norms:
                Sc = normed(S_raw[pool], mode)
                for kn in CFG.tag_k_noun_grid:
                    for kv in CFG.tag_k_verb_grid:
                        P, R, F1 = _prf(topk(Sc, va, kn, kv), tv)
                        grid.append(dict(pool=pool, norm=mode, k_noun=kn, k_verb=kv, val_P=P, val_R=R, val_F1=F1))
        best = max(grid, key=lambda g: (round(g["val_F1"], 9), -(g["k_noun"] + g["k_verb"])))
        select = {k: best[k] for k in ("pool", "norm", "k_noun", "k_verb", "val_P", "val_R", "val_F1")}
    Sc = normed(S_raw[select["pool"]], select["norm"])
    tags = topk(Sc, np.arange(N), select["k_noun"], select["k_verb"])
    return {v: sorted(t) for v, t in enumerate(tags)}, select, grid


def tagger_report(tags, vids, vocab, purpose):
    truth = caption_truth(CAPS, vids, vocab, purpose)
    P, R, F1 = _prf([set(tags[v]) for v in vids], [truth[v] for v in vids])
    return dict(P=P, R=R, F1=F1, n_videos=len(vids))

In [72]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
REL_CATS = ("agent_of", "object_of", "svo", "prep")
_PARSE2_CACHE = {}


def ckey(lemma_, pos):
    """Khoá node lớp: 'dog#n', 'play#v'."""
    return f"{lemma_}#{pos}"


def parse_caption_v2(cap):
    """Như parse_caption (v1) nhưng (1) khoá theo (lemma, POS), (2) mỗi quan hệ giữ LOẠI và VỊ TỪ.
    Trả về (danh từ, {động từ: dạng V-ing}, {(nguồn, loại, đích, vị từ)})."""
    if cap in _PARSE2_CACHE:
        return _PARSE2_CACHE[cap]
    toks = re.findall(r"[a-z]+", str(cap).lower())
    tags = (_TAGGER.tag(toks) if _TAGGER is not None else _fallback_pos(toks)) if toks else []
    nouns, verbs, triples = set(), {}, set()
    subj, last_n, preds, pred_has_verb, cur_v = None, None, [], False, None
    for w, t in tags:
        if t in ("NN", "NNS") and len(w) > 2 and w not in NOT_OBJECTS and w not in STOP:
            n = lemma(w, "n")
            if n in NOT_OBJECTS or not is_visual_noun(n):
                continue
            nouns.add(n)
            if last_n is not None and 1 <= len(preds) <= 2 and last_n != n:
                triples.add((ckey(last_n, "n"), "svo" if pred_has_verb else "prep", ckey(n, "n"), " ".join(preds)))
            if cur_v is not None and n != subj:
                triples.add((ckey(n, "n"), "object_of", ckey(cur_v, "v"), cur_v))
                cur_v = None
            subj = subj or n
            last_n, preds, pred_has_verb = n, [], False
        elif t.startswith("VB"):
            v = lemma(w, "v")
            if v in AUX_VERBS or len(v) < 3 or not is_action_verb(v):
                continue
            if t == "VBG":
                verbs[v] = w
            else:
                verbs.setdefault(v, None)
            if subj is not None:
                triples.add((ckey(subj, "n"), "agent_of", ckey(v, "v"), v))
            if last_n is not None:
                preds.append(v); pred_has_verb = True
            cur_v = v
        elif t in ("IN", "TO", "RP") and w not in _SKIP_P and last_n is not None:
            preds.append(w)
    res = (frozenset(nouns), dict(verbs), frozenset(triples))
    _PARSE2_CACHE[cap] = res
    return res


def build_vocab_v2(store, split):
    """Từ vựng khoá (lemma, POS) từ caption TRAIN: một lemma có thể là cả 'x#n' và 'x#v' nếu cả hai đạt ngưỡng."""
    tr = split["train"]
    caps = store.by_video(tr, "vocab")
    df, ger = Counter(), defaultdict(Counter)
    for v in tr:
        cnt = Counter()
        for c in caps[v]:
            nouns, verbs, _ = parse_caption_v2(c)
            cnt.update(ckey(n, "n") for n in nouns)
            for vb, surf in verbs.items():
                cnt[ckey(vb, "v")] += 1
                if surf:
                    ger[vb][surf] += 1
        df.update(k for k, m in cnt.items() if m >= CFG.vocab_min_caps)
    terms = sorted(k for k, m in df.items() if m >= CFG.min_term_df)
    lem = {k: k.rsplit("#", 1)[0] for k in terms}
    types = {k: ("verb" if k.endswith("#v") else "noun") for k in terms}
    gerund = {k: (ger[lem[k]].most_common(1)[0][0] if ger[lem[k]] else naive_gerund(lem[k])) for k in terms if types[k] == "verb"}
    both = sorted({lem[k] for k in terms if types[k] == "noun"} & {lem[k] for k in terms if types[k] == "verb"})
    return dict(terms=terms, types=types, lemma=lem, gerund=gerund, df={k: int(df[k]) for k in terms},
                lemmas_with_two_pos=both, built_from="train", n_train_videos=len(tr), sha1=sha1_of([terms, types]))


def build_relations_v2(store, split, vocab):
    """{vid TRAIN: [(nguồn, loại, đích, vị từ)]} — chỉ video TRAIN (mục đích 'relations')."""
    vset = set(vocab["terms"])
    caps = store.by_video(split["train"], "relations")
    rel = {}
    for v in split["train"]:
        trip = set()
        for c in caps[v]:
            trip |= {t for t in parse_caption_v2(c)[2] if t[0] in vset and t[2] in vset and t[0] != t[2]}
        rel[v] = sorted(trip)
    return rel


def caption_truth_v2(store, vids, vocab, purpose):
    vset, caps, out = set(vocab["terms"]), store.by_video(vids, purpose), {}
    for v in vids:
        cnt = Counter()
        for c in caps[v]:
            n, vb, _ = parse_caption_v2(c)
            cnt.update({ckey(x, "n") for x in n} | {ckey(x, "v") for x in vb})
        out[v] = {t for t, k in cnt.items() if k >= CFG.truth_min_caps and t in vset}
    return out


_TERM_FEAT2 = {}


def term_text_features_v2(vocab):
    """CLIP-text (ensemble 3 prompt) cho từng khoá: danh từ dùng lemma, động từ dùng dạng V-ing."""
    key = vocab["sha1"] + sha1_of(vocab["gerund"])
    if key not in _TERM_FEAT2:
        terms, T = vocab["terms"], None
        for typ, prompts in (("noun", CFG.noun_prompts), ("verb", CFG.verb_prompts)):
            idx = [i for i, t in enumerate(terms) if vocab["types"][t] == typ]
            if not idx:
                continue
            words = [vocab["lemma"][terms[i]] if typ == "noun" else vocab["gerund"][terms[i]] for i in idx]
            E = sum(clip_encode_texts([p.format(w) for w in words]) for p in prompts)
            if T is None:
                T = np.zeros((len(terms), E.shape[1]), np.float32)
            T[idx] = E / np.linalg.norm(E, axis=1, keepdims=True)
        _TERM_FEAT2[key] = T
    return _TERM_FEAT2[key]


def run_tagger_v2(frame_feat, vid_feat, n_frames, vocab, split, store, select=None):
    """Như run_tagger (v1) với khoá (lemma, POS). Trả thêm điểm đã chuẩn hoá của từng nhãn được gắn
    (dùng làm trọng số contains ở S3). Thống kê chuẩn hoá chỉ trên video TRAIN; lựa chọn theo F1 trên VAL."""
    terms = vocab["terms"]
    T = term_text_features_v2(vocab)
    tr = np.array(split["train"])
    N, D = vid_feat.shape
    X = frame_feat[:, frame_subset(n_frames)].astype(np.float32)
    X /= np.linalg.norm(X, axis=-1, keepdims=True).clip(1e-8)
    S_raw = {"video": vid_feat @ T.T, "frame_max": (X.reshape(-1, D) @ T.T).reshape(N, X.shape[1], -1).max(1)}
    cols = {t: np.array([i for i, w in enumerate(terms) if vocab["types"][w] == t], int) for t in ("noun", "verb")}

    def normed(S, mode):
        mu, sd = S[tr].mean(0), S[tr].std(0) + 1e-6
        return S if mode == "raw" else ((S - mu) / sd if mode == "z" else S / S[tr].std() + (S - mu) / sd)

    def topk(Sc, rows, kn, kv):
        out = [dict() for _ in rows]
        for typ, k in (("noun", kn), ("verb", kv)):
            c = cols[typ]
            if k <= 0 or len(c) == 0:
                continue
            k = min(k, len(c))
            sub = Sc[np.ix_(rows, c)]
            top = np.argpartition(-sub, k - 1, axis=1)[:, :k]
            for j, row in enumerate(top):
                out[j].update({terms[c[i]]: float(sub[j, i]) for i in row})
        return out

    grid = []
    if select is None:
        va = np.array(split["val"])
        truth = caption_truth_v2(store, va, vocab, "tagger_select")
        tv = [truth[v] for v in va]
        for pool in CFG.tag_pools:
            for mode in CFG.tag_norms:
                Sc = normed(S_raw[pool], mode)
                for kn in CFG.tag_k_noun_grid:
                    for kv in CFG.tag_k_verb_grid:
                        P, R, F1 = _prf([set(d) for d in topk(Sc, va, kn, kv)], tv)
                        grid.append(dict(pool=pool, norm=mode, k_noun=kn, k_verb=kv, val_P=P, val_R=R, val_F1=F1))
        best = max(grid, key=lambda g: (round(g["val_F1"], 9), -(g["k_noun"] + g["k_verb"])))
        select = {k: best[k] for k in ("pool", "norm", "k_noun", "k_verb", "val_P", "val_R", "val_F1")}
    Sc = normed(S_raw[select["pool"]], select["norm"])
    tagged = topk(Sc, np.arange(N), select["k_noun"], select["k_verb"])
    tags = {v: sorted(d) for v, d in enumerate(tagged)}
    scores = {v: d for v, d in enumerate(tagged)}
    return tags, scores, select, grid


def tagger_report_v2(tags, vids, vocab, purpose):
    truth = caption_truth_v2(CAPS, vids, vocab, purpose)
    P, R, F1 = _prf([set(tags[v]) for v in vids], [truth[v] for v in vids])
    return dict(P=P, R=R, F1=F1, n_videos=len(vids))

In [73]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# --- Chất lượng quan hệ (Nhiệm vụ B): mẫu cho NGƯỜI chấm + đọc nhãn đã chấm. Chỉ caption TRAIN. ---
# Mẫu KHÔNG phụ thuộc CLIP/tagger (chỉ parser + từ vựng train) ⇒ giống nhau giữa máy local và Kaggle.
# Đơn vị mẫu = MỘT quan hệ (họ, nguồn, vị từ, đích) đủ điều kiện thành cạnh (support ≥ scene_min_count video TRAIN),
# kèm MỘT caption TRAIN chọn ngẫu nhiên có chứa nó; người chấm trả lời: caption này có diễn đạt đúng quan hệ đó không?
CaptionStore.ALLOWED["annotation"] = {"train"}
ANNOT_LABEL_COLS = ("annotator_1", "annotator_2", "adjudicated")


def relation_instances(store, split, vocab):
    """[(vid, caption, (nguồn, họ, đích, vị từ))] trên caption TRAIN; support = số video TRAIN có quan hệ (đếm 1 lần / video);
    pred_support[(họ, vị từ)] = số video TRAIN có ít nhất một quan hệ thuộc họ đó với vị từ đó."""
    vset = set(vocab["terms"])
    caps = store.by_video(split["train"], "annotation")
    inst, sup, psup = [], Counter(), Counter()
    for v in split["train"]:
        seen = set()
        for c in caps[v]:
            for t in sorted(parse_caption_v2(c)[2]):
                if t[0] in vset and t[2] in vset and t[0] != t[2]:
                    inst.append((v, c, t)); seen.add(t)
        sup.update(seen)
        psup.update({(t[1], t[3]) for t in seen})
    return inst, sup, psup


def support_stratum(s):
    for lo, hi, name in CFG.annot_strata:
        if lo <= s <= hi:
            return name
    return "below_threshold"


def _display_relation(src, rel, dst, pred):
    a, b = src.rsplit("#", 1)[0], dst.rsplit("#", 1)[0]
    if rel == "agent_of":
        return a, pred, "", f"'{a}' là CHỦ THỂ thực hiện hành động '{pred}'"
    if rel == "object_of":
        return "", pred, a, f"'{a}' là ĐỐI TƯỢNG chịu hành động '{pred}'"
    return a, pred, b, f"{a} —{pred}→ {b}"


def build_annotation_sample(store, split, vocab, records):
    """Phân tầng: họ quan hệ × mức phổ biến (support); trong mỗi ô chọn luân phiên theo vị từ để đa dạng vị từ."""
    inst, sup, psup = relation_instances(store, split, vocab)
    by_trip = defaultdict(list)
    for v, c, t in inst:
        if sup[t] >= CFG.scene_min_count:
            by_trip[t].append((v, c))
    rng = np.random.RandomState(CFG.annot_seed)
    rows, cells = [], []
    for rel in REL_CATS:
        for lo, hi, name in CFG.annot_strata:
            trips = sorted(t for t in by_trip if t[1] == rel and lo <= sup[t] <= hi)
            by_pred = defaultdict(list)
            for t in trips:
                by_pred[t[3]].append(t)
            preds = sorted(by_pred)
            rng.shuffle(preds)
            for p in preds:
                rng.shuffle(by_pred[p])
            chosen = []
            while len(chosen) < CFG.annot_n_per_cell and any(by_pred[p] for p in preds):
                for p in preds:                              # luân phiên vị từ ⇒ không vị từ nào chiếm cả ô
                    if by_pred[p] and len(chosen) < CFG.annot_n_per_cell:
                        chosen.append(by_pred[p].pop())
            dbl = set(rng.choice(len(chosen), min(CFG.annot_n_double_per_cell, len(chosen)), replace=False).tolist()) if chosen else set()
            cells.append(dict(family=rel, stratum=name, support_lo=lo, support_hi=hi, n_relations_in_cell=len(trips),
                              n_predicates_in_cell=len(preds), n_sampled=len(chosen)))
            for j, t in enumerate(chosen):
                v, c = by_trip[t][rng.randint(len(by_trip[t]))]
                s_, p_, o_, txt = _display_relation(t[0], t[1], t[2], t[3])
                vid = records[v]["video_id"]
                rows.append(dict(sample_id=hashlib.sha1("|".join([vid, c, t[1], t[0], t[3], t[2]]).encode()).hexdigest()[:12],
                                 family=rel, caption=c, subject=s_, predicate=p_, object=o_, relation=txt,
                                 double_annotate=int(j in dbl), annotator_1="", annotator_2="", adjudicated="", note="",
                                 video_id=vid, src_node=t[0], dst_node=t[2], stratum=name, relation_support_videos=int(sup[t]),
                                 predicate_support_videos=int(psup[(t[1], t[3])])))
    df = pd.DataFrame(rows)
    df = df.iloc[rng.permutation(len(df))].reset_index(drop=True)      # trộn thứ tự: người chấm không thấy khối theo ô
    return df, pd.DataFrame(cells), sup, psup


def _label(x):
    x = str(x).strip().lower()
    return 1 if x in ("1", "1.0", "đúng", "dung", "true", "yes", "y") else (0 if x in ("0", "0.0", "sai", "false", "no", "n") else None)


def _read_table(p):
    if p.suffix.lower() == ".xlsx":
        return pd.read_excel(p, dtype=str, keep_default_na=False, sheet_name=0)
    for enc in ("utf-8-sig", "cp1252"):                # Excel có thể lưu ';' hoặc mã hoá khác — nhãn chỉ là 1/0/?
        try:
            return pd.read_csv(p, dtype=str, keep_default_na=False, sep=None, engine="python", encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"không đọc được {p}")


def find_label_file():
    """CFG.annot_file, hoặc relation_annotation*.(csv|xlsx) trong /kaggle/input có ít nhất một nhãn ở annotator_1."""
    cands = [Path(CFG.annot_file)] if CFG.annot_file else []
    if RUN_ON_KAGGLE:
        cands += sorted(p for p in Path("/kaggle/input").rglob("relation_annotation*") if p.suffix.lower() in (".csv", ".xlsx"))
    for p in cands:
        try:
            df = _read_table(p)
            if "annotator_1" in df and df["annotator_1"].map(_label).notna().any():
                return p, df
        except Exception as e:
            print(f"⚠ Không đọc được {p}: {type(e).__name__}: {e}")
    return None, None


def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, c - h, c + h


def cohen_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if len(a) == 0:
        return float("nan")
    po = float((a == b).mean())
    pe = float(a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean()))
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")


def relation_quality(labels):
    """Precision theo họ và theo ô (họ × mức phổ biến) từ nhãn NGƯỜI chấm. Nhãn cuối = adjudicated nếu có; nếu không:
    annotator_1 khi chỉ có một người chấm hoặc hai người đồng ý; hai người bất đồng mà chưa phân xử ⇒ loại khỏi precision."""
    L = labels.copy()
    for c in ANNOT_LABEL_COLS:
        L[c + "_v"] = L[c].map(_label) if c in L else None
    final, status = [], []
    for a1, a2, ad in zip(L["annotator_1_v"], L["annotator_2_v"], L["adjudicated_v"]):
        if not pd.isna(ad):
            final.append(ad); status.append("adjudicated")
        elif pd.isna(a1):
            final.append(None); status.append("unlabeled_or_unclear")
        elif pd.isna(a2):
            final.append(a1); status.append("single")
        elif a1 == a2:
            final.append(a1); status.append("agree")
        else:
            final.append(None); status.append("disagree_unresolved")
    L["final"], L["label_status"] = final, status
    both = L[L["annotator_1_v"].notna() & L["annotator_2_v"].notna()]
    rows = []
    for keys, sub in [(("ALL", "ALL"), L)] + [((f, "ALL"), g) for f, g in L.groupby("family")] + \
            [((f, s), g) for (f, s), g in L.groupby(["family", "stratum"])]:
        lab = sub[sub["final"].notna()]
        k, n = int(lab["final"].sum()), len(lab)
        p, lo, hi = wilson(k, n)
        rows.append(dict(family=keys[0], stratum=keys[1], n_rows=len(sub), n_labeled=n, n_correct=k, precision=p, wilson_lo=lo,
                         wilson_hi=hi, n_unclear_or_unlabeled=int((sub["label_status"] == "unlabeled_or_unclear").sum()),
                         n_disagree_unresolved=int((sub["label_status"] == "disagree_unresolved").sum()),
                         verified=bool(keys[1] != "ALL" and n > 0 and lo >= CFG.prec_min_lower)))
    q = pd.DataFrame(rows)
    agree = dict(n_double=len(both), kappa=cohen_kappa(both["annotator_1_v"].astype(int), both["annotator_2_v"].astype(int))
                 if len(both) else float("nan"), raw_agreement=float((both["annotator_1_v"] == both["annotator_2_v"]).mean())
                 if len(both) else float("nan"))
    return q, agree, L

In [74]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# --- Kiểm chứng TỰ ĐỘNG chất lượng quan hệ (không cần người chấm): đồng thuận với một parser phụ thuộc cú pháp ĐỘC LẬP. ---
# Parser của đồ thị là heuristic POS (v2). spaCy (dependency parse, mô hình thống kê khác hẳn) trích lại quan hệ CÙNG schema trên
# CHÍNH caption TRAIN đó; một quan hệ được "xác nhận" nếu spaCy cũng trích ra nó. Đây là nhãn SILVER (đồng thuận giữa hai
# phương pháp) — KHÔNG phải precision do người chấm: hai parser có thể cùng sai. Chỉ caption TRAIN (mục đích 'agreement').
import importlib

CaptionStore.ALLOWED["agreement"] = {"train"}
_NLP, _DEP_CACHE, DEP_STATUS = None, {}, "chưa nạp"
AGREE, AGREE_COVER = None, None                 # gán ở cell chạy từ vựng (chỉ TRAIN)
_NOUN_POS, _SUBJ, _OBJ = ("NOUN", "PROPN"), ("nsubj",), ("dobj", "obj", "nsubjpass")


def dep_parser():
    """Nạp spaCy + en_core_web_sm (tải nếu thiếu và có Internet). Không được ⇒ None (kiểm chứng tự động = NOT RUN)."""
    global _NLP, DEP_STATUS
    if _NLP is None:
        try:
            spacy = ensure_package("spacy", "spacy")
            try:
                _NLP = spacy.load(CFG.dep_model, disable=["ner", "lemmatizer"])
            except OSError:
                if not INTERNET:
                    raise
                r = subprocess.run([sys.executable, "-m", "spacy", "download", CFG.dep_model], capture_output=True, text=True)
                if r.returncode != 0:
                    raise RuntimeError(r.stderr[-500:])
                importlib.invalidate_caches()
                _NLP = spacy.load(CFG.dep_model, disable=["ner", "lemmatizer"])
            DEP_STATUS = f"spaCy {spacy.__version__} / {CFG.dep_model} {_NLP.meta.get('version')}"
        except Exception as e:
            _NLP, DEP_STATUS = False, f"NOT RUN — không nạp được spaCy/{CFG.dep_model}: {type(e).__name__}: {str(e)[:200]}"
        ENV_INFO["dep_parser"] = DEP_STATUS
    return _NLP or None


def _nkey(t):
    return ckey(lemma(t.text.lower(), "n"), "n")


def _vkey(t):
    return ckey(lemma(t.text.lower(), "v"), "v")


def _with_conj(t):
    return [t] + [c for c in t.conjuncts if c.pos_ in _NOUN_POS]


def _subjects(v, depth=0):
    s = [c for c in v.children if c.dep_ in _SUBJ and c.pos_ in _NOUN_POS]
    if not s and depth < 3:
        if v.dep_ in ("conj", "xcomp", "advcl") and v.head.pos_ in ("VERB", "AUX"):
            return _subjects(v.head, depth + 1)                  # "sitting and wagging", "tries to ride"
        if v.dep_ in ("acl", "relcl") and v.head.pos_ in _NOUN_POS:
            return _with_conj(v.head)                            # "a lady making a dish"
    return [x for t in s for x in _with_conj(t)]


def dep_relations(doc):
    """Quan hệ theo CÙNG schema với parser heuristic: agent {(danh, động)}, object {(danh, động)},
    svo {(chủ, đối tượng, động từ)}, prep {(danh, danh, giới từ)} — giới từ phải gắn vào DANH TỪ (không vào động từ)."""
    out = dict(agent=set(), object=set(), svo=set(), prep=set())
    for t in doc:
        if t.pos_ == "VERB":
            v = lemma(t.text.lower(), "v")
            subs = _subjects(t)
            objs = [x for c in t.children if c.dep_ in _OBJ and c.pos_ in _NOUN_POS for x in _with_conj(c)]
            pobjs = [(p.text, x) for p in t.children if p.dep_ == "prep" for c in p.children
                     if c.dep_ == "pobj" and c.pos_ in _NOUN_POS for x in _with_conj(c)]
            out["agent"] |= {(_nkey(s), _vkey(t)) for s in subs}
            out["object"] |= {(_nkey(o), _vkey(t)) for o in objs}
            out["svo"] |= {(_nkey(s), _nkey(o), v) for s in subs for o in objs + [x for _, x in pobjs]}
        elif t.dep_ == "prep" and t.head.pos_ in _NOUN_POS:
            for c in t.children:
                if c.dep_ == "pobj" and c.pos_ in _NOUN_POS:
                    out["prep"] |= {(_nkey(t.head), _nkey(x), t.text.lower()) for x in _with_conj(c)}
    return out


def dep_parse_all(texts):
    nlp = dep_parser()
    if nlp is None:
        return False
    todo = sorted({x.lower() for x in texts} - set(_DEP_CACHE))
    for txt, doc in zip(todo, nlp.pipe(todo, batch_size=512)):
        _DEP_CACHE[txt] = dep_relations(doc)
    return True


def triple_confirmed(t, dep):
    """Quan hệ heuristic (nguồn, họ, đích, vị từ) được spaCy xác nhận trên cùng caption?
    svo: cùng (chủ, đối tượng) và CÙNG động từ đầu của vị từ; prep: cùng (danh, danh) và giới từ thuộc vị từ."""
    s, rel, o, p = t
    if rel == "agent_of":
        return (s, o) in dep["agent"]
    if rel == "object_of":
        return (s, o) in dep["object"]
    if rel == "svo":
        return (s, o, p.split()[0]) in dep["svo"]
    return any((s, o, w) in dep["prep"] for w in p.split())


def build_agreement(store, split, vocab):
    """AGREE[(họ, nguồn, đích, vị từ)] = [số caption TRAIN spaCy xác nhận, số caption TRAIN có quan hệ]; cùng điều kiện từ
    vựng với build_relations_v2. Kèm độ phủ ngược: tỉ lệ quan hệ spaCy (trong từ vựng) mà heuristic cũng trích được."""
    vset = set(vocab["terms"])
    caps = store.by_video(split["train"], "agreement")
    texts = [c for v in split["train"] for c in caps[v]]
    if not dep_parse_all(texts):
        return None, None
    agree, cover = defaultdict(lambda: [0, 0]), Counter()
    for c in texts:
        dep = _DEP_CACHE[c.lower()]
        heur = {t for t in parse_caption_v2(c)[2] if t[0] in vset and t[2] in vset and t[0] != t[2]}
        for t in heur:
            a = agree[(t[1], t[0], t[2], t[3])]
            a[0] += int(triple_confirmed(t, dep)); a[1] += 1
        hk = {(t[1], t[0], t[2]) for t in heur}
        for fam, rel in (("agent", "agent_of"), ("object", "object_of")):
            for s, o in dep[fam]:
                if s in vset and o in vset:
                    cover[(rel, "total")] += 1; cover[(rel, "found")] += int((rel, s, o) in hk)
        for fam in ("svo", "prep"):
            for s, o, _ in dep[fam]:
                if s in vset and o in vset and s != o:
                    cover[(fam, "total")] += 1; cover[(fam, "found")] += int((fam, s, o) in hk)
    return {k: tuple(v) for k, v in agree.items()}, dict(cover)


def agreement_rate(rel, s, o, p):
    a = AGREE.get((rel, s, o, p)) if AGREE else None
    return a[0] / a[1] if a and a[1] else float("nan")


def agreement_quality(agree, cover, sup):
    """Bảng SILVER theo họ và theo ô (họ × mức phổ biến): tỉ lệ caption-instance được parser độc lập xác nhận (Wilson 95%)."""
    rows = []
    cells = defaultdict(lambda: [0, 0])
    for (rel, s, o, p), (k, n) in agree.items():
        st = support_stratum(sup.get((s, rel, o, p), 0))
        for key in ((rel, "ALL"), (rel, st), ("ALL", "ALL")):
            cells[key][0] += k; cells[key][1] += n
    for (rel, st), (k, n) in sorted(cells.items()):
        p_, lo, hi = wilson(k, n)
        cv = (cover.get((rel, "found"), 0) / cover[(rel, "total")]) if st == "ALL" and cover.get((rel, "total")) else float("nan")
        rows.append(dict(kind="silver_parser_agreement", family=rel, stratum=st, n_instances=n, n_confirmed=k, agreement=p_,
                         wilson_lo=lo, wilson_hi=hi, heuristic_covers_spacy=cv,
                         note="đồng thuận heuristic ↔ spaCy trên caption TRAIN; KHÔNG phải precision do người chấm"))
    return pd.DataFrame(rows)

## 4.8 Đặc tả đồ thị v3 (schema)

**Node.** Canonical inventory **cố định**: video MSVD + video tập ngoài, lớp `(lemma, POS)` từ train, khái niệm theo synset WordNet. Ablation chỉ **tắt cạnh**.

**Cạnh.** Mỗi bản ghi cạnh trong `graph_edges_v3.csv` gồm `(src, relation_family, relation, predicate, dst, direction, weight, confidence, source, status)`. Mỗi dòng là **một** quan hệ: một cặp có nhiều vị từ thành nhiều dòng.
* Trường `predicate` **không bao giờ** chứa tên loại quan hệ; có kiểm tra G2 cho điều này.
* `weight` là support (tần suất), **không** phải độ tin cậy ngữ nghĩa.
* `confidence` là `silver <tỉ lệ> (k/n)`: tỉ lệ caption train của quan hệ được parser độc lập xác nhận. Nếu có nhãn người chấm thì là `human <precision>`; không có cả hai thì là `unverified`.
* Cạnh bị loại vẫn được ghi với `status = excluded:<lý do>`.

**Mẫu đường đi.** Gồm 12 mẫu của v2 cộng `P1m_svo` và `P1m_prep`: đường đi `video → a –f[p]→ b` chỉ được tính khi truy vấn nhắc tới vị từ `p`.

In [75]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
FAMILIES = ("agent", "object", "svo", "prep")
FAMILY_REL = {"agent": "agent_of", "object": "object_of", "svo": "svo", "prep": "prep"}
_SCENE_SRC = "caption TRAIN (POS parser heuristic) — đếm số video TRAIN hỗ trợ; giữ nếu ≥ scene_min_count video, ≤ scene_top_k cạnh / (nguồn, loại)"
_SCENE_W = ("support weight = min(1, support / df_train(nguồn)); chiều _inv: min(1, support / df_train(đích)). "
            "Đây là TẦN SUẤT, KHÔNG phải độ tin cậy ngữ nghĩa")
_CONF_REL = "precision của ô (họ × mức phổ biến) theo nhãn NGƯỜI chấm (Wilson 95%); không có nhãn ⇒ 'unverified'"
_PRED_VERB = "lemma động từ = CHÍNH node đích/nguồn động từ ⇒ không mang thông tin riêng của cạnh; ghi lại, KHÔNG đưa vào message"
_PRED_FREE = ("phân phối vị từ của cặp (vị từ → số video TRAIN); embedding vị từ = CLIP-text đóng băng theo prompt "
              "(pred_mode 'clip') hoặc + MLP dư dùng chung (pred_mode 'clip_adapt'); vào message của GNN và mẫu đường đi P1m")
REL_SPEC = {
    "contains":      dict(src="video", tgt="class", direction="video → lớp được gắn", family="contains", inverse=None,
                          source="CLIP zero-shot trên frame của CHÍNH video × từ vựng train (tagger chọn trên val)",
                          weight="1 (binary) | độ tin cậy tagger đã chuẩn hoá (contains = conf, chọn trên val)",
                          predicate="—", confidence="điểm tagger CLIP (chỉ khi contains = conf)",
                          split="frame mọi video; từ vựng + thống kê chuẩn hoá: train", usage="gnn+path"),
    "contains_inv":  dict(src="class", tgt="video", direction="lớp → video chứa nó", family="contains", inverse="contains",
                          source="chiều ngược của contains", weight="= contains", predicate="—", confidence="= contains",
                          split="như contains", usage="gnn"),
    "agent_of":      dict(src="class:noun", tgt="class:verb", direction="chủ thể → hành động", family="agent", inverse=None,
                          source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_VERB, confidence=_CONF_REL, split="train", usage="gnn+path"),
    "agent_of_inv":  dict(src="class:verb", tgt="class:noun", direction="hành động → chủ thể (done_by)", family="agent",
                          inverse="agent_of", source="chiều ngược của agent_of", weight=_SCENE_W, predicate=_PRED_VERB,
                          confidence=_CONF_REL, split="train", usage="gnn+path"),
    "object_of":     dict(src="class:noun", tgt="class:verb", direction="đồ vật → hành động tác động lên nó", family="object",
                          inverse=None, source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_VERB, confidence=_CONF_REL,
                          split="train", usage="gnn+path"),
    "object_of_inv": dict(src="class:verb", tgt="class:noun", direction="hành động → đồ vật (acts_on)", family="object",
                          inverse="object_of", source="chiều ngược của object_of", weight=_SCENE_W, predicate=_PRED_VERB,
                          confidence=_CONF_REL, split="train", usage="gnn+path"),
    "svo":           dict(src="class:noun", tgt="class:noun", direction="chủ thể –vị từ→ đối tượng", family="svo",
                          inverse=None, source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_FREE, confidence=_CONF_REL,
                          split="train", usage="gnn+path"),
    "svo_inv":       dict(src="class:noun", tgt="class:noun", direction="đối tượng –bị vị từ bởi→ chủ thể", family="svo",
                          inverse="svo", source="chiều ngược của svo", weight=_SCENE_W, predicate=_PRED_FREE,
                          confidence=_CONF_REL, split="train", usage="gnn+path"),
    "prep":          dict(src="class:noun", tgt="class:noun", direction="danh từ –giới từ→ danh từ", family="prep",
                          inverse=None, source=_SCENE_SRC, weight=_SCENE_W, predicate=_PRED_FREE, confidence=_CONF_REL,
                          split="train", usage="gnn+path"),
    "prep_inv":      dict(src="class:noun", tgt="class:noun", direction="chiều ngược của prep", family="prep", inverse="prep",
                          source="chiều ngược của prep", weight=_SCENE_W, predicate=_PRED_FREE, confidence=_CONF_REL,
                          split="train", usage="gnn+path"),
    "related":       dict(src="class", tgt="class", direction="đối xứng (lưu cả hai chiều)", family="related", inverse=None,
                          source="NPMI ≥ ngưỡng của nhãn tagger trên video TRAIN; ≥ min_count; ≤ cooc_top_k / nguồn",
                          weight="P(đích | nguồn) = count / df_train(nguồn)", predicate="—",
                          confidence="— (đồng xuất hiện ≠ quan hệ ngữ nghĩa trực tiếp)", split="train (nhãn tagger từ frame)",
                          usage="gnn+path"),
    "isa":           dict(src="class", tgt="concept|class (cùng POS)", direction="lớp → khái niệm tổng quát hơn",
                          family="isa", inverse=None, source="WordNet (hypernym theo biến thể taxonomy)", weight="1",
                          predicate="—", confidence="— (sense mơ hồ ⇒ KHÔNG nối)",
                          split="tri thức ngoài (chọn sense: ngữ cảnh caption TRAIN)", usage="gnn+path"),
    "isa_inv":       dict(src="concept|class", tgt="class", direction="khái niệm → lớp cụ thể", family="isa", inverse="isa",
                          source="chiều ngược của isa", weight="1", predicate="—", confidence="—", split="như isa", usage="gnn"),
}
EDGE_TYPES = list(REL_SPEC)
ETYPE_ID = {t: i for i, t in enumerate(EDGE_TYPES)}
OPEN_TYPES = ("contains", "contains_inv")            # cổng mở sẵn (như v1)
RELATION_TYPES = [t for t in EDGE_TYPES if REL_SPEC[t]["family"] != "contains"]
PRED_FAMILIES = tuple(CFG.pred_families)             # họ có vị từ tự do (svo, prep)
PRED_TYPES = [t for f in PRED_FAMILIES for t in (f, f + "_inv")]
MASKS = {
    "none": (),
    "no_scene": tuple(t for t in EDGE_TYPES if REL_SPEC[t]["family"] in FAMILIES),
    "no_related": ("related",),
    "no_taxonomy": ("isa", "isa_inv"),
    "contains_only": tuple(RELATION_TYPES),
    "no_roles": ("agent_of", "agent_of_inv", "object_of", "object_of_inv"),   # Level 4(b): họ cạnh vai
    "no_svo_prep": tuple(PRED_TYPES),                                          # Level 4(b): họ cạnh có vị từ tự do
}
# P1m_f: đường đi video → lớp a –f[vị từ p]→ lớp b CHỈ được tính khi truy vấn nhắc tới p (khớp vị từ); dùng ở path_mode 'pred_match'
PATTERNS = (["P0_contains"] + [f"P1_{f}" for f in FAMILIES] + ["P2_isa", "P3_related"]
            + [f"P4_{f}_isa" for f in FAMILIES] + ["P5_related_isa"] + [f"P1m_{f}" for f in PRED_FAMILIES])
PATTERN_LEN = [1] + [2] * 4 + [2, 2] + [3] * 4 + [3] + [2] * len(PRED_FAMILIES)
PIDX = {p: i for i, p in enumerate(PATTERNS)}
REL_TEXT = {"contains": "chứa", "agent_of": "là_chủ_thể_của", "agent_of_inv": "do", "object_of": "là_đối_tượng_của",
            "object_of_inv": "tác_động_lên", "svo": "{pred}", "svo_inv": "bị_{pred}_bởi", "prep": "{pred}",
            "prep_inv": "có_{pred}", "related": "đồng_xuất_hiện", "isa": "là_một"}

### 4.8b Taxonomy (giữ nguyên cách dựng của v2)
Khái niệm được khoá theo synset WordNet (ví dụ `animal.n.01`), nên danh từ và động từ không bao giờ bị trộn với nhau. Có bốn biến thể trên **cùng** một tập node (tập node là hợp của các biến thể):

| Biến thể | Cách chọn hypernym |
|---|---|
| `heuristic` | quy tắc v1, lấy sense đầu tiên |
| `filtered` | `heuristic` bỏ các khái niệm "hub" (hơn 15% video train với tới) và khái niệm danh từ quá nông |
| `sense` | chọn sense bằng Extended Lesk trên ngữ cảnh caption **train**; mapping **mơ hồ** thì không nối `isa` và được ghi rõ |
| `none` | không có `isa` |

Mọi mapping (synset, gloss, điểm Lesk, cờ mơ hồ, hub) được xuất ra `graph_taxonomy_audit.csv`.

Ở v3, `filtered` thắng val nhưng thua trên test đã nhìn, nên **không** còn là ứng viên. Stage V4 so sánh `none` (mặc định, đơn giản hơn) với taxonomy không lọc (`heuristic`). `heuristic` chỉ được giữ khi thắng theo luật đăng ký trước. Kiểm tra mơ hồ, tương thích danh/động từ, hypernym quá chung, hub và chu trình vẫn được ghi trong audit.

In [76]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
CaptionStore.ALLOWED["taxonomy"] = {"train"}          # chọn sense WordNet theo ngữ cảnh caption TRAIN
_WTOK = re.compile(r"[a-z]+")


def _head(s):
    return s.lemmas()[0].name().replace("_", " ").lower()


def _noun_targets(s, word):
    """Quy tắc hypernym danh từ của v1 (cha/ông đủ sâu, nhìn thấy được + BASIC_CATEGORIES) — trả về synset."""
    hy = []
    for path in sorted(s.hypernym_paths(), key=_path_key)[:2]:
        for depth in range(len(path) - 2, -1, -1):
            h = path[depth]
            name = _head(h)
            near = depth >= len(path) - 3
            if name == word or name in GENERIC_CONCEPTS or h in hy:
                continue
            if (near and depth >= CFG.isa_min_depth and (h.lexname() in VISUAL_LEX or h.lexname() == "noun.Tops")) \
                    or name in BASIC_CATEGORIES:
                hy.append(h)
    return hy[:5]


def _verb_targets(s, word):
    """Quy tắc hypernym động từ của v1 (cha/ông, bỏ động từ quá chung)."""
    hy = []
    for path in sorted(s.hypernym_paths(), key=_path_key)[:2]:
        for depth in (len(path) - 2, len(path) - 3):
            if depth < 0:
                continue
            h = path[depth]
            name = _head(h)
            if name != word and name not in GENERIC_VERBS and " " not in name and h not in hy:
                hy.append(h)
    return hy[:3]


def _signature(s):
    """Extended Lesk: từ trong định nghĩa, ví dụ, lemma của synset + cha + 10 con (đã lemma hoá)."""
    words = set(_WTOK.findall(s.definition().lower()))
    for e in s.examples():
        words |= set(_WTOK.findall(e.lower()))
    for r in [s] + sorted(s.hypernyms(), key=_syn_key) + sorted(s.hyponyms(), key=_syn_key)[:10]:
        words |= {l.name().lower() for l in r.lemmas() if "_" not in l.name()}
    return ({lemma(w, "n") for w in words} | {lemma(w, "v") for w in words}) - STOP


def build_sense_context(store, split, vset):
    """Ngữ cảnh của mỗi lớp = các danh từ/động từ khác cùng xuất hiện trong caption TRAIN chứa lớp đó."""
    caps = store.by_video(split["train"], "taxonomy")
    ctx = defaultdict(Counter)
    for v in split["train"]:
        for c in caps[v]:
            nouns, verbs, _ = parse_caption_v2(c)
            words = set(nouns) | set(verbs)
            for k in {ckey(n, "n") for n in nouns} | {ckey(x, "v") for x in verbs}:
                if k in vset:
                    ctx[k].update(words - {k.rsplit("#", 1)[0]})
    return ctx


def choose_sense(lem, pos, ctx):
    wn = _wordnet()
    syns = wn.synsets(lem.replace(" ", "_"), pos=pos)[:CFG.sense_max_candidates] if wn is not None else []
    if not syns:
        return None, True, []
    scores = [sum(math.log1p(ctx[w]) for w in sorted(_signature(s)) if w in ctx) for s in syns]   # thứ tự cố định
    order = np.argsort(-np.array(scores), kind="stable")
    b = scores[order[0]]
    s2 = scores[order[1]] if len(syns) > 1 else -1e9
    ambiguous = len(syns) > 1 and (b <= 0 or b - s2 < CFG.sense_margin)
    return syns[order[0]], ambiguous, [(syns[i].name(), round(scores[i], 2)) for i in order[:3]]


def build_taxonomy_v2(vocab, store, split, train_tags):
    """Ba biến thể taxonomy trên cùng một tập khái niệm (hợp) ⇒ inventory không đổi giữa các biến thể:
    heuristic (= quy tắc v1, sense đầu tiên), filtered (bỏ hub/khái niệm nông), sense (chọn sense theo ngữ cảnh
    caption TRAIN; mơ hồ ⇒ KHÔNG nối isa và đánh dấu), none (không isa)."""
    wn = _wordnet()
    vset = set(vocab["terms"])
    ctx = build_sense_context(store, split, vset)
    concept_meta, var = {}, {"heuristic": {}, "sense": {}, "filtered": {}, "none": {}}
    rows = []

    def target_key(h, pos):
        k = ckey(_head(h), pos)
        if k in vset:
            return k                                   # hypernym trùng một lớp cùng POS ⇒ nối thẳng vào lớp đó
        concept_meta.setdefault(h.name(), dict(key=h.name(), lemma=_head(h), pos=pos, synset=h.name(),
                                               gloss=h.definition(), depth=h.min_depth(), source="wordnet"))
        return h.name()

    def fallback(lem, pos):
        names = (FALLBACK_ISA if pos == "n" else FALLBACK_VERB_ISA).get(lem, [])
        out = []
        for nm in names:
            k = ckey(nm, pos)
            if k not in vset:
                k = f"{nm}.{pos}.fallback"
                concept_meta.setdefault(k, dict(key=k, lemma=nm, pos=pos, synset="", gloss="", depth=-1, source="fallback"))
            out.append(k)
        return out

    for k in vocab["terms"]:
        lem, pos = k.rsplit("#", 1)
        syns = wn.synsets(lem.replace(" ", "_"), pos=pos) if wn is not None else []
        tf = _noun_targets if pos == "n" else _verb_targets
        heur = [target_key(h, pos) for h in tf(syns[0], lem)] if syns else fallback(lem, pos)
        s_syn, amb, scores = choose_sense(lem, pos, ctx[k])
        sense = [] if (amb or s_syn is None) else [target_key(h, pos) for h in tf(s_syn, lem)]
        heur = list(dict.fromkeys(t for t in heur if t != k))
        sense = list(dict.fromkeys(t for t in sense if t != k))
        var["heuristic"][k], var["sense"][k] = heur, sense
        rows.append(dict(class_key=k, pos=pos, train_df=vocab["df"][k], n_synsets=len(syns),
                         heuristic_synset=syns[0].name() if syns else "", heuristic_gloss=syns[0].definition() if syns else "",
                         sense_synset=s_syn.name() if s_syn is not None else "", sense_ambiguous=bool(amb),
                         sense_differs=bool(syns) and s_syn is not None and s_syn != syns[0], lesk_top3=scores,
                         heuristic_targets=heur, sense_targets=sense))
    # filtered: bỏ khái niệm 'hub' (với tới > tax_hub_frac video TRAIN) và khái niệm danh từ quá nông
    tr = list(split["train"])
    reach = defaultdict(set)
    for v in tr:
        for c in train_tags.get(v, []):
            for t in var["heuristic"].get(c, []):
                reach[t].add(v)
    hub = {t for t, s in reach.items() if len(s) / max(len(tr), 1) > CFG.tax_hub_frac}
    shallow = {t for t, m in concept_meta.items() if m["pos"] == "n" and 0 <= m["depth"] < CFG.tax_min_depth_filtered}
    for row in rows:
        k = row["class_key"]
        var["filtered"][k] = [t for t in var["heuristic"][k] if t not in hub and t not in shallow]
        row["filtered_targets"] = var["filtered"][k]
        row["dropped_hub"] = [t for t in var["heuristic"][k] if t in hub]
        row["dropped_shallow"] = [t for t in var["heuristic"][k] if t in shallow]
    var["none"] = {k: [] for k in vocab["terms"]}
    hub_rows = sorted(((t, len(s), len(s) / max(len(tr), 1)) for t, s in reach.items()), key=lambda x: -x[1])
    cycles = {}
    for name, tax in var.items():                       # chu trình trên các cạnh lớp → lớp
        nxt = {k: [t for t in ts if t in vset] for k, ts in tax.items()}
        seen, stack, cyc = set(), set(), 0

        def dfs(u):
            nonlocal cyc
            seen.add(u); stack.add(u)
            for w in nxt.get(u, []):
                if w in stack:
                    cyc += 1
                elif w not in seen:
                    dfs(w)
            stack.discard(u)
        for u in nxt:
            if u not in seen:
                dfs(u)
        cycles[name] = cyc
    stats = dict(n_concepts=len(concept_meta), hubs=[dict(target=t, train_videos=n, frac=round(f, 3)) for t, n, f in hub_rows[:25]],
                 n_hub=len(hub), n_shallow=len(shallow), cycles=cycles,
                 n_ambiguous=int(sum(r["sense_ambiguous"] for r in rows)),
                 n_sense_differs=int(sum(r["sense_differs"] for r in rows)),
                 isa_edges={n: int(sum(len(v) for v in t.values())) for n, t in var.items()})
    return var, concept_meta, pd.DataFrame(rows), stats

### 4.8c Đồ thị v3 và hai đối chứng ngẫu nhiên có ràng buộc
**`GraphV3`** giữ topology cặp của v2, nên relation-family-only (G2) trùng với `v2_base`. Mỗi cặp (nguồn, họ, đích) mang **phân phối vị từ** `{vị từ: số video train}`.

**Chính sách cạnh:**
* `all`: như v2;
* `support`: chỉ vị từ có ≥ 5 video train;
* `agreement`: chỉ vị từ của quan hệ được parser độc lập xác nhận ở ≥ 50% caption train của nó (kiểm chứng tự động);
* `precision`: chỉ ô đã kiểm chứng bằng nhãn người (tuỳ chọn).

Một cặp không còn vị từ hợp lệ thì bị loại và lưu lại.

**Đối chứng:**
* **Rewiring** (G5): directed double-edge swap trong từng họ và nhóm kiểu node. Phép này giữ bậc vào/ra, trọng số và **đa tập phân phối vị từ**, vì vị từ đi theo cạnh. `contains` không đổi.
* **Hoán vị vị từ** (G4): hoán vị phân phối vị từ giữa các cạnh **cùng họ** svo/prep. Topology, trọng số và bậc giống hệt; chỉ việc ghép vị từ ↔ cạnh bị phá.

`path_mode` gồm `weighted`, `count`, `degnorm`, `dedup` như v2, và `pred_match` (= `weighted` cộng mẫu P1m khớp vị từ). Engine, `compute_R` và phần giải thích dùng **chung** các ma trận này.

In [77]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
class NodeInventory:
    """Tập node CỐ ĐỊNH cho mọi cấu hình so sánh được: video | lớp (lemma, POS) | khái niệm (synset).
    Ablation chỉ tắt cạnh ⇒ node id, số node, kích thước embedding không đổi."""

    def __init__(self, records, vocab, concept_meta):
        self.n_vid = len(records)
        self.class_keys = list(vocab["terms"])
        self.concept_keys = sorted(concept_meta)
        self.n_cls, self.n_con = len(self.class_keys), len(self.concept_keys)
        self.n_nodes = self.n_vid + self.n_cls + self.n_con
        self.key2id = {k: self.n_vid + i for i, k in enumerate(self.class_keys)}
        self.key2id.update({k: self.n_vid + self.n_cls + i for i, k in enumerate(self.concept_keys)})
        self.id2key = {i: k for k, i in self.key2id.items()}
        self.node_type = np.array(["video"] * self.n_vid + ["class"] * self.n_cls + ["concept"] * self.n_con)
        self.pos = np.array([""] * self.n_vid + [k.rsplit("#", 1)[1] for k in self.class_keys]
                            + [concept_meta[k]["pos"] for k in self.concept_keys])
        self.lemma = {**{self.key2id[k]: vocab["lemma"][k] for k in self.class_keys},
                      **{self.key2id[k]: concept_meta[k]["lemma"] for k in self.concept_keys}}
        self.name = {i: (records[i]["video_id"] if i < self.n_vid else self.id2key[i]) for i in range(self.n_nodes)}
        self.concepts_by_lemma = defaultdict(list)
        for k in self.concept_keys:
            self.concepts_by_lemma[(concept_meta[k]["lemma"], concept_meta[k]["pos"])].append(self.key2id[k])
        self.concept_meta = concept_meta
        self.sha1 = sha1_of([self.n_vid, self.class_keys, self.concept_keys])

    def semantic_type(self, i):
        t = self.node_type[i]
        return "video" if t == "video" else ({"n": "noun", "v": "verb"}[self.pos[i]] + ("_concept" if t == "concept" else ""))

    def is_class(self, i):
        return self.n_vid <= i < self.n_vid + self.n_cls


def contains_calibration(tag_scores, train_vids, inv):
    """Thống kê chuẩn hoá điểm tagger của các nhãn ĐÃ GẮN trên video TRAIN, tách danh từ / động từ."""
    by = defaultdict(list)
    for v in train_vids:
        for k, s in tag_scores[v].items():
            by[k.rsplit("#", 1)[1]].append(s)
    return {p: dict(mu=float(np.mean(x)), sd=float(np.std(x) + 1e-6), lo=float(np.percentile(x, 1)),
                    hi=float(np.percentile(x, 99))) for p, x in by.items()}


def _conf_weight(s, pos, cal, mode):
    c = cal[pos]
    if mode == "sigmoid":                      # z bị chặn ±3 để hạn chế ngoại lai
        w = 1.0 / (1.0 + math.exp(-float(np.clip((s - c["mu"]) / c["sd"], -3, 3))))
    else:                                      # minmax giữa phân vị 1–99 của train
        w = (s - c["lo"]) / max(c["hi"] - c["lo"], 1e-6)
    return float(np.clip(w, CFG.conf_floor, 1.0))


def _csr(n_rows, n_cols, rows, cols, vals):
    if len(rows) == 0:
        return sp.csr_matrix((n_rows, n_cols), dtype=np.float32)
    M = sp.csr_matrix((np.asarray(vals, np.float32), (np.asarray(rows), np.asarray(cols))), shape=(n_rows, n_cols))
    M.sum_duplicates()
    return M


def _symnorm(M):
    if M.nnz == 0:
        return M.copy()
    dout = np.asarray(M.sum(1)).ravel(); din = np.asarray(M.sum(0)).ravel()
    a = np.where(dout > 0, 1 / np.sqrt(np.maximum(dout, 1e-12)), 0)
    b = np.where(din > 0, 1 / np.sqrt(np.maximum(din, 1e-12)), 0)
    return (sp.diags(a) @ M @ sp.diags(b)).tocsr().astype(np.float32)


def _maxprod(C, B):
    """out[v, c] = max_o C[v, o] · B[o, c] (thưa) — tổ hợp 'max' thay cho tổng (path_dedup)."""
    C, B = C.tocsr(), B.tocsr()
    rows = []
    for v in range(C.shape[0]):
        sl = slice(C.indptr[v], C.indptr[v + 1])
        idx, val = C.indices[sl], C.data[sl]
        if len(idx) == 0:
            rows.append(sp.csr_matrix((1, B.shape[1]), dtype=np.float32))
            continue
        sub = (sp.diags(val) @ B[idx]).tocsr()
        rows.append(sp.csr_matrix(sub.max(axis=0)))
    return sp.vstack(rows).tocsr()




PATH_TYPES = ["agent_of", "agent_of_inv", "object_of", "object_of_inv", "svo", "svo_inv", "prep", "prep_inv", "related", "isa"]
REL_POLICIES = ("all", "support", "agreement", "precision")


def predicate_tokens(p):
    return tuple(p.split())


class GraphV3:
    """Đồ thị dị thể có kiểu, có hướng, có trọng số trên NodeInventory cố định; cạnh quan hệ mang VỊ TỪ.
    Cạnh quan hệ = cặp (nguồn, họ, đích) như v2 (cùng topology) + phân phối vị từ của cặp {vị từ: số video TRAIN}.
      rel_policy : 'all' | 'support' (vị từ có ≥ pred_min_support video TRAIN) | 'agreement' (≥ agree_min caption TRAIN được
                   parser độc lập xác nhận — kiểm chứng TỰ ĐỘNG) | 'precision' (ô đã kiểm chứng bằng nhãn người, tuỳ chọn)
                   — cặp không còn vị từ hợp lệ bị LOẠI và lưu trong self.excluded
      path_mode  : 'weighted' | 'count' | 'degnorm' | 'dedup' | 'pred_match' (= weighted + mẫu P1m khớp vị từ truy vấn)
      rewire_seed: constrained rewiring (vị từ, trọng số đi theo cạnh) ; pred_shuffle_seed: hoán vị phân phối vị từ giữa
                   các cạnh CÙNG họ (topology, trọng số giữ nguyên) ; related_cfg: (min_count, min_npmi)."""

    def __init__(self, inv, records, train_vids, relations, taxonomy, *, tag_scores=None, contains="binary",
                 conf_mode="sigmoid", conf_cal=None, rel_policy="all", path_mode="weighted", masked=(),
                 rewire_seed=None, pred_shuffle_seed=None, tags_override=None, related_cfg=None, verified_cells=None, name=""):
        assert rel_policy in REL_POLICIES, rel_policy
        if rel_policy == "precision" and verified_cells is None:
            raise ValueError("rel_policy = precision cần nhãn người chấm (verified_cells) — NOT RUN khi chưa có nhãn")
        if rel_policy == "agreement" and not AGREE:
            raise ValueError("rel_policy = agreement cần kiểm chứng tự động (spaCy) — NOT RUN khi không nạp được parser")
        self.inv, self.name, self.path_mode, self.masked = inv, name, path_mode, tuple(sorted(set(masked)))
        self.contains_mode, self.conf_mode, self.rel_policy = contains, conf_mode, rel_policy
        self.rewire_seed, self.pred_shuffle_seed = rewire_seed, pred_shuffle_seed
        self.related_cfg = tuple(related_cfg or (CFG.cooc_min_count, CFG.cooc_min_npmi))
        n, N = inv.n_nodes, inv.n_vid
        self.n_nodes, self.n_vid = n, N
        train_set = set(int(v) for v in train_vids)
        tr_rows = np.array(sorted(train_set))
        n_tr = max(len(tr_rows), 1)
        tags = tags_override if tags_override is not None else {r["vid"]: r["tags"] for r in records}
        freq = Counter(t for v in train_set for t in tags[v])
        self.active = {inv.key2id[k] for k, c in freq.items() if c >= CFG.min_tag_freq and k in inv.key2id}

        # ---------- contains (video → lớp), từ nhãn tagger của chính video ----------
        rows, cols, vals, self.vid_tags = [], [], [], []
        for v in range(N):
            ids = []
            for k in tags[v]:
                i = inv.key2id.get(k)
                if i is None or i not in self.active:
                    continue
                w = 1.0 if contains == "binary" else _conf_weight(tag_scores[v][k], k.rsplit("#", 1)[1], conf_cal, conf_mode)
                rows.append(v); cols.append(i); vals.append(w); ids.append(i)
            self.vid_tags.append(sorted(ids))
        self.A_con = _csr(N, n, rows, cols, vals)
        Cb = (self.A_con > 0).astype(np.float32)
        cls_df = np.asarray(Cb[tr_rows].sum(0)).ravel()
        self.cls_df = cls_df

        # ---------- quan hệ từ caption TRAIN, TÁCH theo họ, giữ phân phối vị từ ----------
        sup, preds = defaultdict(Counter), defaultdict(Counter)
        for v in train_set:
            seen = set()
            for s, rel, o, pred in relations.get(v, []):
                a, b = inv.key2id.get(s), inv.key2id.get(o)
                if a is None or b is None or a not in self.active or b not in self.active or a == b:
                    continue
                preds[(rel, a, b)][pred] += 1             # = số video TRAIN có đúng (nguồn, họ, đích, vị từ)
                if (rel, a, b) not in seen:
                    sup[rel][(a, b)] += 1; seen.add((rel, a, b))
        self.preds = preds

        def kept_preds(rel, a, b):
            pc = preds[(rel, a, b)]
            if rel_policy == "support":
                return {p: c for p, c in pc.items() if PRED_SUPPORT.get((rel, p), 0) >= CFG.pred_min_support}
            if rel_policy == "precision":
                return {p: c for p, c in pc.items() if (rel, support_stratum(c)) in verified_cells}
            if rel_policy == "agreement":
                ka, kb = inv.id2key[a], inv.id2key[b]
                return {p: c for p, c in pc.items() if agreement_rate(rel, ka, kb, p) >= CFG.agree_min}
            return dict(pc)

        fwd, self.excluded = {}, []
        for rel in REL_CATS:
            by_src = defaultdict(list)
            for (a, b), c in sup[rel].items():
                if c < CFG.scene_min_count:
                    continue
                kp = kept_preds(rel, a, b)
                if not kp:
                    self.excluded.append(dict(src=a, family=rel, dst=b, support=c, predicates=dict(preds[(rel, a, b)]),
                                              reason=f"policy:{rel_policy}"))
                    continue
                by_src[a].append((c, b, kp))
            E = [(a, b, c, kp) for a, lst in by_src.items() for c, b, kp in sorted(lst, key=lambda x: (-x[0], x[1]))[:CFG.scene_top_k]]
            for a, lst in by_src.items():
                for c, b, kp in sorted(lst, key=lambda x: (-x[0], x[1]))[CFG.scene_top_k:]:
                    self.excluded.append(dict(src=a, family=rel, dst=b, support=c, predicates=kp, reason="top_k"))
            f = dict(src=np.array([e[0] for e in E], int), dst=np.array([e[1] for e in E], int),
                     support=np.array([e[2] for e in E], float))
            f["w"] = np.minimum(1.0, f["support"] / np.maximum(cls_df[f["src"]], 1)) if E else np.zeros(0)
            f["w_inv"] = np.minimum(1.0, f["support"] / np.maximum(cls_df[f["dst"]], 1)) if E else np.zeros(0)
            f["pdist"] = [e[3] for e in E]
            fwd[rel] = f
        self.n_candidate_relations = {r: len(sup[r]) for r in REL_CATS}

        # ---------- related: NPMI đồng xuất hiện nhãn tagger trên video TRAIN (đối xứng) ----------
        min_count, min_npmi = self.related_cfg
        cooc = Counter()
        for v in train_set:
            for a, b in itertools.combinations(sorted(self.vid_tags[v]), 2):
                cooc[(a, b)] += 1
        und = {}
        for (a, b), c in cooc.items():
            if c < min_count:
                continue
            p_ab = c / n_tr
            npmi = math.log(p_ab / ((cls_df[a] / n_tr) * (cls_df[b] / n_tr))) / max(-math.log(p_ab), 1e-6)
            if npmi >= min_npmi:
                und[(a, b)] = (c, npmi)
        keep, by_src = set(), defaultdict(list)
        for (a, b), (c, s) in und.items():
            by_src[a].append((s, b)); by_src[b].append((s, a))
        for a, lst in by_src.items():
            for s, b in sorted(lst, reverse=True)[:CFG.cooc_top_k]:
                keep.add((min(a, b), max(a, b)))
        RE = sorted(keep)
        related = dict(src=np.array([e[0] for e in RE], int), dst=np.array([e[1] for e in RE], int),
                       count=np.array([und[e][0] for e in RE], float))
        related["w_ab"] = np.minimum(1.0, related["count"] / np.maximum(cls_df[related["src"]], 1)) if RE else np.zeros(0)
        related["w_ba"] = np.minimum(1.0, related["count"] / np.maximum(cls_df[related["dst"]], 1)) if RE else np.zeros(0)

        # ---------- isa (biến thể taxonomy) ----------
        IE = sorted({(inv.key2id[c], inv.key2id[t]) for c, ts in taxonomy.items() if c in inv.key2id
                     and inv.key2id[c] in self.active for t in ts if t in inv.key2id
                     and (inv.node_type[inv.key2id[t]] == "concept" or inv.key2id[t] in self.active)})
        isa = dict(src=np.array([e[0] for e in IE], int), dst=np.array([e[1] for e in IE], int))
        isa_true = _csr(n, n, isa["src"], isa["dst"], np.ones(len(IE)))

        # ---------- đối chứng 1: constrained rewiring (contains giữ nguyên; vị từ + trọng số đi theo cạnh) ----------
        self.rewire_audit = []
        if rewire_seed is not None:
            rng = np.random.RandomState(rewire_seed)
            for rel in REL_CATS:
                fwd[rel], st = rewire_directed(fwd[rel], inv, rng, ["w", "w_inv", "support", "pdist"], rel)
                self.rewire_audit.append(st)
            related, st = rewire_undirected(related, inv, rng, "related")
            self.rewire_audit.append(st)
            isa, st = rewire_directed(isa, inv, rng, [], "isa")
            self.rewire_audit.append(st)

        # ---------- đối chứng 2: hoán vị phân phối vị từ giữa các cạnh CÙNG họ (topology, trọng số giữ nguyên) ----------
        self.shuffle_audit = []
        if pred_shuffle_seed is not None:
            rng = np.random.RandomState(pred_shuffle_seed)
            for rel in PRED_FAMILIES:
                old = fwd[rel]["pdist"]
                perms = [rng.permutation(len(old)) for _ in range(10)]     # chọn hoán vị đổi được nhiều cạnh nhất
                perm = max(perms, key=lambda pm: sum(old[i] != old[j] for i, j in enumerate(pm)))
                new = [old[i] for i in perm]
                fwd[rel] = dict(fwd[rel], pdist=new)
                ms = lambda L: sorted(json.dumps(sorted(d.items())) for d in L)
                self.shuffle_audit.append(dict(family=rel, n_edges=len(old),
                                               frac_edges_predicates_changed=float(np.mean([a != b for a, b in zip(old, new)])) if old else 0.0,
                                               predicate_multiset_equal=ms(old) == ms(new)))
        self._fwd, self._related, self._isa = fwd, related, isa

        # ---------- ma trận theo từng loại cạnh + bảng vị từ theo cặp ----------
        A = {}
        for rel in REL_CATS:
            f = fwd[rel]
            A[rel] = _csr(n, n, f["src"], f["dst"], f["w"])
            A[rel + "_inv"] = _csr(n, n, f["dst"], f["src"], f["w_inv"])
        rs, rd = related["src"], related["dst"]
        A["related"] = _csr(n, n, np.r_[rs, rd], np.r_[rd, rs], np.r_[related["w_ab"], related["w_ba"]])
        A["isa"] = _csr(n, n, isa["src"], isa["dst"], np.ones(len(isa["src"])))
        A["isa_inv"] = A["isa"].T.tocsr()
        for t in self.masked:
            if t != "contains" and t != "contains_inv":
                A[t] = sp.csr_matrix((n, n), dtype=np.float32)
        self.A = A
        if "contains" in self.masked:
            raise ValueError("không tắt contains trong ablation cạnh quan hệ")
        # pdist[(họ, a, b)] = {vị từ: số video}; pshare[họ][(a, b)] = {pred_id: tỉ phần} cho họ có vị từ tự do
        self.pdist = {(rel, int(a), int(b)): d for rel in REL_CATS for a, b, d in zip(fwd[rel]["src"], fwd[rel]["dst"], fwd[rel]["pdist"])}
        self.pshare = {}
        for rel in PRED_FAMILIES:
            sh = {}
            for (r_, a, b), d in self.pdist.items():
                if r_ == rel and A[rel][a, b] != 0:
                    tot = float(sum(d.values()))
                    sh[(a, b)] = {PRED_ID[(rel, p)]: c / tot for p, c in sorted(d.items()) if (rel, p) in PRED_ID}
            self.pshare[rel] = sh

        # ---------- IDF (taxonomy, chỉ TRAIN) — KHÔNG phụ thuộc mask/rewiring ----------
        df_i = np.asarray(((Cb @ isa_true)[tr_rows] > 0).sum(0)).ravel()
        df = np.where(np.arange(n) < N + inv.n_cls, cls_df, df_i)
        idf = np.log((n_tr + 1.0) / (df + 1.0)).astype(np.float32)
        idf[:N] = 0.0
        self.idf = idf
        self.PM = self.path_mats()
        self.R = self.compute_R()

    # ---------- đường đi ----------
    def pred_expand(self, A, rel):
        """X[u, pid·n + v] = A_t[u, v] · tỉ phần vị từ pid của cặp, cho t ∈ {rel, rel_inv} (đường đi khớp vị từ)."""
        n, rows, cols, vals = self.n_nodes, [], [], []
        sh = self.pshare.get(rel, {})
        for t, inv_dir in ((rel, False), (rel + "_inv", True)):
            m = A[t].tocoo()
            for u, v, w in zip(m.row.tolist(), m.col.tolist(), m.data.tolist()):
                for pid, s in sh.get((v, u) if inv_dir else (u, v), {}).items():
                    rows.append(u); cols.append(pid * n + v); vals.append(w * s)
        return _csr(n, len(PRED_VOCAB) * n, rows, cols, vals)

    def path_mats(self, A=None, A_con=None):
        A = A or self.A
        C = (A_con if A_con is not None else self.A_con).tocsr().astype(np.float32)
        if self.path_mode == "count":
            C = (C > 0).astype(np.float32)
        PM = {"C": C}
        for t in PATH_TYPES:
            M = A[t].tocsr().astype(np.float32)
            if self.path_mode == "count":
                M = (M > 0).astype(np.float32)
            elif self.path_mode == "degnorm":
                M = _symnorm(M)
            PM[t] = M
        if self.path_mode == "pred_match":
            for rel in PRED_FAMILIES:
                PM[f"X_{rel}"] = self.pred_expand(A, rel)
        return PM

    def _fam(self, PM, f):
        a, b = PM[FAMILY_REL[f]], PM[FAMILY_REL[f] + "_inv"]
        return a.maximum(b) if self.path_mode == "dedup" else (a + b)

    def compute_R(self, rows=None, PM=None):
        PM = PM or self.PM
        C = PM["C"] if rows is None else PM["C"][np.asarray(rows)]
        I, Rl = PM["isa"], PM["related"]
        prod = _maxprod if self.path_mode == "dedup" else (lambda X, Y: (X @ Y).tocsr())
        R = {"P0_contains": C.tocsr()}
        for f in FAMILIES:
            CF = prod(C, self._fam(PM, f))
            R[f"P1_{f}"] = CF
            R[f"P4_{f}_isa"] = prod(CF, I)
        R["P2_isa"] = prod(C, I)
        CR = prod(C, Rl)
        R["P3_related"] = CR
        R["P5_related_isa"] = prod(CR, I)
        direct = (C > 0).astype(np.float32)
        if CFG.drop_redundant_indirect:
            for p in PATTERNS[1:]:
                if p in R:
                    R[p] = (R[p] - R[p].multiply(direct)).tocsr()
        D = sp.diags(self.idf)
        out = {}
        for p in PATTERNS:
            if p.startswith("P1m_"):
                continue
            M = (R[p] @ D).tocsr(); M.eliminate_zeros(); out[p] = M
        n, n_p = self.n_nodes, len(PRED_VOCAB)
        for rel in PRED_FAMILIES:
            key = f"X_{rel}"
            if key not in PM:
                out[f"P1m_{rel}"] = sp.csr_matrix((C.shape[0], n_p * n), dtype=np.float32)
                continue
            M = (C @ PM[key]).tocoo()
            b = M.col % n
            keep = np.ones(M.nnz, bool)
            if CFG.drop_redundant_indirect and M.nnz:
                keep = np.asarray(direct[M.row, b]).ravel() == 0
            X = sp.csr_matrix(((M.data * self.idf[b])[keep], (M.row[keep], M.col[keep])), shape=M.shape, dtype=np.float32)
            X.eliminate_zeros()
            out[f"P1m_{rel}"] = X
        return out

    # ---------- cạnh cho GNN (có loại, có trọng số, có chỉ số vị từ) ----------
    def edge_arrays(self):
        """(src, dst, type, weight, pidx): pidx = hàng trong self.pmix (phân phối vị từ của cạnh) hoặc −1."""
        if not hasattr(self, "_edges"):
            parts, mix_rows = [], []
            c = self.A_con.tocoo()
            parts.append((c.row, c.col, np.full(c.nnz, ETYPE_ID["contains"]), c.data, np.full(c.nnz, -1)))
            parts.append((c.col, c.row, np.full(c.nnz, ETYPE_ID["contains_inv"]), c.data, np.full(c.nnz, -1)))
            for t in RELATION_TYPES:
                m = self.A[t].tocoo()
                if not m.nnz:
                    continue
                pidx = np.full(m.nnz, -1)
                if t in PRED_TYPES:
                    base, inv_dir = (t[:-4], True) if t.endswith("_inv") else (t, False)
                    for k, (u, v) in enumerate(zip(m.row.tolist(), m.col.tolist())):
                        d = self.pshare[base].get((v, u) if inv_dir else (u, v), {})
                        if d:
                            pidx[k] = len(mix_rows); mix_rows.append(d)
                parts.append((m.row, m.col, np.full(m.nnz, ETYPE_ID[t]), m.data, pidx))
            s, d_, e, w, pi = (np.concatenate([p[i] for p in parts]) for i in range(5))
            r_, c_, v_ = [], [], []
            for i, dd in enumerate(mix_rows):
                for pid, sh in dd.items():
                    r_.append(i); c_.append(pid); v_.append(sh)
            self.pmix = _csr(len(mix_rows), len(PRED_VOCAB), r_, c_, v_)
            self._edges = (s.astype(np.int64), d_.astype(np.int64), e.astype(np.int64), w.astype(np.float32), pi.astype(np.int64))
        return self._edges

    def edge_counts(self):
        out = {"contains": int(self.A_con.nnz), "contains_inv": int(self.A_con.nnz)}
        out.update({t: int(self.A[t].nnz) for t in RELATION_TYPES})
        return out

    def predicates_of(self, rel, a, b):
        """{vị từ: số video} của cạnh (rel, a → b) THEO ĐÚNG CHIỀU lưu trong đồ thị (đã tính rewiring / hoán vị vị từ)."""
        base = rel[:-4] if rel.endswith("_inv") else rel
        return self.pdist.get((base, b, a) if rel.endswith("_inv") else (base, a, b), {})

    def predicate(self, rel, a, b):
        d = self.predicates_of(rel, a, b)
        return max(sorted(d), key=lambda p: d[p]) if d else ""

    def summary(self):
        return dict(name=self.name, nodes=self.n_nodes, videos=self.n_vid, classes=self.inv.n_cls, concepts=self.inv.n_con,
                    active_classes=len(self.active), edges=self.edge_counts(), contains=self.contains_mode,
                    path_mode=self.path_mode, rel_policy=self.rel_policy, related_cfg=list(self.related_cfg),
                    masked=list(self.masked), rewired=self.rewire_seed is not None,
                    predicate_shuffled=self.pred_shuffle_seed is not None, excluded=len(self.excluded),
                    paths={p: int(self.R[p].nnz) for p in PATTERNS})

In [78]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def rewire_directed(E, inv, rng, attrs, rel):
    """Directed double-edge swap TRONG một loại quan hệ: (a→b),(c→d) ⇒ (a→d),(c→b) khi a,c cùng kiểu và b,d cùng
    kiểu (semantic type: noun / verb / noun_concept / verb_concept). Giữ out-degree, in-degree, kiểu đầu/cuối,
    số cạnh; cấm self-loop và cạnh trùng. Thuộc tính (trọng số, support, phân phối vị từ) gắn với cạnh theo NGUỒN."""
    src, dst = E["src"].copy(), E["dst"].copy()
    m = len(src)
    st = dict(relation=rel, n_edges=m, attempts=0, swaps=0, fail_self=0, fail_dup=0)
    if m < 2:
        st.update(frac_changed=0.0, success_rate=0.0)
        return dict(E, src=src, dst=dst), st
    key = np.array([f"{inv.semantic_type(a)}>{inv.semantic_type(b)}" for a, b in zip(src, dst)])
    edges = set(zip(src.tolist(), dst.tolist()))
    for g in np.unique(key):
        I = np.where(key == g)[0]
        if len(I) < 2:
            continue
        for _ in range(CFG.rewire_swaps_per_edge * len(I)):
            i, j = rng.choice(I, 2, replace=False)
            a, b, c, d = src[i], dst[i], src[j], dst[j]
            st["attempts"] += 1
            if a == d or c == b:
                st["fail_self"] += 1; continue
            if (a, d) in edges or (c, b) in edges:
                st["fail_dup"] += 1; continue
            edges -= {(a, b), (c, d)}; edges |= {(a, d), (c, b)}
            dst[i], dst[j] = d, b
            st["swaps"] += 1
    orig = set(zip(E["src"].tolist(), E["dst"].tolist()))
    st["frac_changed"] = 1 - len(orig & set(zip(src.tolist(), dst.tolist()))) / m
    st["success_rate"] = st["swaps"] / max(st["attempts"], 1)
    return dict(E, src=src, dst=dst), st


def rewire_undirected(E, inv, rng, rel):
    """Double-edge swap cho quan hệ ĐỐI XỨNG (related): giữ bậc vô hướng, cặp kiểu, không self-loop/trùng.
    Cạnh lưu (a, b) có kiểu(a) ≤ kiểu(b); số đếm đồng xuất hiện gắn với cạnh."""
    src, dst = E["src"].copy(), E["dst"].copy()
    wab, wba = E["w_ab"].copy(), E["w_ba"].copy()            # P(b|a), P(a|b) gắn với cạnh, đổi chỗ khi lật hướng
    m = len(src)
    st = dict(relation=rel, n_edges=m, attempts=0, swaps=0, fail_self=0, fail_dup=0)
    if m < 2:
        st.update(frac_changed=0.0, success_rate=0.0)
        return dict(E, src=src, dst=dst, w_ab=wab, w_ba=wba), st
    T = lambda x: inv.semantic_type(x)

    def flip(k):
        src[k], dst[k] = dst[k], src[k]
        wab[k], wba[k] = wba[k], wab[k]

    for k in range(m):                                  # định hướng theo kiểu để swap giữ cặp kiểu
        if T(src[k]) > T(dst[k]):
            flip(k)
    key = np.array([f"{T(a)}~{T(b)}" for a, b in zip(src, dst)])
    edges = {(min(a, b), max(a, b)) for a, b in zip(src.tolist(), dst.tolist())}
    for g in np.unique(key):
        I = np.where(key == g)[0]
        if len(I) < 2:
            continue
        same_type = g.split("~")[0] == g.split("~")[1]
        for _ in range(CFG.rewire_swaps_per_edge * len(I)):
            i, j = rng.choice(I, 2, replace=False)
            if same_type and rng.rand() < 0.5:
                flip(j)                                 # cùng kiểu ⇒ cho phép cả hai cách ghép
            a, b, c, d = src[i], dst[i], src[j], dst[j]
            st["attempts"] += 1
            n1, n2 = (min(a, d), max(a, d)), (min(c, b), max(c, b))
            if a == d or c == b or n1 == n2:
                st["fail_self"] += 1; continue
            if n1 in edges or n2 in edges:
                st["fail_dup"] += 1; continue
            edges -= {(min(a, b), max(a, b)), (min(c, d), max(c, d))}; edges |= {n1, n2}
            dst[i], dst[j] = d, b                       # (a, d) và (c, b); trọng số theo nguồn a, c
            st["swaps"] += 1
    orig = {(min(a, b), max(a, b)) for a, b in zip(E["src"].tolist(), E["dst"].tolist())}
    st["frac_changed"] = 1 - len(orig & edges) / m
    st["success_rate"] = st["swaps"] / max(st["attempts"], 1)
    return dict(E, src=src, dst=dst, w_ab=wab, w_ba=wba), st


def _pred_multiset(g, rel):
    """Đa tập phân phối vị từ của các cạnh thuộc họ rel (chỉ cạnh còn trong đồ thị)."""
    if rel not in REL_CATS:
        return []
    return sorted(json.dumps(sorted(d.items())) for (r_, a, b), d in g.pdist.items() if r_ == rel and g.A[rel][a, b] != 0)


def randomization_audit(g0, g1):
    """So sánh đồ thị gốc và đồ thị rewired theo từng loại cạnh; áp tiêu chí chấp nhận đăng ký trước."""
    inv, rows = g0.inv, []
    stats = {s["relation"]: s for s in g1.rewire_audit}
    for t in ["contains"] + RELATION_TYPES:
        M0 = g0.A_con if t == "contains" else g0.A[t]
        M1 = g1.A_con if t == "contains" else g1.A[t]
        c0, c1 = M0.tocoo(), M1.tocoo()
        e0, e1 = set(zip(c0.row.tolist(), c0.col.tolist())), set(zip(c1.row.tolist(), c1.col.tolist()))
        tp = lambda c: Counter(f"{inv.semantic_type(a) if t != 'contains' else 'video'}>{inv.semantic_type(b)}"
                               for a, b in zip(c.row, c.col))
        base = t[:-4] if t.endswith("_inv") else t
        s = stats.get(base, {})
        row = dict(relation=t, edges_orig=len(e0), edges_rand=len(e1),
                   out_degree_equal=bool(np.array_equal(np.asarray((M0 > 0).sum(1)).ravel(), np.asarray((M1 > 0).sum(1)).ravel())),
                   in_degree_equal=bool(np.array_equal(np.asarray((M0 > 0).sum(0)).ravel(), np.asarray((M1 > 0).sum(0)).ravel())),
                   type_pairs_equal=tp(c0) == tp(c1), weights_multiset_equal=bool(np.allclose(np.sort(c0.data), np.sort(c1.data))),
                   self_loops=int(sum(a == b for a, b in e1)), duplicates=int(c1.nnz - len(e1)),
                   predicates_multiset_equal=_pred_multiset(g0, base) == _pred_multiset(g1, base),
                   frac_changed=1 - len(e0 & e1) / max(len(e0), 1), swaps=s.get("swaps"), attempts=s.get("attempts"),
                   success_rate=s.get("success_rate"), fail_dup=s.get("fail_dup"), fail_self=s.get("fail_self"))
        n_s, n_t = len({a for a, _ in e0}), len({b for _, b in e0})
        row["density"] = len(e0) / max(n_s * n_t, 1)      # ≈ 1 ⇒ gần như KHÔNG có cách sắp xếp khác giữ nguyên bậc
        must_change = t != "contains" and len(e0) >= 10
        row["accepted"] = bool(row["edges_orig"] == row["edges_rand"] and row["out_degree_equal"] and row["in_degree_equal"]
                               and row["type_pairs_equal"] and row["weights_multiset_equal"] and row["self_loops"] == 0
                               and row["duplicates"] == 0 and row["predicates_multiset_equal"]
                               and (row["frac_changed"] >= CFG.rewire_min_changed or not must_change)
                               and (t != "contains" or row["frac_changed"] == 0))
        row["note"] = ("contains phải giữ nguyên" if t == "contains" else "< 10 cạnh: không yêu cầu đổi" if not must_change
                       else ("đổi < ngưỡng: đồ thị quá đặc (mật độ cao) — không có đủ cách sắp xếp giữ bậc"
                             if not row["accepted"] and row["density"] > 0.5 else ""))
        rows.append(row)
    df = pd.DataFrame(rows)
    return df, bool(df.accepted.all())


def predicate_shuffle_audit(g0, g1):
    """Đối chứng hoán vị vị từ: topology, trọng số, bậc của MỌI loại cạnh phải giống hệt; chỉ phân phối vị từ của cạnh
    thuộc họ có vị từ tự do được đổi chỗ (đa tập giữ nguyên)."""
    rows = []
    stats = {s["family"]: s for s in g1.shuffle_audit}
    for t in ["contains"] + RELATION_TYPES:
        M0 = g0.A_con if t == "contains" else g0.A[t]
        M1 = g1.A_con if t == "contains" else g1.A[t]
        base = t[:-4] if t.endswith("_inv") else t
        same_topology = (M0 != M1).nnz == 0
        moved = 0
        if base in PRED_FAMILIES:
            c = M0.tocoo()
            for u, v in zip(c.row.tolist(), c.col.tolist()):
                moved += int(g0.predicates_of(t, u, v) != g1.predicates_of(t, u, v))
        st = stats.get(base, {})
        row = dict(relation=t, n_edges=int(M0.nnz), topology_and_weights_equal=bool(same_topology),
                   predicates_multiset_equal=_pred_multiset(g0, base) == _pred_multiset(g1, base),
                   edges_with_changed_predicates=moved, frac_edges_predicates_changed=moved / max(int(M0.nnz), 1),
                   shuffled=base in PRED_FAMILIES, family_frac_changed=st.get("frac_edges_predicates_changed"))
        row["accepted"] = bool(same_topology and row["predicates_multiset_equal"]
                               and (moved == 0 if base not in PRED_FAMILIES else (M0.nnz < 10 or row["frac_edges_predicates_changed"] >= CFG.rewire_min_changed)))
        rows.append(row)
    df = pd.DataFrame(rows)
    return df, bool(df.accepted.all())

In [79]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def query_tokens_v2(text):
    """[(POS dự đoán, lemma danh từ, lemma động từ)] — POS tagger trên CHÍNH câu truy vấn (input của hệ thống)."""
    toks = re.findall(r"[a-z]+", str(text).lower())
    tags = (_TAGGER.tag(toks) if _TAGGER is not None else _fallback_pos(toks)) if toks else []
    out = []
    for w, t in tags:
        if w in STOP or len(w) < 2:
            continue
        p = "v" if t.startswith("VB") else ("n" if t.startswith("NN") else "x")
        out.append((p, lemma(w, "n"), lemma(w, "v")))
    return out


def parse_tokens_v2(toks, inv):
    """Khớp token với node: lớp đúng POS trước (danh từ → 'x#n', động từ → 'x#v'), không có thì POS còn lại,
    không có lớp thì khái niệm taxonomy cùng lemma + POS. Trả về [(node_id, tên node, token)]."""
    hits, seen = [], set()
    for p, ln, lv in toks:
        order = [("v", lv), ("n", ln)] if p == "v" else [("n", ln), ("v", lv)]
        found = None
        for pos, lm in order:
            i = inv.key2id.get(ckey(lm, pos))
            if i is not None:
                found = [i]; break
        if found is None:
            for pos, lm in order:
                if inv.concepts_by_lemma.get((lm, pos)):
                    found = inv.concepts_by_lemma[(lm, pos)]; break
        for i in found or []:
            if i not in seen:
                seen.add(i); hits.append((i, inv.name[i], lv if p == "v" else ln))
    return hits


def query_pred_tokens(text):
    """Tập token của truy vấn để khớp VỊ TỪ: từ gốc + lemma động từ (giới từ giữ nguyên; KHÔNG bỏ stop-word)."""
    w = re.findall(r"[a-z]+", str(text).lower())
    return frozenset(w) | frozenset(lemma(x, "v") for x in w)


_PRED_BY_FIRST = {}


def match_predicates(qset, rel):
    """pred_id của họ rel có MỌI token nằm trong truy vấn ('ride on' khớp 'a man is riding on a horse')."""
    if rel not in _PRED_BY_FIRST:
        idx = defaultdict(list)
        for i, (r, p) in enumerate(PRED_VOCAB):
            if r == rel:
                t = predicate_tokens(p)
                idx[t[0]].append((i, t))
        _PRED_BY_FIRST[rel] = idx
    idx = _PRED_BY_FIRST[rel]
    return sorted({i for tok in qset for i, t in idx.get(tok, ()) if all(x in qset for x in t)})

## 4.9 Mô hình v3

**`TypedSAGEv3`.** Công thức là `h'_v = W₀h_v + Σ_t σ(g_t)·[W_t·AGG_t(h_u) + U_t·AGG_t(e_uv)]`.
* `e_uv` là biểu diễn vị từ của cạnh. Nó chỉ có ở các loại cạnh svo, svo_inv, prep, prep_inv, và chỉ khi `pred_mode ≠ none`.
* `AGG_t` dùng **cùng hệ số** cho `h_u` và `e_uv`, nên trọng số cạnh chỉ nhân một lần.
* Vị từ của agent_of/object_of chính là node động từ đích, nên không đưa vào message, tránh trùng thông tin.

**`pred_mode`:**
* `none` = relation-family-only, đúng mô hình v2;
* `clip` (phương án A) = CLIP-text **đóng băng** của vị từ theo prompt, qua một phép chiếu tuyến tính;
* `clip_adapt` (phương án B) = A cộng MLP dư **dùng chung** cho mọi vị từ, lớp cuối khởi tạo 0. Không có ID riêng cho từng vị từ, nên vị từ hiếm hoặc chưa gặp vẫn mã hoá được.

`mp_weight` (`binary` / `weighted_mean` / `weighted_symnorm`) và `init` như v2.
### 4.9b Mô hình v4 — QC-RGAT (GNN quan hệ có attention điều kiện theo truy vấn)

**Vì sao đổi mô hình.** Ở v3, truy vấn chỉ đi vào GNN dưới dạng **một vector CLIP toàn cục**. Cạnh quan hệ nối lớp với lớp và là thống kê **dùng chung** cho mọi video, nên mọi video có cùng nhãn đều nhận cùng "ngữ cảnh quan hệ". Vì vậy quan hệ chỉ làm mượt embedding lớp, và kết quả Kaggle v3 cho G3 ≈ G2 ≈ G4 ≈ G5. Ở v4, **chính truy vấn quyết định cạnh quan hệ nào được truyền tin**.

**Công thức** (T = 1 hoặc 2 lớp; chọn trên VAL):
* `H_q⁽⁰⁾(c) = E_c` và `H_q⁽ᵗ⁺¹⁾(c) = E_c + Σ_(u –τ[p]→ c) a_q(u,τ,p,c) · z_{t,τ} ⊙ H_q⁽ᵗ⁾(u)`. Ở đây `z_{t,τ}` là toán tử của loại quan hệ τ, phép nhân theo chiều kiểu CompGCN/DistMult.
* `a_q = σ(β_τ + κ_τ·log w + [W_g u_q]_τ + ρ_{τ, vai_q(u)} + γ_τ·1[p ∈ vị từ truy vấn] + γ'_τ·cos(e_p, e_P(q)))` gồm năm thành phần:
  * cổng theo loại quan hệ do truy vấn chọn (`W_g u_q`);
  * **vai** của node nguồn trong truy vấn (chủ thể / động từ / đối tượng / tân ngữ giới từ, do parser caption áp lên chính truy vấn);
  * **khớp vị từ** giữa cạnh và truy vấn, cả khớp đúng lẫn khớp mềm bằng CLIP-text;
  * support của cạnh (`κ_τ·log w`);
  * độ lệch `β_τ`.
* `s(q, v) = ⟨û_q, W_v x_v + Σ_c Ā_con[v,c] · H_q⁽ᵀ⁾(c)⟩`. Video **chỉ nhận** qua nhãn của chính nó (inductive). Điểm cuối = `α·z(cos CLIP) + (1−α)·z(s)`, với α chọn trên VAL.

**Khởi tạo và học các cổng.** γ_τ khởi tạo bằng 1, tức prior "ưu tiên cạnh có vị từ khớp truy vấn"; dữ liệu có thể làm γ giảm. Các cổng vô hướng (β, κ, ρ, γ, γ') học với lr × `scalar_lr_mult`, như cổng của v3. Nếu khởi tạo γ = 0 và học với lr thường, γ gần như không đổi sau huấn luyện. Khi đó G2, G4, G6 cho kết quả trùng G3, và các đối chứng không còn đo được gì; lần thử dev đã gặp đúng trường hợp này.

**Giải thích từ chính cơ chế.** Attention chỉ phụ thuộc truy vấn và thuộc tính tĩnh của cạnh, nên phép truyền là **tuyến tính theo E**. Vì vậy `s` tách **chính xác** thành tổng các số hạng: thị giác + từng nhãn contains + từng **đường đi quan hệ** (`u –τ[p]→ c ←chứa– v`, có thêm một bước khi T = 2). Giải thích là top-m số hạng quan hệ. Xoá một cạnh làm điểm đổi **đúng bằng** tổng đóng góp của các đường đi qua cạnh đó (T = 1); điều này có kiểm thử Q3 và X3.

**Đối chứng tách từng thành phần** (cùng mô hình, cùng giao thức huấn luyện):
* G1 `v4_rel=False`: không cạnh quan hệ;
* G2 `v4_pred=False`: không khớp vị từ;
* G4: hoán vị vị từ giữa các cạnh cùng họ;
* G5: rewiring có ràng buộc;
* G6 `v4_qcond=False`: attention tĩnh, không theo truy vấn — tức là quan hệ như v3;
* R3: chính mô hình v3 cuối.

Liên hệ tài liệu: NBFNet (Zhu et al., NeurIPS 2021) với message điều kiện theo truy vấn; KGAT (Wang et al., KDD 2019) với attention nhận biết quan hệ; CompGCN (Vashishth et al., ICLR 2020) với toán tử quan hệ. Thuộc họ **GAT quan hệ** của cấp độ 1.


In [80]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def scatter_sum(src, index, n):
    out = torch.zeros((n,) + tuple(src.shape[1:]), device=src.device, dtype=src.dtype)
    return out.index_add_(0, index, src)


PRED_MODES = ("none", "clip", "clip_adapt")


class TypedSAGEv3(nn.Module):
    """h'_v = W₀ h_v + Σ_t σ(g_t) · [ W_t · AGG_t(h_u) + U_t · AGG_t(e_uv) ]
    e_uv = embedding vị từ của cạnh (chỉ loại cạnh thuộc họ có vị từ tự do, khi pred_mode ≠ 'none'); AGG_t dùng CÙNG hệ số
    chuẩn hoá cho h_u và e_uv ⇒ message(u → v) = W_t h_u + U_t e_uv đúng như thiết kế, trọng số cạnh chỉ nhân MỘT lần.
    AGG theo mp_weight: 'binary' (trung bình, bỏ trọng số) | 'weighted_mean' (Σ w·x / Σ w) | 'weighted_symnorm' (Σ w·x / √(d_u d_v)).
    Nhánh W₀ h_v giữ biểu diễn của node không có hàng xóm (không sinh NaN)."""

    def __init__(self, din, dout, n_types, open_types, gate_init, mp_weight, pred_type_ids=()):
        super().__init__()
        assert mp_weight in ("binary", "weighted_mean", "weighted_symnorm"), mp_weight
        self.mp_weight, self.n_types = mp_weight, n_types
        self.self_w = nn.Linear(din, dout)
        self.neigh_w = nn.ModuleList([nn.Linear(din, dout, bias=False) for _ in range(n_types)])
        self.pred_type_ids = list(pred_type_ids)
        self.pred_w = nn.ModuleDict({str(t): nn.Linear(din, dout, bias=False) for t in self.pred_type_ids})
        g = torch.full((n_types,), float(gate_init))
        for t in open_types:
            g[t] = 3.0
        self.gate = nn.Parameter(g)

    def coef(self, src, dst, w, n):
        """Hệ số của từng cạnh trong AGG (cùng cho h_u và e_uv)."""
        if self.mp_weight == "binary":
            w = torch.ones_like(w)
        if self.mp_weight in ("binary", "weighted_mean"):
            return w / scatter_sum(w, dst, n)[dst].clamp(min=1e-12)
        d_out, d_in = scatter_sum(w, src, n), scatter_sum(w, dst, n)
        return w / torch.sqrt(d_out[src] * d_in[dst]).clamp(min=1e-12)

    def forward(self, x, edge_index, edge_type, edge_weight, edge_pidx=None, pred_h=None):
        src, dst = edge_index
        n = x.size(0)
        out = self.self_w(x)
        for t in range(self.n_types):
            m = edge_type == t
            if not bool(m.any()):
                continue
            s, d = src[m], dst[m]
            c = self.coef(s, d, edge_weight[m], n).unsqueeze(-1)
            msg = self.neigh_w[t](scatter_sum(x[s] * c, d, n))
            if pred_h is not None and t in self.pred_type_ids:
                pi = edge_pidx[m]
                has = pi >= 0
                if bool(has.any()):
                    msg = msg + self.pred_w[str(t)](scatter_sum(pred_h[pi[has]] * c[has], d[has], n))
            out = out + torch.sigmoid(self.gate[t]) * msg
        return out


class PredicateEncoder(nn.Module):
    """Biểu diễn vị từ của cạnh = Σ_p tỉ phần(p) · φ(e_p), e_p = CLIP-text ĐÓNG BĂNG của vị từ theo prompt.
      'clip'       (phương án A): φ = phép chiếu tuyến tính học được (về chiều GNN)
      'clip_adapt' (phương án B): φ(e) = chiếu(e + MLP(e)); MLP DÙNG CHUNG cho mọi vị từ, lớp cuối khởi tạo 0
                                  ⇒ không có tham số riêng cho vị từ hiếm; vị từ mới (ngoài từ vựng) vẫn mã hoá được."""

    def __init__(self, clip_dim, dout, mode, hidden):
        super().__init__()
        self.mode = mode
        self.proj = nn.Linear(clip_dim, dout)
        if mode == "clip_adapt":
            self.res = nn.Sequential(nn.Linear(clip_dim, hidden), nn.GELU(), nn.Linear(hidden, clip_dim))
            nn.init.zeros_(self.res[2].weight); nn.init.zeros_(self.res[2].bias)

    def forward(self, pmix, pemb):
        e = pemb + self.res(pemb) if self.mode == "clip_adapt" else pemb
        return torch.sparse.mm(pmix, self.proj(e)) if pmix.is_sparse else pmix @ self.proj(e)


class GraphScorerV3(nn.Module):
    """Như GraphScorer v2 (12 mẫu đường đi + 2 mẫu khớp vị từ; khởi tạo node 'learned' | 'clip_frozen' | 'clip_residual')
    cộng nhánh vị từ (pred_mode). pred_mode = 'none' ⇒ đúng mô hình v2 (đối chứng relation-family-only)."""

    def __init__(self, n_nodes, clip_dim, vid_feat, sym_feat, pred_emb=None, cfg=CFG, mp_weight="binary", init="learned",
                 pred_mode="none"):
        super().__init__()
        assert pred_mode in PRED_MODES, pred_mode
        self.n_vid, self.init, self.pred_mode = vid_feat.shape[0], init, pred_mode
        self.register_buffer("vid_feat", vid_feat.clone())
        self.vid_proj = nn.Linear(clip_dim, cfg.gnn_dim)
        n_sym = n_nodes - self.n_vid
        if init == "learned":
            self.sym_emb = nn.Embedding(n_sym, cfg.gnn_dim)
            nn.init.normal_(self.sym_emb.weight, std=0.02)
        else:
            self.register_buffer("sym_feat", torch.as_tensor(sym_feat, dtype=torch.float32).clone())
            self.sym_proj = nn.Linear(sym_feat.shape[1], cfg.gnn_dim)
            if init == "clip_residual":
                self.sym_res = nn.Embedding(n_sym, cfg.gnn_dim)
                nn.init.zeros_(self.sym_res.weight)
        pred_ids = [ETYPE_ID[t] for t in PRED_TYPES] if pred_mode != "none" else []
        if pred_mode != "none":
            self.register_buffer("pred_emb", torch.as_tensor(pred_emb, dtype=torch.float32).clone())
            self.pred_enc = PredicateEncoder(self.pred_emb.shape[1], cfg.gnn_dim, pred_mode, cfg.pred_adapt_hidden)
        open_ids = [ETYPE_ID[t] for t in OPEN_TYPES]
        self.layers = nn.ModuleList([TypedSAGEv3(cfg.gnn_dim, cfg.gnn_dim, len(EDGE_TYPES), open_ids, cfg.gate_init, mp_weight, pred_ids)
                                     for _ in range(cfg.gnn_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(cfg.gnn_dim) for _ in range(cfg.gnn_layers)])
        self.drop = nn.Dropout(cfg.gnn_dropout)
        self.q_proj = nn.Sequential(nn.Linear(clip_dim, cfg.gnn_dim), nn.GELU(), nn.Linear(cfg.gnn_dim, cfg.gnn_dim))
        init_w = torch.full((len(PATTERNS),), float(cfg.path_init_indirect)); init_w[0] = 0.0
        self.path_w = nn.Parameter(init_w)
        self.decay = nn.Parameter(torch.tensor(0.5))
        self.register_buffer("path_gain", torch.tensor(1.0))
        self.clip_logit = nn.Parameter(torch.tensor(0.5413))

    def node_features(self):
        if self.init == "learned":
            sym = self.sym_emb.weight
        else:
            sym = self.sym_proj(self.sym_feat)
            if self.init == "clip_residual":
                sym = sym + self.sym_res.weight
        return torch.cat([self.vid_proj(self.vid_feat), sym], 0)

    def edge_gates(self):
        return [torch.sigmoid(l.gate).detach().cpu().numpy() for l in self.layers]

    def predicate_features(self, pmix):
        return self.pred_enc(pmix, self.pred_emb) if self.pred_mode != "none" and pmix is not None else None

    def encode_graph(self, edge_index, edge_type, edge_weight, edge_pidx=None, pmix=None):
        h = self.node_features()
        ph = self.predicate_features(pmix)
        for i, (l, nrm) in enumerate(zip(self.layers, self.norms)):
            h = nrm(l(h, edge_index, edge_type, edge_weight, edge_pidx, ph))
            if i < len(self.layers) - 1:
                h = self.drop(F.gelu(h))
        return F.normalize(h, dim=-1)

    def pattern_weights(self):
        d = torch.sigmoid(self.decay)
        lens = torch.tensor(PATTERN_LEN, device=self.path_w.device, dtype=torch.float32)
        w = F.softplus(self.path_w) * d.pow(lens - 1)
        return torch.cat([torch.ones(1, device=w.device, dtype=w.dtype), w[1:]])

    def gnn_score(self, h, q_clip, cand_idx):
        q = F.normalize(self.q_proj(q_clip), dim=-1)
        return torch.einsum("bd,bkd->bk", q, h[cand_idx])

    def path_score(self, path_mat):
        return F.softplus(self.path_gain) * (path_mat * self.pattern_weights()).sum(-1)


_SYM_FEAT = {}


def symbolic_features(inv, vocab):
    """CLIP-text của node lớp (= prompt ensemble của tagger) và khái niệm ('lemma: định nghĩa WordNet')."""
    key = inv.sha1
    if key not in _SYM_FEAT:
        T = term_text_features_v2(vocab)
        cls = np.stack([T[vocab["terms"].index(k)] for k in inv.class_keys])
        texts = [f"{inv.concept_meta[k]['lemma']}: {inv.concept_meta[k]['gloss']}".strip(": ") for k in inv.concept_keys]
        con = clip_encode_texts(texts) if texts else np.zeros((0, cls.shape[1]), np.float32)
        _SYM_FEAT[key] = np.concatenate([cls, con]).astype(np.float32)
    return _SYM_FEAT[key]


def predicate_phrase(rel, p, vocab):
    """Cụm chữ cho prompt: svo → dạng V-ing của động từ đầu + phần còn lại ('ride on' → 'riding on'); prep → nguyên văn."""
    toks = p.split()
    if rel == "svo" and toks:
        g = vocab["gerund"].get(ckey(toks[0], "v")) or naive_gerund(toks[0])
        toks = [g] + toks[1:]
    return " ".join(toks)


def predicate_embeddings(pred_vocab, vocab):
    """[n_vị_từ, D] CLIP-text ĐÓNG BĂNG (trung bình các prompt CFG.pred_prompts[họ], chuẩn L2)."""
    out = None
    for rel in PRED_FAMILIES:
        idx = [i for i, (r, _) in enumerate(pred_vocab) if r == rel]
        if not idx:
            continue
        words = [predicate_phrase(rel, pred_vocab[i][1], vocab) for i in idx]
        E = sum(clip_encode_texts([pr.format(w) for w in words]) for pr in CFG.pred_prompts[rel])
        if out is None:
            out = np.zeros((len(pred_vocab), E.shape[1]), np.float32)
        out[idx] = E / np.linalg.norm(E, axis=1, keepdims=True)
    return out if out is not None else np.zeros((len(pred_vocab), 1), np.float32)

# ---------------------------------- v4: QC-RGAT (GNN quan hệ có attention ĐIỀU KIỆN THEO TRUY VẤN) ----------------------------------
# H_q^(0)(c) = E_c ;  H_q^(t+1)(c) = E_c + Σ_(u –τ[p]→ c) a_q(u,τ,p,c) · z_{t,τ} ⊙ H_q^(t)(u)            (cạnh giữa lớp/khái niệm)
# a_q = σ( β_τ + κ_τ·log w_uc + [W_g u_q]_τ + ρ_{τ, vai_q(u)} + γ_τ·1[p ∈ vị từ truy vấn] + γ'_τ·cos(e_p, e_P(q)) )
# s(q, v) = ⟨û_q, W_v x_v + Σ_c Ā_con[v, c] · H_q^(T)(c)⟩        (Ā_con = contains chuẩn hoá theo hàng, video CHỈ nhận)
# Phép truyền TUYẾN TÍNH theo E (attention chỉ phụ thuộc truy vấn + thuộc tính tĩnh của cạnh) ⇒ s = Σ đóng góp của từng
# thành phần: thị giác + từng nhãn contains + từng ĐƯỜNG ĐI quan hệ — phân rã CHÍNH XÁC, giải thích = đúng các số hạng của điểm.
V4_REL_TYPES = list(RELATION_TYPES)
V4_TID = {ETYPE_ID[t]: i for i, t in enumerate(V4_REL_TYPES)}
Q_ROLES = ("none", "subj", "verb", "obj", "pobj", "other")


def query_roles_v4(texts, inv):
    """Vai của node lớp trong CHÍNH truy vấn (parser caption áp lên văn bản truy vấn lúc suy luận — mục đích query_input)."""
    out = []
    for text in texts:
        _, _, trip = parse_caption_v2(text)
        role = {}
        for s_, rel, o_, _p in trip:
            a, b = inv.key2id.get(s_), inv.key2id.get(o_)
            if a is None or b is None:
                continue
            if rel == "agent_of":
                role.setdefault(a, "subj"); role.setdefault(b, "verb")
            elif rel == "object_of":
                role.setdefault(a, "obj"); role.setdefault(b, "verb")
            elif rel == "svo":
                role.setdefault(a, "subj")
                role[b] = role[b] if role.get(b) not in (None, "other") else "obj"
            elif rel == "prep":
                role.setdefault(b, "pobj")
        out.append({k: Q_ROLES.index(r) for k, r in role.items()})
    return out


class RelDataV4:
    """Tensor của một đồ thị cho QC-RGAT: cạnh quan hệ (giữa node lớp/khái niệm), vị từ của cạnh, contains chuẩn hoá theo hàng."""

    def __init__(self, graph, drop_edges=None):
        self.graph = graph
        self.N, self.n = graph.n_vid, graph.n_nodes
        self.ns = self.n - self.N
        s, d, e, w, pi = graph.edge_arrays()
        keep = np.isin(e, list(V4_TID))
        assert (s[keep] >= self.N).all() and (d[keep] >= self.N).all(), "cạnh quan hệ chỉ được nối node lớp/khái niệm"
        self.gidx = np.where(keep)[0]                       # chỉ số cạnh trong graph.edge_arrays() (để giải thích / xoá cạnh)
        if drop_edges is not None:
            self.gidx = np.setdiff1d(self.gidx, np.asarray(list(drop_edges), np.int64))
        s, d, e, w, pi = s[self.gidx], d[self.gidx], e[self.gidx], w[self.gidx], pi[self.gidx]
        self.E = len(self.gidx)
        self.src_np, self.dst_np, self.pidx_np = s - self.N, d - self.N, pi
        self.src = torch.from_numpy(self.src_np).to(DEVICE)
        self.dst = torch.from_numpy(self.dst_np).to(DEVICE)
        self.et = torch.tensor([V4_TID[int(x)] for x in e], dtype=torch.long, device=DEVICE)
        self.logw = torch.from_numpy(np.log(np.maximum(w, 1e-3)).astype(np.float32)).to(DEVICE)
        PE = np.zeros((self.E, PRED_EMB.shape[1]), np.float32)
        rows = np.where(pi >= 0)[0]
        if len(rows):
            X = np.asarray(graph.pmix[pi[rows]] @ PRED_EMB)
            PE[rows] = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-8)
        self.PE = torch.from_numpy(PE).to(DEVICE)
        M = (graph.pmix[np.maximum(pi, 0)].multiply((pi >= 0)[:, None]) if self.E and (pi >= 0).any()
             else sp.csr_matrix((self.E, len(PRED_VOCAB)), dtype=np.float32))   # G8: không cạnh nào mang vị từ
        self.Mex = sp.csr_matrix(M).T.tocsr()               # [n_vị_từ, E]: phần vị từ p trên cạnh e
        A = (graph.A_con.tocsr()[:, self.N:] > 0).astype(np.float32).toarray()
        self.Acon = torch.from_numpy(A / np.maximum(A.sum(1, keepdims=True), 1)).to(DEVICE)

    def query_batch(self, q_idx, use_pred=True):
        """Đầu vào phía truy vấn: node khớp + vai, vị từ truy vấn (khớp đúng + CLIP mềm) trên TỪNG cạnh, CLIP-text."""
        q_idx = np.asarray(q_idx, np.int64)
        B = len(q_idx)
        bi, ci, ri = [], [], []
        role_src = torch.zeros(B, self.ns, dtype=torch.long)
        for b, q in enumerate(q_idx):
            for h in V4_HITS[q]:
                r = V4_ROLES[q].get(h, Q_ROLES.index("verb" if INV.pos[h] == "v" else "other"))
                bi.append(b); ci.append(h - self.N); ri.append(r)
                role_src[b, h - self.N] = r
        ex = np.zeros((B, self.E), np.float32)
        qp = np.zeros((B, PRED_EMB.shape[1]), np.float32)
        if use_pred and self.E:
            for b, q in enumerate(q_idx):
                ids = [p for rel in PRED_FAMILIES for p in QPREDS[q][rel]]
                if ids:
                    ex[b] = np.minimum(np.asarray(self.Mex[ids].sum(0)).ravel(), 1.0)
                    m = PRED_EMB[ids].mean(0)
                    qp[b] = m / max(np.linalg.norm(m), 1e-8)
        qp_t = torch.from_numpy(qp).to(DEVICE)
        return dict(q=q_idx, bi=torch.tensor(bi, dtype=torch.long, device=DEVICE), ci=torch.tensor(ci, dtype=torch.long, device=DEVICE),
                    ri=torch.tensor(ri, dtype=torch.long, device=DEVICE), role_src=role_src.to(DEVICE),
                    ex=torch.from_numpy(ex).to(DEVICE), soft=qp_t @ self.PE.T,
                    tq=torch.from_numpy(np.ascontiguousarray(TXT_FEAT[q_idx], dtype=np.float32)).to(DEVICE))


class QCRGAT(nn.Module):
    """rel=False ⇒ G1 (chỉ contains); use_pred=False ⇒ G2 (không vị từ); qcond=False ⇒ G6 (attention tĩnh, không theo truy vấn)."""

    SCALARS = ("beta", "kappa", "rgate", "g_ex", "g_soft")      # cổng vô hướng: lr × CFG.scalar_lr_mult (như cổng của v3)

    def __init__(self, data, clip_dim, d=64, T=1, rel=True, qcond=True, use_pred=True):
        super().__init__()
        assert T in (1, 2), "phân rã đường đi hiện cài cho T = 1 hoặc 2"
        self.D, self.T, self.rel, self.qcond, self.use_pred = data, T, rel, qcond, use_pred
        nt = len(V4_REL_TYPES)
        self.vid = nn.Linear(clip_dim, d)
        self.q = nn.Sequential(nn.Linear(clip_dim, d), nn.GELU(), nn.Linear(d, d))
        self.qo = nn.Linear(d, d)
        self.cls = nn.Embedding(data.ns, d)
        nn.init.normal_(self.cls.weight, std=0.02)
        self.z = nn.Parameter(1 + 0.1 * torch.randn(T, nt, d))
        self.beta = nn.Parameter(torch.full((nt,), -1.0))
        self.kappa = nn.Parameter(torch.zeros(nt))
        self.qgate = nn.Linear(d, nt)
        self.rgate = nn.Parameter(torch.zeros(len(Q_ROLES), nt))
        self.g_ex = nn.Parameter(torch.ones(nt))         # prior: cạnh có vị từ KHỚP truy vấn được ưu tiên (học tiếp từ dữ liệu)
        self.g_soft = nn.Parameter(torch.zeros(nt))
        self.clip_logit = nn.Parameter(torch.tensor(0.5413))

    def attention(self, qb, u, D=None):
        """a_q(cạnh) ∈ (0, 1), [B, E]. Chỉ phụ thuộc truy vấn + thuộc tính tĩnh của cạnh (KHÔNG phụ thuộc trạng thái node)."""
        D = D or self.D
        a = (self.beta[D.et] + self.kappa[D.et] * D.logw)[None].expand(u.shape[0], -1)
        if self.qcond:
            a = a + self.qgate(u)[:, D.et] + self.rgate[qb["role_src"][:, D.src], D.et]
            if self.use_pred:
                a = a + self.g_ex[D.et][None] * qb["ex"] + self.g_soft[D.et][None] * qb["soft"]
        return torch.sigmoid(a)

    def class_states(self, qb, u, D=None):
        D = D or self.D
        E = self.cls.weight[None].expand(u.shape[0], -1, -1)
        if not self.rel or D.E == 0:
            return E, None
        a = self.attention(qb, u, D)
        H = E
        for t in range(self.T):
            H = E.index_add(1, D.dst, a[:, :, None] * self.z[t][D.et][None] * H[:, D.src, :])
        return H, a

    def score(self, qb, cand, D=None):
        """cand: [B, K] video id (tensor) ⇒ [B, K]."""
        D = D or self.D
        u = self.q(qb["tq"])
        H, _ = self.class_states(qb, u, D)
        V = self.vid(VID_T[cand]) + torch.einsum("bkc,bcd->bkd", D.Acon[cand], H)
        return torch.einsum("bd,bkd->bk", F.normalize(self.qo(u), dim=-1), V)

    @torch.no_grad()
    def decompose(self, q, vid, D=None):
        """Phân rã CHÍNH XÁC s(q, vid) = thị giác + Σ nhãn + Σ đường đi quan hệ. Mỗi đường đi: list chỉ số cạnh (trong D)
        theo chiều đi từ node nguồn tới node nhãn c của video, kèm đóng góp."""
        D = D or self.D
        qb = D.query_batch([q], self.use_pred)
        u = self.q(qb["tq"])
        uo = F.normalize(self.qo(u), dim=-1)[0]
        E = self.cls.weight
        vt = int(vid)
        terms = [dict(kind="visual", node=None, edges=(), contrib=float(uo @ self.vid(VID_T[vt])))]
        acon = D.Acon[vt]
        tags = torch.nonzero(acon > 0).flatten().tolist()
        for c in tags:
            terms.append(dict(kind="contains", node=c, edges=(), contrib=float(acon[c] * (uo @ E[c]))))
        if self.rel and D.E:
            a = self.attention(qb, u, D)[0]
            inc = defaultdict(list)
            for k in range(D.E):
                inc[int(D.dst_np[k])].append(k)
            for c in tags:
                for e1 in inc.get(c, []):                     # đường đi 1 bước: u –e1→ c   (dùng z của lớp cuối)
                    zz = self.z[self.T - 1][D.et[e1]]
                    terms.append(dict(kind="relation", node=c, edges=(e1,),
                                      contrib=float(acon[c] * a[e1] * (uo @ (zz * E[D.src_np[e1]])))))
                    if self.T == 2:                           # đường đi 2 bước: w –e0→ u –e1→ c
                        for e0 in inc.get(int(D.src_np[e1]), []):
                            zz2 = self.z[1][D.et[e1]] * self.z[0][D.et[e0]]
                            terms.append(dict(kind="relation", node=c, edges=(e0, e1),
                                              contrib=float(acon[c] * a[e1] * a[e0] * (uo @ (zz2 * E[D.src_np[e0]])))))
        return terms


VID_T = None


def v4_tensors():
    global VID_T
    VID_T = torch.from_numpy(np.ascontiguousarray(VID_FEAT, dtype=np.float32)).to(DEVICE)


## 4.10 Engine và hai giao thức suy luận
* **strict inductive** (mặc định): video val, test và tập ngoài **chỉ nhận** message qua `contains_inv`, nên không ảnh hưởng node dùng chung hay video khác. Có unit test cho cả video test MSVD và video MSR-VTT.
* **transductive**: chỉ để báo cáo.

Engine mang thêm chỉ số vị từ của từng cạnh (`edge_pidx`) và bảng phân phối vị từ (`pmix`) cho GNN. Nó cũng có ma trận truy vấn × (vị từ, node) cho mẫu P1m. Vị từ của truy vấn được khớp khi mọi token của vị từ có mặt trong truy vấn, ví dụ `ride on` khớp "a man is riding on a horse".

In [81]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def znorm_per_row(x, eps=1e-6):
    if isinstance(x, torch.Tensor):
        return (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + eps)
    return (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + eps)


def combine(s_clip, s_gnn, s_path, alpha, lam):
    return alpha * s_clip + (1 - alpha) * (lam * s_gnn + (1 - lam) * s_path)


PROTOCOLS = ("inductive", "transductive")


def _torch_sparse(M):
    M = M.tocoo()
    idx = torch.from_numpy(np.vstack([M.row, M.col]).astype(np.int64))
    return torch.sparse_coo_tensor(idx, torch.from_numpy(M.data.astype(np.float32)), M.shape).coalesce()


class EngineV3:
    """Truy vấn = caption; ứng viên = video. Ba tập cạnh cho GNN (như v2):
      train        : bỏ MỌI cạnh chạm video không thuộc train;
      inductive    : cạnh của train + cạnh lớp → video (contains_inv) của video val/test/ext, KHÔNG có chiều video → lớp
                     ⇒ video mới chỉ NHẬN message, không ảnh hưởng node lớp/khái niệm dùng chung hay video khác;
      transductive : đồ thị đầy đủ.
    Thêm ở v3: chỉ số vị từ của từng cạnh (edge_pidx) + bảng phân phối vị từ (pmix) cho GNN; ma trận truy vấn × (vị từ, node)
    cho mẫu đường đi khớp vị từ P1m."""

    _QCACHE = {}

    def __init__(self, graph, vid_feat, txt_feat, qtok, qpreds):
        self.graph = graph
        self.vid_feat = torch.from_numpy(np.ascontiguousarray(vid_feat, dtype=np.float32))
        self.txt_feat = torch.from_numpy(np.ascontiguousarray(txt_feat, dtype=np.float32))
        n = graph.n_nodes
        key = (graph.inv.sha1, len(qtok), id(qtok), id(qpreds))
        if key not in EngineV3._QCACHE:                      # chỉ phụ thuộc inventory + truy vấn ⇒ dùng chung mọi biến thể đồ thị
            qc = [parse_tokens_v2(t, graph.inv) for t in qtok]
            rows, cols = [], []
            for j, hits in enumerate(qc):
                for nid, _, _ in hits:
                    rows.append(j); cols.append(nid)
            QT = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(len(qtok), n))
            QPB = {}
            for rel in PRED_FAMILIES:
                r_, c_ = [], []
                for j, (hits, qp) in enumerate(zip(qc, qpreds)):
                    for pid in qp[rel]:
                        for nid, _, _ in hits:
                            r_.append(j); c_.append(pid * n + nid)
                QPB[rel] = sp.csr_matrix((np.ones(len(r_), np.float32), (r_, c_)), shape=(len(qtok), len(PRED_VOCAB) * n))
            EngineV3._QCACHE[key] = (qc, QT, QPB)
        self.q_concepts, self.QT, self.QPB = EngineV3._QCACHE[key]
        self.q_preds = qpreds
        s, d, e, w, pi = graph.edge_arrays()
        N = graph.n_vid
        is_train_vid = np.zeros(n, bool)
        is_train_vid[np.array(SPLIT["train"])] = True
        other_vid = (np.arange(n) < N) & ~is_train_vid
        touch = other_vid[s] | other_vid[d]
        keep = {"train": ~touch, "transductive": np.ones(len(s), bool),
                "inductive": ~touch | ((e == ETYPE_ID["contains_inv"]) & other_vid[d] & ~other_vid[s])}
        self.edges = {k: (torch.from_numpy(np.stack([s[m], d[m]])), torch.from_numpy(e[m]), torch.from_numpy(w[m]),
                          torch.from_numpy(pi[m])) for k, m in keep.items()}
        self.edge_stats = {k: int(m.sum()) for k, m in keep.items()}
        self.pmix = _torch_sparse(graph.pmix)

    def edges_on(self, name):
        ei, et, ew, epi = self.edges[name]
        return ei.to(DEVICE), et.to(DEVICE), ew.to(DEVICE), epi.to(DEVICE), self.pmix.to(DEVICE)

    def path_feats(self, cand, q_idx):
        cand, q_idx = np.asarray(cand), np.asarray(q_idx)
        if cand.ndim == 1:
            cand = cand[:, None]
        B, K = cand.shape
        out = np.zeros((B, K, len(PATTERNS)), np.float32)
        for s0 in range(0, B, CFG.eval_chunk):
            c, q = cand[s0:s0 + CFG.eval_chunk], q_idx[s0:s0 + CFG.eval_chunk]
            rows, inv_ = np.unique(c.ravel(), return_inverse=True)
            inv_ = inv_.reshape(c.shape)
            col = np.arange(len(q))[:, None]
            Qb = self.QT[q].T.tocsr()
            for p, name in enumerate(PATTERNS):
                if name.startswith("P1m_"):
                    M = self.graph.R[name]
                    if M.nnz == 0:
                        continue
                    out[s0:s0 + len(q), :, p] = (M[rows] @ self.QPB[name[4:]][q].T).toarray()[inv_, col]
                else:
                    out[s0:s0 + len(q), :, p] = (self.graph.R[name][rows] @ Qb).toarray()[inv_, col]
        return np.log1p(out)

    def clip_candidates(self, q_idx, gallery, k):
        gallery, q_idx = np.asarray(gallery), np.asarray(q_idx)
        G = self.vid_feat[torch.from_numpy(gallery)]
        k = min(int(k), len(gallery))
        C, S = [], []
        for s0 in range(0, len(q_idx), CFG.eval_chunk):
            top = (self.txt_feat[torch.from_numpy(q_idx[s0:s0 + CFG.eval_chunk])] @ G.T).topk(k, dim=1)
            C.append(gallery[top.indices.numpy()]); S.append(top.values.numpy())
        return np.concatenate(C), np.concatenate(S)


def make_model(engine, sym_feat, mp_weight="binary", init="learned", pred_mode="none"):
    return GraphScorerV3(engine.graph.n_nodes, engine.txt_feat.shape[1], engine.vid_feat, sym_feat, pred_emb=PRED_EMB,
                         mp_weight=mp_weight, init=init, pred_mode=pred_mode).to(DEVICE)


def train_graph_scorer_v3(engine, train_q, sym_feat, seed=0, mp_weight="binary", init="learned", pred_mode="none", verbose=False):
    """InfoNCE như v1/v2 (âm trong batch không trùng video + âm khó CLIP trong gallery TRAIN); chỉ dùng cạnh 'train'."""
    train_q = np.asarray(train_q)
    CAPS.use(train_q, "gnn_train")
    set_seed(seed)
    gold_all = CAP_VID[train_q]
    train_vids = np.unique(gold_all)
    assert all(SPLIT_OF[int(v)] == "train" for v in train_vids)
    model = make_model(engine, sym_feat, mp_weight, init, pred_mode)
    model.clip_logit.requires_grad_(bool(CFG.learn_clip_w))
    params = [(n_, p) for n_, p in model.named_parameters() if p.requires_grad]
    scalar = [p for n_, p in params if n_.endswith("gate") or n_ in ("path_w", "decay", "clip_logit")]
    sid = {id(p) for p in scalar}
    opt = torch.optim.AdamW([dict(params=[p for _, p in params if id(p) not in sid]),
                             dict(params=scalar, lr=CFG.lr * CFG.scalar_lr_mult, weight_decay=0.0)],
                            lr=CFG.lr, weight_decay=CFG.weight_decay)
    edges = engine.edges_on("train")
    vid_dev = engine.vid_feat.to(DEVICE)
    cand_hn, _ = engine.clip_candidates(train_q, train_vids, CFG.hard_neg_k + 1)
    hard = np.stack([np.concatenate([row[row != gv], row[:1]])[:CFG.hard_neg_k] for row, gv in zip(cand_hn, gold_all)])

    def logits_for(cand_np, qi, h, qf):
        cand = torch.from_numpy(cand_np).to(DEVICE)
        pm = torch.from_numpy(engine.path_feats(cand_np, qi)).to(DEVICE)
        z = znorm_per_row(model.gnn_score(h, qf, cand)) + znorm_per_row(model.path_score(pm))
        if CFG.train_with_clip:
            z = z + F.softplus(model.clip_logit) * znorm_per_row(torch.einsum("bd,bkd->bk", qf, vid_dev[cand]))
        return z / CFG.temperature

    by_vid = defaultdict(list)
    for j, v in enumerate(gold_all):
        by_vid[int(v)].append(j)
    rng, hist = np.random.RandomState(seed), []
    for ep in range(CFG.epochs):
        model.train()
        tot, nb = 0.0, 0
        for _ in range(CFG.caption_passes):
            perm = rng.permutation(train_vids)
            for s0 in range(0, len(perm), CFG.train_bs):
                bv = perm[s0:s0 + CFG.train_bs]
                if len(bv) < 8:
                    continue
                sel = np.array([by_vid[int(v)][rng.randint(len(by_vid[int(v)]))] for v in bv])
                qi, gi = train_q[sel], gold_all[sel]
                qf = engine.txt_feat[torch.from_numpy(qi)].to(DEVICE)
                h = model.encode_graph(*edges)
                loss = F.cross_entropy(logits_for(np.tile(gi, (len(gi), 1)), qi, h, qf), torch.arange(len(gi), device=DEVICE))
                loss = loss + F.cross_entropy(logits_for(np.concatenate([gi[:, None], hard[sel]], 1), qi, h, qf),
                                              torch.zeros(len(gi), dtype=torch.long, device=DEVICE))
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                tot += loss.item(); nb += 1
        hist.append(tot / max(nb, 1))
        if verbose and (ep % 5 == 0 or ep == CFG.epochs - 1):
            print(f"    epoch {ep:2d} | loss {hist[-1]:.4f} | w_path " +
                  " ".join(f"{x:.2f}" for x in model.pattern_weights().detach().cpu().numpy()))
    model.eval()
    return model, hist


@torch.no_grad()
def score_queries_v3(model, engine, q_idx, gallery, k, protocol="inductive"):
    if isinstance(model, QCRGAT):                       # v4: cùng đầu ra, s_path = 0 (λ = 1)
        return score_queries_v4(model, engine, q_idx, gallery, k, protocol)
    model.eval()
    h = model.encode_graph(*engine.edges_on(protocol))
    q_idx = np.asarray(q_idx)
    cand, s_clip = engine.clip_candidates(q_idx, gallery, k)
    s_gnn, s_path = np.zeros_like(s_clip), np.zeros_like(s_clip)
    for s0 in range(0, len(q_idx), CFG.eval_chunk):
        sl = slice(s0, s0 + CFG.eval_chunk)
        c = torch.from_numpy(cand[sl]).to(DEVICE)
        qf = engine.txt_feat[torch.from_numpy(q_idx[sl])].to(DEVICE)
        s_gnn[sl] = model.gnn_score(h, qf, c).cpu().numpy()
        s_path[sl] = model.path_score(torch.from_numpy(engine.path_feats(cand[sl], q_idx[sl])).to(DEVICE)).cpu().numpy()
    return dict(cand=cand, clip_raw=s_clip, path_raw=s_path, gnn_raw=s_gnn,
                s_clip=znorm_per_row(s_clip), s_gnn=znorm_per_row(s_gnn), s_path=znorm_per_row(s_path))


def tune_on_val_v3(model, engine, part, k, clip_rank, lam_grid=None, protocol="inductive"):
    """(α, λ) tối đa MRR trên VAL; lam_grid cố định cho ablation path-only (λ=0) / GNN-only (λ=1)."""
    assert part["name"] == "val", "tune chỉ nhận VAL"
    CAPS.use(part["q_idx"], "tune")
    sc = score_queries_v3(model, engine, part["q_idx"], part["gallery"], k, protocol)
    grid, best = [], None
    for a in CFG.alpha_grid:
        for l in (lam_grid or CFG.lambda_grid):
            r = ranks_with_fallback(sc["cand"], combine(sc["s_clip"], sc["s_gnn"], sc["s_path"], a, l), part["gold"], clip_rank)
            m = metrics_from_ranks(r, part["gold"])
            grid.append(dict(alpha=a, lam=l, k=k, **{x: m[x] for x in ("R@1", "R@5", "R@10", "MRR")}, rank=r))
            if best is None or m["MRR"] > best["MRR"]:
                best = grid[-1]
    return (best["alpha"], best["lam"]), [{k_: v for k_, v in g_.items() if k_ != "rank"} for g_ in grid], best

# ---------------------------------- v4: huấn luyện + chấm điểm QC-RGAT (cùng giao thức v3: InfoNCE, âm khó CLIP, chỉ video TRAIN) ----------------------------------
_RELDATA = {}


def reldata_v4(engine):
    if id(engine) not in _RELDATA:
        _RELDATA[id(engine)] = RelDataV4(engine.graph)
    return _RELDATA[id(engine)]


def make_model_v4(engine, design):
    return QCRGAT(reldata_v4(engine), TXT_FEAT.shape[1], d=CFG.v4_dim, T=design["v4_layers"], rel=design["v4_rel"],
                  qcond=design["v4_qcond"], use_pred=design["v4_pred"]).to(DEVICE)


def train_v4(engine, train_q, design, seed=0, verbose=False):
    """Như train_graph_scorer_v3: in-batch (video không trùng) + 31 âm khó CLIP trong gallery TRAIN; logit = z(s) + c·z(cos CLIP)."""
    train_q = np.asarray(train_q)
    CAPS.use(train_q, "gnn_train")
    set_seed(seed)
    gold_all = CAP_VID[train_q]
    train_vids = np.unique(gold_all)
    assert all(SPLIT_OF[int(v)] == "train" for v in train_vids)
    model = make_model_v4(engine, design)
    model.clip_logit.requires_grad_(bool(CFG.learn_clip_w))
    named = [(n_, p) for n_, p in model.named_parameters() if p.requires_grad]
    scal = [p for n_, p in named if n_ in QCRGAT.SCALARS]
    opt = torch.optim.AdamW([dict(params=[p for n_, p in named if n_ not in QCRGAT.SCALARS]),
                             dict(params=scal, lr=CFG.v4_lr * CFG.scalar_lr_mult, weight_decay=0.0)],
                            lr=CFG.v4_lr, weight_decay=CFG.weight_decay)
    cand_hn, _ = engine.clip_candidates(train_q, train_vids, CFG.hard_neg_k + 1)
    hard = np.stack([np.concatenate([row[row != gv], row[:1]])[:CFG.hard_neg_k] for row, gv in zip(cand_hn, gold_all)])
    txt_dev = torch.from_numpy(np.ascontiguousarray(TXT_FEAT, dtype=np.float32)).to(DEVICE)

    def logits_for(qb, cand_np):
        cand = torch.from_numpy(cand_np).to(DEVICE)
        z = znorm_per_row(model.score(qb, cand))
        if CFG.train_with_clip:
            z = z + F.softplus(model.clip_logit) * znorm_per_row(torch.einsum("bd,bkd->bk", txt_dev[qb["q"]], VID_T[cand]))
        return z / CFG.temperature

    by_vid = defaultdict(list)
    for j, v in enumerate(gold_all):
        by_vid[int(v)].append(j)
    rng, hist = np.random.RandomState(seed), []
    for ep in range(CFG.epochs):
        model.train()
        tot, nb = 0.0, 0
        for _ in range(CFG.caption_passes):
            perm = rng.permutation(train_vids)
            for s0 in range(0, len(perm), CFG.train_bs):
                bv = perm[s0:s0 + CFG.train_bs]
                if len(bv) < 8:
                    continue
                sel = np.array([by_vid[int(v)][rng.randint(len(by_vid[int(v)]))] for v in bv])
                gi = gold_all[sel]
                qb = model.D.query_batch(train_q[sel], model.use_pred)
                loss = F.cross_entropy(logits_for(qb, np.tile(gi, (len(gi), 1))), torch.arange(len(gi), device=DEVICE))
                loss = loss + F.cross_entropy(logits_for(qb, np.concatenate([gi[:, None], hard[sel]], 1)),
                                              torch.zeros(len(gi), dtype=torch.long, device=DEVICE))
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                tot += loss.item(); nb += 1
        hist.append(tot / max(nb, 1))
        if verbose and (ep % 5 == 0 or ep == CFG.epochs - 1):
            print(f"    epoch {ep:2d} | loss {hist[-1]:.4f}")
    model.eval()
    return model, hist


@torch.no_grad()
def score_queries_v4(model, engine, q_idx, gallery, k, protocol="inductive", D=None):
    """Cùng đầu ra với score_queries_v3; s_gnn = điểm QC-RGAT, s_path = 0 (λ cố định = 1)."""
    model.eval()
    D = D or model.D
    q_idx = np.asarray(q_idx)
    cand, s_clip = engine.clip_candidates(q_idx, gallery, k)
    s = np.zeros_like(s_clip)
    bs = max(1, CFG.v4_score_bs)
    for s0 in range(0, len(q_idx), bs):
        qb = D.query_batch(q_idx[s0:s0 + bs], model.use_pred)
        s[s0:s0 + bs] = model.score(qb, torch.from_numpy(cand[s0:s0 + bs]).to(DEVICE), D).cpu().numpy()
    z = np.zeros_like(s)
    return dict(cand=cand, clip_raw=s_clip, path_raw=z, gnn_raw=s, s_clip=znorm_per_row(s_clip), s_gnn=znorm_per_row(s), s_path=z)


## 4.11 Chỉ số đánh giá

* Mỗi **caption là một truy vấn**, có đúng một video dương. Notebook báo cáo R@1, R@5, R@10, MRR, nDCG@10, MedR và `MRR_video_macro` (macro-MRR theo video).
* Thứ hạng tính trên **toàn bộ gallery**: top-K được sắp lại, ngoài top-K giữ thứ hạng CLIP.
* Kiểm định cặp dùng **bootstrap theo cụm video**. Họ so sánh chính được hiệu chỉnh **Holm–Bonferroni**; luật chọn từng stage cũng hiệu chỉnh Holm trên các ứng viên của stage.
* Notebook báo cáo thêm: candidate recall, nhóm direct/indirect/other, khái niệm hiếm, vị từ hiếm/phổ biến/không có vị từ, số node/cạnh/bậc, thời gian chạy và bộ nhớ GPU.

In [82]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def ranks_with_fallback(cand, scores, gold, clip_rank):
    """Thứ hạng trên TOÀN gallery: trong top-K theo điểm xếp lại; ngoài top-K giữ thứ hạng CLIP (> K)."""
    order = np.argsort(-scores, axis=1, kind="stable")
    hit = np.take_along_axis(cand, order, axis=1) == np.asarray(gold)[:, None]
    return np.where(hit.any(1), hit.argmax(1) + 1, np.maximum(clip_rank, cand.shape[1] + 1))


def metrics_from_ranks(rank, gold=None, ks=(1, 5, 10)):
    rank = np.asarray(rank, np.float64)
    out = {f"R@{k}": float((rank <= k).mean()) for k in ks}
    out["MRR"] = float((1.0 / rank).mean())
    out["nDCG@10"] = float(np.where(rank <= 10, 1.0 / np.log2(rank + 1), 0.0).mean())
    out["MedR"] = float(np.median(rank))
    out["n_queries"] = int(len(rank))
    if gold is not None:
        _, inv = np.unique(np.asarray(gold), return_inverse=True)
        out["MRR_video_macro"] = float((np.bincount(inv, 1.0 / rank) / np.bincount(inv)).mean())
        out["n_videos"] = int(inv.max() + 1) if len(inv) else 0
    return out


METRIC_KEYS = ["R@1", "R@5", "R@10", "MRR", "nDCG@10", "MedR", "MRR_video_macro"]


def fmt_mean_std(vals):
    vals = np.asarray(vals, float)
    return f"{vals.mean():.4f} ± {vals.std():.4f}"


def cluster_bootstrap_diff(rr_a, rr_b, gold, n_boot=5000, seed=0):
    """Δ = mean(rr_a − rr_b) theo truy vấn; khoảng tin cậy 95% bằng bootstrap THEO VIDEO (cụm caption)."""
    d = np.asarray(rr_a) - np.asarray(rr_b)
    _, inv = np.unique(np.asarray(gold), return_inverse=True)
    S, C = np.bincount(inv, d), np.bincount(inv).astype(float)
    idx = np.random.RandomState(seed).randint(0, len(S), (n_boot, len(S)))
    boot = S[idx].sum(1) / C[idx].sum(1)
    return dict(diff=float(d.mean()), ci_lo=float(np.percentile(boot, 2.5)), ci_hi=float(np.percentile(boot, 97.5)),
                p_not_better=float((boot <= 0).mean()), a_wins=int((d > 0).sum()), b_wins=int((d < 0).sum()))


def holm(pvals):
    """p đã hiệu chỉnh Holm–Bonferroni (đơn điệu), cùng thứ tự với đầu vào."""
    p = np.asarray(pvals, float)
    m = len(p)
    if m == 0:
        return p
    order = np.argsort(p)
    adj = np.empty(m)
    run = 0.0
    for k, i in enumerate(order):
        run = max(run, min(1.0, (m - k) * p[i]))
        adj[i] = run
    return adj

## 4.12 Giải thích v3
* Đường đi được liệt kê theo **chiều** cạnh, kèm loại cạnh và **vị từ** ở bước quan hệ.
  * Mẫu P1m chỉ dùng vị từ có trong truy vấn.
  * Giải thích hợp lệ khi: cạnh có thật, đúng loại, đúng chiều, đúng kiểu node; vị từ thuộc **đúng** cạnh theo **đúng** chiều và có trong truy vấn.
  * Có đối chứng âm: sửa pred_id thì phải bị phát hiện.
* Điểm liệt kê phải khớp ma trận ở mọi `path_mode`.
* Fidelity báo riêng ba mức: thay đổi **điểm**, **thứ hạng**, **quyết định** top-1 / top-10.
* Có phần đóng góp theo mẫu đường đi và độ nhạy theo seed (Jaccard).
* Xoá cạnh giải thích được so với xoá cạnh **đối chứng** cùng loại.
* Notebook báo cáo trọng số hiệu dụng `(1−α)(1−λ)` của nhánh đường đi.

In [83]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
def enumerate_paths_v3(g, vid, concept_ids, qpred=None, PM=None):
    """Đường đi CỤ THỂ từ video tới khái niệm truy vấn, đi theo CHIỀU cạnh, ghi loại cạnh từng bước và VỊ TỪ của bước
    quan hệ. Trọng số = idf(đích) × Π trọng số trong ma trận đường đi (PM) của chế độ path hiện tại ⇒ khớp graph.R.
    Mẫu P1m_f (path_mode 'pred_match'): bước a –f→ b chỉ tính phần vị từ có trong truy vấn (qpred[f] = tập pred_id)."""
    PM = PM or g.PM
    target = {int(c) for c in concept_ids}
    C = PM["C"]
    own = set(C.indices[C.indptr[vid]:C.indptr[vid + 1]].tolist())
    out = []

    def nbrs(M, x):
        sl = slice(M.indptr[x], M.indptr[x + 1])
        return zip(M.indices[sl].tolist(), M.data[sl].tolist())

    def add(pattern, nodes, rels, w, pred=None):
        c = nodes[-1]
        if pattern != "P0_contains" and CFG.drop_redundant_indirect and c in own:
            return
        w = float(g.idf[c]) * float(w)
        if w > 0:
            out.append(dict(pattern=pattern, nodes=[int(x) for x in nodes], rels=list(rels), w=w, pred=pred))

    for o, w0 in nbrs(C, vid):
        if o in target:
            add("P0_contains", [o], ["contains"], w0)
        for f in FAMILIES:
            for t in (FAMILY_REL[f], FAMILY_REL[f] + "_inv"):
                for o2, w1 in nbrs(PM[t], o):
                    if o2 in target:
                        add(f"P1_{f}", [o, o2], ["contains", t], w0 * w1)
                    for c, w2 in nbrs(PM["isa"], o2):
                        if c in target:
                            add(f"P4_{f}_isa", [o, o2, c], ["contains", t, "isa"], w0 * w1 * w2)
        for c, w1 in nbrs(PM["isa"], o):
            if c in target:
                add("P2_isa", [o, c], ["contains", "isa"], w0 * w1)
        for o2, w1 in nbrs(PM["related"], o):
            if o2 in target:
                add("P3_related", [o, o2], ["contains", "related"], w0 * w1)
            for c, w2 in nbrs(PM["isa"], o2):
                if c in target:
                    add("P5_related_isa", [o, o2, c], ["contains", "related", "isa"], w0 * w1 * w2)
        if g.path_mode == "pred_match" and qpred:
            for rel in PRED_FAMILIES:
                want = set(qpred.get(rel, ()))
                if not want:
                    continue
                for t in (rel, rel + "_inv"):
                    for o2, w1 in nbrs(PM[t], o):
                        if o2 not in target:
                            continue
                        a, b = (o2, o) if t.endswith("_inv") else (o, o2)
                        for pid, s in g.pshare[rel].get((a, b), {}).items():
                            if pid in want:
                                add(f"P1m_{rel}", [o, o2], ["contains", t], w0 * w1 * s, pred=(rel, int(pid), float(s)))
    return out


def paths_to_counts_v3(paths, mode):
    """log1p(tổng theo mẫu) — với path_dedup: tổng theo khái niệm của đường đi MẠNH NHẤT (khớp _maxprod)."""
    v = np.zeros(len(PATTERNS), np.float64)
    if mode == "dedup":
        best = defaultdict(float)
        for p in paths:
            k = (p["pattern"], p["nodes"][-1])
            best[k] = max(best[k], p["w"])
        for (pat, _), w in best.items():
            v[PIDX[pat]] += w
    else:
        for p in paths:
            v[PIDX[p["pattern"]]] += p["w"]
    return np.log1p(v).astype(np.float32)


def path_edges_v3(p, vid):
    """[(loại, u, v)] gồm cạnh contains đầu tiên."""
    nodes = [int(vid)] + p["nodes"]
    return list(zip(p["rels"], nodes[:-1], nodes[1:]))


def edge_valid_v3(g, rel, u, v):
    """Cạnh có thật trong ma trận GỐC đúng loại + đúng chiều, và kiểu đầu/cuối khớp REL_SPEC."""
    M = g.A_con if rel == "contains" else g.A[rel]
    if M[u, v] == 0:
        return False
    ok_src = REL_SPEC[rel]["src"].split(":")[0].split("|")
    ok_tgt = REL_SPEC[rel]["tgt"].split(" ")[0].split(":")[0].split("|")
    return g.inv.node_type[u] in ok_src and g.inv.node_type[v] in ok_tgt


def pred_valid_v3(g, p, vid, qtokens):
    """Đường đi khớp vị từ: vị từ ghi trong giải thích phải (1) thuộc phân phối vị từ của ĐÚNG cạnh theo ĐÚNG chiều và
    (2) có mặt trong truy vấn. Đường đi không mang vị từ ⇒ True."""
    if p.get("pred") is None:
        return True
    rel, pid, share = p["pred"]
    t, u, v = path_edges_v3(p, vid)[1]
    a, b = (v, u) if t.endswith("_inv") else (u, v)
    ok_edge = abs(g.pshare[rel].get((a, b), {}).get(pid, -1.0) - share) < 1e-9
    return bool(ok_edge and all(x in qtokens for x in predicate_tokens(PRED_VOCAB[pid][1])) and PRED_VOCAB[pid][0] == rel)


def path_text_v3(g, p):
    nm = lambda i: g.inv.name[i]
    s = "video"
    nodes = p["nodes"]
    for rel, a, b in zip(p["rels"], [None] + nodes[:-1], nodes):
        lab = REL_TEXT.get(rel, rel)
        if "{pred}" in lab:
            if p.get("pred") is not None:
                pr = f"{PRED_VOCAB[p['pred'][1]][1]}({p['pred'][2]:.2f})"
            else:
                d = g.predicates_of(rel, a, b)
                tot = max(sum(d.values()), 1)
                pr = "/".join(f"{k}({c / tot:.2f})" for k, c in sorted(d.items(), key=lambda x: (-x[1], x[0]))[:3]) or "?"
            lab = lab.format(pred=pr.replace(" ", "_"))
        s += f" –{lab}→ {nm(b)}"
    return s


def q_args(eng, q):
    """(node khái niệm của truy vấn, {họ: pred_id khớp}, tập token truy vấn)."""
    return [nid for nid, _, _ in eng.q_concepts[q]], eng.q_preds[q], QPRED_TOK[q]


@torch.no_grad()
def explain_and_score_v3(model, g, vid, concept_ids, qpred=None, qtokens=frozenset(), top_m=CFG.top_m_paths, rng_seed=0):
    paths = enumerate_paths_v3(g, vid, concept_ids, qpred)
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    score = lambda ps: float(gain * (paths_to_counts_v3(ps, g.path_mode) * w).sum())
    full = score(paths)
    contrib = [full - score(paths[:i] + paths[i + 1:]) for i in range(len(paths))]
    order = np.argsort(-np.array(contrib), kind="stable") if contrib else np.array([], int)
    ex = [int(i) for i in order[:top_m]]
    expl = [dict(text=path_text_v3(g, paths[i]), pattern=paths[i]["pattern"], contribution=float(contrib[i]),
                 nodes=paths[i]["nodes"], rels=paths[i]["rels"], pred=paths[i]["pred"]) for i in ex]
    keep = [paths[i] for i in ex]
    drop = [p for i, p in enumerate(paths) if i not in set(ex)]
    rng, rnd = np.random.RandomState(rng_seed), []
    for _ in range(10):
        r = set(rng.choice(len(paths), size=min(len(ex), len(paths)), replace=False).tolist()) if paths else set()
        rnd.append(score([p for i, p in enumerate(paths) if i not in r]))
    cset = {int(c) for c in concept_ids}
    valid = [p["nodes"][-1] in cset and all(edge_valid_v3(g, *e) for e in path_edges_v3(p, vid)) for p in keep]
    pvalid = [pred_valid_v3(g, p, vid, qtokens) for p in keep]
    share = np.zeros(len(PATTERNS))
    for i in range(len(paths)):
        share[PIDX[paths[i]["pattern"]]] += max(contrib[i], 0)
    detail = dict(n_paths=len(paths), score_full=full, score_without_explanation=score(drop),
                  score_only_explanation=score(keep), score_without_random=float(np.mean(rnd)) if rnd else full,
                  sparsity=1 - len(ex) / max(len(paths), 1), validity=float(np.mean(valid)) if valid else 0.0,
                  predicate_validity=float(np.mean(pvalid)) if pvalid else 1.0,
                  n_predicate_paths=int(sum(p.get("pred") is not None for p in keep)),
                  pattern_share={p: float(s / max(share.sum(), 1e-12)) for p, s in zip(PATTERNS, share) if s > 0})
    return expl, full, detail


def _rank_in(fin, j):
    return int((fin > fin[j]).sum()) + 1


def path_score_from_R(g, R, cids, qpred, gain, w):
    """s_path (chưa chuẩn hoá) từ một R đã tính lại (hàng = ứng viên) — cùng công thức với engine.path_feats."""
    n, cols = g.n_nodes, np.array(sorted({int(c) for c in cids}))
    feats = []
    for p in PATTERNS:
        if p.startswith("P1m_"):
            pc = np.array(sorted(pid * n + c for pid in qpred.get(p[4:], ()) for c in cols), np.int64)
            feats.append(np.asarray(R[p][:, pc].sum(1)).ravel() if len(pc) and R[p].nnz else np.zeros(R["P0_contains"].shape[0]))
        else:
            feats.append(np.asarray(R[p][:, cols].sum(1)).ravel())
    return gain * (np.log1p(np.stack(feats, 1)) * w).sum(1)


@torch.no_grad()
def edge_deletion_v3(run, part, groups, n=CFG.n_edge_deletion, seed=0, protocol="inductive"):
    """Với mỗi truy vấn (video đúng trong top-K): xoá THẬT (a) cạnh then chốt của đường đi giải thích top-1 và
    (b) đường đi QUAN HỆ đứng đầu; đối chứng: xoá một cạnh CÙNG LOẠI, KHÔNG nằm trên đường đi nào tới khái niệm
    của truy vấn. Tính lại R cho mọi ứng viên từ ma trận đã sửa (cùng công thức), ghi Δ s_path, Δ điểm cuối, hạng,
    và quyết định top-1 / top-10. GNN giữ cố định để cô lập nhánh đường đi."""
    g, eng, model = run["graph"], run["engine"], run["model"]
    a, l, z = run["alpha"], run["lam"], znorm_per_row
    h = model.encode_graph(*eng.edges_on(protocol))
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    rng = np.random.RandomState(seed)
    rows, used, mism = [], set(), 0

    def delete(rel, u, v):
        A2 = dict(g.A)
        Acon = g.A_con
        if rel == "contains":
            M = g.A_con.tolil(copy=True); M[u, v] = 0; Acon = M.tocsr(); Acon.eliminate_zeros()
        else:
            M = g.A[rel].tolil(copy=True); M[u, v] = 0; A2[rel] = M.tocsr(); A2[rel].eliminate_zeros()
        return g.path_mats(A=A2, A_con=Acon)

    for i in rng.permutation(len(part["q_idx"])):
        q, gold = int(part["q_idx"][i]), int(part["gold"][i])
        cids, qpred, qtok = q_args(eng, q)
        if gold in used or not cids:
            continue
        cand, s_clip = eng.clip_candidates(np.array([q]), part["gallery"], CFG.rerank_k)
        cand, s_clip = cand[0], s_clip[0]
        hit = np.where(cand == gold)[0]
        if len(hit) == 0:
            continue
        expl, _, _ = explain_and_score_v3(model, g, gold, cids, qpred, qtok)
        expl = [e for e in expl if e["contribution"] > 0]
        if not expl:
            continue
        j = int(hit[0])
        sp0 = path_score_from_R(g, g.compute_R(rows=cand), cids, qpred, gain, w)
        sp_eng = gain * (eng.path_feats(cand[None], np.array([q]))[0] * w).sum(-1)
        mism += int(not np.allclose(sp0, sp_eng, atol=1e-4))
        sg = model.gnn_score(h, eng.txt_feat[[q]].to(DEVICE), torch.from_numpy(cand[None]).to(DEVICE)).cpu().numpy()
        f0 = combine(z(s_clip[None]), z(sg), z(sp0[None]), a, l)[0]
        r0, top0 = _rank_in(f0, j), int(cand[np.argmax(f0)])
        targets = [("top1", expl[0])]
        relp = next((e for e in expl if e["pattern"] != "P0_contains"), None)
        if relp is not None and relp is not expl[0]:
            targets.append(("top_relational", relp))
        on_path = {(t_, x, y) for p in enumerate_paths_v3(g, gold, cids, qpred) for t_, x, y in path_edges_v3(p, gold)}
        for which, p in targets:
            edges = path_edges_v3(p, gold)
            rel, u, v = edges[1] if len(edges) > 1 else edges[0]
            ctrl = None
            if rel != "contains":                        # cạnh đối chứng cùng loại, không nằm trên đường đi nào
                cand_e = [(x, y) for x, y in zip(*g.A[rel].nonzero()) if (rel, x, y) not in on_path]
                if cand_e:
                    ctrl = cand_e[rng.randint(len(cand_e))]
            todo = [("explanation", (u, v))] + ([("control", ctrl)] if ctrl is not None else [])
            for kind, (uu, vv) in todo:
                sp1 = path_score_from_R(g, g.compute_R(rows=cand, PM=delete(rel, uu, vv)), cids, qpred, gain, w)
                f1 = combine(z(s_clip[None]), z(sg), z(sp1[None]), a, l)[0]
                r1 = _rank_in(f1, j)
                rows.append(dict(q=q, video_id=RECORDS[gold]["video_id"], group=str(groups[i]), which=which, deleted=kind,
                                 explanation=p["text"], pattern=p["pattern"], relation=rel,
                                 predicate=PRED_VOCAB[p["pred"][1]][1] if p.get("pred") else "",
                                 edge=f"{g.inv.name[uu]} → {g.inv.name[vv]}", edge_valid=edge_valid_v3(g, rel, uu, vv),
                                 path_contribution=p["contribution"], s_path_before=float(sp0[j]), s_path_after=float(sp1[j]),
                                 d_s_path=float(sp1[j] - sp0[j]), d_final=float(f1[j] - f0[j]), rank_before=r0, rank_after=r1,
                                 rank_changed=r1 != r0, top1_changed=int(cand[np.argmax(f1)]) != top0,
                                 top10_changed=(r0 <= 10) != (r1 <= 10),
                                 w_path_effective=(1 - a) * (1 - l), alpha=a, lam=l))
        used.add(gold)
        if len(used) >= n:
            break
    return pd.DataFrame(rows), mism


@torch.no_grad()
def path_fidelity_v3(run, part, groups, n_queries=CFG.n_explain_queries, top_m=CFG.top_m_paths, seed=0, protocol="inductive"):
    """Fidelity mức đường đi (score) và mức thứ hạng của điểm cuối cho cùng một tập truy vấn; đóng góp theo mẫu đường đi."""
    g, eng, model = run["graph"], run["engine"], run["model"]
    a, l, z = run["alpha"], run["lam"], znorm_per_row
    h = model.encode_graph(*eng.edges_on(protocol))
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    sc = lambda ps: gain * float((paths_to_counts_v3(ps, g.path_mode) * w).sum())
    rng, rows, used = np.random.RandomState(seed), [], set()
    for i in rng.permutation(len(part["q_idx"])):
        q, gold = int(part["q_idx"][i]), int(part["gold"][i])
        cids, qpred, qtok = q_args(eng, q)
        if gold in used or not cids:
            continue
        cand, s_clip = eng.clip_candidates(np.array([q]), part["gallery"], CFG.rerank_k)
        cand, s_clip = cand[0], s_clip[0]
        hit = np.where(cand == gold)[0]
        expl, s, det = explain_and_score_v3(model, g, gold, cids, qpred, qtok, top_m=top_m)
        if len(hit) == 0 or det["n_paths"] <= top_m:
            continue
        paths = enumerate_paths_v3(g, gold, cids, qpred)
        j = int(hit[0])
        sp_raw = gain * (eng.path_feats(cand[None], np.array([q]))[0] * w).sum(-1)
        s_gnn = model.gnn_score(h, eng.txt_feat[[q]].to(DEVICE), torch.from_numpy(cand[None]).to(DEVICE)).cpu().numpy()

        def rank_with(kept):
            s_ = sp_raw.copy(); s_[j] = sc(kept)
            return _rank_in(combine(z(s_clip[None]), z(s_gnn), z(s_[None]), a, l)[0], j)

        contrib = np.array([sc(paths) - sc(paths[:k] + paths[k + 1:]) for k in range(len(paths))])
        top = set(np.argsort(-contrib, kind="stable")[:top_m].tolist())
        rr = [rank_with([p for k, p in enumerate(paths) if k not in set(rng.choice(len(paths), top_m, replace=False).tolist())])
              for _ in range(10)]
        rf, rd = rank_with(paths), rank_with([p for k, p in enumerate(paths) if k not in top])
        rows.append(dict(q=q, video_id=RECORDS[gold]["video_id"], group=str(groups[i]),
                         **{k_: v for k_, v in det.items() if k_ != "pattern_share"},
                         **{f"share_{k_}": v for k_, v in det["pattern_share"].items()},
                         matrix_vs_enumeration_ok=bool(abs(sc(paths) - sp_raw[j]) < 1e-4),
                         drop_explanation=det["score_full"] - det["score_without_explanation"],
                         drop_random=det["score_full"] - det["score_without_random"],
                         rank_full=rf, rank_drop_expl=rd, rank_drop_rand=float(np.mean(rr)),
                         rank_only_expl=rank_with([paths[k] for k in top]),
                         top10_changed_drop_expl=(rf <= 10) != (rd <= 10), w_path_effective=(1 - a) * (1 - l),
                         explanation=" || ".join(e["text"] for e in expl[:3])))
        used.add(gold)
        if len(rows) >= n_queries:
            break
    return pd.DataFrame(rows)


@torch.no_grad()
def explanation_sensitivity(runs_by_seed, part, n=CFG.n_seed_sensitivity, top_m=CFG.top_m_paths, seed=0):
    """Jaccard của tập đường đi giải thích top-m cho CÙNG (truy vấn, video đúng) giữa các seed / cấu hình."""
    keys = list(runs_by_seed)
    if len(keys) < 2:
        return pd.DataFrame()
    eng0 = runs_by_seed[keys[0]]["engine"]
    rows, used = [], set()
    for i in np.random.RandomState(seed).permutation(len(part["q_idx"])):
        q, gold = int(part["q_idx"][i]), int(part["gold"][i])
        cids, qpred, qtok = q_args(eng0, q)
        if gold in used or not cids:
            continue
        sets = {}
        for k in keys:
            r = runs_by_seed[k]
            ex, _, _ = explain_and_score_v3(r["model"], r["graph"], gold, cids, qpred, qtok, top_m=top_m)
            sets[k] = {e["text"] for e in ex}
        if not any(sets.values()):
            continue
        js = [len(sets[x] & sets[y]) / max(len(sets[x] | sets[y]), 1) for x, y in itertools.combinations(keys, 2)]
        rows.append(dict(q=q, video_id=RECORDS[gold]["video_id"], n_configs=len(keys), mean_jaccard=float(np.mean(js)),
                         min_jaccard=float(np.min(js))))
        used.add(gold)
        if len(rows) >= n:
            break
    return pd.DataFrame(rows)

# ---------------------------------- v4: giải thích = CHÍNH các số hạng của điểm (phân rã chính xác) ----------------------------------
def v4_edge_info(D, k):
    """(nguồn, loại, đích, vị từ) của cạnh k trong RelDataV4 — đọc lại từ đồ thị gốc."""
    g = D.graph
    s, d, e, w, pi = g.edge_arrays()
    j = int(D.gidx[k])
    preds = {}
    if pi[j] >= 0:
        row = g.pmix[pi[j]]
        preds = {PRED_VOCAB[c][1]: float(v) for c, v in zip(row.indices, row.data)}
    return dict(src=int(s[j]), rel=EDGE_TYPES[int(e[j])], dst=int(d[j]), weight=float(w[j]), predicates=preds)


def v4_path_text(D, t, vid):
    inv = D.graph.inv
    if t["kind"] == "visual":
        return f"[thị giác] {inv.name[vid]}"
    c = D.N + t["node"]
    if t["kind"] == "contains":
        return f"{inv.name[vid]} –chứa→ {inv.name[c]}"
    hops = []
    for k in t["edges"]:
        ei = v4_edge_info(D, k)
        p = max(ei["predicates"], key=ei["predicates"].get) if ei["predicates"] else ""
        hops.append((inv.name[ei["src"]], ei["rel"] + (f"[{p}]" if p else ""), inv.name[ei["dst"]]))
    chain = hops[0][0] + "".join(f" –{r}→ {b}" for _, r, b in hops)
    return f"{chain} ←chứa– {inv.name[vid]}"


def v4_edge_valid(D, k, q):
    """Cạnh có thật (đúng loại, chiều, kiểu node theo REL_SPEC); nếu truy vấn có vị từ khớp cạnh thì vị từ đó thuộc ĐÚNG cạnh."""
    ei = v4_edge_info(D, k)
    g, inv = D.graph, D.graph.inv
    ok = ei["rel"] in g.A and g.A[ei["rel"]][ei["src"], ei["dst"]] > 0
    ok = ok and inv.node_type[ei["src"]] != "video" and inv.node_type[ei["dst"]] != "video"
    ids = {PRED_VOCAB[p][1] for rel in PRED_FAMILIES for p in QPREDS[q][rel]}
    matched = ids & set(ei["predicates"])
    return bool(ok), bool(matched), bool(not matched or matched <= set(ei["predicates"]))


@torch.no_grad()
def explain_v4(model, q, vid, top_m=CFG.top_m_paths):
    """Giải thích "vì sao video được trả về" = top-m đường đi quan hệ ỦNG HỘ (đóng góp > 0); đường đi PHẢN ĐỐI (< 0) ghi riêng.
    Σ mọi số hạng (thị giác + nhãn + mọi đường đi) = điểm mô hình."""
    terms = model.decompose(q, vid)
    tot = {k: sum(t["contrib"] for t in terms if t["kind"] == k) for k in ("visual", "contains", "relation")}
    rel = [t for t in terms if t["kind"] == "relation"]
    sup = sorted([t for t in rel if t["contrib"] > 0], key=lambda t: -t["contrib"])[:top_m]
    opp = sorted([t for t in rel if t["contrib"] < 0], key=lambda t: t["contrib"])[:top_m]
    return dict(terms=terms, top_relation=sup, opposing=opp, totals=tot, score=sum(t["contrib"] for t in terms))


@torch.no_grad()
def v4_rank_after(run, part_q, gold, cand, s_clip_z, D2):
    """Thứ hạng video đúng trong top-K (điểm cuối với α đã đóng băng) khi mô hình dùng dữ liệu đồ thị D2."""
    m = run["model"]
    qb = D2.query_batch([part_q], m.use_pred)
    s = m.score(qb, torch.from_numpy(cand[None]).to(DEVICE), D2).cpu().numpy()
    fin = run["alpha"] * s_clip_z[None] + (1 - run["alpha"]) * znorm_per_row(s)
    j = int(np.where(cand == gold)[0][0])
    return int((fin[0] > fin[0, j]).sum()) + 1, s[0, j], fin[0]


def xai_v4(run, part, groups, n_queries=CFG.n_explain_queries, top_m=CFG.top_m_paths, seed=0):
    """Với ≥ n truy vấn có video đúng trong top-K và có đường đi quan hệ ỦNG HỘ: (1) hợp lệ (cạnh thật, vị từ đúng cạnh), (2) độ thưa,
    (3) fidelity: xoá THẬT các cạnh của top-m đường đi giải thích ⇒ Δ điểm mô hình so với dự đoán từ phân rã (khớp = chính xác),
    so với xoá cùng số cạnh quan hệ NGẪU NHIÊN nối vào nhãn của video; (4) thứ hạng / top-1 / top-10 đổi (báo riêng)."""
    m, eng = run["model"], run["engine"]
    D = m.D
    rng = np.random.RandomState(seed)
    q_all = part["q_idx"]
    order = rng.permutation(len(q_all))
    cand_all, s_clip = eng.clip_candidates(q_all, part["gallery"], CFG.rerank_k)
    rows = []
    for i in order:
        if len(rows) >= n_queries:
            break
        q, gold = int(q_all[i]), int(part["gold"][i])
        cand = cand_all[i]
        if gold not in cand:
            continue
        ex = explain_v4(m, q, gold, top_m)
        if not ex["top_relation"]:
            continue
        zc = znorm_per_row(s_clip[i:i + 1])[0]
        r_full, s_full, _ = v4_rank_after(run, q, gold, cand, zc, D)
        e_del = sorted({k for t in ex["top_relation"] for k in t["edges"]})
        D_ex = RelDataV4(D.graph, drop_edges=D.gidx[e_del])
        r_ex, s_ex, fin_ex = v4_rank_after(run, q, gold, cand, zc, D_ex)
        tags = set(torch.nonzero(D.Acon[gold] > 0).flatten().tolist())
        pool = [k for k in range(D.E) if int(D.dst_np[k]) in tags and k not in e_del]
        e_rand = list(rng.choice(pool, min(len(e_del), len(pool)), replace=False)) if pool else []
        D_rd = RelDataV4(D.graph, drop_edges=D.gidx[e_rand]) if e_rand else D
        r_rd, s_rd, _ = v4_rank_after(run, q, gold, cand, zc, D_rd)
        expect = sum(t["contrib"] for t in ex["terms"] if t["kind"] == "relation" and set(t["edges"]) & set(e_del))
        val = [v4_edge_valid(D, k, q) for k in e_del]
        _, _, fin_full = v4_rank_after(run, q, gold, cand, zc, D)
        rows.append(dict(q=q, gold=gold, group=groups[i], n_terms=len(ex["terms"]), n_relation_terms=sum(t["kind"] == "relation" for t in ex["terms"]),
                         n_expl_paths=len(ex["top_relation"]), n_expl_edges=len(e_del), sparsity=len(ex["top_relation"]) / max(1, sum(t["kind"] == "relation" for t in ex["terms"])),
                         validity=float(np.mean([v[0] for v in val])), predicate_validity=float(np.mean([v[2] for v in val])),
                         n_predicate_edges=int(sum(v[1] for v in val)),
                         share_relation=ex["totals"]["relation"] / max(1e-9, sum(abs(x) for x in ex["totals"].values())),
                         s_full=s_full, d_s_expl=s_ex - s_full, d_s_expected=-expect, d_s_rand=s_rd - s_full,
                         decomposition_ok=float(abs(ex["score"] - s_full) < 1e-4), deletion_matches_decomposition=float(abs((s_ex - s_full) + expect) < 1e-4) if m.T == 1 else float("nan"),
                         rank_full=r_full, rank_drop_expl=r_ex, rank_drop_rand=r_rd,
                         top1_changed_expl=float(np.argmax(fin_ex) != np.argmax(fin_full)),
                         top10_changed_expl=float(set(np.argsort(-fin_ex)[:10]) != set(np.argsort(-fin_full)[:10])),
                         explanation=" || ".join(f"{t['contrib']:+.4f} {v4_path_text(D, t, gold)}" for t in ex["top_relation"]),
                         opposing=" || ".join(f"{t['contrib']:+.4f} {v4_path_text(D, t, gold)}" for t in ex["opposing"][:3])))
    return pd.DataFrame(rows)


@torch.no_grad()
def relation_analysis_v4(run, part, max_q=3000, seed=0):
    """Theo họ quan hệ: phần đóng góp vào điểm của video đúng, knockout lúc suy luận (bỏ mọi cạnh của họ) ⇒ ΔMRR;
    attention trung bình trên cạnh có vị từ KHỚP truy vấn vs không khớp; theo vai của node nguồn trong truy vấn."""
    m, eng = run["model"], run["engine"]
    D = m.D
    q = part["q_idx"]
    if len(q) > max_q:
        q = np.sort(np.random.RandomState(seed).choice(q, max_q, replace=False))
    gold = CAP_VID[q]
    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, q, part["gallery"], gold)
    sc = score_queries_v4(m, eng, q, part["gallery"], CFG.rerank_k)
    base = metrics_from_ranks(ranks_with_fallback(sc["cand"], combine(sc["s_clip"], sc["s_gnn"], sc["s_path"], run["alpha"], 1.0),
                                                  gold, clip_rank))["MRR"]
    rows = []
    fam_of = np.array([REL_SPEC[V4_REL_TYPES[int(t)]]["family"] for t in D.et.cpu().numpy()])
    for fam in sorted(set(fam_of)):
        D2 = RelDataV4(D.graph, drop_edges=D.gidx[np.where(fam_of == fam)[0]])
        sc2 = score_queries_v4(m, eng, q, part["gallery"], CFG.rerank_k, D=D2)
        mrr = metrics_from_ranks(ranks_with_fallback(sc2["cand"], combine(sc2["s_clip"], sc2["s_gnn"], sc2["s_path"], run["alpha"], 1.0),
                                                     gold, clip_rank))["MRR"]
        rows.append(dict(section="family_knockout", split=part["name"], item=fam, n_edges=int((fam_of == fam).sum()),
                         knockout_MRR=mrr, d_MRR_knockout=mrr - base))
    D0 = RelDataV4(D.graph, drop_edges=D.gidx)
    sc0 = score_queries_v4(m, eng, q, part["gallery"], CFG.rerank_k, D=D0)
    mrr0 = metrics_from_ranks(ranks_with_fallback(sc0["cand"], combine(sc0["s_clip"], sc0["s_gnn"], sc0["s_path"], run["alpha"], 1.0),
                                                  gold, clip_rank))["MRR"]
    rows.append(dict(section="family_knockout", split=part["name"], item="ALL_relations", n_edges=D.E, knockout_MRR=mrr0, d_MRR_knockout=mrr0 - base))
    att_m, att_n, by_role = [], [], defaultdict(list)
    for s0 in range(0, min(len(q), 1000), 128):
        qb = D.query_batch(q[s0:s0 + 128], m.use_pred)
        a = m.attention(qb, m.q(qb["tq"])) if m.rel and D.E else None
        if a is None:
            break
        pe = (D.pidx_np >= 0)
        exm = qb["ex"].cpu().numpy() > 0
        A = a.cpu().numpy()
        att_m.extend(A[exm & pe[None]].tolist()); att_n.extend(A[~exm & pe[None]].tolist())
        rs = qb["role_src"][:, D.src].cpu().numpy()
        for r in range(len(Q_ROLES)):
            if (rs == r).any():
                by_role[Q_ROLES[r]].extend(A[rs == r].tolist())
    rows.append(dict(section="predicate_attention", split=part["name"], item="pred_edges",
                     att_pred_match=float(np.mean(att_m)) if att_m else float("nan"),
                     att_pred_nomatch=float(np.mean(att_n)) if att_n else float("nan"), n_match=len(att_m)))
    for r, v in by_role.items():
        rows.append(dict(section="role_attention", split=part["name"], item=r, att_mean=float(np.mean(v)), n=len(v)))
    return pd.DataFrame(rows), dict(base_MRR=base, all_relations_knockout_MRR=mrr0,
                                    att_pred_match=float(np.mean(att_m)) if att_m else float("nan"),
                                    att_pred_nomatch=float(np.mean(att_n)) if att_n else float("nan"))


## 4.13 Khung thí nghiệm
Mỗi thí nghiệm có: `experiment_id`, stage, giả thuyết, thiết kế đầy đủ, seed, checkpoint fp16, số cạnh theo loại, số node, số tham số, (α, λ) chọn trên val, thời gian chạy, đỉnh bộ nhớ GPU, trạng thái. Thí nghiệm không chạy được ghi `NOT RUN` kèm lý do.

`decide()` áp luật đăng ký trước trên **val**; incumbent là phương án đơn giản hơn. `PRIMARY_COMPARISONS` là họ so sánh chính ở mục 4.2.

In [84]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
DESIGN_BASE = dict(taxonomy="heuristic", contains="binary", conf_mode="sigmoid", rel_policy="all", path_mode="weighted",
                   mp_weight="binary", init="learned", mask="none", rewire=False, pred_shuffle=False, pred_mode="none",
                   related="base", protocol="inductive", tags="base",
                   model="v3", v4_layers=1, v4_rel=True, v4_qcond=True, v4_pred=True)        # model: v3 (TypedSAGE) | v4 (QC-RGAT)
V3_FINAL = dict(DESIGN_BASE, taxonomy="none", mask="no_related")   # cấu hình cuối của v3 (E44, chọn trên VAL ở Kaggle) = đồ thị của v4
RELATED_CFGS = {"base": (CFG.cooc_min_count, CFG.cooc_min_npmi), "npmi20": (CFG.cooc_min_count, 0.20),
                "count10": (10, CFG.cooc_min_npmi)}
GRAPHS, EXPS, DECISIONS, NOT_RUN = {}, {}, [], []
REWIRE_SEED_BASE, SHUFFLE_SEED_BASE = 1000, 2000
VERIFIED_CELLS = None            # {(họ, mức phổ biến)} đạt ngưỡng precision — chỉ có khi có nhãn người chấm


def masked_types(mask):
    """mask = tên trong MASKS, có thể ghép bằng '+' (ví dụ 'no_related+no_taxonomy')."""
    return tuple(sorted({t for m in str(mask).split("+") for t in MASKS[m]}))


def add_mask(mask, extra):
    parts = sorted({m for m in str(mask).split("+") + [extra] if m != "none"})
    return "+".join(parts) or "none"


def graph_key(d, seed):
    return (d["taxonomy"], d["contains"], d["conf_mode"] if d["contains"] == "conf" else "", d["rel_policy"],
            d["path_mode"], d["mask"], (REWIRE_SEED_BASE + seed) if d["rewire"] else None,
            (SHUFFLE_SEED_BASE + seed) if d["pred_shuffle"] else None, d["tags"], d["related"])


def graph_for(d, seed):
    key = graph_key(d, seed)
    if key not in GRAPHS:
        tags = ALT_TAGS[d["tags"]] if d["tags"] != "base" else None
        scores = ALT_SCORES[d["tags"]] if d["tags"] != "base" else TAG_SCORES
        g = GraphV3(INV, RECORDS, SPLIT["train"], RELATIONS2, TAXONOMY[d["taxonomy"]], tag_scores=scores,
                    contains=d["contains"], conf_mode=d["conf_mode"], conf_cal=CONF_CAL, rel_policy=d["rel_policy"],
                    path_mode=d["path_mode"], masked=masked_types(d["mask"]), rewire_seed=key[6], pred_shuffle_seed=key[7],
                    tags_override=tags, related_cfg=RELATED_CFGS[d["related"]], verified_cells=VERIFIED_CELLS,
                    name="|".join(str(x) for x in key))
        assert g.n_nodes == INV.n_nodes and g.inv is INV, "inventory phải cố định"
        GRAPHS[key] = (g, EngineV3(g, VID_FEAT, TXT_FEAT, QTOK2, QPREDS))
    return GRAPHS[key]


def make_part(name, std_only=False):
    """Tập truy vấn + gallery. 'test' (MSVD, thăm dò) và 'ext' (xác nhận) chỉ mở sau khi đóng băng.
    std_only: chỉ caption chuẩn 1k-A của tập ngoài (truy vấn PHỤ)."""
    if name in ("test", "ext"):
        TEST_GATE.check(f"make_part({name})", name)
    q = np.where((CAP_SPLIT == name) & (CAP_STD if std_only else True))[0]
    return dict(name=name + ("_std" if std_only else ""), split=name, q_idx=q, gold=CAP_VID[q], gallery=np.array(SPLIT[name]))


def _mean_std(vals):
    return float(np.mean(vals)), float(np.std(vals))


def not_run(exp_id, stage, desc, reason, design=None, hypothesis=""):
    """Ghi thí nghiệm KHÔNG chạy được (kèm lý do) vào manifest — không tạo số liệu thay thế."""
    EXPS[exp_id] = dict(experiment_id=exp_id, stage=stage, description=desc, hypothesis=hypothesis, design=design, seeds=[],
                        runs={}, status="NOT RUN", reason=reason, val_MRR=(float("nan"), float("nan")),
                        val_R1=(float("nan"), float("nan")), edges={}, n_params=0, seconds=0.0)
    NOT_RUN.append(dict(stage=stage, experiment_id=exp_id, reason=reason))
    print(f"  [{stage}] {exp_id:28s} NOT RUN — {reason}")
    return EXPS[exp_id]


def run_exp(exp_id, stage, design, desc, hypothesis="", seeds=None, lam_fixed=None, reuse=None):
    """Huấn luyện (hoặc tái dùng mô hình của `reuse`) cho từng seed; chọn (α, λ) trên VAL; ghi manifest."""
    seeds = seeds or CFG.seeds
    rec = dict(experiment_id=exp_id, stage=stage, description=desc, hypothesis=hypothesis, design=dict(design),
               seeds=list(seeds), lam_fixed=lam_fixed, reuse=reuse, status="RUNNING", runs={})
    EXPS[exp_id] = rec
    for seed in seeds:
        g, eng = graph_for(design, seed)
        t0 = time.time()
        if DEVICE == "cuda":
            torch.cuda.reset_peak_memory_stats()
        if reuse is not None:
            model, hist = EXPS[reuse]["runs"][seed]["model"], EXPS[reuse]["runs"][seed]["hist"]
        elif design.get("model") == "v4":
            model, hist = train_v4(eng, TRAIN_Q, design, seed=seed)
        else:
            model, hist = train_graph_scorer_v3(eng, TRAIN_Q, SYM_FEAT, seed=seed, mp_weight=design["mp_weight"],
                                                init=design["init"], pred_mode=design["pred_mode"])
        lam_v = 1.0 if design.get("model") == "v4" else lam_fixed           # v4: một điểm đồ thị duy nhất (s_path = 0)
        (a, l), grid, best = tune_on_val_v3(model, eng, PV, CFG.rerank_k, CLIP_RANK_VAL,
                                            lam_grid=[lam_v] if lam_v is not None else None, protocol=design["protocol"])
        ck = None
        if CFG.save_checkpoints and reuse is None:
            ck = CFG.out_dir / "checkpoints" / f"{exp_id}_s{seed}.pt"
            torch.save({k_: v.half() if v.is_floating_point() else v for k_, v in model.state_dict().items()}, ck)
        rec["runs"][seed] = dict(graph=g, engine=eng, model=model, hist=hist, alpha=a, lam=l, grid=grid,
                                 val={k_: best[k_] for k_ in ("R@1", "R@5", "R@10", "MRR")}, rr_val=1.0 / best["rank"],
                                 seconds=round(time.time() - t0, 1), checkpoint=str(ck) if ck else None,
                                 gpu_peak_mb=round(torch.cuda.max_memory_allocated() / 2 ** 20, 1) if DEVICE == "cuda" else None)
    r0 = rec["runs"][seeds[0]]
    rec.update(status="DONE", val_MRR=_mean_std([r["val"]["MRR"] for r in rec["runs"].values()]),
               val_R1=_mean_std([r["val"]["R@1"] for r in rec["runs"].values()]),
               rr_val=np.mean([r["rr_val"] for r in rec["runs"].values()], 0),
               edges=r0["graph"].edge_counts(), n_nodes=r0["graph"].n_nodes,
               n_params=int(sum(p.numel() for p in r0["model"].parameters())),
               alpha_lam=[(r["alpha"], r["lam"]) for r in rec["runs"].values()],
               seconds=float(sum(r["seconds"] for r in rec["runs"].values())),
               gpu_peak_mb=max([r["gpu_peak_mb"] or 0 for r in rec["runs"].values()]) or None)
    m, s = rec["val_MRR"]
    print(f"  [{stage}] {exp_id:28s} VAL MRR {m:.4f} ± {s:.4f} | α,λ {rec['alpha_lam']} | "
          f"{rec['seconds']:.0f}s | cạnh quan hệ {sum(v for k_, v in rec['edges'].items() if not k_.startswith('contains'))}")
    return rec


def design_value(d, field):
    return "/".join(str((d or {}).get(f)) for f in field.split("|"))


def decide(stage, incumbent, candidates, field):
    """Luật đăng ký trước: incumbent = phương án ĐƠN GIẢN hơn (hoặc thiết kế đang dùng). Ứng viên chỉ được chấp nhận khi
    Δ MRR val (trung bình seed) ≥ adopt_min_delta VÀ p(Δ ≤ 0) — bootstrap theo video, hiệu chỉnh Holm trên các ứng viên
    của stage — < adopt_max_p. Nhiều ứng viên đạt ⇒ chọn Δ lớn nhất. Ứng viên NOT RUN bị bỏ qua (ghi lại)."""
    base = EXPS[incumbent]
    cands = [c for c in candidates if EXPS[c]["status"] == "DONE"]
    rows = []
    for c in cands:
        e = EXPS[c]
        d = e["val_MRR"][0] - base["val_MRR"][0]
        bs = cluster_bootstrap_diff(e["rr_val"], base["rr_val"], PV["gold"])
        rows.append(dict(stage=stage, field=field, incumbent=incumbent, candidate=c, value=design_value(e["design"], field),
                         delta_val_MRR=d, ci_lo=bs["ci_lo"], ci_hi=bs["ci_hi"], p_not_better=bs["p_not_better"]))
    adj = holm([r["p_not_better"] for r in rows])
    best = None
    for r, pa in zip(rows, adj):
        r["p_holm"] = float(pa)
        r["passes"] = bool(r["delta_val_MRR"] >= CFG.adopt_min_delta and pa < CFG.adopt_max_p)
        if r["passes"] and (best is None or r["delta_val_MRR"] > best[1]):
            best = (r["candidate"], r["delta_val_MRR"])
    for r in rows:
        r["adopted"] = best is not None and r["candidate"] == best[0]
    for c in candidates:
        if EXPS[c]["status"] != "DONE":
            rows.append(dict(stage=stage, field=field, incumbent=incumbent, candidate=c, value=None, delta_val_MRR=float("nan"),
                             ci_lo=float("nan"), ci_hi=float("nan"), p_not_better=float("nan"), p_holm=float("nan"),
                             passes=False, adopted=False, note=f"NOT RUN: {EXPS[c].get('reason', '')}"))
    DECISIONS.extend(rows)
    winner = best[0] if best else incumbent
    print(f"  → {stage}: {'CHẤP NHẬN ' + winner if best else 'giữ ' + incumbent} ({field} = {design_value(EXPS[winner]['design'], field)})")
    return winner


@torch.no_grad()
def evaluate_exp(exp_id, part, clip_rank, protocol=None):
    """Đánh giá MỘT lần mỗi seed với (α, λ) đã đóng băng; trả về thứ hạng theo truy vấn."""
    e = EXPS[exp_id]
    out = {}
    for seed, r in e["runs"].items():
        sc = score_queries_v3(r["model"], r["engine"], part["q_idx"], part["gallery"], CFG.rerank_k,
                              protocol or e["design"]["protocol"])
        fin = combine(sc["s_clip"], sc["s_gnn"], sc["s_path"], r["alpha"], r["lam"])
        out[seed] = dict(rank=ranks_with_fallback(sc["cand"], fin, part["gold"], clip_rank),
                         cand_hash=hashlib.sha1(sc["cand"].tobytes()).hexdigest()[:12])
    return out


def query_groups_v3(engine, q_idx):
    """direct | indirect | other (như v1/v2) trên đồ thị của engine."""
    g, q_idx = engine.graph, np.asarray(q_idx)
    gold = CAP_VID[q_idx]
    pf = engine.path_feats(gold[:, None], q_idx)[:, 0]
    out = []
    for j, q in enumerate(q_idx):
        cids = {n_ for n_, _, _ in engine.q_concepts[q]}
        out.append("direct" if cids & set(g.vid_tags[gold[j]]) else ("indirect" if pf[j, 1:].sum() > 0 else "other"))
    return np.array(out)


def rare_mask(engine, q_idx, thr=5):
    """Truy vấn 'khái niệm hiếm': mọi node lớp khớp được có tần suất gắn nhãn trên video TRAIN ≤ thr (hoặc chỉ khớp khái niệm)."""
    g = engine.graph
    out = []
    for q in q_idx:
        cls = [n_ for n_, _, _ in engine.q_concepts[q] if g.inv.is_class(n_)]
        out.append(bool(engine.q_concepts[q]) and (not cls or max(g.cls_df[c] for c in cls) <= thr))
    return np.array(out)


def predicate_groups(engine, q_idx, thr=None):
    """Chỉ xét vị từ của truy vấn CÓ trên cạnh của đồ thị (svo/prep): 'no_pred' nếu không có; 'pred_rare' nếu MỌI vị từ khớp
    đều có support TRAIN ≤ thr (truy vấn chỉ dựa vào vị từ hiếm); ngược lại 'pred_common'."""
    thr = thr or 2 * CFG.pred_min_support
    used = {pid for rel in PRED_FAMILIES for d in engine.graph.pshare[rel].values() for pid in d}
    out = []
    for q in q_idx:
        pids = [p for rel in PRED_FAMILIES for p in QPREDS[q][rel] if p in used]
        if not pids:
            out.append("no_pred")
        else:
            out.append("pred_rare" if max(PRED_SUPPORT.get(PRED_VOCAB[p], 0) for p in pids) <= thr else "pred_common")
    return np.array(out)


@torch.no_grad()
def path_analysis(run, part, protocol="inductive", max_q=4000, seed=0):
    """Theo mẫu: phần đóng góp, knockout từng mẫu và từng HỌ quan hệ lúc suy luận; theo node: bậc, số đường đi đi qua,
    phạm vi video; tổng hợp: ρ(degree, s_path), ρ(degree, điểm cuối), P(degree video đúng > video âm)."""
    g, eng, model = run["graph"], run["engine"], run["model"]
    q = part["q_idx"]
    if len(q) > max_q:
        q = np.sort(np.random.RandomState(seed).choice(q, max_q, replace=False))
    gold = CAP_VID[q]
    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, q, part["gallery"], gold)
    sc = score_queries_v3(model, eng, q, part["gallery"], CFG.rerank_k, protocol)
    pf = eng.path_feats(sc["cand"], q)
    w = model.pattern_weights().detach().cpu().numpy()
    gain = F.softplus(model.path_gain).item()
    contrib = gain * pf * w
    tot = contrib.sum(-1, keepdims=True)
    share = np.where(tot > 0, contrib / np.maximum(tot, 1e-12), 0).mean((0, 1))
    reldeg_out = sum(np.asarray((g.A[t] > 0).sum(1)).ravel() for t in PATH_TYPES)
    reldeg_in = sum(np.asarray((g.A[t] > 0).sum(0)).ravel() for t in PATH_TYPES)
    con_in = np.asarray((g.A_con > 0).sum(0)).ravel()
    vdeg = np.array([reldeg_out[g.vid_tags[v]].sum() for v in range(g.n_vid)])
    cand_deg = vdeg[sc["cand"]]
    from scipy.stats import spearmanr
    is_gold = (sc["cand"] == gold[:, None])
    base_fin = combine(sc["s_clip"], sc["s_gnn"], sc["s_path"], run["alpha"], run["lam"])
    base_mrr = metrics_from_ranks(ranks_with_fallback(sc["cand"], base_fin, gold, clip_rank))["MRR"]

    def knock(idx):
        pf2 = pf.copy(); pf2[..., idx] = 0
        sp2 = znorm_per_row(gain * (pf2 * w).sum(-1))
        return metrics_from_ranks(ranks_with_fallback(sc["cand"], combine(sc["s_clip"], sc["s_gnn"], sp2, run["alpha"], run["lam"]),
                                                      gold, clip_rank))["MRR"]

    rows = []
    for p_i, p in enumerate(PATTERNS):
        mrr = knock([p_i])
        reach = np.asarray((g.R[p] > 0).sum(0)).ravel()
        rows.append(dict(section="pattern", split=part["name"], item=p, weight=float(w[p_i]), nnz=int(g.R[p].nnz),
                         frac_pairs_nonzero=float((pf[..., p_i] > 0).mean()), contribution_share=float(share[p_i]),
                         knockout_MRR=mrr, d_MRR_knockout=mrr - base_mrr, max_video_reach=int(reach.max()) if reach.size else 0))
    groups = {f"family:{f}": [PIDX[p] for p in PATTERNS if p.startswith(f"P1_{f}") or p.startswith(f"P4_{f}")
                              or p == f"P1m_{FAMILY_REL[f]}"] for f in FAMILIES}
    groups.update({"family:related": [PIDX["P3_related"], PIDX["P5_related_isa"]],
                   "family:isa": [PIDX["P2_isa"]] + [PIDX[f"P4_{f}_isa"] for f in FAMILIES] + [PIDX["P5_related_isa"]],
                   "all_indirect": list(range(1, len(PATTERNS)))})
    for nm, idx in groups.items():
        mrr = knock(idx)
        rows.append(dict(section="family_knockout", split=part["name"], item=nm, knockout_MRR=mrr, d_MRR_knockout=mrr - base_mrr,
                         contribution_share=float(share[idx].sum())))
    reach_all = sum(np.asarray((g.R[p] > 0).sum(0)).ravel()[:g.n_nodes] for p in PATTERNS[1:] if not p.startswith("P1m_"))
    through = con_in * reldeg_out                         # số đường đi 2 bước video → node → lớp/khái niệm đi QUA node
    for i in np.argsort(-through)[:50]:
        if through[i] <= 0:
            break
        rows.append(dict(section="node", split=part["name"], item=g.inv.name[i], node_type=g.inv.semantic_type(i),
                         videos_tagged=int(con_in[i]), rel_out_degree=int(reldeg_out[i]), rel_in_degree=int(reldeg_in[i]),
                         paths_through=int(through[i]), videos_reached_indirect=int(reach_all[i]), idf=float(g.idf[i])))
    flat_deg, flat_sp, flat_fin = cand_deg.ravel(), sc["path_raw"].ravel(), base_fin.ravel()
    gold_rank = ranks_with_fallback(sc["cand"], base_fin, gold, clip_rank)
    gold_deg = vdeg[gold]
    deg_auc = float(np.mean([(cand_deg[k][is_gold[k]] > cand_deg[k][~is_gold[k]]).mean()
                              for k in range(len(q)) if is_gold[k].any()])) if is_gold.any() else float("nan")
    summary = dict(split=part["name"], n_queries=len(q), base_MRR=base_mrr,
                   spearman_degree_vs_s_path=float(spearmanr(flat_deg, flat_sp).correlation),
                   spearman_degree_vs_final=float(spearmanr(flat_deg, flat_fin).correlation),
                   spearman_gold_degree_vs_gold_rank=float(spearmanr(gold_deg, gold_rank).correlation),
                   p_gold_degree_gt_negative=deg_auc, w_path_effective=(1 - run["alpha"]) * (1 - run["lam"]),
                   hubs=[dict(node=g.inv.name[i], paths_through=int(through[i])) for i in np.argsort(-through)[:10]])
    rows.append(dict(section="summary", split=part["name"], item="degree_bias",
                     **{k: v for k, v in summary.items() if k not in ("hubs", "split")}))
    return pd.DataFrame(rows), summary


# ---- họ so sánh CHÍNH (đăng ký trước, một phía: a > b), hiệu chỉnh Holm; chỉ trên tập ngoài mới là XÁC NHẬN ----
PRIMARY_COMPARISONS = [("H1_graph_vs_clip", "G3", "G0"), ("H2_relations_vs_contains_only", "G3", "G1"),
                       ("H3_predicate_vs_family_only", "G3", "G2"), ("H4_true_vs_shuffled_predicates", "G3", "G4"),
                       ("H5_true_vs_rewired_topology", "G3", "G5"), ("H6_query_conditioned_vs_static", "G3", "G6"),
                       ("H7_v4_vs_v3", "G3", "V3REF"),
                       ("H8_role_edges", "G3", "G7"), ("H9_svo_prep_edges", "G3", "G8")]
H_V4 = ("quan hệ được dùng CÓ ĐIỀU KIỆN THEO TRUY VẤN (loại cạnh, vị từ khớp truy vấn, vai của node trong truy vấn) cải thiện "
        "retrieval so với cùng mô hình không có quan hệ / quan hệ sai")

## 4.14 Self-test v3 (mục 12)
| mục 12 | kiểm tra | id |
|---|---|---|
| 1 | embedding vị từ tác động forward; pred_mode none (≡ v2) **không** bị ảnh hưởng | P1 |
| 2 | hai vị từ khác nhau trên cùng cặp node ⇒ thông điệp khác | P2 |
| 3 | rewiring giữ kiểu node, họ, bậc, trọng số, đa tập vị từ | G6 |
| 4 | hoán vị vị từ không đổi topology / trọng số | G7 |
| 5 | mỗi ablation tắt đúng thành phần; inventory cố định | AB, G1 |
| 6 | trọng số chỉ nhân một lần (khớp tính tay, bất biến tỉ lệ) | G4 |
| 7 | node không có hàng xóm không sinh NaN; cô lập inductive | G4, G5 |
| 8 | không rò rỉ caption train/val/test/tập ngoài | C1, C1b, C2, C3, C8, C9 |
| 9 | giải thích đúng vị từ, đúng chiều; điểm khớp ma trận | G3, X1 |
| 10 | số trong báo cáo khớp CSV đầu ra | R1 |

Một kiểm thử phần mềm PASS **không** có nghĩa giả thuyết khoa học đã được xác nhận.

In [85]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
CANARY_NOUNS = ["flamingo", "accordion", "kayak", "harmonica", "trombone", "anvil", "walrus", "xylophone",
                "gondola", "hedgehog", "unicycle", "periscope", "tuba", "iguana", "llama"]
CANARY_VERBS = ["juggle", "knit", "yodel", "whittle", "embroider", "lasso"]


def _same_graph(g1, g2):
    if g1.n_nodes != g2.n_nodes or g1.inv.sha1 != g2.inv.sha1:
        return False
    if (g1.A_con != g2.A_con).nnz:
        return False
    return (all((g1.A[t] != g2.A[t]).nnz == 0 for t in RELATION_TYPES) and np.array_equal(g1.idf, g2.idf)
            and g1.pdist == g2.pdist)


def _pred_vocab_of(rel):
    s = Counter()
    for v in SPLIT["train"]:
        s.update({(r, p) for _, r, _, p in rel[v]})
    return sorted(k for k in s if k[0] in PRED_FAMILIES)


def test_canary(splits=("test", "ext"), n_videos=40):
    """Canary chỉ có trong caption TEST / tập ngoài không được lọt vào từ vựng, quan hệ, từ vựng vị từ; đối chứng dương:
    đặt canary vào TRAIN thì phải phát hiện được."""
    terms = {k.rsplit("#", 1)[0] for k in VOCAB2["terms"]}
    a, b = [w for w in CANARY_NOUNS if w not in terms and is_visual_noun(w)][:2]
    seen_pred = {t for _, p in PRED_VOCAB for t in p.split()}           # động từ canary cũng không được là vị từ train
    v = next((w for w in CANARY_VERBS if w not in terms and w not in seen_pred and is_action_verb(w)),
             next(w for w in CANARY_VERBS if w not in terms and is_action_verb(w)))
    canary = [f"a {a} is {naive_gerund(v)} a {b}"] * 3
    tv = [x for s in splits for x in SPLIT[s][:n_videos]]
    store = CAPS.replaced({x: CAPS._caps[x] + canary for x in tv})
    voc = build_vocab_v2(store, SPLIT)
    rel = build_relations_v2(store, SPLIT, voc)
    keys = {ckey(a, "n"), ckey(b, "n"), ckey(v, "v")}
    leak_v = sorted(keys & set(voc["terms"]))
    leak_r = [t for trip in rel.values() for t in trip if {t[0], t[2]} & keys]
    leak_p = sorted(set(_pred_vocab_of(rel)) - set(PRED_VOCAB))          # vị từ MỚI so với từ vựng vị từ train
    fake = dict(SPLIT, train=list(SPLIT["train"]) + list(tv))
    store_pos = store.replaced({}, new_split={x: "train" for x in tv})
    voc_pos = build_vocab_v2(store_pos, fake)
    rel_pos = build_relations_v2(store_pos, fake, voc_pos)
    detected = {ckey(a, "n"), ckey(b, "n")} <= set(voc_pos["terms"]) and any({t[0], t[2]} & keys for trip in rel_pos.values() for t in trip)
    return dict(canary=f"{a} –{v}→ {b}", splits=list(splits), n_videos=len(tv), leak_vocab=leak_v, leak_relations=leak_r[:3],
                leak_predicates=leak_p[:3], positive_control_detected=bool(detected))


def test_invariance():
    """Xáo / thay caption TEST và tập ngoài rồi dựng lại TOÀN BỘ: từ vựng, quan hệ, vị từ, tagger, taxonomy, đồ thị."""
    tv = list(SPLIT["test"]) + list(SPLIT["ext"])
    perm = np.random.RandomState(0).permutation(tv)
    junk = ["a purple elephant is skiing on the moon", "zebra piano astronaut volcano juggling"] * 4
    store = CAPS.replaced({x: (CAPS._caps[int(p)] if k % 2 == 0 else junk) for k, (x, p) in enumerate(zip(tv, perm))})
    voc = build_vocab_v2(store, SPLIT)
    rel = build_relations_v2(store, SPLIT, voc)
    tags, scores, sel, _ = run_tagger_v2(FRAME_FEAT, VID_FEAT, POOL_SEL["n_frames"], voc, SPLIT, store)
    tax, cmeta, _, _ = build_taxonomy_v2(voc, store, SPLIT, {v: tags[v] for v in SPLIT["train"]})
    recs = [dict(r, tags=tags[r["vid"]]) for r in RECORDS]
    inv = NodeInventory(recs, voc, cmeta)
    g2 = GraphV3(inv, recs, SPLIT["train"], rel, tax["heuristic"], tag_scores=scores)
    g1 = graph_for(DESIGN_BASE, CFG.seeds[0])[0]
    return dict(vocab=voc["terms"] == VOCAB2["terms"], relations=rel == RELATIONS2, predicate_vocab=_pred_vocab_of(rel) == PRED_VOCAB,
                tagger_selection=sel == TAG_SELECT2, tags=tags == TAGS2, taxonomy=tax == TAXONOMY, inventory=inv.sha1 == INV.sha1,
                graph=_same_graph(g1, g2), annotation_sample=build_annotation_sample(store, SPLIT, voc, RECORDS)[0]
                .sample_id.tolist() == ANN_SAMPLE.sample_id.tolist(),
                automatic_verification=(build_agreement(store, SPLIT, voc)[0] == AGREE) if AGREE else True)


VARIANT_DESIGNS = ([dict(DESIGN_BASE, mask=m) for m in MASKS] + [dict(DESIGN_BASE, rewire=True), dict(DESIGN_BASE, pred_shuffle=True),
                   dict(DESIGN_BASE, rel_policy="support"), dict(DESIGN_BASE, taxonomy="none"), dict(DESIGN_BASE, contains="conf"),
                   dict(DESIGN_BASE, related="npmi20"), dict(DESIGN_BASE, path_mode="pred_match")])


def test_inventory_constant():
    """Mọi biến thể đồ thị dùng CÙNG inventory; với CÙNG cấu hình mô hình, số tham số không đổi giữa các biến thể đồ thị."""
    gs = [graph_for(d, CFG.seeds[0]) for d in VARIANT_DESIGNS]
    params = {pm: sorted({sum(p.numel() for p in make_model(e, SYM_FEAT, pred_mode=pm).parameters()) for _, e in gs[:4]})
              for pm in PRED_MODES}
    return dict(n_graphs=len(gs), node_counts=sorted({g.n_nodes for g, _ in gs}), same_inventory=all(g.inv is INV for g, _ in gs),
                param_counts_by_pred_mode=params)


def test_schema_conformance(g):
    """Mọi cạnh: kiểu nút đầu/cuối + POS đúng REL_SPEC; không self-loop; trọng số trong (0, 1]; vị từ đúng họ:
    svo/prep ⇒ mọi vị từ thuộc từ vựng vị từ của họ đó (không lẫn tên loại quan hệ); agent/object ⇒ vị từ = lemma động từ."""
    bad = Counter()
    inv = g.inv
    fam_names = set(REL_SPEC) | set(FAMILIES)
    for t in ["contains"] + RELATION_TYPES:
        M = (g.A_con if t == "contains" else g.A[t]).tocoo()
        spec = REL_SPEC[t]
        base = t[:-4] if t.endswith("_inv") else t
        for u, v, w in zip(M.row.tolist(), M.col.tolist(), M.data.tolist()):
            src_ok = spec["src"].split(":")[0].split("|")
            tgt_ok = spec["tgt"].split(" ")[0].split(":")[0].split("|")
            if inv.node_type[u] not in src_ok or inv.node_type[v] not in tgt_ok:
                bad[f"{t}:type"] += 1
            if ":" in spec["src"] and inv.semantic_type(u) != spec["src"].split(":")[1]:
                bad[f"{t}:src_pos"] += 1
            if ":" in spec["tgt"] and inv.semantic_type(v) != spec["tgt"].split(":")[1]:
                bad[f"{t}:tgt_pos"] += 1
            if t in ("isa", "isa_inv") and inv.pos[u] != inv.pos[v]:
                bad[f"{t}:pos_mismatch"] += 1
            if u == v:
                bad[f"{t}:self_loop"] += 1
            if not (0 < w <= 1 + 1e-6):
                bad[f"{t}:weight"] += 1
            if base in REL_CATS:
                d = g.predicates_of(t, u, v)
                if not d:
                    bad[f"{t}:no_predicate"] += 1
                if any(p in fam_names for p in d):
                    bad[f"{t}:predicate_is_relation_name"] += 1
                if base in PRED_FAMILIES and any((base, p) not in PRED_ID for p in d):
                    bad[f"{t}:predicate_not_in_vocab"] += 1
                if base in ("agent_of", "object_of"):
                    verb = inv.lemma[v if not t.endswith("_inv") else u]
                    if set(d) != {verb}:
                        bad[f"{t}:predicate_not_verb_node"] += 1
    return dict(violations=dict(bad), n_checked=int(g.A_con.nnz + sum(g.A[t].nnz for t in RELATION_TYPES)))


def test_path_identity(n=200):
    """Điểm ma trận (engine) = tổng/max liệt kê đường đi ở MỌI path_mode (kể cả pred_match); tính lại R khớp graph.R;
    mọi đường đi khớp vị từ ghi đúng vị từ của đúng cạnh theo đúng chiều; đối chứng âm: sửa pred_id ⇒ bị phát hiện."""
    out = {}
    rng = np.random.RandomState(0)
    pool = np.where(np.isin(CAP_SPLIT, ["train", "val"]))[0]
    qs = rng.choice(pool, min(n, len(pool)), replace=False)
    pairs = [(q, int(CAP_VID[q])) for q in qs] + [(q, int(v)) for q in qs for v in rng.choice(SPLIT["train"], 2)]
    for mode in ("weighted", "count", "degnorm", "dedup", "pred_match"):
        g, eng = graph_for(dict(DESIGN_BASE, path_mode=mode), CFG.seeds[0])
        bad = cnt = npred = pbad = neg = 0
        for q, v in pairs:                               # video đúng + video ngẫu nhiên (có nhiều đường đi gián tiếp hơn)
            cids, qpred, qtok = q_args(eng, q)
            if not cids:
                continue
            cnt += 1
            paths = enumerate_paths_v3(g, v, cids, qpred)
            bad += int(not np.allclose(eng.path_feats(np.array([[v]]), np.array([q]))[0, 0],
                                       paths_to_counts_v3(paths, mode), atol=1e-5))
            for p in paths:
                if p.get("pred") is not None:
                    npred += 1
                    pbad += int(not pred_valid_v3(g, p, v, qtok))
                    wrong = dict(p, pred=(p["pred"][0], (p["pred"][1] + 1) % len(PRED_VOCAB), p["pred"][2]))
                    neg += int(not pred_valid_v3(g, wrong, v, qtok))
        R2 = g.compute_R(rows=np.arange(g.n_vid), PM=g.path_mats())
        diff = max(float(abs(R2[p] - g.R[p]).max()) if (R2[p] - g.R[p]).nnz else 0.0 for p in PATTERNS)
        out[mode] = dict(pairs=cnt, mismatches=bad, recompute_maxdiff=diff, predicate_paths=npred, predicate_invalid=pbad,
                         negative_control_detected=neg)
    return out


def test_mp_weights():
    """Unit test TypedSAGEv3: trọng số cạnh dùng trong forward (weighted), bị bỏ qua (binary), w=0 ≡ không cạnh, node cô lập
    hữu hạn, bất biến thứ tự cạnh, gradient hữu hạn; trọng số chỉ nhân MỘT lần: (i) khớp công thức tính tay, (ii) weighted_mean
    bất biến khi nhân mọi trọng số với hằng số (nhân lặp sẽ làm đổi), (iii) cùng hệ số cho h_u và vị từ e_uv."""
    torch.manual_seed(0)
    res = {}
    x = torch.randn(4, 8)
    ei = torch.tensor([[1, 2, 3], [0, 0, 0]])
    et = torch.zeros(3, dtype=torch.long)
    pidx = torch.tensor([0, 1, 2])
    ph = torch.randn(3, 8)
    for mode in ("binary", "weighted_mean", "weighted_symnorm"):
        L = TypedSAGEv3(8, 8, 1, [0], 0.0, mode, pred_type_ids=[0])
        w1 = torch.tensor([1.0, 1.0, 1.0]); w2 = torch.tensor([1.0, 0.0, 1.0]); w3 = torch.tensor([3.0, 1.0, 1.0])
        o1, o2, o3 = L(x, ei, et, w1, pidx, ph), L(x, ei, et, w2, pidx, ph), L(x, ei, et, w3, pidx, ph)
        perm = torch.tensor([2, 0, 1])
        o_perm = L(x, ei[:, perm], et[perm], w3[perm], pidx[perm], ph)
        o_drop = L(x, ei[:, [0, 2]], et[[0, 2]], torch.tensor([1.0, 1.0]), pidx[[0, 2]], ph)
        o_scaled = L(x, ei, et, 5 * w3, pidx, ph)
        ww = torch.ones(3) if mode == "binary" else w3
        coef = ww / ww.sum() if mode != "weighted_symnorm" else ww / torch.sqrt(ww * ww.sum())
        manual = L.self_w(x[0]) + torch.sigmoid(L.gate[0]) * (L.neigh_w[0]((coef[:, None] * x[1:4]).sum(0))
                                                               + L.pred_w["0"]((coef[:, None] * ph).sum(0)))
        xg = x.clone().requires_grad_(True)
        L(xg, ei, et, w3, pidx, ph).sum().backward()
        changes = bool((o1[0] - o3[0]).abs().max() > 1e-6)
        zero_ok = bool((o2[0] - o_drop[0]).abs().max() < 1e-5)
        res[mode] = dict(weight_changes_output=changes, zero_weight_equals_edge_removed=zero_ok,
                         isolated_nodes_finite=bool(torch.isfinite(o1[1:]).all()),
                         edge_order_invariant=bool((o3 - o_perm).abs().max() < 1e-5),
                         grad_finite=bool(torch.isfinite(xg.grad).all()),
                         matches_formula=bool((o3[0] - manual).abs().max() < 1e-5),
                         scale_invariant=bool((o3[0] - o_scaled[0]).abs().max() < 1e-5),
                         as_designed=((not changes) if mode == "binary" else (changes and zero_ok))
                         and bool((o3[0] - manual).abs().max() < 1e-5)
                         and (mode != "weighted_mean" or bool((o3[0] - o_scaled[0]).abs().max() < 1e-5)))
    return res


def test_predicate_forward():
    """(1) Embedding vị từ thật sự tác động forward: đổi embedding của các vị từ đang dùng ⇒ biểu diễn node đổi khi
    pred_mode ∈ {clip, clip_adapt}; pred_mode = none (relation-family-only, ≡ v2) PHẢI không đổi (đối chứng âm).
    (2) Hai vị từ khác nhau trên CÙNG cặp node tạo thông điệp khác nhau (trường hợp kiểm soát, embedding CLIP thật)."""
    g, eng = graph_for(DESIGN_BASE, CFG.seeds[0])
    used = sorted({pid for rel in PRED_FAMILIES for d in g.pshare[rel].values() for pid in d})
    out = {}
    for pm in PRED_MODES:
        torch.manual_seed(0)
        model = make_model(eng, SYM_FEAT, pred_mode=pm).eval()
        if pm == "clip_adapt":                         # MLP dư khởi tạo 0 ⇒ cho trọng số khác 0 để kiểm tra cả nhánh dư
            with torch.no_grad():
                for prm in model.pred_enc.res.parameters():
                    prm.normal_(0, 0.05)
        with torch.no_grad():
            h1 = model.encode_graph(*eng.edges_on("train"))
            if pm != "none" and used:
                old = model.pred_emb.clone()
                model.pred_emb[used] = torch.roll(model.pred_emb[used], 1, 0)
                h2 = model.encode_graph(*eng.edges_on("train"))
                model.pred_emb.copy_(old)
            else:
                ed = list(eng.edges_on("train"))
                ed[3] = torch.full_like(ed[3], -1)                       # gỡ mọi chỉ số vị từ
                h2 = model.encode_graph(*ed)
        out[pm] = dict(max_change=float((h1 - h2).abs().max()), finite=bool(torch.isfinite(h1).all()))
    out["predicate_used_as_designed"] = (out["clip"]["max_change"] > 1e-5 and out["clip_adapt"]["max_change"] > 1e-5
                                         and out["none"]["max_change"] < 1e-7)
    pick = {}
    for want in ("ride", "feed", "play", "hold"):
        if ("svo", want) in PRED_ID:
            pick[want] = PRED_ID[("svo", want)]
    pair = list(pick.items())[:2] if len(pick) >= 2 else [(PRED_VOCAB[i][1], i) for i in used[:2]]
    torch.manual_seed(0)
    L = TypedSAGEv3(8, 8, 1, [0], 0.0, "binary", pred_type_ids=[0])
    enc = PredicateEncoder(PRED_EMB.shape[1], 8, "clip", 16)
    pe = enc(torch.eye(len(PRED_VOCAB))[[pair[0][1], pair[1][1]]], torch.from_numpy(PRED_EMB))
    x = torch.randn(2, 8)
    ei, et, w = torch.tensor([[0], [1]]), torch.zeros(1, dtype=torch.long), torch.ones(1)
    with torch.no_grad():
        a = L(x, ei, et, w, torch.tensor([0]), pe)[1]
        b = L(x, ei, et, w, torch.tensor([1]), pe)[1]
    out["two_predicates"] = dict(predicates=[pair[0][0], pair[1][0]], message_diff=float((a - b).abs().max()),
                                 differ=bool((a - b).abs().max() > 1e-6))
    return out


def test_ablations(g0):
    """Mỗi ablation tắt ĐÚNG thành phần đã mô tả: mask ⇒ loại cạnh bị tắt = 0, loại khác giữ nguyên; support ⇒ mọi vị từ còn
    lại đạt ngưỡng; pred_mode none ⇒ mô hình không có tham số vị từ; contains_only ⇒ chỉ còn contains."""
    out = {}
    for m, types in MASKS.items():
        g = graph_for(dict(DESIGN_BASE, mask=m), CFG.seeds[0])[0]
        off_ok = all(g.A[t].nnz == 0 for t in types)
        rest_ok = all((g.A[t] != g0.A[t]).nnz == 0 for t in RELATION_TYPES if t not in types) and (g.A_con != g0.A_con).nnz == 0
        out[f"mask:{m}"] = bool(off_ok and rest_ok)
    gs = graph_for(dict(DESIGN_BASE, rel_policy="support"), CFG.seeds[0])[0]
    out["policy:support"] = all(PRED_SUPPORT[(r_, p)] >= CFG.pred_min_support for (r_, a, b), d in gs.pdist.items()
                                if gs.A[r_][a, b] != 0 for p in d)
    if AGREE:
        ga = graph_for(dict(DESIGN_BASE, rel_policy="agreement"), CFG.seeds[0])[0]
        out["policy:agreement"] = all(agreement_rate(r_, INV.id2key[a], INV.id2key[b], p) >= CFG.agree_min
                                      for (r_, a, b), d in ga.pdist.items() if ga.A[r_][a, b] != 0 for p in d)
    m0 = make_model(E_BASE, SYM_FEAT, pred_mode="none")
    out["pred_mode:none_has_no_predicate_params"] = not any("pred" in n_ for n_, _ in m0.named_parameters())
    m1 = make_model(E_BASE, SYM_FEAT, pred_mode="clip")
    out["pred_mode:clip_has_predicate_params"] = any("pred" in n_ for n_, _ in m1.named_parameters())
    out["pred_mode:clip_no_adapter"] = not any("pred_enc.res" in n_ for n_, _ in m1.named_parameters())
    gp = graph_for(dict(DESIGN_BASE, path_mode="pred_match"), CFG.seeds[0])[0]
    out["path:weighted_has_no_P1m"] = all(g0.R[f"P1m_{r}"].nnz == 0 for r in PRED_FAMILIES)
    out["path:pred_match_has_P1m"] = any(gp.R[f"P1m_{r}"].nnz > 0 for r in PRED_FAMILIES)
    out["path:pred_match_same_other_patterns"] = all((gp.R[p] != g0.R[p]).nnz == 0 for p in PATTERNS if not p.startswith("P1m_"))
    return out


def test_inductive_isolation():
    """Strict inductive: đổi embedding của MỘT video đánh giá không làm đổi node lớp/khái niệm hay video khác;
    transductive thì có (để chứng minh phép thử nhạy). Chạy cho cả video test MSVD và video tập ngoài."""
    g, eng = graph_for(DESIGN_BASE, CFG.seeds[0])
    model = make_model(eng, SYM_FEAT, pred_mode="clip").eval()
    out = {}
    for s in ("test", "ext"):
        tv = np.array(SPLIT[s])
        if len(tv) < 2:
            continue
        v0, v1 = int(tv[0]), int(tv[1])
        for proto in PROTOCOLS:
            with torch.no_grad():
                h1 = model.encode_graph(*eng.edges_on(proto))
                old = model.vid_feat[v0].clone()
                model.vid_feat[v0] = torch.randn_like(old)
                h2 = model.encode_graph(*eng.edges_on(proto))
                model.vid_feat[v0] = old
            sym = slice(g.n_vid, g.n_nodes)
            out[f"{s}:{proto}"] = dict(symbolic_changed=float((h1[sym] - h2[sym]).abs().max()),
                                       other_video_changed=float((h1[v1] - h2[v1]).abs().max()),
                                       own_video_changed=float((h1[v0] - h2[v0]).abs().max()),
                                       all_finite=bool(torch.isfinite(h1).all()))
    out["edges"] = eng.edge_stats
    return out


def test_guards():
    out = {}
    for s in ("test", "ext"):
        if not SPLIT[s]:
            continue
        try:
            make_part(s); out[f"{s}_before_freeze_blocked"] = False
        except RuntimeError:
            out[f"{s}_before_freeze_blocked"] = True
        sq = np.where(CAP_SPLIT == s)[0][:3]
        for name, fn in ((f"{s}→gnn_train", lambda: CAPS.use(sq, "gnn_train")), (f"{s}→tune", lambda: CAPS.use(sq, "tune")),
                         (f"{s}→vocab", lambda: CAPS.by_video(SPLIT[s][:2], "vocab")),
                         (f"{s}→relations", lambda: CAPS.by_video(SPLIT[s][:2], "relations")),
                         (f"{s}→annotation", lambda: CAPS.by_video(SPLIT[s][:2], "annotation")),
                         (f"{s}→agreement", lambda: CAPS.by_video(SPLIT[s][:2], "agreement"))):
            try:
                fn(); out[name + "_blocked"] = False
            except PermissionError:
                out[name + "_blocked"] = True
        try:
            CAPS.use(sq, "eval"); out[f"{s}_eval_before_freeze_blocked"] = False
        except RuntimeError:
            out[f"{s}_eval_before_freeze_blocked"] = True
    vq = np.where(CAP_SPLIT == "val")[0][:3]
    for name, fn in (("val→gnn_train", lambda: CAPS.use(vq, "gnn_train")),
                     ("val→relations", lambda: CAPS.by_video(SPLIT["val"][:2], "relations")),
                     ("train→tune", lambda: CAPS.use(np.where(CAP_SPLIT == "train")[0][:3], "tune"))):
        try:
            fn(); out[name + "_blocked"] = False
        except PermissionError:
            out[name + "_blocked"] = True
    return out

# ---------------------------------- Self-test v4 (QC-RGAT) — nhanh (< 1 phút CPU), không huấn luyện ----------------------------------
def _v4_model_for_test(design, seed=0):
    """Mô hình KHỞI TẠO NGẪU NHIÊN nhưng mọi cổng đều khác 0 (để phép thử không qua vì tham số bằng 0)."""
    g, eng = graph_for(design, CFG.seeds[0])
    set_seed(seed)
    m = make_model_v4(eng, design).eval()
    with torch.no_grad():
        for p in (m.beta, m.kappa, m.g_ex, m.g_soft, m.rgate):
            p.copy_(torch.randn_like(p))
    return g, eng, m


def _queries_with_pred_edges(D, n, seed=0):
    """Truy vấn TRAIN có ≥ 1 cạnh mang vị từ khớp vị từ của truy vấn (để thử nhánh vị từ)."""
    rng = np.random.RandomState(seed)
    out = []
    for q in rng.permutation(TRAIN_Q)[:4000]:
        ids = [p for rel in PRED_FAMILIES for p in QPREDS[q][rel]]
        if ids and D.E and np.asarray(D.Mex[ids].sum(0)).ravel().max() > 0 and V4_HITS[q]:
            out.append(int(q))
            if len(out) >= n:
                break
    return out


def test_v4_decomposition(n=30):
    """Σ đóng góp (thị giác + nhãn + đường đi) = điểm của mô hình, cho T = 1 và 2; và xoá một cạnh ⇒ điểm giảm ĐÚNG bằng tổng đóng
    góp các đường đi qua cạnh đó (T = 1: tuyệt đối; T = 2: tính lại trên đồ thị đã xoá)."""
    out = {}
    for T in (1, 2):
        g, eng, m = _v4_model_for_test(dict(V3_FINAL, model="v4", v4_layers=T))
        qs = _queries_with_pred_edges(m.D, n)
        maxdiff, del_diff, n_rel, n_del = 0.0, 0.0, 0, 0
        vids = [int(v) for v in SPLIT["train"] if m.D.Acon[int(v)].gt(0).any()]
        for q in qs:
            v = vids[q % len(vids)]
            terms = m.decompose(q, v)
            with torch.no_grad():
                s = float(m.score(m.D.query_batch([q], m.use_pred), torch.tensor([[v]], device=DEVICE))[0, 0])
            maxdiff = max(maxdiff, abs(s - sum(t["contrib"] for t in terms)))
            rel = [t for t in terms if t["kind"] == "relation"]
            n_rel += len(rel)
            if rel:
                e = max(rel, key=lambda t: abs(t["contrib"]))["edges"][-1]
                D2 = RelDataV4(g, drop_edges=[int(m.D.gidx[e])])
                with torch.no_grad():
                    s2 = float(m.score(D2.query_batch([q], m.use_pred), torch.tensor([[v]], device=DEVICE), D2)[0, 0])
                expect = sum(t["contrib"] for t in rel if e in t["edges"])
                del_diff = max(del_diff, abs((s - s2) - expect)); n_del += 1
        out[f"T{T}"] = dict(queries=len(qs), relation_terms=n_rel, max_abs_diff=maxdiff, deletions=n_del, deletion_max_abs_diff=del_diff)
    return out


def test_v4_predicate_and_query():
    """(1) Đổi vị từ của truy vấn ⇒ attention trên cạnh mang vị từ đó đổi; use_pred = False ⇒ KHÔNG đổi (đối chứng âm).
    (2) Hai truy vấn khác nhau ⇒ attention khác (qcond); qcond = False ⇒ giống hệt. (3) Hoán vị vị từ (G4) đổi tín hiệu khớp."""
    out = {}
    for use_pred in (True, False):
        g, eng, m = _v4_model_for_test(dict(V3_FINAL, model="v4", v4_layers=1, v4_pred=use_pred))
        qs = _queries_with_pred_edges(m.D, 10)
        diffs = []
        for q in qs:
            qb = m.D.query_batch([q], True)
            qb2 = dict(qb, ex=torch.zeros_like(qb["ex"]), soft=torch.zeros_like(qb["soft"]))
            with torch.no_grad():
                u = m.q(qb["tq"])
                diffs.append(float((m.attention(qb, u) - m.attention(qb2, u)).abs().max()))
        out[f"pred_change_use_pred={use_pred}"] = max(diffs) if diffs else float("nan")
    for qcond in (True, False):
        g, eng, m = _v4_model_for_test(dict(V3_FINAL, model="v4", v4_layers=1, v4_qcond=qcond))
        qs = _queries_with_pred_edges(m.D, 2)
        with torch.no_grad():
            qb = m.D.query_batch(qs, True)
            a = m.attention(qb, m.q(qb["tq"]))
        out[f"two_queries_attention_diff_qcond={qcond}"] = float((a[0] - a[1]).abs().max())
    g0 = graph_for(dict(V3_FINAL, model="v4"), CFG.seeds[0])[0]
    gs = graph_for(dict(V3_FINAL, model="v4", pred_shuffle=True), CFG.seeds[0])[0]
    D0, Ds = RelDataV4(g0), RelDataV4(gs)
    qs = _queries_with_pred_edges(D0, 50)
    out["pred_shuffle_changes_match"] = float((D0.query_batch(qs)["ex"] - Ds.query_batch(qs)["ex"]).abs().sum())
    out["pred_shuffle_same_topology"] = bool(np.array_equal(D0.src_np, Ds.src_np) and np.array_equal(D0.dst_np, Ds.dst_np))
    return out


def test_v4_inductive_and_masks():
    """Video chỉ NHẬN (node lớp không phụ thuộc video) ⇒ đổi đặc trưng / nhãn của MỘT video test (hoặc tập ngoài) không đổi điểm
    của video khác; rel = False ⇒ không có số hạng quan hệ; điểm hữu hạn, gradient hữu hạn.
    So với dung sai 1e-6 (như G5 của v3): trên GPU, index_add_ cộng dồn bằng atomic nên hai lần gọi cùng đầu vào có thể lệch ~1e-8."""
    g, eng, m = _v4_model_for_test(dict(V3_FINAL, model="v4", v4_layers=2))
    out = {}
    q = _queries_with_pred_edges(m.D, 1)[0]
    qb = m.D.query_batch([q])
    for s in ("test", "ext"):
        tv = [int(v) for v in SPLIT.get(s, [])]
        if len(tv) < 2:
            continue
        v0, v1 = tv[0], tv[1]
        cand = torch.tensor([[v0, v1]], device=DEVICE)
        with torch.no_grad():
            s1 = m.score(qb, cand)
            s1b = m.score(qb, cand)                          # cùng đầu vào ⇒ đo sai số số học khi chạy lại (GPU không tất định)
            old_x, old_a = VID_T[v0].clone(), m.D.Acon[v0].clone()
            VID_T[v0] = torch.randn_like(old_x); m.D.Acon[v0] = torch.roll(old_a, 7)
            s2 = m.score(qb, cand)
            VID_T[v0] = old_x; m.D.Acon[v0] = old_a
        out[s] = dict(other_video_changed=float((s1[0, 1] - s2[0, 1]).abs()), own_video_changed=float((s1[0, 0] - s2[0, 0]).abs()),
                      rerun_noise=float((s1 - s1b).abs().max()))
    g1, e1, m1 = _v4_model_for_test(dict(V3_FINAL, model="v4", v4_rel=False))
    v = int(SPLIT["train"][0])
    out["no_relation_terms_when_rel_false"] = not any(t["kind"] == "relation" for t in m1.decompose(q, v))
    m.train()
    qb = m.D.query_batch(TRAIN_Q[:16])
    loss = m.score(qb, torch.from_numpy(np.tile(CAP_VID[TRAIN_Q[:16]], (16, 1))).to(DEVICE)).logsumexp(1).mean()
    loss.backward()
    out["grad_finite"] = bool(all(torch.isfinite(p.grad).all() for p in m.parameters() if p.grad is not None))
    out["score_finite"] = bool(torch.isfinite(loss).item())
    m.eval()
    return out


## 4.15 Level 4(b): bộ truy vấn đăng ký trước, xAI theo thứ hạng, phân tích ca, demo, kiểm kê

Các hàm dưới đây chỉ là **định nghĩa**, được gọi ở phần THỰC THI. Mọi file đầu ra sinh từ chính lần chạy; không có số liệu điền tay.

In [86]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b): công cụ bổ sung (chỉ ĐỊNH NGHĨA — chạy ở phần THỰC THI) ==============================
import copy as _copy

# Nguồn / giấy phép: dataset card HF (kiểm qua API ngày 2026-10-25) KHÔNG khai báo license ⇒ ghi đúng như vậy, không tự điền.
DATA_SOURCES = {
    "msvd": dict(name="MSVD (Microsoft Research Video Description Corpus)", paper="Chen & Dolan, ACL 2011",
                 repo=CFG.hf_repo, revision=CFG.hf_revision,
                 license="KHÔNG XÁC NHẬN ĐƯỢC — dataset card HF không khai báo license; dữ liệu gốc (clip YouTube) phát hành cho nghiên cứu",
                 role="train: từ vựng, quan hệ, huấn luyện GNN | val: chọn cấu hình | test: THĂM DÒ"),
    "msrvtt": dict(name="MSR-VTT", paper="Xu et al., CVPR 2016", repo=CFG.msrvtt_repo, revision=CFG.msrvtt_revision,
                   license="KHÔNG XÁC NHẬN ĐƯỢC — dataset card HF không khai báo license; dữ liệu gốc của Microsoft Research cho nghiên cứu",
                   role="chỉ đánh giá (chuyển giao zero-shot): không caption nào dùng để học hay chọn"),
}
# Số lần mỗi benchmark đã được nhìn TRƯỚC notebook này (lịch sử v1–v4, xem README). Đã nhìn ⇒ kết quả chỉ là THĂM DÒ.
BENCHMARK_LOOKS_BEFORE = {"msvd_test": 4, "msrvtt_1kA": 1, "msrvtt_fresh1k": 1, "msrvtt_fresh1kB": 0}
_PROV_RULES["evaluation"] = {"val", "test", "ext"}   # bộ truy vấn đánh giá: dựng từ văn bản truy vấn của tập đánh giá, không học / chọn gì
FINDINGS = []          # kết quả KHOA HỌC được báo cáo (không phải kiểm thử phần mềm): id, câu hỏi, kết quả, diễn giải


def finding(fid, question, result, interpretation):
    FINDINGS.append(dict(id=fid, question=question, result=str(result)[:900], interpretation=interpretation))
    print(f"  [FINDING] {fid} — {question}: {result}\n            ⇒ {interpretation}")


def benchmark_status():
    """Trạng thái từng tập đánh giá của lần chạy này: CONFIRMATORY chỉ khi tập CHƯA từng được nhìn."""
    ext_key = {"fresh1kB": "msrvtt_fresh1kB", "fresh1k": "msrvtt_fresh1k", "1kA": "msrvtt_1kA"}.get(EXT_REPORT.get("set"))
    out = {"msvd_test": dict(looks_before=BENCHMARK_LOOKS_BEFORE["msvd_test"], evaluated=not CFG.dev_mode,
                             status="EXPLORATORY" if not CFG.dev_mode else "NOT RUN (dev_mode: test khoá)")}
    if EXT_RECORDS and not EXT_REPORT.get("dev"):
        lb = BENCHMARK_LOOKS_BEFORE[ext_key]
        out[ext_key] = dict(looks_before=lb, evaluated=True, status="CONFIRMATORY" if lb == 0 else "EXPLORATORY",
                            label=EXT_REPORT.get("label"))
    elif EXT_RECORDS:
        out["msrvtt_dev"] = dict(looks_before=None, evaluated=True, status="DEV (thử đường ống)", label=EXT_REPORT.get("label"))
    else:
        out["msrvtt"] = dict(evaluated=False, status=EXT_STATUS)
    return out


def node_name(i):
    return INV.name[int(i)]


def edges_into_tags(g, src, tags):
    """Cạnh quan hệ src –t→ c với c ∈ tags (đúng chiều QC-RGAT truyền tin: nguồn → nhãn của video)."""
    out = []
    for t in RELATION_TYPES:
        A = g.A[t]
        if A.nnz == 0:
            continue
        row = A.getrow(int(src))
        for c in row.indices:
            if int(c) in tags:
                base = t[:-4] if t.endswith("_inv") else t
                key = (base, int(c), int(src)) if t.endswith("_inv") else (base, int(src), int(c))
                pd_ = g.pdist.get(key, {}) if base in PRED_FAMILIES else {}
                p = max(pd_, key=pd_.get) if pd_ else ""
                out.append((int(src), t, int(c), p))
    return out


def edge_str(e):
    s_, t, d_, p = e
    return f"{node_name(s_)} –{t}{'[' + p + ']' if p else ''}→ {node_name(d_)}"


def build_eval_queries(part, g, n_per_group=None, seed=None):
    """Bộ truy vấn ĐĂNG KÝ TRƯỚC (mục 7.3). Chỉ dùng: văn bản truy vấn (mục đích query_input), nhãn CLIP zero-shot của video đúng,
    cạnh đồ thị dựng từ caption TRAIN. KHÔNG dùng điểm hay thứ hạng của bất kỳ mô hình nào.
    trực tiếp : mọi khái niệm khớp được của truy vấn đều là nhãn của video đúng.
    gián tiếp : truy vấn nêu ≥ 1 quan hệ (bộ ba), nhãn video phủ MỘT PHẦN khái niệm truy vấn, và có cạnh quan hệ nối một khái
                niệm còn thiếu vào một nhãn của video (= quan hệ dự kiến phải khai thác)."""
    n_per_group = n_per_group or CFG.n_eval_queries
    seed = CFG.eval_query_seed if seed is None else seed
    q_all, gold_all = part["q_idx"], part["gold"]
    texts = CAPS.texts(q_all, "query_input")
    pools = {"direct": [], "indirect": []}
    for j, q in enumerate(q_all):
        v, hits = int(gold_all[j]), set(V4_HITS[int(q)])
        tags = set(g.vid_tags[v])
        if not hits:
            continue
        seen, miss = hits & tags, hits - tags
        base = dict(q_idx=int(q), text=texts[j], gold_vid=v, gold_video_id=RECORDS[v]["video_id"],
                    query_concepts="; ".join(sorted(node_name(h) for h in hits)),
                    concepts_seen_in_video="; ".join(sorted(node_name(h) for h in seen)),
                    concepts_missing="; ".join(sorted(node_name(h) for h in miss)))
        if not miss:
            pools["direct"].append(dict(base, type="direct", expected_relations="",
                                        reason=f"mọi khái niệm của truy vấn ({base['query_concepts']}) là nhãn CLIP zero-shot của video đúng"))
            continue
        trip = [t for t in parse_caption_v2(texts[j])[2] if INV.key2id.get(t[0]) is not None and INV.key2id.get(t[2]) is not None]
        if not trip or not seen:
            continue
        bridge = [e for m_ in sorted(miss) for e in edges_into_tags(g, m_, tags)]
        if not bridge:
            continue
        rel_txt = "; ".join(f"{s_.split('#')[0]} –{r_}{'[' + p_ + ']' if p_ else ''}→ {o_.split('#')[0]}" for s_, r_, o_, p_ in sorted(trip))
        pools["indirect"].append(dict(base, type="indirect", expected_relations=" || ".join(edge_str(e) for e in bridge[:5]),
                                      reason=(f"truy vấn nêu quan hệ ({rel_txt}); video chỉ có nhãn {base['concepts_seen_in_video']}, "
                                              f"thiếu {base['concepts_missing']}; {len(bridge)} cạnh quan hệ nối khái niệm thiếu vào nhãn video")))
    rows, info = [], {}
    for k, (grp, pool) in enumerate(pools.items()):
        order = np.random.RandomState(seed + k).permutation(len(pool))
        used, take = set(), []
        for i in order:
            if pool[i]["gold_vid"] in used:
                continue
            used.add(pool[i]["gold_vid"]); take.append(pool[i])
            if len(take) >= n_per_group:
                break
        for n_, r in enumerate(take, 1):
            rows.append(dict(query_id=f"{part['name']}-{grp[0].upper()}{n_:02d}", part=part["name"], **r))
        info[grp] = dict(eligible=len(pool), eligible_videos=len({r["gold_vid"] for r in pool}), selected=len(take))
    return pd.DataFrame(rows), info


def eval_query_metrics(E, QS):
    """R@1/5/10, MRR (trung bình ± std theo seed) của MỌI thí nghiệm trên bộ truy vấn đăng ký trước, theo nhóm."""
    part = E["part"]
    pos = {int(q): i for i, q in enumerate(part["q_idx"])}
    rows, pairs = [], []
    for grp in ("all", "direct", "indirect"):
        sel = QS if grp == "all" else QS[QS.type == grp]
        if not len(sel):
            continue
        idx = np.array([pos[int(q)] for q in sel.q_idx])
        gold = part["gold"][idx]
        for k, ev_k in E["ev"].items():
            ms = [metrics_from_ranks(v["rank"][idx], gold) for v in ev_k.values()]
            rows.append(dict(part=part["name"], kind=E["info"]["kind"], experiment_id=k, group=grp, n_queries=len(idx),
                             n_videos=len(set(gold.tolist())), n_seeds=len(ms),
                             **{f"{x}_mean": float(np.mean([mm[x] for mm in ms])) for x in ("R@1", "R@5", "R@10", "MRR")},
                             **{f"{x}_std": float(np.std([mm[x] for mm in ms])) for x in ("R@1", "R@5", "R@10", "MRR")}))
        for a in (G_IDS.get("G3"), G_IDS.get("V3REF")):                 # QC-RGAT và GraphSAGE có kiểu cạnh, mỗi cái vs CLIP-only
            if a in E["ev"]:
                rr = lambda k_: np.mean([1.0 / v["rank"][idx] for v in E["ev"][k_].values()], 0)
                bs = cluster_bootstrap_diff(rr(a), rr("G0_clip_only"), gold)
                pairs.append(dict(part=part["name"], kind=E["info"]["kind"], group=grp, a=a, b="G0_clip_only", n=len(idx),
                                  MRR_a=float(rr(a).mean()), MRR_b=float(rr("G0_clip_only").mean()), **bs))
    return pd.DataFrame(rows), pd.DataFrame(pairs)


def drop_graph_parts(D, vid, cls_drop=(), rel_drop=()):
    """Bản sao dữ liệu đồ thị với (1) cạnh contains video –chứa→ c (c ∈ cls_drop) của RIÊNG video `vid` bị xoá (chuẩn hoá lại hàng
    như lúc dựng), (2) cạnh quan hệ (chỉ số trong D) bị xoá. Mô hình và tham số giữ nguyên."""
    D2 = RelDataV4(D.graph, drop_edges=D.gidx[list(rel_drop)]) if len(rel_drop) else _copy.copy(D)
    if len(cls_drop):
        A = D2.Acon.clone()
        row = (A[vid] > 0).float()
        row[list(cls_drop)] = 0.0
        A[vid] = row / row.sum().clamp(min=1.0)
        D2 = _copy.copy(D2)
        D2.Acon = A
    return D2


def term_text(D, t, vid):
    if t["kind"] == "contains":
        return f"{RECORDS[vid]['video_id']} –chứa→ {node_name(D.N + t['node'])}"
    return v4_path_text(D, t, vid)


def term_valid(D, t, vid, q):
    """Cạnh của số hạng có thật trong đồ thị đang chạy, đúng loại, đúng chiều, đúng kiểu node; vị từ khớp (nếu có) nằm trên ĐÚNG cạnh."""
    if t["kind"] == "contains":
        c = D.N + t["node"]
        ok = D.graph.A_con[vid, c] > 0 and INV.node_type[c] != "video" and INV.node_type[vid] == "video"
        return bool(ok), True
    v = [v4_edge_valid(D, k, q) for k in t["edges"]]
    return all(x[0] for x in v), all(x[2] for x in v)


@torch.no_grad()
def explain_full(model, q, vid, top_m=None):
    """Giải thích ĐẦY ĐỦ "vì sao video được xếp hạng như vậy" = top-m số hạng ĐỒ THỊ ủng hộ (contains + đường đi quan hệ).
    Số hạng thị giác (W_v x_v) không phải đồ thị nên được báo riêng. Σ mọi số hạng = điểm đồ thị của mô hình."""
    top_m = top_m or CFG.top_m_paths
    terms = model.decompose(q, vid)
    graph_terms = [t for t in terms if t["kind"] != "visual"]
    sup = sorted([t for t in graph_terms if t["contrib"] > 0], key=lambda t: -t["contrib"])[:top_m]
    opp = sorted([t for t in graph_terms if t["contrib"] < 0], key=lambda t: t["contrib"])[:top_m]
    tot = {k: sum(t["contrib"] for t in terms if t["kind"] == k) for k in ("visual", "contains", "relation")}
    return dict(terms=terms, support=sup, oppose=opp, totals=tot, score=sum(t["contrib"] for t in terms),
                n_graph_terms=len(graph_terms), pos_graph=sum(t["contrib"] for t in graph_terms if t["contrib"] > 0))


def _deletion_sets(sup):
    return (sorted({t["node"] for t in sup if t["kind"] == "contains"}),
            sorted({k for t in sup if t["kind"] == "relation" for k in t["edges"]}))


@torch.no_grad()
def xai_rank_fidelity(run, part, n_queries=None, top_m=None, seed=0):
    """Fidelity theo QUYẾT ĐỊNH xếp hạng (mục 8.3): với ≥ n truy vấn có video đúng trong top-K ứng viên, xoá THẬT các cạnh của
    giải thích đầy đủ (contains của video + cạnh quan hệ) rồi chấm lại TOÀN BỘ ứng viên ⇒ Δ điểm đồ thị, Δ điểm cuối, Δ hạng,
    đổi top-1 / top-10; đối chứng = xoá cùng số cạnh contains khác của video + cùng số cạnh quan hệ ngẫu nhiên vào nhãn của video.
    Báo riêng: hạng không đổi thì ghi đúng là không đổi."""
    n_queries, top_m = n_queries or CFG.n_explain_queries, top_m or CFG.top_m_paths
    m, eng = run["model"], run["engine"]
    D = m.D
    rng = np.random.RandomState(seed)
    q_all = part["q_idx"]
    cand_all, s_clip = eng.clip_candidates(q_all, part["gallery"], CFG.rerank_k)
    rows = []
    for i in rng.permutation(len(q_all)):
        if len(rows) >= n_queries:
            break
        q, gold = int(q_all[i]), int(part["gold"][i])
        cand = cand_all[i]
        if gold not in cand:
            continue
        ex = explain_full(m, q, gold, top_m)
        if not ex["support"]:
            continue
        zc = znorm_per_row(s_clip[i:i + 1])[0]
        r_full, s_full, fin_full = v4_rank_after(run, q, gold, cand, zc, D)
        j = int(np.where(cand == gold)[0][0])
        con_del, rel_del = _deletion_sets(ex["support"])
        r_ex, s_ex, fin_ex = v4_rank_after(run, q, gold, cand, zc, drop_graph_parts(D, gold, con_del, rel_del))
        tags = torch.nonzero(D.Acon[gold] > 0).flatten().tolist()
        other = [c for c in tags if c not in con_del]
        rc = [int(x) for x in rng.choice(other, min(len(con_del), len(other)), replace=False)] if (other and con_del) else []
        pool = [k for k in range(D.E) if int(D.dst_np[k]) in set(tags) and k not in rel_del]
        rr = [int(x) for x in rng.choice(pool, min(len(rel_del), len(pool)), replace=False)] if (pool and rel_del) else []
        r_rd, s_rd, fin_rd = v4_rank_after(run, q, gold, cand, zc, drop_graph_parts(D, gold, rc, rr))
        val = [term_valid(D, t, gold, q) for t in ex["support"]]
        expl_edges = len(con_del) + len(rel_del)
        top10 = lambda f: set(np.argsort(-f)[:10].tolist())
        rows.append(dict(q=q, gold=gold, gold_video_id=RECORDS[gold]["video_id"], n_graph_terms=ex["n_graph_terms"],
                         n_expl_terms=len(ex["support"]), n_expl_contains=sum(t["kind"] == "contains" for t in ex["support"]),
                         n_expl_relation=sum(t["kind"] == "relation" for t in ex["support"]), n_expl_edges=expl_edges,
                         n_rand_edges=len(rc) + len(rr), sparsity=len(ex["support"]) / max(1, ex["n_graph_terms"]),
                         coverage=sum(t["contrib"] for t in ex["support"]) / max(1e-9, ex["pos_graph"]),
                         validity=float(np.mean([v[0] for v in val])), predicate_validity=float(np.mean([v[1] for v in val])),
                         decomposition_ok=float(abs(ex["score"] - s_full) < 1e-4),
                         s_graph=float(s_full), d_s_expl=float(s_ex - s_full), d_s_rand=float(s_rd - s_full),
                         d_final_expl=float(fin_ex[j] - fin_full[j]), d_final_rand=float(fin_rd[j] - fin_full[j]),
                         rank_full=r_full, rank_drop_expl=r_ex, rank_drop_rand=r_rd,
                         rank_worse_expl=float(r_ex > r_full), rank_worse_rand=float(r_rd > r_full),
                         top1_changed_expl=float(np.argmax(fin_ex) != np.argmax(fin_full)),
                         top1_changed_rand=float(np.argmax(fin_rd) != np.argmax(fin_full)),
                         top10_changed_expl=float(top10(fin_ex) != top10(fin_full)), top10_changed_rand=float(top10(fin_rd) != top10(fin_full)),
                         share_graph=(ex["totals"]["contains"] + ex["totals"]["relation"]) / max(1e-9, sum(abs(x) for x in ex["totals"].values())),
                         share_relation=ex["totals"]["relation"] / max(1e-9, sum(abs(x) for x in ex["totals"].values())),
                         explanation=" || ".join(f"{t['contrib']:+.4f} {term_text(D, t, gold)}" for t in ex["support"]),
                         opposing=" || ".join(f"{t['contrib']:+.4f} {term_text(D, t, gold)}" for t in ex["oppose"][:3])))
    return pd.DataFrame(rows)


def diagnose_case(run, q, gold, cand, rank, clip_rank, text):
    """Nguyên nhân (luật cố định): candidate_recall_low | tag_missing | missing_edge | predicate_not_in_graph | wrong_predicate |
    noisy_relation | distractor_shares_concepts | graph_helped | clip_already_correct."""
    g, D = run["graph"], run["model"].D
    hits, tags = set(V4_HITS[q]), set(g.vid_tags[gold])
    causes, notes = [], []
    if gold not in cand:
        causes.append("candidate_recall_low"); notes.append(f"video đúng ở hạng CLIP {clip_rank} > k = {CFG.rerank_k}")
    if hits and not hits & tags:
        causes.append("tag_missing"); notes.append("không khái niệm nào của truy vấn là nhãn của video đúng")
    for s_, r_, o_, p_ in sorted(parse_caption_v2(text)[2]):
        a, b = INV.key2id.get(s_), INV.key2id.get(o_)
        if a is None or b is None or a == b:
            continue
        on = [t for t in RELATION_TYPES if g.A[t][a, b] != 0 or g.A[t][b, a] != 0]
        if not on:
            causes.append("missing_edge"); notes.append(f"không có cạnh nào giữa {s_} và {o_} (quan hệ '{r_}' của truy vấn)")
        elif r_ in PRED_FAMILIES and p_:
            if (r_, p_) not in PRED_ID:
                causes.append("predicate_not_in_graph"); notes.append(f"vị từ '{p_}' không có trong từ vựng vị từ (train)")
            else:
                pd_ = g.pdist.get((r_, a, b), {}) or g.pdist.get((r_, b, a), {})
                if pd_ and p_ not in pd_:
                    causes.append("wrong_predicate")
                    notes.append(f"cạnh {s_}–{r_}–{o_} mang vị từ {sorted(pd_)[:3]}, không có '{p_}' của truy vấn")
    if gold in cand:
        ex = explain_full(run["model"], q, gold)
        if ex["totals"]["relation"] < 0 and rank > clip_rank:
            causes.append("noisy_relation"); notes.append(f"tổng đóng góp quan hệ cho video đúng âm ({ex['totals']['relation']:+.4f})")
        if rank < clip_rank:
            causes.append("graph_helped"); notes.append(f"đồ thị kéo video đúng từ hạng {clip_rank} lên {rank}")
    if clip_rank == 1 and rank == 1:
        causes.append("clip_already_correct")
    return list(dict.fromkeys(causes)), notes


FAIL_CAUSES = ("candidate_recall_low", "missing_edge", "wrong_predicate", "noisy_relation", "tag_missing", "predicate_not_in_graph")


@torch.no_grad()
def case_analysis_l4b(run, part, n_each=None, seed=0, max_q=4000):
    """5 thành công (hạng 1) + 5 thất bại (hạng > 10), chọn theo LUẬT: thành công ưu tiên ca đồ thị kéo hạng lên rồi tới ca CLIP đã đúng;
    thất bại lấy mỗi nguyên nhân trong FAIL_CAUSES một ca (nếu có) rồi bổ sung ngẫu nhiên — không chọn ví dụ đẹp."""
    n_each = n_each or CFG.n_cases
    m, eng = run["model"], run["engine"]
    rng = np.random.RandomState(seed)
    q = part["q_idx"]
    sel = np.sort(rng.choice(len(q), min(max_q, len(q)), replace=False))
    q, gold = q[sel], part["gold"][sel]
    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, q, part["gallery"], gold)
    sc = score_queries_v4(m, eng, q, part["gallery"], CFG.rerank_k)
    fin = combine(sc["s_clip"], sc["s_gnn"], sc["s_path"], run["alpha"], 1.0)
    rank = ranks_with_fallback(sc["cand"], fin, gold, clip_rank)
    texts = CAPS.texts(q, "report")
    order = rng.permutation(len(q))
    succ = [i for i in order if rank[i] == 1 and clip_rank[i] > 1][:3]
    succ += [i for i in order if rank[i] == 1 and i not in succ][:n_each - len(succ)]
    fail_pool = [i for i in order if rank[i] > 10]
    diag = {}
    for i in fail_pool[:400]:
        diag[i] = diagnose_case(run, int(q[i]), int(gold[i]), sc["cand"][i], int(rank[i]), int(clip_rank[i]), texts[i])
    fail = []
    for c in FAIL_CAUSES:
        hit = next((i for i in fail_pool[:400] if c in diag[i][0] and i not in fail), None)
        if hit is not None and len(fail) < n_each:
            fail.append(hit)
    fail += [i for i in fail_pool if i not in fail][:n_each - len(fail)]
    cases = []
    for kind, idx in (("THÀNH CÔNG", succ), ("THẤT BẠI", fail)):
        for i in idx:
            qi, gi = int(q[i]), int(gold[i])
            causes, notes = diag.get(i) or diagnose_case(run, qi, gi, sc["cand"][i], int(rank[i]), int(clip_rank[i]), texts[i])
            top = sc["cand"][i][np.argsort(-fin[i])[:3]].tolist()
            ex = explain_full(m, qi, gi) if gi in sc["cand"][i] else None
            ex_top1 = explain_full(m, qi, top[0]) if top and top[0] != gi else None
            cases.append(dict(kind=kind, q=qi, query=texts[i], gold_vid=gi, gold_video_id=RECORDS[gi]["video_id"],
                              rank_clip=int(clip_rank[i]), rank_final=int(rank[i]), in_candidates=bool(gi in sc["cand"][i]),
                              top3=[RECORDS[v]["video_id"] for v in top], causes=causes, notes=notes,
                              gold_totals={k_: round(v_, 4) for k_, v_ in ex["totals"].items()} if ex else None,
                              gold_explanation=[f"{t['contrib']:+.4f} {term_text(m.D, t, gi)}" for t in ex["support"]] if ex else [],
                              gold_opposing=[f"{t['contrib']:+.4f} {term_text(m.D, t, gi)}" for t in ex["oppose"][:3]] if ex else [],
                              top1_explanation=[f"{t['contrib']:+.4f} {term_text(m.D, t, top[0])}" for t in ex_top1["support"]] if ex_top1 else []))
    stats = dict(n_queries=len(q), top1=float(np.mean(rank == 1)), fail_gt10=float(np.mean(rank > 10)),
                 cause_counts_in_fail_sample=dict(Counter(c for i in diag for c in diag[i][0])), n_fail_diagnosed=len(diag))
    return cases, stats


def cases_markdown(cases, stats, part_name):
    L = [f"# Phân tích 5 thành công + 5 thất bại — {part_name}", "",
         "Chọn theo luật cố định (xem `case_analysis_l4b`), KHÔNG chọn tay. Thành công = video đúng ở hạng 1; thất bại = hạng > 10.",
         f"Thống kê trên {stats['n_queries']} truy vấn mẫu: hạng 1 = {stats['top1']:.1%}, hạng > 10 = {stats['fail_gt10']:.1%}; "
         f"nguyên nhân trong mẫu thất bại đã chẩn đoán ({stats['n_fail_diagnosed']}): {stats['cause_counts_in_fail_sample']}", ""]
    for n_, c in enumerate(cases, 1):
        L += [f"## {n_}. [{c['kind']}] \"{c['query']}\"", "",
              f"- Video đúng: `{c['gold_video_id']}` — hạng CLIP **{c['rank_clip']}** → hạng cuối **{c['rank_final']}**"
              f"{'' if c['in_candidates'] else ' (KHÔNG nằm trong top-k ứng viên CLIP ⇒ đồ thị không thể cứu)'}",
              f"- Top-3 trả về: {', '.join('`' + v + '`' for v in c['top3'])}",
              f"- Chẩn đoán (luật cố định{'; nguyên nhân thất bại' if c['kind'] == 'THẤT BẠI' else ''}): {', '.join(c['causes']) or '—'}"] + \
            [f"  - {x}" for x in c["notes"]]
        if c["gold_totals"]:
            L.append(f"- Phân rã điểm đồ thị của video đúng: {c['gold_totals']}")
        L += ["- Đường đi / quan hệ ỦNG HỘ video đúng (đóng góp thật vào điểm):"] + [f"  - {x}" for x in c["gold_explanation"] or ["(không có số hạng đồ thị dương)"]]
        if c["gold_opposing"]:
            L += ["- Số hạng PHẢN ĐỐI:"] + [f"  - {x}" for x in c["gold_opposing"]]
        if c["top1_explanation"]:
            L += [f"- Vì sao `{c['top3'][0]}` (hạng 1, sai) được xếp trên:"] + [f"  - {x}" for x in c["top1_explanation"]]
        L.append("")
    return "\n".join(L)


# ---------------------------------- Demo: văn bản tự do → top-K video + giải thích ----------------------------------
def thumb_path(vid):
    """Thumbnail khung giữa: dùng file đã lưu lúc trích đặc trưng; nếu không có (dùng cache) thì đọc lại từ video."""
    r = RECORDS[vid]
    p = CFG.out_dir / "thumbs" / f"{r['video_id']}.jpg"
    if p.exists():
        return p
    try:
        src = SOURCE if vid < N_MSVD else EXT_SOURCE
        with src.open(r["video_id"]) as path:
            frames, _ = read_sampled_frames(path, 3)
        mid = frames[1]
        h, w = mid.shape[:2]
        th = cv2.resize(mid, (CFG.thumb_width, max(1, int(h * CFG.thumb_width / w))), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(p), cv2.cvtColor(th, cv2.COLOR_RGB2BGR))
        return p
    except Exception:
        return None


@torch.no_grad()
def demo_query(text, run=None, part=None, k=None, show=True, save_png=None):
    """Nhập văn bản tự do ⇒ top-k video của gallery `part` (mặc định tập xác nhận, nếu không thì tập thăm dò) với điểm CLIP, điểm đồ thị,
    điểm cuối (α đã đóng băng) và giải thích = số hạng đồ thị ủng hộ. Văn bản demo KHÔNG đi vào CaptionStore và không được lưu làm dữ liệu."""
    global TXT_FEAT
    run = run or EXPS[G_IDS["G3"]]["runs"][CFG.seeds[0]]
    part = part or (EVALS["confirm"]["part"] if "confirm" in EVALS else EVALS["explore"]["part"])
    k = k or CFG.demo_topk
    m = run["model"]
    D = m.D
    t = clip_encode_texts([text])[0].astype(TXT_FEAT.dtype)
    q = len(TXT_FEAT)
    toks = query_tokens_v2(text)
    saved = (TXT_FEAT,)
    TXT_FEAT = np.vstack([TXT_FEAT, t[None]])
    V4_HITS.append(sorted({nid for nid, _, _ in parse_tokens_v2(toks, INV)}))
    V4_ROLES.append(query_roles_v4([text], INV)[0])
    QPREDS.append({rel: match_predicates(query_pred_tokens(text), rel) for rel in PRED_FAMILIES})
    try:
        gal = np.asarray(part["gallery"])
        cos = VID_FEAT[gal] @ t
        top = np.argsort(-cos)[:CFG.rerank_k]
        cand, s_clip = gal[top], cos[top]
        qb = D.query_batch([q], m.use_pred)
        s = m.score(qb, torch.from_numpy(cand[None]).to(DEVICE)).cpu().numpy()[0]
        fin = run["alpha"] * znorm_per_row(s_clip[None])[0] + (1 - run["alpha"]) * znorm_per_row(s[None])[0]
        order = np.argsort(-fin)[:k]
        out = dict(query=text, gallery=part["name"], alpha=run["alpha"], matched_concepts=[node_name(h) for h in V4_HITS[q]],
                   results=[])
        for r_, j in enumerate(order, 1):
            v = int(cand[j])
            ex = explain_full(m, q, v)
            out["results"].append(dict(rank=r_, video_id=RECORDS[v]["video_id"], clip_cos=float(s_clip[j]), graph_score=float(s[j]),
                                       final=float(fin[j]), clip_rank=int(j) + 1,          # cand đã xếp theo cosine giảm dần
                                       totals={k_: round(v_, 4) for k_, v_ in ex["totals"].items()},
                                       explanation=[f"{tt['contrib']:+.4f} {term_text(D, tt, v)}" for tt in ex["support"]],
                                       note="" if ex["support"] else "KHÔNG có đường đi giải thích hợp lệ (không số hạng đồ thị nào ủng hộ)"))
    finally:
        TXT_FEAT = saved[0]
        V4_HITS.pop(); V4_ROLES.pop(); QPREDS.pop()
    if show:
        print(f"\nTruy vấn: '{text}' | gallery {out['gallery']} ({len(gal)} video) | khái niệm khớp: {out['matched_concepts'] or '—'} | α = {out['alpha']}")
        for r in out["results"]:
            print(f"  #{r['rank']} {r['video_id']:>12s} | CLIP cos {r['clip_cos']:.3f} (hạng CLIP {r['clip_rank']}) | đồ thị {r['graph_score']:+.3f} "
                  f"| cuối {r['final']:+.3f} | phân rã {r['totals']}")
            for x in r["explanation"]:
                print(f"       • {x}")
            if r["note"]:
                print(f"       ⚠ {r['note']}")
        try:
            import matplotlib.pyplot as plt
            fig, axes = plt.subplots(1, len(out["results"]), figsize=(3 * len(out["results"]), 2.6))
            for ax, r in zip(np.atleast_1d(axes), out["results"]):
                v = next(x["vid"] for x in RECORDS if x["video_id"] == r["video_id"])
                p = thumb_path(v)
                if p is not None:
                    ax.imshow(plt.imread(str(p)))
                ax.set_title(f"#{r['rank']} {r['video_id']}\nCLIP {r['clip_cos']:.2f} | cuối {r['final']:+.2f}", fontsize=8)
                ax.axis("off")
            fig.suptitle(text, fontsize=9)
            fig.tight_layout()
            if save_png:
                fig.savefig(save_png, dpi=110)
            plt.show()
            plt.close(fig)
        except Exception as e:
            print(f"  (không vẽ được thumbnail: {type(e).__name__}: {e})")
    return out


def demo_widget():
    """Ô nhập truy vấn tương tác (Jupyter / Colab / Kaggle interactive). Không tự chạy khi Save & Run All."""
    import ipywidgets as w
    box, btn, out = w.Text(placeholder="ví dụ: a man is riding a horse", layout=w.Layout(width="60%")), w.Button(description="Tìm"), w.Output()

    def go(_):
        with out:
            out.clear_output()
            demo_query(box.value)
    btn.on_click(go)
    display(w.HBox([box, btn]), out)


# ---------------------------------- Kiểm kê, audit, kết luận, manifest ----------------------------------
def file_sha1(p, n=1 << 20):
    h = hashlib.sha1()
    with open(p, "rb") as f:
        while True:
            b = f.read(n)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def split_counts():
    out = {}
    for s in sorted(set(CAP_SPLIT.tolist())):
        vids = sorted(set(CAP_VID[CAP_SPLIT == s].tolist()))
        out[s] = dict(videos=len(vids), captions=int((CAP_SPLIT == s).sum()),
                      video_ids_sha1=sha1_of([RECORDS[v]["video_id"] for v in vids]))
    return out


def graph_inventory(g):
    s, d, e, w, pi = g.edge_arrays()
    et = Counter(EDGE_TYPES[int(x)] for x in e)
    nt = Counter(INV.node_type)
    deg = np.bincount(np.concatenate([s, d]), minlength=INV.n_nodes) if len(s) else np.zeros(INV.n_nodes, int)
    nonvid = np.array([t != "video" for t in INV.node_type])
    return dict(nodes_by_type=dict(nt), edges_by_type=dict(et), n_edges=int(len(e)), self_loops=int((s == d).sum()),
                duplicate_edges=int(len(e) - len(set(zip(s.tolist(), d.tolist(), e.tolist())))),
                isolated_non_video_nodes=int(((deg == 0) & nonvid).sum()), videos_without_tag=int(sum(len(x) == 0 for x in g.vid_tags)),
                degree_non_video=dict(mean=float(deg[nonvid].mean()), median=float(np.median(deg[nonvid])), max=int(deg[nonvid].max())),
                edges_with_predicate=int((pi >= 0).sum()), schema_sha1=SCHEMA["sha1"], inventory_sha1=INV.sha1)

---
## 4.16 Thực thi

Thứ tự chạy:
1. Dữ liệu: MSVD và MSR-VTT → split → đặc trưng → pooling.
2. Từ vựng, vị từ, tagger, mẫu cho người chấm và nhãn.
3. Đồ thị v3, schema, audit → self-test.
4. Các stage V0–V7 trên VAL → **đóng băng**.
5. **Bộ truy vấn đăng ký trước** (trực tiếp / gián tiếp) → đánh giá test MSVD (thăm dò) và MSR-VTT fresh1k-B (xác nhận).
6. Phân tích quan hệ → xAI → metric bộ truy vấn → xAI theo thứ hạng → 5 + 5 ca → demo.
7. Xuất file → báo cáo, kết luận → kiểm kê dữ liệu, audit rò rỉ, nghiệm thu, 7 câu kết luận, manifest → nén zip.

In [87]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== CỔNG A: dữ liệu thật, đọc được ==============================
mark_stage("data", requires=("config",))
if CFG.dataset == "msvd":
    RECORDS, _caps, SOURCE, DATA_REPORT = load_msvd()
else:
    RECORDS, _caps, SOURCE, DATA_REPORT = make_synthetic_msvd()
N_MSVD = len(RECORDS)

# ---- benchmark xác nhận độc lập (MSR-VTT, tập theo CFG.confirm_set — mặc định fresh1k); lỗi tải ⇒ NOT RUN, ghi rõ lý do, KHÔNG thay bằng tập khác ----
EXT_RECORDS, EXT_SOURCE, EXT_REPORT, EXT_STATUS = [], None, {}, "NOT RUN — CFG.confirm = none"
if CFG.confirm == "msrvtt":
    try:
        if CFG.dataset == "msvd":
            EXT_RECORDS, _ext_caps, EXT_SOURCE, EXT_REPORT = load_msrvtt(N_MSVD)
        else:
            EXT_RECORDS, _ext_caps, EXT_SOURCE, EXT_REPORT = make_synthetic_ext(N_MSVD)
        _caps.update(_ext_caps)
        EXT_STATUS = "DEV (không phải tập xác nhận)" if EXT_REPORT["dev"] else "OK"
    except Exception as e:
        EXT_RECORDS, EXT_SOURCE, EXT_STATUS = [], None, f"NOT RUN — không nạp được MSR-VTT: {type(e).__name__}: {e}"
        print("⚠", EXT_STATUS)
RECORDS = RECORDS + EXT_RECORDS
SPLIT_OF = {r["vid"]: r["split"] for r in RECORDS}
CAPS = CaptionStore(_caps, SPLIT_OF)
del _caps                                                  # caption chỉ còn trong CAPS
CAP_VID, CAP_SPLIT = CAPS.vid_of_cap, CAPS.split_of_cap
CAP_STD = np.zeros(len(CAP_VID), bool)                     # caption chuẩn 1k-A (truy vấn PHỤ của tập ngoài)
_first_cap = {v: i for i, v in reversed(list(enumerate(CAP_VID.tolist())))}
for r in EXT_RECORDS:
    CAP_STD[_first_cap[r["vid"]] + r["std_caption"]] = True

rep = DATA_REPORT["splits"]
print(f"Nguồn video: {DATA_REPORT['source']} ({DATA_REPORT['source_label']}) | {DATA_REPORT['n_video_files']} file video")
display(pd.DataFrame(rep).T)
if CFG.dataset == "msvd":
    got = {s: (rep[s]["videos"], rep[s]["captions"]) for s in rep}
    check("A1", "số video / caption theo split khớp dataset card MSVD", got == {s: tuple(v) for s, v in CFG.expected.items()},
          f"thực tế {got} | kỳ vọng {CFG.expected}")
    check("A1b", "cột 'video' của HF chỉ là tên file .avi (video thật lấy từ zip/thư mục)",
          all(rep[s]["video_field_is_filename"] for s in rep), DATA_REPORT["source_label"])
else:
    check("A1", "số video / caption khớp MSVD", False, "dữ liệu SYNTHETIC — không áp dụng", hard=False, status="NOT RUN")
check("A2", "mỗi video_id có đúng một file video (không thiếu file)",
      all(rep[s]["missing_files"] == 0 for s in rep), {s: rep[s]["missing_files"] for s in rep})
print(f"Tập ngoài: {EXT_STATUS} | {EXT_REPORT.get('label', '—')} | {EXT_REPORT.get('videos', 0)} video / "
      f"{EXT_REPORT.get('captions', 0)} caption")
if EXT_RECORDS:
    if EXT_REPORT["dev"]:
        ok_ext = EXT_REPORT["overlap_with_test_1kA"] == 0
    elif EXT_REPORT.get("set") in ("fresh1k", "fresh1kB"):
        ok_ext = (EXT_REPORT["videos"] == CFG.fresh_n and EXT_REPORT["overlap_with_test_1kA"] == 0
                  and EXT_REPORT["overlap_with_dev"] == 0
                  and (EXT_REPORT["set"] == "fresh1k" or EXT_REPORT["overlap_with_fresh1k"] == 0))
    else:
        ok_ext = EXT_REPORT["videos"] == 1000 and EXT_REPORT["overlap_with_test_1kA"] == 1000
    check("A6", "tập ngoài: đủ file video; fresh1k / fresh1k-B = đúng CFG.fresh_n video, rời 1k-A, rời 200 video dev (fresh1k-B rời cả fresh1k); "
                "1kA = đúng 1000 video test 1k-A; dev = rời hẳn 1k-A",
          EXT_REPORT["missing_files"] == 0 and ok_ext,
          {k: EXT_REPORT.get(k) for k in ("label", "set", "videos", "captions", "captions_per_video", "std_captions_added",
                                          "missing_files", "overlap_with_test_1kA", "overlap_with_dev", "overlap_with_fresh1k")})
else:
    check("A6", "tập ngoài (benchmark xác nhận độc lập)", False, EXT_STATUS, hard=False, status="NOT RUN")

probes = []
for s in ("train", "val", "test", "ext"):
    src = EXT_SOURCE if s == "ext" else SOURCE
    for r in [r for r in RECORDS if r["split"] == s][:3]:
        with src.open(r["video_id"]) as p:
            probes.append(dict(split=s, video_id=r["video_id"], **probe_video(p)))
display(pd.DataFrame(probes))
check("A3", "giải mã được video mẫu của TỪNG split (3 video/split, đếm frame bằng grab)",
      all(p["ok"] for p in probes),
      f"{sum(p['ok'] for p in probes)}/{len(probes)} video | codec {sorted({str(p.get('codec')) for p in probes})} | "
      f"frame {[p.get('frames_decoded') for p in probes]}")
DATA_REPORT["probes"], DATA_REPORT["ext"] = probes, dict(EXT_REPORT, status=EXT_STATUS)
save_json(DATA_REPORT, RESULTS / "gateA_dataset_report.json")
provenance("RECORDS + CaptionStore", "external", f"{CFG.hf_repo}@{CFG.hf_revision[:8]}" if CFG.dataset == "msvd"
           else "synthetic", note="metadata + caption gốc; caption chỉ đọc qua CaptionStore")
if EXT_RECORDS:
    provenance("tập ngoài (MSR-VTT)", "external", EXT_REPORT["label"], note="chỉ frame (encode) trước đóng băng; caption chỉ để đánh giá")

In [88]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
mark_stage("split", requires=("data",))
SPLIT = make_split(RECORDS)
check("A5", "split THEO VIDEO, không video nào ở hai split; manifest đã lưu",
      sum(len(SPLIT[k]) for k in ("train", "val", "test", "ext")) == len(RECORDS), f"sha1={SPLIT['sha1']} → split_manifest.json")

In [89]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== CỔNG B (1/2): frame + đặc trưng ==============================
mark_stage("features", requires=("split",))
FRAME_FEAT, FRAME_META, TXT_FEAT, FEAT_CACHES = all_features()
n_bad = len(FRAME_META["bad"])
check("A4", "TOÀN BỘ video giải mã được và lấy đủ F_max frame", n_bad == 0,
      f"{len(RECORDS) - n_bad}/{len(RECORDS)} video đọc được | nguồn {FRAME_META['source']} | {FRAME_META['seconds']}s")
save_json(dict(policy=CFG.frame_policy, n_frames_max=CFG.n_frames_max,
               formula="idx_k = floor((k + 0.5) / F_max · n_decoded), t_k = idx_k / fps",
               per_video=FRAME_META["per_video"], bad=FRAME_META["bad"]), RESULTS / "frame_sampling.json")
provenance("FRAME_FEAT [N,12,D]", "frames", f"CLIP-image đóng băng {CFG.clip_model}/{CFG.clip_weights} trên 12 frame/video",
           video_splits=tuple(k for k in ("train", "val", "test", "ext") if SPLIT[k]), note=" | ".join(map(str, FEAT_CACHES)))
provenance("TXT_FEAT [N_caption,D]", "encoded", "CLIP-text đóng băng — mã hoá để cache, không học",
           caption_splits=tuple(k for k in ("train", "val", "test", "ext") if SPLIT[k]), note="việc DÙNG tuân theo CaptionStore")
dur = np.array([m["duration_s"] for m in FRAME_META["per_video"].values() if m])
print(f"Thời lượng video: min {dur.min():.1f}s | trung vị {np.median(dur):.1f}s | max {dur.max():.1f}s")

In [90]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== CỔNG B (2/2): gộp theo thời gian + kiểm tra ==============================
mark_stage("pooling", requires=("features",))
POOL_SEL, POOL_TABLE = select_pooling_on_val(FRAME_FEAT, TXT_FEAT, SPLIT)
VID_FEAT = pool_frames(FRAME_FEAT, POOL_SEL["n_frames"], POOL_SEL["pooling"])
POOL_TABLE.to_csv(RESULTS / "pooling_selection_val.csv", index=False)
display(POOL_TABLE.round(4))
print(f"→ chọn trên VAL: {POOL_SEL}")
provenance("n_frames + pooling", "selected", "MRR CLIP-only text→video trên VAL", caption_splits=("val",))
provenance("VID_FEAT [N,D]", "frames", f"gộp {POOL_SEL['n_frames']} frame ({POOL_SEL['pooling']})",
           video_splits=tuple(k for k in ("train", "val", "test", "ext") if SPLIT[k]))
gate_b_checks(FRAME_FEAT, FRAME_META, VID_FEAT, TXT_FEAT, SPLIT)
save_json([c for c in CHECKS if c["id"].startswith("B")], RESULTS / "gateB_checks.json")

In [91]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
mark_stage("vocab_tagger", requires=("pooling",))
t0 = time.time()
VOCAB2 = build_vocab_v2(CAPS, SPLIT)
RELATIONS2 = build_relations_v2(CAPS, SPLIT, VOCAB2)
TAGS2, TAG_SCORES, TAG_SELECT2, TAG_GRID2 = run_tagger_v2(FRAME_FEAT, VID_FEAT, POOL_SEL["n_frames"], VOCAB2, SPLIT, CAPS)
for r in RECORDS:
    r["tags"] = TAGS2[r["vid"]]
TAXONOMY, CONCEPT_META, TAX_AUDIT, TAX_STATS = build_taxonomy_v2(VOCAB2, CAPS, SPLIT, {v: TAGS2[v] for v in SPLIT["train"]})
INV = NodeInventory(RECORDS, VOCAB2, CONCEPT_META)
CONF_CAL = contains_calibration(TAG_SCORES, SPLIT["train"], INV)
_qtexts = CAPS.texts(np.arange(len(CAP_VID)), "query_input")
QTOK2 = [query_tokens_v2(t) for t in _qtexts]
SYM_FEAT = symbolic_features(INV, VOCAB2)
TRAIN_Q = np.where(CAP_SPLIT == "train")[0]

# ---- vị từ (v3): từ vựng + support chỉ từ quan hệ caption TRAIN; embedding = CLIP-text đóng băng theo prompt ----
PRED_SUPPORT = Counter()
for v in SPLIT["train"]:
    PRED_SUPPORT.update({(rel, p) for _, rel, _, p in RELATIONS2[v]})
PRED_VOCAB = sorted(k for k in PRED_SUPPORT if k[0] in PRED_FAMILIES)
PRED_ID = {k: i for i, k in enumerate(PRED_VOCAB)}
PRED_EMB = predicate_embeddings(PRED_VOCAB, VOCAB2)
QPRED_TOK = [query_pred_tokens(t) for t in _qtexts]
QPREDS = [{rel: match_predicates(qs, rel) for rel in PRED_FAMILIES} for qs in QPRED_TOK]
del _qtexts
n_v = sum(t == "verb" for t in VOCAB2["types"].values())
print(f"Từ vựng: {len(VOCAB2['terms'])} khoá = {len(VOCAB2['terms']) - n_v} danh từ + {n_v} động từ; "
      f"{len(VOCAB2['lemmas_with_two_pos'])} lemma có CẢ hai POS")
print(f"Tagger (chọn trên VAL): {TAG_SELECT2}")
print(f"Taxonomy: {TAX_STATS['n_concepts']} khái niệm | isa theo biến thể {TAX_STATS['isa_edges']} | mơ hồ {TAX_STATS['n_ambiguous']} | "
      f"sense khác sense đầu {TAX_STATS['n_sense_differs']} | hub {TAX_STATS['n_hub']} | chu trình {TAX_STATS['cycles']}")
print(f"Vị từ: {len(PRED_VOCAB)} (họ có vị từ tự do {PRED_FAMILIES}; "
      f"{ {f: sum(r == f for r, _ in PRED_VOCAB) for f in PRED_FAMILIES} }); ≥ {CFG.pred_min_support} video TRAIN: "
      f"{ {f: sum(r == f and PRED_SUPPORT[(r, p)] >= CFG.pred_min_support for r, p in PRED_VOCAB) for f in PRED_FAMILIES} } | "
      f"truy vấn khớp ≥ 1 vị từ: {np.mean([any(q.values()) for q in QPREDS]):.1%}")
print(f"Inventory: {INV.n_nodes} node = {INV.n_vid} video ({N_MSVD} MSVD + {INV.n_vid - N_MSVD} tập ngoài) + {INV.n_cls} lớp + "
      f"{INV.n_con} khái niệm (sha1 {INV.sha1}) | {time.time() - t0:.0f}s")

# ---- Nhiệm vụ B: kiểm chứng TỰ ĐỘNG (parser độc lập, chỉ TRAIN) + mẫu phân tầng; nhãn người chấm chỉ là TUỲ CHỌN ----
t1 = time.time()
AGREE, AGREE_COVER = build_agreement(CAPS, SPLIT, VOCAB2)
ANN_SAMPLE, ANN_CELLS, TRIPLE_SUP, _ = build_annotation_sample(CAPS, SPLIT, VOCAB2, RECORDS)
if AGREE is not None:
    ANN_SAMPLE.insert(7, "auto_parser_agrees", [int(triple_confirmed((r.src_node, r.family, r.dst_node, r.predicate),
                                                                       _DEP_CACHE[r.caption.lower()])) for r in ANN_SAMPLE.itertuples()])
    REL_QUALITY_AUTO = agreement_quality(AGREE, AGREE_COVER, TRIPLE_SUP)
    _fam = REL_QUALITY_AUTO[(REL_QUALITY_AUTO.stratum == "ALL") & (REL_QUALITY_AUTO.family != "ALL")]
    AUTO_STATUS = (f"kiểm chứng TỰ ĐỘNG ({DEP_STATUS}): tỉ lệ quan hệ heuristic được parser độc lập xác nhận — "
                   + "; ".join(f"{r.family} {r.agreement:.2f} [{r.wilson_lo:.2f}, {r.wilson_hi:.2f}]" for r in _fam.itertuples())
                   + f" ({time.time() - t1:.0f}s). SILVER — không phải precision do người chấm.")
else:
    REL_QUALITY_AUTO = pd.DataFrame([dict(kind="silver_parser_agreement", family="ALL", stratum="ALL", status="NOT RUN", note=DEP_STATUS)])
    AUTO_STATUS = DEP_STATUS
print("Kiểm chứng tự động:", AUTO_STATUS)
ANN_SAMPLE.to_csv(RESULTS / "relation_annotation_sample.csv", index=False, encoding="utf-8-sig")
ANN_CELLS.to_csv(RESULTS / "relation_annotation_cells.csv", index=False)
LABEL_FILE, LABELS = find_label_file()           # tuỳ chọn: chỉ dùng nếu có file nhãn trong /kaggle/input hoặc MSVD_G3_LABELS
if LABELS is not None:
    REL_QUALITY, KAPPA, LABELED = relation_quality(LABELS)
    VERIFIED_CELLS = {(r.family, r.stratum) for r in REL_QUALITY.itertuples() if r.verified}
    match = float(LABELS["sample_id"].isin(set(ANN_SAMPLE["sample_id"])).mean()) if "sample_id" in LABELS else float("nan")
    REL_QUALITY_STATUS = (f"nhãn người chấm: {LABEL_FILE} | {int(REL_QUALITY.iloc[0].n_labeled)} dòng có nhãn | "
                          f"kappa {KAPPA['kappa']:.3f} trên {KAPPA['n_double']} dòng chấm đôi | ô đạt ngưỡng: {sorted(VERIFIED_CELLS)} | "
                          f"sample_id khớp mẫu tái tạo: {match:.1%}")
    LABELED.to_csv(RESULTS / "relation_annotation_labeled.csv", index=False)
else:
    KAPPA, REL_QUALITY = dict(n_double=0, kappa=float("nan"), raw_agreement=float("nan")), None
    REL_QUALITY_STATUS = "không có nhãn người chấm (tuỳ chọn) — chỉ có kiểm chứng tự động"
pd.concat([REL_QUALITY_AUTO] + ([REL_QUALITY.assign(kind="human_precision")] if REL_QUALITY is not None else []),
          ignore_index=True).to_csv(RESULTS / "relation_quality_audit.csv", index=False)
print("Nhãn người chấm:", REL_QUALITY_STATUS)

pd.DataFrame(TAG_GRID2).to_csv(RESULTS / "tagger_selection_val_v2.csv", index=False)
save_json(dict(VOCAB2, relations_example={RECORDS[v]["video_id"]: RELATIONS2[v][:8] for v in SPLIT["train"][:5]}),
          RESULTS / "vocab_v2.json")
save_json(dict(selection=TAG_SELECT2, calibration=CONF_CAL), RESULTS / "tagger_info_v2.json")
TAX_AUDIT.to_csv(RESULTS / "graph_taxonomy_audit.csv", index=False)
save_json(TAX_STATS, RESULTS / "graph_taxonomy_stats.json")
provenance("từ vựng (lemma, POS)", "learned", "caption TRAIN (POS + WordNet)", caption_splits=("train",))
provenance("quan hệ (loại + vị từ)", "learned", "caption TRAIN", caption_splits=("train",))
provenance("từ vựng + support vị từ", "learned", "quan hệ caption TRAIN", caption_splits=("train",))
provenance("embedding vị từ (CLIP-text đóng băng)", "external", "prompt CFG.pred_prompts × vị từ train")
provenance("mẫu quan hệ phân tầng", "learned", "caption TRAIN (phân tầng họ × mức phổ biến)", caption_splits=("train",))
if AGREE is not None:
    provenance("đồng thuận heuristic ↔ spaCy (kiểm chứng tự động)", "learned", f"caption TRAIN; {DEP_STATUS}", caption_splits=("train",))
if LABELS is not None:
    provenance("precision quan hệ + ô đã kiểm chứng", "external", f"nhãn NGƯỜI chấm trên caption TRAIN ({LABEL_FILE.name})")
provenance("tham số tagger", "selected", "F1 trên caption VAL", caption_splits=("val",))
provenance("thống kê hiệu chỉnh contains (μ, σ, phân vị)", "frames", "điểm tagger trên video TRAIN", video_splits=("train",))
provenance("taxonomy: sense theo ngữ cảnh + hub", "learned", "caption TRAIN + nhãn tagger TRAIN + WordNet",
           caption_splits=("train",), video_splits=("train",))
provenance("đặc trưng ngữ nghĩa node (CLIP-text)", "external", "prompt tagger / 'lemma: định nghĩa WordNet'")
# ---- v4: node khớp + VAI của từng node trong CHÍNH truy vấn (đọc văn bản truy vấn — mục đích query_input, như QTOK2) ----
V4_HITS = [sorted({nid for nid, _, _ in parse_tokens_v2(t, INV)}) for t in QTOK2]
V4_ROLES = query_roles_v4(CAPS.texts(np.arange(len(CAP_VID)), "query_input"), INV)
print(f"v4: truy vấn có ≥ 1 node khớp {np.mean([bool(h) for h in V4_HITS]):.3f}; có vai chủ thể/đối tượng "
      f"{np.mean([any(r in (1, 3) for r in d.values()) for d in V4_ROLES]):.3f}")


In [92]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== ĐỒ THỊ v3: schema · node · cạnh có vị từ · audit cấu trúc · audit vị từ ==============================
mark_stage("graph_v3", requires=("vocab_tagger",))
G_BASE, E_BASE = graph_for(DESIGN_BASE, CFG.seeds[0])
print("Đồ thị v3 (thiết kế gốc = v2_base):", json.dumps(G_BASE.summary(), ensure_ascii=False))
SCHEMA = dict(version="v3", node_types=dict(video="video MSVD / tập ngoài (embedding CLIP gộp frame, đóng băng)",
                                            **{"class": "lớp (lemma, POS) từ caption TRAIN", "concept": "synset WordNet"}),
              semantic_types=["video", "noun", "verb", "noun_concept", "verb_concept"], relations=REL_SPEC,
              edge_record=["src", "relation_family", "relation", "predicate", "dst", "direction", "weight", "confidence",
                           "source", "status"],
              predicate_families=list(PRED_FAMILIES), predicate_vocab_size=len(PRED_VOCAB),
              predicate_prompts=CFG.pred_prompts, pred_modes=list(PRED_MODES), rel_policies=list(REL_POLICIES),
              patterns=dict(zip(PATTERNS, PATTERN_LEN)), masks={k: list(v) for k, v in MASKS.items()},
              inventory=dict(n_nodes=INV.n_nodes, videos=INV.n_vid, msvd_videos=N_MSVD, ext_videos=INV.n_vid - N_MSVD,
                             classes=INV.n_cls, concepts=INV.n_con, sha1=INV.sha1),
              path_modes=["weighted", "count", "degnorm", "dedup", "pred_match"],
              mp_weight_modes=["binary", "weighted_mean", "weighted_symnorm"], init_modes=["learned", "clip_frozen", "clip_residual"],
              related_cfgs=RELATED_CFGS, protocols=list(PROTOCOLS))
SCHEMA["sha1"] = sha1_of(SCHEMA)
save_json(SCHEMA, RESULTS / "graph_schema_v3.json")

deg_out = {t: np.asarray((G_BASE.A[t] > 0).sum(1)).ravel() for t in RELATION_TYPES}
deg_in = {t: np.asarray((G_BASE.A[t] > 0).sum(0)).ravel() for t in RELATION_TYPES}
con_in = np.asarray((G_BASE.A_con > 0).sum(0)).ravel()
node_rows = []
for i in range(INV.n_nodes):
    t = INV.node_type[i]
    node_rows.append(dict(node_id=i, node_type=t, canonical_name=INV.name[i], semantic_type=INV.semantic_type(i),
                          lemma=INV.lemma.get(i, ""), split_provenance=SPLIT_OF.get(i, "") if t == "video" else
                          ("train" if t == "class" else "external"),
                          synset=INV.concept_meta.get(INV.name[i], {}).get("synset", ""),
                          train_df=int(G_BASE.cls_df[i]) if t == "class" else "", active=i in G_BASE.active,
                          videos_tagged=int(con_in[i]), rel_out_degree=int(sum(deg_out[x][i] for x in RELATION_TYPES)),
                          rel_in_degree=int(sum(deg_in[x][i] for x in RELATION_TYPES))))
NODES_DF = pd.DataFrame(node_rows)
NODES_DF.to_csv(RESULTS / "graph_nodes_v3.csv", index=False)


def _confidence(rel, c, s=None, o=None, p=None):
    """Nhãn người chấm (nếu có) của ô (họ × mức phổ biến); ngược lại tỉ lệ đồng thuận SILVER với parser độc lập của đúng
    quan hệ đó; không có cả hai ⇒ 'unverified'."""
    if rel not in REL_CATS:
        return ""
    if REL_QUALITY is not None:
        r = REL_QUALITY[(REL_QUALITY.family == rel) & (REL_QUALITY.stratum == support_stratum(c))]
        if len(r) and r.iloc[0].n_labeled:
            return f"human {r.iloc[0].precision:.3f} [{r.iloc[0].wilson_lo:.3f}, {r.iloc[0].wilson_hi:.3f}]"
    a = AGREE.get((rel, s, o, p)) if AGREE and s is not None else None
    return f"silver {a[0] / a[1]:.2f} ({a[0]}/{a[1]})" if a and a[1] else "unverified"


def edge_table(g):
    """Một dòng = MỘT quan hệ (nguồn, họ, vị từ, đích, chiều): cạnh quan hệ tách theo từng vị từ của cặp; cạnh bị loại
    (chính sách / top-k) vẫn được ghi với status = excluded:<lý do>."""
    rows = []
    c = g.A_con.tocoo()
    for u, v, w in zip(c.row.tolist(), c.col.tolist(), c.data.tolist()):
        for rel, a, b, dr in (("contains", u, v, "forward"), ("contains_inv", v, u, "inverse")):
            rows.append(dict(src=INV.name[a], relation_family="contains", relation=rel, predicate="", dst=INV.name[b],
                             direction=dr, weight=w, confidence="tagger" if g.contains_mode == "conf" else "",
                             source=REL_SPEC[rel]["source"][:60], status="kept", pair_support="", predicate_videos="",
                             predicate_share=""))
    for t in RELATION_TYPES:
        m = g.A[t].tocoo()
        base = t[:-4] if t.endswith("_inv") else t
        dr = "symmetric" if t == "related" else ("inverse" if t.endswith("_inv") else "forward")
        for u, v, w in zip(m.row.tolist(), m.col.tolist(), m.data.tolist()):
            d = g.predicates_of(t, u, v) if base in REL_CATS else {}
            tot = float(sum(d.values())) or 1.0
            ka, kb = (INV.name[v], INV.name[u]) if t.endswith("_inv") else (INV.name[u], INV.name[v])
            for p, cnt in (sorted(d.items(), key=lambda x: (-x[1], x[0])) or [("", "")]):
                rows.append(dict(src=INV.name[u], relation_family=REL_SPEC[t]["family"], relation=t, predicate=p,
                                 dst=INV.name[v], direction=dr, weight=w, confidence=_confidence(base, cnt, ka, kb, p) if p else "",
                                 source=REL_SPEC[t]["source"][:60], status="kept",
                                 pair_support=int(sum(d.values())) if d else "", predicate_videos=cnt,
                                 predicate_share=round(cnt / tot, 4) if p else ""))
    for x in g.excluded:
        for p, cnt in sorted(x["predicates"].items(), key=lambda y: (-y[1], y[0])):
            rows.append(dict(src=INV.name[x["src"]], relation_family=REL_SPEC[x["family"]]["family"], relation=x["family"],
                             predicate=p, dst=INV.name[x["dst"]], direction="forward", weight="",
                             confidence=_confidence(x["family"], cnt, INV.name[x["src"]], INV.name[x["dst"]], p),
                             source=REL_SPEC[x["family"]]["source"][:60],
                             status=f"excluded:{x['reason']}", pair_support=x["support"], predicate_videos=cnt, predicate_share=""))
    return pd.DataFrame(rows)


EDGES_DF = edge_table(G_BASE)
EDGES_DF.to_csv(RESULTS / "graph_edges_v3.csv", index=False)

struct = []
for t in ["contains"] + RELATION_TYPES:
    M = G_BASE.A_con if t == "contains" else G_BASE.A[t]
    m = M.tocoo()
    do, di = np.asarray((M > 0).sum(1)).ravel(), np.asarray((M > 0).sum(0)).ravel()
    srcs, tgts = np.unique(m.row), np.unique(m.col)
    base = t[:-4] if t.endswith("_inv") else t
    npred = [len(G_BASE.predicates_of(t, u, v)) for u, v in zip(m.row.tolist(), m.col.tolist())] if base in REL_CATS else []
    struct.append(dict(section="v3_relation", relation=t, family=REL_SPEC[t]["family"], usage=REL_SPEC[t]["usage"], n_edges=int(M.nnz),
                       src_types=dict(Counter(INV.semantic_type(i) for i in srcs)) if t != "contains" else {"video": len(srcs)},
                       tgt_types=dict(Counter(INV.semantic_type(i) for i in tgts)),
                       w_mean=float(m.data.mean()) if m.nnz else None, w_nonbinary_frac=float((m.data < 1 - 1e-9).mean()) if m.nnz else 0,
                       out_deg_mean=float(do[do > 0].mean()) if (do > 0).any() else 0, out_deg_max=int(do.max()) if do.size else 0,
                       in_deg_mean=float(di[di > 0].mean()) if (di > 0).any() else 0, in_deg_max=int(di.max()) if di.size else 0,
                       predicates_per_edge_mean=float(np.mean(npred)) if npred else None,
                       frac_edges_multi_predicate=float(np.mean([k > 1 for k in npred])) if npred else None,
                       predicate_in_message=t in PRED_TYPES, self_loops=int((m.row == m.col).sum()),
                       duplicates=int(m.nnz - len(set(zip(m.row, m.col))))))
act = np.array(sorted(G_BASE.active))
rel_deg = sum(deg_out[t] + deg_in[t] for t in RELATION_TYPES)
vid_no_tag = int(sum(len(x) == 0 for x in G_BASE.vid_tags))
struct.append(dict(section="v3_nodes", relation="—", note=(
    f"lớp active {len(act)}/{INV.n_cls}; lớp active KHÔNG có cạnh quan hệ: {int((rel_deg[act] == 0).sum())}; video không có contains: "
    f"{vid_no_tag}; khái niệm không có isa vào: {int((deg_in['isa'][INV.n_vid + INV.n_cls:] == 0).sum())}/{INV.n_con}; "
    f"cặp quan hệ bị loại (chính sách/top-k): {len(G_BASE.excluded)}")))
STRUCT_DF = pd.DataFrame(struct)
STRUCT_DF.to_csv(RESULTS / "graph_structure_audit.csv", index=False)
display(STRUCT_DF[STRUCT_DF.section == "v3_relation"][["relation", "n_edges", "w_mean", "predicates_per_edge_mean",
                                                       "frac_edges_multi_predicate", "predicate_in_message", "self_loops", "duplicates"]])

# ---- audit embedding vị từ: vị từ nào dùng trong đồ thị gốc, support, láng giềng gần nhất (kiểm tra ngữ nghĩa) ----
_used = Counter()
for rel in PRED_FAMILIES:
    for d in G_BASE.pshare[rel].values():
        _used.update({PRED_VOCAB[pid] for pid in d})
_E = PRED_EMB / np.linalg.norm(PRED_EMB, axis=1, keepdims=True).clip(1e-8)
pe_rows = []
for rel in PRED_FAMILIES:
    idx = np.array([i for i, (r, _) in enumerate(PRED_VOCAB) if r == rel])
    if not len(idx):
        continue
    used_idx = np.array([i for i in idx if _used[PRED_VOCAB[i]] > 0])
    S = _E[idx] @ _E[idx].T
    off = S[~np.eye(len(idx), dtype=bool)] if len(idx) > 1 else np.array([1.0])
    for i in (used_idx if len(used_idx) else idx[:0]):
        s = _E[idx] @ _E[i]
        nn_ = [PRED_VOCAB[idx[j]][1] for j in np.argsort(-s) if idx[j] != i][:3]
        pe_rows.append(dict(family=rel, predicate=PRED_VOCAB[i][1], phrase=predicate_phrase(rel, PRED_VOCAB[i][1], VOCAB2),
                            prompt=CFG.pred_prompts[rel][0].format(predicate_phrase(rel, PRED_VOCAB[i][1], VOCAB2)),
                            train_videos=int(PRED_SUPPORT[PRED_VOCAB[i]]), graph_edges=int(_used[PRED_VOCAB[i]]),
                            passes_support_policy=PRED_SUPPORT[PRED_VOCAB[i]] >= CFG.pred_min_support,
                            nearest=", ".join(nn_), family_mean_cos=float(off.mean()), family_min_cos=float(off.min()),
                            family_max_cos=float(off.max())))
PRED_AUDIT = pd.DataFrame(pe_rows)
PRED_AUDIT.to_csv(RESULTS / "predicate_embedding_audit.csv", index=False)
if len(PRED_AUDIT):
    print("Embedding vị từ:", PRED_AUDIT.groupby("family").agg(n=("predicate", "size"), mean_cos=("family_mean_cos", "first"),
                                                              min_cos=("family_min_cos", "first")).to_dict("index"))

# ---- thống kê quan hệ caption TRAIN ----
tq = np.where(CAP_SPLIT == "train")[0]
tr_texts = CAPS.texts(tq, "relations")
all_trip, oov = Counter(), 0
vset = set(VOCAB2["terms"])
for txt in tr_texts:
    for s, rel, o, pred in parse_caption_v2(txt)[2]:
        all_trip[rel] += 1
        oov += int(s not in vset or o not in vset)
REL_STATS = dict(train_captions=len(tq), triples_extracted=int(sum(all_trip.values())), by_category=dict(all_trip),
                 frac_dropped_oov=oov / max(sum(all_trip.values()), 1),
                 kept_relation_pairs=sum(int(G_BASE.A[rel].nnz) for rel in REL_CATS),
                 kept_relation_predicate_rows=int(((EDGES_DF.status == "kept") & EDGES_DF.relation.isin(REL_CATS)).sum()),
                 excluded_pairs=len(G_BASE.excluded), annotation_sample_rows=len(ANN_SAMPLE),
                 automatic_verification=AUTO_STATUS, human_precision=REL_QUALITY_STATUS,
                 heuristic_covers_spacy={k: round(AGREE_COVER.get((k, "found"), 0) / max(AGREE_COVER.get((k, "total"), 0), 1), 3)
                                         for k in REL_CATS} if AGREE_COVER else "NOT RUN",
                 recall_f1="NOT RUN — không có tập quan hệ đúng đầy đủ; 'heuristic_covers_spacy' chỉ là độ phủ tương đối so với spaCy")
save_json(REL_STATS, RESULTS / "graph_relation_quality_stats.json")
print("Quan hệ caption TRAIN:", json.dumps(REL_STATS, ensure_ascii=False))
provenance("cạnh quan hệ v3 (agent/object/svo/prep + _inv, phân phối vị từ)", "learned", "caption TRAIN", caption_splits=("train",))
provenance("cạnh related (NPMI)", "learned", "nhãn tagger của video TRAIN", video_splits=("train",))
provenance("IDF", "learned", "contains + isa thật của video TRAIN", video_splits=("train",))
provenance("cạnh isa", "learned", "WordNet + chọn sense từ ngữ cảnh caption TRAIN", caption_splits=("train",))

In [93]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== SELF-TEST v3 (mục 12: kiểm thử 1–10) ==============================
# PASS ở đây chỉ xác nhận PHẦN MỀM làm đúng như mô tả — KHÔNG xác nhận giả thuyết khoa học.
mark_stage("self_tests", requires=("graph_v3",))
ST = {}
t = ST["canary"] = test_canary()
check("C1", "[12.8] canary chỉ có trong caption TEST / tập ngoài không lọt vào từ vựng, quan hệ, từ vựng vị từ",
      not t["leak_vocab"] and not t["leak_relations"] and not t["leak_predicates"], t)
check("C1b", "đối chứng dương: canary đặt vào TRAIN thì bị phát hiện", t["positive_control_detected"], t)
t = ST["invariance"] = test_invariance()
check("C2", "[12.8] xáo/thay caption TEST + tập ngoài: từ vựng, quan hệ, vị từ, tagger, taxonomy, inventory, đồ thị, mẫu gán nhãn GIỐNG HỆT",
      all(t.values()), t)
t = ST["guards"] = test_guards()
check("C3", "[12.8] hàng rào: chặn mở test / tập ngoài trước đóng băng + caption sai mục đích", all(t.values()), t)
t = ST["inventory"] = test_inventory_constant()
check("G1", "[12.5] inventory CỐ ĐỊNH giữa mọi biến thể đồ thị; cùng cấu hình mô hình ⇒ cùng số tham số",
      len(t["node_counts"]) == 1 and t["same_inventory"] and all(len(v) == 1 for v in t["param_counts_by_pred_mode"].values()), t)
t = ST["schema"] = test_schema_conformance(G_BASE)
check("G2", "mọi cạnh tuân thủ schema (kiểu node, POS, trọng số, vị từ đúng họ, không lẫn tên loại quan hệ)", not t["violations"], t)
t = ST["path_identity"] = test_path_identity()
check("G3", "[12.9] điểm đường đi ma trận = liệt kê ở cả 5 path_mode; tính lại R khớp; đường đi khớp vị từ ghi đúng vị từ + chiều",
      all(v["mismatches"] == 0 and v["recompute_maxdiff"] < 1e-5 and v["pairs"] > 0 and v["predicate_invalid"] == 0
          and v["negative_control_detected"] == v["predicate_paths"] for v in t.values())
      and t["pred_match"]["predicate_paths"] > 0, t)
t = ST["mp_weights"] = test_mp_weights()
check("G4", "[12.6, 12.7] message passing: trọng số dùng đúng công thức, chỉ nhân MỘT lần (khớp tính tay, bất biến tỉ lệ), "
      "w=0 ≡ không cạnh, node cô lập hữu hạn, bất biến thứ tự, gradient hữu hạn",
      all(v["as_designed"] and v["isolated_nodes_finite"] and v["edge_order_invariant"] and v["grad_finite"] for v in t.values()), t)
t = ST["predicate_forward"] = test_predicate_forward()
check("P1", "[12.1] embedding vị từ TÁC ĐỘNG forward khi pred_mode ≠ none; pred_mode none (≡ v2) KHÔNG dùng vị từ",
      t["predicate_used_as_designed"] and all(t[m]["finite"] for m in PRED_MODES), t)
check("P2", "[12.2] hai vị từ khác nhau trên cùng cặp node ⇒ thông điệp khác nhau", t["two_predicates"]["differ"], t["two_predicates"])
t = ST["ablations"] = test_ablations(G_BASE)
check("AB", "[12.5] mỗi ablation tắt ĐÚNG thành phần mô tả (mask, chính sách support, pred_mode, path_mode)", all(t.values()), t)
t = ST["inductive"] = test_inductive_isolation()
ind = [v for k, v in t.items() if k.endswith(":inductive")]
tra = [v for k, v in t.items() if k.endswith(":transductive")]
check("G5", "[12.7] strict inductive (video test MSVD + tập ngoài): KHÔNG đổi node lớp/khái niệm hay video khác; transductive thì đổi; "
      "không NaN", all(v["symbolic_changed"] < 1e-6 and v["other_video_changed"] < 1e-6 and v["own_video_changed"] > 1e-6
                        and v["all_finite"] for v in ind) and all(v["symbolic_changed"] > 1e-6 for v in tra), t)
R_AUD, ok_list = [], []
for s in CFG.seeds:
    df_, ok_ = randomization_audit(G_BASE, graph_for(dict(DESIGN_BASE, rewire=True), s)[0])
    R_AUD.append(df_.assign(control="rewire", seed=s, rand_seed=REWIRE_SEED_BASE + s)); ok_list.append(ok_)
S_AUD, ok_s = [], []
for s in CFG.seeds:
    df_, ok_ = predicate_shuffle_audit(G_BASE, graph_for(dict(DESIGN_BASE, pred_shuffle=True), s)[0])
    S_AUD.append(df_.assign(control="predicate_shuffle", seed=s, rand_seed=SHUFFLE_SEED_BASE + s)); ok_s.append(ok_)
RAND_AUDIT = pd.concat(R_AUD + S_AUD, ignore_index=True)
RAND_AUDIT.to_csv(RESULTS / "graph_randomization_audit.csv", index=False)
RANDOM_VALID, SHUFFLE_VALID = all(ok_list), all(ok_s)
display(RAND_AUDIT[(RAND_AUDIT.seed == CFG.seeds[0]) & (RAND_AUDIT.control == "rewire")][
    ["relation", "edges_orig", "edges_rand", "out_degree_equal", "in_degree_equal", "type_pairs_equal", "weights_multiset_equal",
     "predicates_multiset_equal", "duplicates", "self_loops", "frac_changed", "accepted"]].round(3))
check("G6", "[12.3] đối chứng rewiring giữ kiểu node, họ quan hệ, bậc, trọng số, đa tập vị từ; đổi ≥ ngưỡng (nếu không: KHÔNG kết luận về topology)",
      RANDOM_VALID, f"{int((~RAND_AUDIT[RAND_AUDIT.control == 'rewire'].accepted).sum())} dòng không đạt", hard=False)
check("G7", "[12.4] đối chứng hoán vị vị từ KHÔNG đổi topology / trọng số; chỉ đổi vị từ trong họ svo/prep (nếu không: KHÔNG kết luận về vị từ)",
      SHUFFLE_VALID, RAND_AUDIT[RAND_AUDIT.control == "predicate_shuffle"].groupby("relation").frac_edges_predicates_changed.mean()
      .round(3).to_dict(), hard=False)
check("C7", "split MSVD tái lập được (sha1 giống v2)", make_split(RECORDS, write=False)["sha1"] == SPLIT["sha1"], SPLIT["sha1"])
v4_tensors()
t = ST["v4_decomposition"] = test_v4_decomposition()
check("Q3", "[v4] Σ số hạng (thị giác + nhãn + đường đi quan hệ) = điểm mô hình (T = 1, 2); xoá một cạnh ⇒ điểm đổi ĐÚNG bằng phần "
      "đóng góp các đường đi qua cạnh đó", all(v["max_abs_diff"] < 1e-4 and v["deletion_max_abs_diff"] < 1e-4 and v["relation_terms"] > 0
                                             for v in t.values()), t)
t = ST["v4_predicate_query"] = test_v4_predicate_and_query()
check("Q1", "[v4] vị từ của truy vấn tác động attention trên ĐÚNG cạnh mang vị từ đó; v4_pred = False thì không (đối chứng âm); "
      "hoán vị vị từ đổi tín hiệu khớp nhưng giữ topology",
      t["pred_change_use_pred=True"] > 1e-6 and t["pred_change_use_pred=False"] < 1e-6 and t["pred_shuffle_changes_match"] > 0
      and t["pred_shuffle_same_topology"], t)
check("Q2", "[v4] attention phụ thuộc truy vấn khi v4_qcond = True; giống hệt giữa các truy vấn khi False (đối chứng âm)",
      t["two_queries_attention_diff_qcond=True"] > 1e-6 and t["two_queries_attention_diff_qcond=False"] < 1e-6, t)
t = ST["v4_inductive"] = test_v4_inductive_and_masks()
check("Q4", "[v4] inductive: đổi đặc trưng/nhãn của MỘT video test (tập ngoài) không đổi điểm video khác (dung sai 1e-6 như G5: GPU cộng dồn atomic không tất định); rel = False ⇒ không số hạng "
      "quan hệ; điểm + gradient hữu hạn",
      all(v["other_video_changed"] < 1e-6 and v["own_video_changed"] > 1e-6 for k, v in t.items() if k in ("test", "ext"))
      and t["no_relation_terms_when_rel_false"] and t["grad_finite"] and t["score_finite"], t)
save_json(ST, RESULTS / "self_test_v4.json")

In [94]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== V0 — CLIP-only (G0) + v3 cuối (tham chiếu R3) ==============================
mark_stage("V0", requires=("self_tests",))
PV = make_part("val")
CLIP_RANK_VAL = full_gallery_ranks(TXT_FEAT, VID_FEAT, PV["q_idx"], PV["gallery"], PV["gold"])
CLIP_VAL = metrics_from_ranks(CLIP_RANK_VAL, PV["gold"])
ALT_TAGS, ALT_SCORES = {}, {}
EXPS["G0_clip_only"] = dict(experiment_id="G0_clip_only", stage="V0", description="CLIP-only (không xếp lại bằng đồ thị, α = 1)",
                            design=None, hypothesis="baseline chính", seeds=[], runs={}, status="DONE",
                            val_MRR=(CLIP_VAL["MRR"], 0.0), val_R1=(CLIP_VAL["R@1"], 0.0), rr_val=1.0 / CLIP_RANK_VAL,
                            edges={}, n_params=0, seconds=0.0)
print(f"CLIP-only VAL: MRR {CLIP_VAL['MRR']:.4f} R@1 {CLIP_VAL['R@1']:.4f}")
CFG.rerank_k = CFG.v4_rerank_k                       # k = 50: chọn trên VAL ở v3 (Kaggle), giữ cố định ở v4 — không nhìn test
CAND_RECALL_VAL = float(np.mean(CLIP_RANK_VAL <= CFG.rerank_k))
provenance("k (số ứng viên rerank)", "selected", "chọn trên VAL ở v3 (k = 50), giữ nguyên ở v4", caption_splits=("val",))
v4_tensors()
run_exp("R3_v3_final", "V0", V3_FINAL, "v3 cuối (TypedSAGE + 12 mẫu đường đi, đồ thị E44_no_related) — tham chiếu", "v4 tốt hơn v3")



In [95]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== V1 — v4 (QC-RGAT): chọn số lớp trên VAL ==============================
mark_stage("V1", requires=("V0",))
V4_T1 = dict(V3_FINAL, model="v4", v4_layers=1)
run_exp("E10_v4_T1", "V1", V4_T1, "QC-RGAT 1 lớp (đường đi quan hệ ≤ 1 bước tới nhãn của video)", H_V4)
if "V1" in CFG.stages:
    run_exp("E11_v4_T2", "V1", dict(V4_T1, v4_layers=2), "QC-RGAT 2 lớp (đường đi ≤ 2 bước)", "đường đi 2 bước có thêm thông tin")
    FINAL = decide("V1", "E10_v4_T1", ["E11_v4_T2"], "v4_layers")
else:
    NOT_RUN.append(dict(stage="V1", experiment_id="E11_v4_T2", reason="không có trong CFG.stages"))
    FINAL = "E10_v4_T1"
CURRENT = dict(EXPS[FINAL]["design"])



In [96]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== V7 — ma trận đối chứng dưới CÙNG cấu hình v4 (G3 = v4 đầy đủ) ==============================
mark_stage("V7", requires=("V1",))
G3_DESIGN = dict(CURRENT)
G_IDS = {"G0": "G0_clip_only", "V3REF": "R3_v3_final"}


def control(gid, design, desc, hyp, same_as=None):
    G_IDS[gid.split("_")[0]] = gid
    src = same_as or next((k for k, e in EXPS.items() if e["status"] == "DONE" and e["design"] == design
                           and sorted(e["runs"]) == sorted(CFG.seeds)), None)
    if src is not None:
        return run_exp(gid, "V7", EXPS[src]["design"], f"{desc} (≡ {src}, dùng lại mô hình)", hyp, reuse=src)
    return run_exp(gid, "V7", design, desc, hyp)


control("G3_v4_full", G3_DESIGN, "v4 đầy đủ: quan hệ thật + vị từ + attention theo truy vấn", H_V4)
if "V7" in CFG.stages:
    control("G1_contains_only", dict(G3_DESIGN, v4_rel=False), "cùng mô hình, KHÔNG cạnh quan hệ (chỉ contains + thị giác)",
            "quan hệ đóng góp ngoài contains")
    control("G2_no_predicate", dict(G3_DESIGN, v4_pred=False), "cùng topology + vai + cổng theo truy vấn, KHÔNG khớp vị từ", H_V4)
    control("G4_pred_shuffled", dict(G3_DESIGN, pred_shuffle=True), "phân phối vị từ hoán vị giữa các cạnh cùng họ",
            "ghép ĐÚNG vị từ vào ĐÚNG cạnh có ích")
    control("G5_rewired", dict(G3_DESIGN, rewire=True), "constrained rewiring (giữ kiểu node, họ, bậc, trọng số, đa tập vị từ)",
            "cấu trúc liên kết thật có ích hơn ngẫu nhiên có ràng buộc")
    control("G6_static_attention", dict(G3_DESIGN, v4_qcond=False), "attention tĩnh (chỉ loại cạnh + trọng số), KHÔNG theo truy vấn",
            "điều kiện theo truy vấn làm quan hệ có ích")
    control("G7_no_role_edges", dict(G3_DESIGN, mask=add_mask(G3_DESIGN["mask"], "no_roles")),
            "HUẤN LUYỆN LẠI không có họ cạnh vai (agent_of, object_of); cùng inventory, các họ khác, giao thức", "họ cạnh vai có giá trị")
    control("G8_no_svo_prep_edges", dict(G3_DESIGN, mask=add_mask(G3_DESIGN["mask"], "no_svo_prep")),
            "HUẤN LUYỆN LẠI không có họ cạnh svo / prep (có vị từ); cùng inventory, các họ khác, giao thức", "họ cạnh svo/prep có giá trị")
    control("V3REF", V3_FINAL, "v3 cuối (tham chiếu)", "v4 tốt hơn v3", same_as="R3_v3_final")
    g3 = graph_for(G3_DESIGN, CFG.seeds[0])[0]
    a1, RANDOM_VALID_G3 = randomization_audit(g3, graph_for(dict(G3_DESIGN, rewire=True), CFG.seeds[0])[0])
    a2, SHUFFLE_VALID_G3 = predicate_shuffle_audit(g3, graph_for(dict(G3_DESIGN, pred_shuffle=True), CFG.seeds[0])[0])
    RAND_AUDIT = pd.concat([RAND_AUDIT, a1.assign(control="rewire", config="G3", seed=CFG.seeds[0]),
                            a2.assign(control="predicate_shuffle", config="G3", seed=CFG.seeds[0])], ignore_index=True)
    RAND_AUDIT.to_csv(RESULTS / "graph_randomization_audit.csv", index=False)
else:
    NOT_RUN.append(dict(stage="V7", experiment_id="G1–G6", reason="không có trong CFG.stages"))
    RANDOM_VALID_G3, SHUFFLE_VALID_G3 = RANDOM_VALID, SHUFFLE_VALID
DEC_DF = pd.DataFrame(DECISIONS)
DEC_DF.to_csv(RESULTS / "graph_stage_decisions.csv", index=False)
VAL_TABLE = pd.DataFrame([dict(experiment_id=k, stage=e["stage"], status=e["status"], description=e["description"],
                               val_MRR=e["val_MRR"][0], val_MRR_std=e["val_MRR"][1], val_R1=e["val_R1"][0],
                               n_seeds=len(e["seeds"]) or (1 if e["status"] == "DONE" else 0), seconds=e.get("seconds"))
                          for k, e in EXPS.items()])
display(VAL_TABLE.round(4))


In [97]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== ĐÓNG BĂNG (trước mọi lần chạm test MSVD / tập ngoài của v4) ==============================
mark_stage("freeze", requires=("V7",))
untouched = not TEST_GATE.log and all(CAPS.log.get((p, s), 0) == 0 for p in ("eval", "report", "display") for s in ("test", "ext"))
FROZEN = dict(version="level4b", model="QC-RGAT", v4=dict(dim=CFG.v4_dim, lr=CFG.v4_lr), v3_final=V3_FINAL, final_experiment=FINAL, final_design=CURRENT, g3_design=G3_DESIGN, g_ids=G_IDS, design_base=DESIGN_BASE,
              rerank_k=CFG.rerank_k, learn_clip_w=CFG.learn_clip_w, seeds=CFG.seeds, split_sha1=SPLIT["sha1"],
              ext_sha1=SPLIT["ext_sha1"], ext_status=EXT_STATUS, inventory_sha1=INV.sha1, schema_sha1=SCHEMA["sha1"],
              clip=f"{CFG.clip_model}/{CFG.clip_weights}", n_frames=POOL_SEL["n_frames"], pooling=POOL_SEL["pooling"],
              tagger=TAG_SELECT2, predicate_vocab=len(PRED_VOCAB), relation_quality=dict(automatic=AUTO_STATUS, human=REL_QUALITY_STATUS),
              verified_cells=sorted(VERIFIED_CELLS) if VERIFIED_CELLS is not None else None,
              random_control_valid=dict(base=RANDOM_VALID, g3=RANDOM_VALID_G3),
              predicate_shuffle_valid=dict(base=SHUFFLE_VALID, g3=SHUFFLE_VALID_G3),
              adoption_rule=dict(min_delta=CFG.adopt_min_delta, max_p_holm=CFG.adopt_max_p, default="phương án đơn giản hơn"),
              confirmatory=PRIMARY_COMPARISONS, holm_alpha=CFG.holm_alpha,
              decisions=DECISIONS, experiments={k: dict(stage=e["stage"], status=e["status"], design=e["design"],
                                                        alpha_lam=e.get("alpha_lam"), val_MRR=e["val_MRR"]) for k, e in EXPS.items()},
              stages_not_run=NOT_RUN, msvd_test_looks_before=BENCHMARK_LOOKS_BEFORE["msvd_test"], benchmark_looks_before=BENCHMARK_LOOKS_BEFORE,
              confirm_set=CFG.confirm_set, eval_query_rule=dict(n_per_group=CFG.n_eval_queries, seed=CFG.eval_query_seed),
              note=("test MSVD đã được nhìn ở v1–v4 ⇒ ở đây chỉ là THĂM DÒ (lần 5), không dùng để chọn; "
                    "xác nhận = MSR-VTT fresh1k-B (1000 video CHƯA dùng ở v1–v4), đánh giá MỘT lần sau đóng băng"))
TEST_GATE.freeze(FROZEN)
check("D1", "mọi lựa chọn của v4 chỉ dựa trên VAL và đóng băng TRƯỚC khi chạm test MSVD / tập ngoài", untouched,
      f"sha1={TEST_GATE.frozen_sha}; final={FINAL}; G3={G_IDS.get('G3')}; quyết định={[(d['stage'], d['candidate'], d['adopted']) for d in DECISIONS]}")

In [98]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b) — BỘ TRUY VẤN ĐĂNG KÝ TRƯỚC (dựng sau đóng băng, TRƯỚC khi chấm bất kỳ mô hình nào) ==============================
mark_stage("eval_queries", requires=("freeze",))
_scored_before = sum(CAPS.log.get(("eval", s_), 0) for s_ in ("val", "test", "ext"))
_G3_GRAPH = graph_for(G3_DESIGN, CFG.seeds[0])[0]
_qs_parts = [("explore", make_part("val" if CFG.dev_mode else "test"))] + ([("confirm", make_part("ext"))] if EXT_RECORDS else [])
QS_ALL, QS_INFO = [], {}
for _k, _p in _qs_parts:
    _df, _inf = build_eval_queries(_p, _G3_GRAPH)
    QS_ALL.append(_df.assign(part_key=_k))
    QS_INFO[_k] = dict(part=_p["name"], **_inf)
QS = pd.concat(QS_ALL, ignore_index=True)
QS.to_csv(RESULTS / "eval_queries_level4b.csv", index=False)
QS_SHA1 = sha1_of(QS[["query_id", "q_idx", "type"]].values.tolist())
save_json(dict(rule=build_eval_queries.__doc__, n_per_group=CFG.n_eval_queries, seed=CFG.eval_query_seed, sha1=QS_SHA1,
               model_scores_used=False, eval_calls_before_build=_scored_before, graph="G3 (đồ thị cuối, cạnh từ caption TRAIN)",
               tags="CLIP zero-shot trên frame của chính video (tagger chọn trên VAL)", info=QS_INFO), RESULTS / "eval_queries_level4b.json")
provenance("bộ truy vấn đánh giá Level 4(b)", "evaluation", "luật cố định: nhãn CLIP của video đúng + cạnh đồ thị train; không dùng điểm mô hình",
           caption_splits=tuple(sorted({p_["split"] for _, p_ in _qs_parts})))
print(f"Bộ truy vấn đăng ký trước (sha1 {QS_SHA1}): {QS_INFO}")
display(QS[["query_id", "type", "text", "gold_video_id", "concepts_seen_in_video", "concepts_missing", "expected_relations"]].head(8))
_main_k = "confirm" if "confirm" in QS_INFO else "explore"
check("QS1", f"bộ truy vấn trực tiếp / gián tiếp ≥ {CFG.n_eval_queries} / nhóm trên tập đánh giá chính, theo luật đăng ký trước, "
             "dựng TRƯỚC khi chấm mô hình (không dùng điểm mô hình)",
      _scored_before == 0 and all(QS_INFO[_main_k][g_]["selected"] >= CFG.n_eval_queries for g_ in ("direct", "indirect")),
      {k_: {g_: v_[g_]["selected"] for g_ in ("direct", "indirect")} for k_, v_ in QS_INFO.items()}, hard=False)

In [99]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== ĐÁNH GIÁ sau đóng băng: test MSVD (THĂM DÒ) + tập ngoài (XÁC NHẬN) ==============================
# dev_mode: test MSVD bị khoá ⇒ phần 'thăm dò' chạy trên VAL (gắn nhãn DEV); tập ngoài = video TRAIN-9k của MSR-VTT (DEV).
mark_stage("evaluate", requires=("freeze",))
EVAL_SPLIT = "val" if CFG.dev_mode else "test"
_, E_G3 = graph_for(G3_DESIGN, CFG.seeds[0])
resolve = lambda x: FINAL if x == "FINAL" else G_IDS.get(x, x)


def eval_part(part, kind):
    """Đánh giá MỌI thí nghiệm đã chạy trên một tập truy vấn; trả về thứ hạng, nhóm truy vấn, bảng metric."""
    CAPS.use(part["q_idx"], "eval")
    clip_rank = full_gallery_ranks(TXT_FEAT, VID_FEAT, part["q_idx"], part["gallery"], part["gold"])
    groups = query_groups_v3(E_G3, part["q_idx"])
    pg = predicate_groups(E_G3, part["q_idx"])
    masks = {"all": np.ones(len(groups), bool), "direct": groups == "direct", "indirect": groups == "indirect",
             "other": groups == "other", "rare_concepts": rare_mask(E_G3, part["q_idx"]),
             "pred_rare": pg == "pred_rare", "pred_common": pg == "pred_common", "no_pred": pg == "no_pred"}
    ev, hashes = {"G0_clip_only": {None: dict(rank=clip_rank)}}, {}
    t0 = time.time()
    for k, e in EXPS.items():
        if e["runs"]:
            ev[k] = evaluate_exp(k, part, clip_rank)
            hashes[k] = {r["cand_hash"] for r in ev[k].values()}
    rows = []
    for k, ev_k in ev.items():
        e = EXPS[k]
        for grp, m in masks.items():
            if m.sum() < 5:
                continue
            ms = [metrics_from_ranks(v["rank"][m], part["gold"][m]) for v in ev_k.values()]
            rows.append(dict(part=part["name"], kind=kind, experiment_id=k, stage=e["stage"], description=e["description"], group=grp,
                             n_queries=int(m.sum()), n_videos=ms[0]["n_videos"], n_seeds=len(ms),
                             **{f"{x}_mean": float(np.mean([mm[x] for mm in ms])) for x in METRIC_KEYS},
                             **{f"{x}_std": float(np.std([mm[x] for mm in ms])) for x in ("R@1", "R@5", "R@10", "MRR", "MRR_video_macro")},
                             val_MRR=e["val_MRR"][0]))
    info = dict(part=part["name"], kind=kind, n_queries=len(part["q_idx"]), n_videos=len(part["gallery"]),
                candidate_recall=float(np.mean(clip_rank <= CFG.rerank_k)), groups=dict(Counter(groups.tolist())),
                predicate_groups=dict(Counter(pg.tolist())), seconds=round(time.time() - t0, 1),
                cand_hashes=sorted(set().union(*hashes.values())) if hashes else [])
    return dict(part=part, ev=ev, masks=masks, rows=pd.DataFrame(rows), info=info)


def rr_of(E, k, m=None):
    rr = np.mean([1.0 / v["rank"] for v in E["ev"][k].values()], 0)
    return rr if m is None else rr[m]


def paired(E, a, b, grp="all"):
    m = E["masks"][grp]
    bs = cluster_bootstrap_diff(rr_of(E, a, m), rr_of(E, b, m), E["part"]["gold"][m])
    return dict(part=E["part"]["name"], kind=E["info"]["kind"], a=a, b=b, group=grp, n=int(m.sum()),
                MRR_a=float(rr_of(E, a, m).mean()), MRR_b=float(rr_of(E, b, m).mean()), **bs)


EVALS = {}
EVALS["explore"] = eval_part(make_part(EVAL_SPLIT), "DEV_val" if CFG.dev_mode else "exploratory_test_5th_look")
if EXT_RECORDS:
    EVALS["confirm"] = eval_part(make_part("ext"), "DEV_ext_train9k" if EXT_REPORT["dev"] else
                                  ("confirmatory" if EXT_REPORT.get("set") == "fresh1kB" else "exploratory_seen_benchmark"))
    EVALS["confirm_std"] = eval_part(make_part("ext", std_only=True), "DEV_ext_train9k_std" if EXT_REPORT["dev"] else
                                      ("confirmatory_secondary" if EXT_REPORT.get("set") == "fresh1kB" else "exploratory_seen_benchmark_std"))
for k_, E in EVALS.items():
    print(f"[{k_}] {E['info']}")
check("D2", "mọi thí nghiệm dùng chung split, gallery, truy vấn, tập ứng viên CLIP và hàm metric (từng tập đánh giá)",
      all(len(E["info"]["cand_hashes"]) == 1 for E in EVALS.values()), {k_: E["info"]["cand_hashes"] for k_, E in EVALS.items()})

COMPARE = []
for d in DECISIONS:
    if d.get("adopted") is not None and d["candidate"] in EXPS and EXPS[d["candidate"]]["runs"] and EXPS[d["incumbent"]].get("runs") is not None:
        COMPARE.append((f"{d['stage']}_variant", d["candidate"], d["incumbent"]))
for gid in ("G1", "G2", "G4", "G5", "G6", "G7", "G8", "V3REF"):
    if gid in G_IDS and EXPS.get(G_IDS[gid], {}).get("runs"):
        COMPARE.append(("G3_vs_control", G_IDS["G3"], G_IDS[gid]))
if "G3" in G_IDS:
    COMPARE.append(("G3_vs_clip", G_IDS["G3"], "G0_clip_only"))       # so sánh PHỤ theo nhóm trực tiếp / gián tiếp
SIG_ROWS, CONF_ROWS = [], []
for k_, E in EVALS.items():
    for c, a, b in COMPARE:
        if a in E["ev"] and b in E["ev"]:
            for g_ in ("all", "direct", "indirect", "rare_concepts", "pred_rare", "pred_common"):
                if E["masks"][g_].sum() >= 5:
                    SIG_ROWS.append(dict(comparison=c, **paired(E, a, b, g_)))
    fam = [(h, resolve(a), resolve(b)) for h, a, b in PRIMARY_COMPARISONS if not (h == "H0_final_vs_clip" and FINAL == G_IDS.get("G3"))]
    ok = [(h, a, b) for h, a, b in fam if a in E["ev"] and b in E["ev"]]
    res = [dict(hypothesis=h, **paired(E, a, b)) for h, a, b in ok]
    adj = holm([r["p_not_better"] for r in res])
    for r, pa in zip(res, adj):
        r.update(p_holm=float(pa), supported=bool(pa < CFG.holm_alpha and r["diff"] > 0),
                 status="CONFIRMATORY" if E["info"]["kind"] == "confirmatory" else
                 ("EXPLORATORY (benchmark đã được nhìn trước đây)" if E["info"]["kind"].startswith("exploratory") else "DEV"))
        CONF_ROWS.append(r)
    for h, a, b in fam:
        if (h, a, b) not in ok:
            CONF_ROWS.append(dict(hypothesis=h, part=E["part"]["name"], kind=E["info"]["kind"], a=a, b=b, status="NOT RUN",
                                  note=f"{a if a not in E['ev'] else b}: {EXPS.get(a if a not in E['ev'] else b, {}).get('reason', 'không có')}"))
if not EXT_RECORDS:
    for h, a, b in PRIMARY_COMPARISONS:
        CONF_ROWS.append(dict(hypothesis=h, part="ext", kind="confirmatory", a=resolve(a), b=resolve(b), status="NOT RUN", note=EXT_STATUS))
SIG_T = pd.DataFrame(SIG_ROWS)
CONF_T = pd.DataFrame(CONF_ROWS)
RES_ALL = pd.concat([E["rows"] for E in EVALS.values()], ignore_index=True)
RES_ALL.to_csv(RESULTS / "graph_experiment_results.csv", index=False)
SIG_T.to_csv(RESULTS / "graph_paired_tests.csv", index=False)
CONF_T.to_csv(RESULTS / "graph_confirmatory_tests.csv", index=False)
save_json({k_: E["info"] for k_, E in EVALS.items()}, RESULTS / "graph_eval_parts.json")
show = ["part", "experiment_id", "n_seeds", "R@1_mean", "R@5_mean", "R@10_mean", "MRR_mean", "MRR_std", "MRR_video_macro_mean", "val_MRR"]
print("\nKẾT QUẢ (nhóm 'all'):")
display(RES_ALL[(RES_ALL.group == "all") & RES_ALL.experiment_id.isin(list(G_IDS.values()) + [FINAL])][show].round(4))
print("Họ so sánh chính (một phía, Holm):")
display(CONF_T[[c for c in ("hypothesis", "part", "status", "a", "b", "MRR_a", "MRR_b", "diff", "ci_lo", "ci_hi", "p_not_better",
                            "p_holm", "supported") if c in CONF_T]].round(4))
TAGGER_REPORT = {k_: tagger_report_v2(TAGS2, SPLIT[E["part"]["split"]], VOCAB2, "report")
                 for k_, E in EVALS.items() if k_ != "confirm_std"}
print("Tagger trên tập đánh giá:", TAGGER_REPORT)
check("D3", "R@1/5/10, MRR, macro-MRR theo video + kiểm định cặp cho G0–G6 + v3 trên tập xác nhận độc lập",
      "confirm" in EVALS and EVALS["confirm"]["info"]["kind"] == "confirmatory",
      f"{len(RES_ALL)} dòng → graph_experiment_results.csv | xác nhận: {EXT_STATUS}", hard=False,
      status=None if ("confirm" in EVALS and EVALS["confirm"]["info"]["kind"] == "confirmatory") else "NOT RUN")

In [100]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Phân tích quan hệ v4: knockout theo họ, attention theo vị từ / vai ==============================
mark_stage("relations", requires=("evaluate",))
P_MAIN = EVALS["confirm"]["part"] if "confirm" in EVALS else EVALS["explore"]["part"]
G3_ID = G_IDS["G3"]
RA_ROWS, RA_SUMM = [], []
for part in (PV, P_MAIN):
    df_, s_ = relation_analysis_v4(EXPS[G3_ID]["runs"][CFG.seeds[0]], part)
    RA_ROWS.append(df_.assign(experiment_id=G3_ID)); RA_SUMM.append(dict(experiment_id=G3_ID, split=part["name"], **s_))
REL_DF = pd.concat(RA_ROWS, ignore_index=True)
REL_DF.to_csv(RESULTS / "graph_relation_analysis.csv", index=False)
save_json(RA_SUMM, RESULTS / "graph_relation_analysis_summary.json")
display(REL_DF[REL_DF.split == P_MAIN["name"]].round(4))
for s_ in RA_SUMM:
    print(f"{s_['split']:8s} MRR {s_['base_MRR']:.4f} | bỏ MỌI cạnh quan hệ lúc suy luận ⇒ {s_['all_relations_knockout_MRR']:.4f} | "
          f"attention cạnh có vị từ KHỚP truy vấn {s_['att_pred_match']:.3f} vs không khớp {s_['att_pred_nomatch']:.3f}")



In [101]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== xAI v4: giải thích = số hạng của điểm; fidelity bằng xoá cạnh THẬT ==============================
mark_stage("xai", requires=("relations",))
RUN_F = EXPS[G3_ID]["runs"][CFG.seeds[0]]
GROUPS_X = query_groups_v3(RUN_F["engine"], P_MAIN["q_idx"])
XAI_DF = xai_v4(RUN_F, P_MAIN, GROUPS_X)
XAI_DF.to_csv(RESULTS / "graph_xai_fidelity.csv", index=False)
SENS = []
if len(CFG.seeds) > 1 and len(XAI_DF):
    for _, r in XAI_DF.head(CFG.n_seed_sensitivity).iterrows():
        sets = [{tuple(t["edges"]) for t in explain_v4(EXPS[G3_ID]["runs"][s]["model"], int(r.q), int(r.gold))["top_relation"]}
                for s in CFG.seeds]
        js = [len(a & b) / max(1, len(a | b)) for a, b in itertools.combinations(sets, 2)]
        SENS.append(dict(q=int(r.q), gold=int(r.gold), mean_jaccard=float(np.mean(js))))
SENS_DF = pd.DataFrame(SENS)
SENS_DF.to_csv(RESULTS / "graph_xai_seed_sensitivity.csv", index=False)
_f = lambda d, c: float(d[c].mean()) if len(d) and c in d else float("nan")
XAI_SUMM = dict(experiment_id=G3_ID, part=P_MAIN["name"], n_queries=len(XAI_DF), validity=_f(XAI_DF, "validity"),
                predicate_validity=_f(XAI_DF, "predicate_validity"), predicate_edges=int(XAI_DF.n_predicate_edges.sum()) if len(XAI_DF) else 0,
                sparsity=_f(XAI_DF, "sparsity"), share_relation=_f(XAI_DF, "share_relation"),
                decomposition_ok=_f(XAI_DF, "decomposition_ok"), deletion_matches_decomposition=_f(XAI_DF, "deletion_matches_decomposition"),
                d_s_expl=_f(XAI_DF, "d_s_expl"), d_s_rand=_f(XAI_DF, "d_s_rand"),
                frac_score_decreased_expl=float((XAI_DF.d_s_expl < -1e-9).mean()) if len(XAI_DF) else float("nan"),
                rank_full=_f(XAI_DF, "rank_full"), rank_drop_expl=_f(XAI_DF, "rank_drop_expl"), rank_drop_rand=_f(XAI_DF, "rank_drop_rand"),
                frac_rank_changed_expl=float((XAI_DF.rank_drop_expl != XAI_DF.rank_full).mean()) if len(XAI_DF) else float("nan"),
                frac_rank_changed_rand=float((XAI_DF.rank_drop_rand != XAI_DF.rank_full).mean()) if len(XAI_DF) else float("nan"),
                frac_top1_changed_expl=_f(XAI_DF, "top1_changed_expl"), frac_top10_changed_expl=_f(XAI_DF, "top10_changed_expl"),
                seed_sensitivity_mean_jaccard=_f(SENS_DF, "mean_jaccard"), alpha=RUN_F["alpha"])
save_json(XAI_SUMM, RESULTS / "graph_xai_summary.json")
print(json.dumps(XAI_SUMM, indent=1, ensure_ascii=False))
if len(XAI_DF):
    for _, r in XAI_DF.head(3).iterrows():
        print(f"  q={r.q} (nhóm {r.group}) hạng {r.rank_full}→{r.rank_drop_expl} khi xoá giải thích: {r.explanation[:300]}")
check("X1", "[12.9] giải thích hợp lệ (cạnh thật đúng loại/chiều; vị từ khớp nằm trên ĐÚNG cạnh); Σ số hạng = điểm; ≥ 10 truy vấn",
      len(XAI_DF) >= 10 and XAI_SUMM["validity"] == 1.0 and XAI_SUMM["predicate_validity"] == 1.0 and XAI_SUMM["decomposition_ok"] == 1.0,
      XAI_SUMM, hard=False)
check("X2", "xoá cạnh của các đường đi ỦNG HỘ (giải thích) làm điểm mô hình giảm nhiều hơn xoá cùng số cạnh quan hệ ngẫu nhiên (vào nhãn của cùng video)",
      XAI_SUMM["d_s_expl"] < XAI_SUMM["d_s_rand"], f"{XAI_SUMM['d_s_expl']:.4f} vs {XAI_SUMM['d_s_rand']:.4f}", hard=False)
check("X3", "T = 1: Δ điểm khi xoá cạnh = đúng tổng đóng góp đã báo (fidelity chính xác theo cấu tạo)",
      RUN_F["model"].T != 1 or XAI_SUMM["deletion_matches_decomposition"] == 1.0, XAI_SUMM["deletion_matches_decomposition"], hard=False)
# X4 (thứ hạng có đổi khi xoá giải thích?) là KẾT QUẢ khoa học, không phải kiểm thử phần mềm ⇒ báo ở FINDING F-X4 (ô xAI theo thứ hạng).


In [102]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b) — METRIC TRÊN BỘ TRUY VẤN ĐĂNG KÝ TRƯỚC + nhóm trực tiếp / gián tiếp ==============================
mark_stage("eval_queries_metrics", requires=("evaluate", "eval_queries"))
QM, QP = [], []
for _k, E in EVALS.items():
    _q = QS[QS.part_key == _k] if _k in ("explore", "confirm") else QS.iloc[:0]
    if len(_q):
        _a, _b = eval_query_metrics(E, _q)
        QM.append(_a); QP.append(_b)
QS_METRICS = pd.concat(QM, ignore_index=True) if QM else pd.DataFrame()
QS_PAIRED = pd.concat(QP, ignore_index=True) if QP else pd.DataFrame()
QS_METRICS.to_csv(RESULTS / "eval_queries_metrics.csv", index=False)
QS_PAIRED.to_csv(RESULTS / "eval_queries_paired.csv", index=False)
_show = [G_IDS[g_] for g_ in ("G0", "G1", "G3", "G7", "G8", "V3REF") if g_ in G_IDS]
for _k in ("explore", "confirm"):
    if _k in EVALS and len(QS_METRICS):
        _t = QS_METRICS[(QS_METRICS.part == EVALS[_k]["part"]["name"]) & QS_METRICS.experiment_id.isin(_show)]
        print(f"\n[{_k}] bộ truy vấn đăng ký trước — MRR (trung bình theo seed):")
        display(_t.pivot_table(index="experiment_id", columns="group", values="MRR_mean").round(4))
if len(QS_PAIRED):
    display(QS_PAIRED[["part", "group", "a", "b", "n", "MRR_a", "MRR_b", "diff", "ci_lo", "ci_hi"]].round(4))
_mk = "confirm" if "confirm" in EVALS else "explore"
_sig = SIG_T[(SIG_T.comparison == "G3_vs_clip") & (SIG_T.part == EVALS[_mk]["part"]["name"])] if len(SIG_T) else SIG_T
for _, r in _sig[_sig.group.isin(["all", "direct", "indirect"])].iterrows():
    finding(f"F-GRP-{r.group}", f"QC-RGAT (G3) vs CLIP-only trên nhóm '{r.group}' của {r.part} (mọi caption; so sánh PHỤ, không Holm)",
            f"n={r.n}: ΔMRR {r['diff']:+.4f} [CI95 theo video {r.ci_lo:+.4f}, {r.ci_hi:+.4f}]",
            "đồ thị giúp nhóm này" if r.ci_lo > 0 else ("đồ thị làm kém nhóm này" if r.ci_hi < 0 else "không khác biệt có ý nghĩa"))
_qi = QS_PAIRED[(QS_PAIRED.group == "indirect") & (QS_PAIRED.a == G_IDS.get("G3")) & (QS_PAIRED.part == EVALS[_mk]["part"]["name"])] \
    if len(QS_PAIRED) else QS_PAIRED
if len(_qi):
    r = _qi.iloc[0]
    finding("F-QS-indirect", f"G3 vs CLIP-only trên {r.n} truy vấn GIÁN TIẾP đăng ký trước ({r.part})",
            f"MRR {r.MRR_b:.4f} → {r.MRR_a:.4f} (Δ {r['diff']:+.4f}, CI95 [{r.ci_lo:+.4f}, {r.ci_hi:+.4f}])",
            "n nhỏ ⇒ CI rộng; chỉ là bằng chứng bổ sung cho kết quả trên mọi caption")

In [103]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b) — xAI theo THỨ HẠNG: giải thích đầy đủ (contains + đường đi quan hệ) ==============================
mark_stage("xai_rank", requires=("xai",))
XR = xai_rank_fidelity(RUN_F, P_MAIN)
XR.to_csv(RESULTS / "xai_rank_fidelity.csv", index=False)
_m = lambda c: float(XR[c].mean()) if len(XR) else float("nan")
XR_SUMM = dict(experiment_id=G3_ID, part=P_MAIN["name"], n_queries=len(XR), top_m=CFG.top_m_paths, alpha=RUN_F["alpha"],
               validity=_m("validity"), predicate_validity=_m("predicate_validity"), decomposition_ok=_m("decomposition_ok"),
               size_edges=_m("n_expl_edges"), sparsity=_m("sparsity"), coverage=_m("coverage"),
               expl_contains_terms=_m("n_expl_contains"), expl_relation_terms=_m("n_expl_relation"),
               d_s_expl=_m("d_s_expl"), d_s_rand=_m("d_s_rand"), d_final_expl=_m("d_final_expl"), d_final_rand=_m("d_final_rand"),
               rank_full=_m("rank_full"), rank_drop_expl=_m("rank_drop_expl"), rank_drop_rand=_m("rank_drop_rand"),
               frac_rank_worse_expl=_m("rank_worse_expl"), frac_rank_worse_rand=_m("rank_worse_rand"),
               frac_top1_changed_expl=_m("top1_changed_expl"), frac_top1_changed_rand=_m("top1_changed_rand"),
               frac_top10_changed_expl=_m("top10_changed_expl"), frac_top10_changed_rand=_m("top10_changed_rand"),
               share_graph=_m("share_graph"), share_relation=_m("share_relation"),
               definitions=dict(explanation="top-m số hạng ĐỒ THỊ có đóng góp dương (contains: video –chứa→ nhãn; quan hệ: u –r→ nhãn ←chứa– video)",
                                validity="tỉ lệ số hạng giải thích mà MỌI cạnh có thật trong đồ thị đang chạy, đúng loại, chiều, kiểu node",
                                sparsity="số hạng giải thích / số hạng đồ thị khác 0 của video (nhỏ = gọn)", size_edges="số cạnh bị xoá",
                                coverage="Σ đóng góp của giải thích / Σ đóng góp đồ thị dương",
                                fidelity="xoá THẬT các cạnh của giải thích rồi chấm lại mọi ứng viên: Δ điểm đồ thị, Δ điểm cuối, Δ hạng, top-1/top-10; "
                                         "đối chứng = xoá cùng số cạnh contains khác + cạnh quan hệ ngẫu nhiên vào nhãn của cùng video"))
save_json(XR_SUMM, RESULTS / "xai_level4b_summary.json")
print(json.dumps({k_: (round(v_, 4) if isinstance(v_, float) else v_) for k_, v_ in XR_SUMM.items() if k_ != "definitions"}, ensure_ascii=False))
if len(XR):
    display(XR[["gold_video_id", "rank_full", "rank_drop_expl", "rank_drop_rand", "d_s_expl", "d_s_rand", "d_final_expl", "explanation"]].head(6))
check("X5", "giải thích đầy đủ hợp lệ (mọi cạnh contains / quan hệ có thật, đúng loại, chiều, kiểu node; vị từ đúng cạnh); "
            "Σ số hạng = điểm; ≥ 10 truy vấn", len(XR) >= 10 and XR_SUMM["validity"] == 1.0 and XR_SUMM["predicate_validity"] == 1.0
      and XR_SUMM["decomposition_ok"] == 1.0, {k_: XR_SUMM[k_] for k_ in ("n_queries", "validity", "predicate_validity", "decomposition_ok")},
      hard=False)
finding("F-X4", "xoá các đường đi QUAN HỆ của giải thích (v4) có đổi thứ hạng video đúng?",
        f"{XAI_SUMM['n_queries']} truy vấn: hạng đổi {XAI_SUMM['frac_rank_changed_expl']:.0%} (xoá ngẫu nhiên {XAI_SUMM['frac_rank_changed_rand']:.0%}); "
        f"Δ điểm đồ thị {XAI_SUMM['d_s_expl']:+.4f} vs ngẫu nhiên {XAI_SUMM['d_s_rand']:+.4f}",
        "điểm giảm đúng như phân rã nhưng quyết định xếp hạng KHÔNG đổi ⇒ quan hệ không phải nguyên nhân của thứ hạng"
        if XAI_SUMM["frac_rank_changed_expl"] == 0 else "xoá quan hệ của giải thích có làm đổi thứ hạng ở một phần truy vấn")
_better = XR_SUMM["frac_rank_worse_expl"] > XR_SUMM["frac_rank_worse_rand"] and XR_SUMM["d_final_expl"] < XR_SUMM["d_final_rand"]
finding("F-X6", "xoá giải thích ĐẦY ĐỦ (contains + quan hệ) có làm video đúng tụt hạng nhiều hơn xoá ngẫu nhiên cùng số cạnh?",
        f"{XR_SUMM['n_queries']} truy vấn: tụt hạng {XR_SUMM['frac_rank_worse_expl']:.0%} vs {XR_SUMM['frac_rank_worse_rand']:.0%}; "
        f"hạng TB {XR_SUMM['rank_full']:.2f} → {XR_SUMM['rank_drop_expl']:.2f} (ngẫu nhiên {XR_SUMM['rank_drop_rand']:.2f}); "
        f"Δ điểm cuối {XR_SUMM['d_final_expl']:+.3f} vs {XR_SUMM['d_final_rand']:+.3f}; top-1 đổi {XR_SUMM['frac_top1_changed_expl']:.0%}",
        "giải thích chỉ ra đúng phần đồ thị quyết định thứ hạng (mạnh hơn ngẫu nhiên)" if _better else
        "KHÔNG mạnh hơn xoá ngẫu nhiên ⇒ không kết luận được giải thích phản ánh nguyên nhân của thứ hạng")

In [104]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b) — 5 THÀNH CÔNG + 5 THẤT BẠI (chọn theo luật, có đường đi thật + nguyên nhân) ==============================
mark_stage("cases", requires=("xai_rank",))
CASES, CASE_STATS = case_analysis_l4b(RUN_F, P_MAIN)
save_json(dict(experiment_id=G3_ID, part=P_MAIN["name"], alpha=RUN_F["alpha"], stats=CASE_STATS, cases=CASES), RESULTS / "case_analysis.json")
(RESULTS / "case_analysis.md").write_text(cases_markdown(CASES, CASE_STATS, P_MAIN["name"]))
for c in CASES:
    print(f"[{c['kind']}] hạng CLIP {c['rank_clip']} → {c['rank_final']} | {c['query'][:70]} | nguyên nhân: {', '.join(c['causes']) or '—'}")
    for x in c["gold_explanation"][:3]:
        print(f"      • {x}")
_ns, _nf = sum(c["kind"] == "THÀNH CÔNG" for c in CASES), sum(c["kind"] == "THẤT BẠI" for c in CASES)
check("K1", f"phân tích ≥ {CFG.n_cases} thành công + ≥ {CFG.n_cases} thất bại, chọn theo luật, có đường đi thật, thứ hạng và nguyên nhân",
      _ns >= CFG.n_cases and _nf >= CFG.n_cases,
      f"{_ns} thành công, {_nf} thất bại; nguyên nhân có mặt: {sorted({x for c in CASES for x in c['causes']})}", hard=False)

In [105]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b) — DEMO: văn bản tự do → top-K video + điểm CLIP / đồ thị / cuối + giải thích ==============================
mark_stage("demo", requires=("cases",))
DEMO_OUT = []
for _i, _t in enumerate(CFG.demo_queries, 1):
    DEMO_OUT.append(demo_query(_t, save_png=RESULTS / f"demo_{_i}.png"))
for _kind in ("THÀNH CÔNG", "THẤT BẠI"):                     # một ca có đáp án mỗi loại (văn bản của tập đánh giá, chỉ hiển thị)
    c = next((c for c in CASES if c["kind"] == _kind), None)
    if c is not None:
        _o = demo_query(c["query"], save_png=RESULTS / f"demo_case_{'ok' if _kind == 'THÀNH CÔNG' else 'fail'}.png")
        _o.update(gold_video_id=c["gold_video_id"], case_kind=_kind)
        DEMO_OUT.append(_o)
save_json(DEMO_OUT, RESULTS / "demo_outputs.json")
_ok = all(len(o["results"]) == CFG.demo_topk and all(r["explanation"] or r["note"] for r in o["results"]) for o in DEMO_OUT)
check("DM1", "demo: văn bản tự do → top-k video + điểm CLIP / đồ thị / cuối + giải thích (hoặc báo rõ không có đường đi hợp lệ)",
      _ok and len(DEMO_OUT) >= len(CFG.demo_queries), f"{len(DEMO_OUT)} truy vấn demo → demo_outputs.json, demo_*.png", hard=False)
print("\nThử truy vấn khác: demo_query('a man is riding a horse')  |  ô nhập tương tác: demo_widget()")

In [106]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== XUẤT KẾT QUẢ THEO CÂU HỎI + MANIFEST ==============================
mark_stage("export", requires=("xai",))
PARTS = [k_ for k_ in ("explore", "confirm", "confirm_std") if k_ in EVALS]


def result_table(ids, groups=("all", "direct", "indirect")):
    """Một dòng / thí nghiệm: thiết kế, trạng thái, VAL, và metric trên từng tập đánh giá (thăm dò / xác nhận) theo nhóm."""
    rows = []
    label = {v: k for k, v in G_IDS.items()}
    for k in dict.fromkeys(ids):
        if k not in EXPS:
            continue
        e = EXPS[k]
        base = dict(experiment_id=k, control=label.get(k, ""), is_final=k == FINAL, stage=e["stage"], status=e["status"],
                    reason=e.get("reason", ""), description=e["description"], hypothesis=e.get("hypothesis", ""),
                    design=json.dumps(e["design"], ensure_ascii=False), n_seeds=len(e["seeds"]),
                    edges=json.dumps(e.get("edges", {})), n_params=e.get("n_params"), runtime_s=e.get("seconds"),
                    gpu_peak_mb=e.get("gpu_peak_mb"), val_MRR=e["val_MRR"][0], val_MRR_std=e["val_MRR"][1], val_R1=e["val_R1"][0])
        for pk in PARTS:
            sub = EVALS[pk]["rows"]
            sub = sub[sub.experiment_id == k]
            for grp in groups:
                r = sub[sub.group == grp]
                if len(r):
                    r = r.iloc[0]
                    tag = f"{pk}_{grp}"
                    base.update({f"{tag}_{x}": r[f"{x}_mean"] for x in ("R@1", "R@5", "R@10", "MRR", "MRR_video_macro")})
                    base[f"{tag}_MRR_std"] = r["MRR_std"]
        rows.append(base)
    return pd.DataFrame(rows)


G_ORDER = [G_IDS[g] for g in ("G0", "G1", "G2", "G3", "G4", "G5", "G6", "G7", "G8", "V3REF") if g in G_IDS]
ABL_DF = result_table(G_ORDER + [FINAL] + [k for k, e in EXPS.items() if e["stage"] == "V1"],
                      ("all", "direct", "indirect", "pred_rare", "pred_common", "no_pred"))
ABL_DF.to_csv(RESULTS / "graph_ablation_results.csv", index=False)

MANIFEST = []
for k, e in EXPS.items():
    MANIFEST.append(dict(experiment_id=k, control={v: g for g, v in G_IDS.items()}.get(k, ""), stage=e["stage"], status=e["status"],
                         reason=e.get("reason"), description=e["description"], hypothesis=e.get("hypothesis", ""), design=e["design"],
                         seeds=e["seeds"], reuse=e.get("reuse"), lam_fixed=e.get("lam_fixed"),
                         checkpoints=[r.get("checkpoint") for r in e["runs"].values()], edges_by_type=e.get("edges"),
                         n_nodes=e.get("n_nodes"), n_params=e.get("n_params"), alpha_lam=e.get("alpha_lam"),
                         val=dict(MRR=e["val_MRR"], R1=e["val_R1"]), runtime_s=e.get("seconds"), gpu_peak_mb=e.get("gpu_peak_mb"),
                         eval={pk: {r_.group: dict(MRR=r_["MRR_mean"], MRR_std=r_["MRR_std"], R1=r_["R@1_mean"])
                                    for _, r_ in EVALS[pk]["rows"][EVALS[pk]["rows"].experiment_id == k].iterrows()} for pk in PARTS},
                         per_seed={str(s): dict(val=r["val"], alpha=r["alpha"], lam=r["lam"], seconds=r["seconds"],
                                                final_loss=r["hist"][-1] if r["hist"] else None) for s, r in e["runs"].items()}))
save_json(dict(version="level4b", final=FINAL, benchmark_status=benchmark_status(), g_ids=G_IDS, rerank_k=CFG.rerank_k, eval_parts={k_: E["info"] for k_, E in EVALS.items()},
               candidate_recall_val=CAND_RECALL_VAL, not_run=NOT_RUN, primary_comparisons=PRIMARY_COMPARISONS,
               experiments=MANIFEST, env=ENV_INFO), RESULTS / "experiment_manifest.json")
PROV = pd.DataFrame(PROVENANCE)
PROV.to_csv(RESULTS / "provenance.csv", index=False)
ACCESS = CAPS.access_table()
ACCESS.to_csv(RESULTS / "caption_access_log.csv", index=False)
bad_access = ACCESS[ACCESS.split.isin(["test", "ext"]) & ~ACCESS.purpose.isin(["encode", "query_input", "eval", "display", "report"])]
check("C8", "[12.8] nhật ký: caption TEST / tập ngoài chỉ dùng cho encode / query_input / eval / report", bad_access.empty,
      {f"{r.purpose}:{r.split}": int(r.n) for r in ACCESS.itertuples()})
bad_prov = PROV[((PROV.kind == "learned") & PROV.caption_splits.str.contains("val|test|ext"))
                | ((PROV.kind == "selected") & PROV.caption_splits.str.contains("test|ext"))]
check("C9", "[12.8] provenance: artifact học được chỉ từ caption TRAIN; lựa chọn chỉ từ VAL", bad_prov.empty, f"{len(PROV)} artifact")
save_json(TEST_GATE.log, RESULTS / "test_access_log.json")
first = min((x["t"] for x in TEST_GATE.log), default=None)
check("D4", "mọi lần chạm test MSVD / tập ngoài sau thời điểm đóng băng", first is None or first >= TEST_GATE.frozen_at,
      f"{len(TEST_GATE.log)} lần ({dict(Counter(x['split'] for x in TEST_GATE.log))})")
r_ = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
(RESULTS / "pip_freeze.txt").write_text(r_.stdout if r_.returncode == 0 else r_.stderr[-2000:])
save_json(dict(ENV_INFO, cfg=cfg_snapshot(), stages=STAGES), RESULTS / "env_versions.json")

In [107]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== BÁO CÁO AUDIT + KẾT LUẬN v4 (tự sinh từ số liệu) ==============================
mark_stage("report", requires=("export",))
official = CFG.dataset == "msvd" and not CFG.dev_mode
_by = {c["id"]: c for c in CHECKS}


def _st(*ids):
    rows = [_by[i] for i in ids if i in _by]
    if not rows:
        return "NOT RUN", "không chạy"
    st = "FAIL" if any(r["status"] == "FAIL" for r in rows) else ("NOT RUN" if any(r["status"] == "NOT RUN" for r in rows) else "PASS")
    return st, " | ".join(f"{r['id']}={r['status']}" for r in rows)


def _verdict(lo, hi):
    return "TỐT HƠN có ý nghĩa" if lo > 0 else ("KÉM HƠN có ý nghĩa" if hi < 0 else "không khác biệt có ý nghĩa")


def _conf(hyp, part_key):
    if part_key not in EVALS:
        return "NOT RUN" + (f" ({EXT_STATUS})" if part_key.startswith("confirm") else "")
    r = CONF_T[(CONF_T.hypothesis == hyp) & (CONF_T.part == EVALS[part_key]["part"]["name"])]
    if r.empty or r.iloc[0].get("status") == "NOT RUN":
        return "NOT RUN" + (f" — {r.iloc[0].get('note', '')}" if len(r) else "")
    r = r.iloc[0]
    return (f"{r.a} vs {r.b}: ΔMRR {r['diff']:+.4f} (CI95 theo video [{r.ci_lo:+.4f}, {r.ci_hi:+.4f}], p Holm {r.p_holm:.3f}) "
            f"⇒ {_verdict(r.ci_lo, r.ci_hi)}{' — ỦNG HỘ giả thuyết' if r.supported else ''}")


def _val(a, b):
    if not (EXPS.get(a, {}).get("runs") or a == "G0_clip_only") or not (EXPS.get(b, {}).get("runs") or b == "G0_clip_only"):
        return "NOT RUN"
    bs = cluster_bootstrap_diff(EXPS[a]["rr_val"], EXPS[b]["rr_val"], PV["gold"])
    return f"{a} vs {b} (VAL): ΔMRR {bs['diff']:+.4f} [{bs['ci_lo']:+.4f}, {bs['ci_hi']:+.4f}]"


CK = "confirm" if "confirm" in EVALS else None
EX = "explore"
L = []
if not official:
    L.append(f"⚠ KHÔNG phải lần chạy chính thức (dataset={CFG.dataset}, dev_mode={CFG.dev_mode}) — số liệu chỉ để kiểm tra mã.")
L.append(f"⚠ Khai báo: test MSVD đã được nhìn ở v1–v4 ⇒ số test MSVD ở đây là THĂM DÒ (lần nhìn thứ 5). Xác nhận độc lập: "
         f"{EXT_STATUS} ({EXT_REPORT.get('label', '—')}).")
L.append(f"Mô hình v4 = QC-RGAT ({CFG.v4_dim} chiều, {CURRENT['v4_layers']} lớp), đồ thị = cấu hình cuối của v3 "
         f"({json.dumps({k: V3_FINAL[k] for k in ('taxonomy', 'mask', 'rel_policy', 'contains')}, ensure_ascii=False)}); k = {CFG.rerank_k}; "
         f"chọn số lớp trên VAL: {FINAL}.")
L.append(f"Chất lượng quan hệ — {AUTO_STATUS}")
HYPS = (("H1_graph_vs_clip", "v4 (G3) vs CLIP-only (G0)"), ("H2_relations_vs_contains_only", "Quan hệ vs chỉ contains (G3 vs G1)"),
        ("H3_predicate_vs_family_only", "Khớp vị từ vs không vị từ (G3 vs G2)"),
        ("H4_true_vs_shuffled_predicates", "Vị từ đúng cạnh vs hoán vị (G3 vs G4)"),
        ("H5_true_vs_rewired_topology", "Topology thật vs rewiring có ràng buộc (G3 vs G5)"),
        ("H6_query_conditioned_vs_static", "Attention theo truy vấn vs tĩnh (G3 vs G6)"),
        ("H7_v4_vs_v3", "QC-RGAT vs GraphSAGE có kiểu cạnh (G3 vs R3)"),
        ("H8_role_edges", "Họ cạnh vai agent_of / object_of (G3 vs G7 huấn luyện lại)"),
        ("H9_svo_prep_edges", "Họ cạnh svo / prep (G3 vs G8 huấn luyện lại)"))
for hyp, txt in HYPS:
    a, b = PRIMARY_COMPARISONS[[h for h, _, _ in PRIMARY_COMPARISONS].index(hyp)][1:]
    L.append(f"{hyp} — {txt}:")
    L.append(f"     VAL (phát triển): {_val(resolve(a), resolve(b))}")
    L.append(f"     {'test MSVD (THĂM DÒ)' if not CFG.dev_mode else 'VAL lặp lại (DEV)'}: {_conf(hyp, EX)}")
    L.append(f"     tập ngoài {'(XÁC NHẬN)' if CK and EVALS[CK]['info']['kind'] == 'confirmatory' else '(DEV)' if CK else ''}: "
             f"{_conf(hyp, CK) if CK else 'NOT RUN — ' + EXT_STATUS}")
L.append(f"Đối chứng (dưới cấu hình G3): rewiring hợp lệ: {RANDOM_VALID_G3}; hoán vị vị từ hợp lệ: {SHUFFLE_VALID_G3}. "
         "Đối chứng không hợp lệ ⇒ KHÔNG rút kết luận tương ứng.")
for s_ in RA_SUMM:
    L.append(f"Phân tích quan hệ ({s_['split']}): bỏ mọi cạnh quan hệ lúc suy luận: MRR {s_['base_MRR']:.4f} → "
             f"{s_['all_relations_knockout_MRR']:.4f}; attention cạnh có vị từ khớp truy vấn {s_['att_pred_match']:.3f} vs không khớp "
             f"{s_['att_pred_nomatch']:.3f}.")
L.append(f"Giải thích ({XAI_SUMM['experiment_id']}, {XAI_SUMM['part']}, {XAI_SUMM['n_queries']} truy vấn): hợp lệ {XAI_SUMM['validity']:.2f}, "
         f"vị từ đúng cạnh {XAI_SUMM['predicate_validity']:.2f}, Σ số hạng = điểm {XAI_SUMM['decomposition_ok']:.2f}, xoá cạnh khớp phân rã "
         f"{XAI_SUMM['deletion_matches_decomposition']:.2f}; Δ điểm khi xoá giải thích {XAI_SUMM['d_s_expl']:+.4f} vs ngẫu nhiên "
         f"{XAI_SUMM['d_s_rand']:+.4f}; hạng đổi {XAI_SUMM['frac_rank_changed_expl']:.0%} (ngẫu nhiên {XAI_SUMM['frac_rank_changed_rand']:.0%}); "
         f"phần điểm từ quan hệ {XAI_SUMM['share_relation']:.1%}; Jaccard theo seed {XAI_SUMM['seed_sensitivity_mean_jaccard']:.2f}.")
conf_sup = {r.hypothesis: bool(r.supported) for r in CONF_T.itertuples()
            if CK and r.part == EVALS[CK]["part"]["name"] and getattr(r, "status", "") == "CONFIRMATORY"}
REL_KEYS = ("H2_relations_vs_contains_only", "H4_true_vs_shuffled_predicates", "H5_true_vs_rewired_topology")
if conf_sup and all(conf_sup.get(h) for h in REL_KEYS) and RANDOM_VALID_G3 and SHUFFLE_VALID_G3:
    VERDICT = ("Trên tập xác nhận độc lập, v4 vượt có ý nghĩa (Holm) cả đối chứng chỉ-contains, vị từ hoán vị và topology rewired ⇒ "
               "quan hệ (đúng cạnh, đúng vị từ) có giá trị thật cho xếp hạng.")
elif conf_sup:
    VERDICT = ("Quan hệ CHƯA được xác nhận đầy đủ: trên tập xác nhận, v4 không vượt có ý nghĩa mọi đối chứng quan hệ "
               f"(ủng hộ: {[h for h, v in conf_sup.items() if v] or '—'}). Cơ chế dùng quan hệ có thật (P/Q/X-test) nhưng tác động lên MRR "
               "không đủ để kết luận.")
else:
    VERDICT = "Chưa có kết luận xác nhận: tập xác nhận độc lập NOT RUN hoặc DEV. Số VAL / test MSVD chỉ là phát triển / thăm dò."
L.append("KẾT LUẬN: " + VERDICT)
_tf1 = ", ".join(f"{k_} {v_['F1']:.3f}" for k_, v_ in TAGGER_REPORT.items())
L.append(f"Giới hạn: nhãn video từ CLIP zero-shot (F1 {_tf1}); quan hệ là thống kê cấp LỚP từ caption train (không có quan hệ cấp video "
         "cho video mới); parser quan hệ heuristic; val chỉ 100 video; PASS của kiểm thử phần mềm không xác nhận giả thuyết khoa học.")
CONCLUSION = "\n".join(L)
(RESULTS / "graph_conclusion.md").write_text("# Kết luận đồ thị — Level 4(b) (tự sinh từ số liệu)\n\n" + "\n".join(f"- {x}" for x in L) + "\n")

_csv_conf = pd.read_csv(RESULTS / "graph_confirmatory_tests.csv")
_csv_abl = pd.read_csv(RESULTS / "graph_ablation_results.csv")
_m1 = CONF_T.dropna(subset=["diff"]) if "diff" in CONF_T else CONF_T.iloc[:0]
_m2 = _csv_conf.dropna(subset=["diff"]) if "diff" in _csv_conf else _csv_conf.iloc[:0]
ok_conf = len(_m1) == len(_m2) and np.allclose(_m1["diff"].values, _m2["diff"].values) and \
    np.allclose(_m1["p_holm"].values, _m2["p_holm"].values)
ok_abl = all(abs(float(_csv_abl.loc[_csv_abl.experiment_id == k, "val_MRR"].iloc[0]) - EXPS[k]["val_MRR"][0]) < 1e-9
             for k in _csv_abl.experiment_id if EXPS[k]["status"] == "DONE")
ok_text = all(f"{r['diff']:+.4f}" in CONCLUSION for _, r in _m1.iterrows()
              if r["hypothesis"] in {h for h, _, _ in PRIMARY_COMPARISONS} and r["part"] in {EVALS[k_]["part"]["name"] for k_ in (EX, CK) if k_})
check("R1", "[12.10] kết quả trong báo cáo / kết luận khớp các file CSV đầu ra (đọc lại từ đĩa)", ok_conf and ok_abl and ok_text,
      dict(confirmatory=ok_conf, ablation_val=ok_abl, conclusion_text=ok_text))

REQUIRED = ["graph_schema_v3.json", "graph_edges_v3.csv", "relation_quality_audit.csv", "graph_randomization_audit.csv",
            "graph_ablation_results.csv", "graph_relation_analysis.csv", "graph_xai_fidelity.csv", "frozen_config.json",
            "experiment_manifest.json", "graph_conclusion.md"]


def _nonempty(p):
    if not p.exists() or p.stat().st_size == 0:
        return False
    if p.suffix == ".csv":
        try:
            return len(pd.read_csv(p)) > 0
        except Exception:
            return False
    return True


files = {f: _nonempty(RESULTS / f) for f in REQUIRED}
criteria = [
    ("1", "Quan hệ được DÙNG có điều kiện theo truy vấn (vị từ, vai, loại cạnh tác động attention)", *_st("Q1", "Q2")),
    ("2", "Điểm = Σ số hạng giải thích (phân rã chính xác); xoá cạnh ⇒ đúng phần đóng góp", *_st("Q3", "X1", "X3")),
    ("3", "Có đối chứng tách quan hệ / vị từ / topology / điều kiện truy vấn (G1, G2, G4, G5, G6 + audit)", *_st("G6", "G7")),
    ("4", "Ablation công bằng (inventory cố định, cùng ứng viên, cùng giao thức huấn luyện)", *_st("G1", "D2")),
    ("5", "Không dùng test đã xem để chọn mô hình", *_st("D1", "D4", "C3")),
    ("6", "Đủ bằng chứng xác định tốt hơn / tương đương / kém hơn (tập xác nhận độc lập, Holm)", *_st("D3")),
    ("7", "Mọi kết luận có nguồn số liệu kiểm chứng", *_st("R1")),
]
md = ["# Audit đồ thị — Level 4(b): QC-RGAT + đối chứng cho text→video (MSVD → MSR-VTT)", "",
      f"{time.strftime('%Y-%m-%d %H:%M:%S')} | dataset={CFG.dataset} | dev_mode={CFG.dev_mode} | Kaggle={RUN_ON_KAGGLE} | "
      f"{ENV_INFO.get('gpu', DEVICE)} | CLIP {CFG.clip_model} | seeds {CFG.seeds} | epochs {CFG.epochs} | tập ngoài: {EXT_STATUS}", "",
      "## Tiêu chí nghiệm thu", "", "| # | Tiêu chí | Trạng thái | Bằng chứng |", "|---|---|---|---|"]
md += [f"| {a} | {b} | **{c}** | {d} |" for a, b, c, d in criteria]
md += ["", "## File bàn giao", "", "| File | Có dữ liệu |", "|---|---|"] + [f"| {f} | {'PASS' if ok else 'FAIL'} |" for f, ok in files.items()]
md += ["", "## Quyết định theo stage (chỉ VAL; incumbent = phương án đơn giản hơn)", "",
       "| stage | incumbent | ứng viên | Δ val MRR | p Holm | chấp nhận |", "|---|---|---|---|---|---|"]
md += [f"| {d['stage']} | {d['incumbent']} | {d['candidate']} | {d['delta_val_MRR']:+.4f} | {d['p_holm']:.3f} | {d['adopted']} |"
       if d.get("value") is not None else f"| {d['stage']} | {d['incumbent']} | {d['candidate']} | NOT RUN | — | False |" for d in DECISIONS]
md += ["", "## Thí nghiệm NOT RUN", ""] + ([f"- {n['stage']} {n.get('experiment_id', '')}: {n['reason']}" for n in NOT_RUN] or ["- (không có)"])
md += ["", "## Kết luận", ""] + [f"- {x}" for x in L]
md += ["", "## Mọi kiểm tra", "", "| id | kiểm tra | trạng thái |", "|---|---|---|"] + [f"| {c['id']} | {c['check']} | {c['status']} |" for c in CHECKS]
(RESULTS / "graph_audit_report.md").write_text("\n".join(md) + "\n")
display(pd.DataFrame(criteria, columns=["#", "tiêu chí", "trạng thái", "bằng chứng"]))
print("\n" + CONCLUSION)
print(f"\nFile bàn giao có dữ liệu: {sum(files.values())}/{len(files)} | kiểm tra FAIL: {sum(c['status'] == 'FAIL' for c in CHECKS)} "
      f"→ {RESULTS / 'graph_audit_report.md'}")


In [108]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== Level 4(b) — KIỂM KÊ DỮ LIỆU · AUDIT RÒ RỈ · NGHIỆM THU · KẾT LUẬN · MANIFEST (tự sinh từ lần chạy này) ==============================
mark_stage("level4b_report", requires=("report", "demo"))
BENCH = benchmark_status()
_g3 = graph_for(G3_DESIGN, CFG.seeds[0])[0]
GRAPH_INV = graph_inventory(_g3)
_cache_dirs = [CFG.out_dir / "cache"] + [Path(d) for d in CFG.extra_cache_dirs] + \
    ([p for p in Path("/kaggle/input").glob("*/**/cache") if p.is_dir()] if RUN_ON_KAGGLE else [])
_caches = [dict(file=p.name, dir=str(p.parent), mb=round(p.stat().st_size / 2 ** 20, 1), used_by_this_run=CFG.clip_model in p.name and
                CFG.clip_weights in p.name) for d in _cache_dirs if d.exists() for p in sorted(d.glob("*.npz"))]
_ck = sorted((CFG.out_dir / "checkpoints").glob("*")) if (CFG.out_dir / "checkpoints").exists() else []
DATA_INVENTORY = dict(
    created=time.strftime("%Y-%m-%d %H:%M:%S"), official_run=CFG.dataset == "msvd" and not CFG.dev_mode, dev_mode=CFG.dev_mode,
    datasets=[dict(key="msvd", **DATA_SOURCES["msvd"], videos_loaded=N_MSVD, video_source=DATA_REPORT.get("source_label"),
                   splits={s: DATA_REPORT["splits"][s] for s in DATA_REPORT["splits"]}),
              dict(key="msrvtt", **DATA_SOURCES["msrvtt"], status=EXT_STATUS,
                   report={k_: v_ for k_, v_ in EXT_REPORT.items() if k_ != "missing_examples"})],
    loaded_counts_by_split=split_counts(), benchmark_status=BENCH, benchmark_looks_before=BENCHMARK_LOOKS_BEFORE,
    split_manifest=dict(file="split_manifest.json", msvd_sha1=SPLIT["sha1"], ext_sha1=SPLIT.get("ext_sha1"),
                        rule="split CHÍNH THỨC của MSVD theo video; tập ngoài theo luật seed cố định (load_msrvtt)"),
    features=dict(clip=f"{CFG.clip_model}/{CFG.clip_weights} (đóng băng)", frames_per_video=CFG.n_frames_max,
                  frame_policy=CFG.frame_policy, pooling_selected_on_val=POOL_SEL, video_dim=int(VID_FEAT.shape[1]),
                  n_video_embeddings=int(VID_FEAT.shape[0]), n_text_embeddings=int(TXT_FEAT.shape[0]), caches=_caches),
    tagger=dict(selected_on_val=TAG_SELECT2, report=TAGGER_REPORT),
    graph=dict(design=G3_DESIGN, **GRAPH_INV), checkpoints=dict(n_files=len(_ck), mb=round(sum(p.stat().st_size for p in _ck) / 2 ** 20, 1),
                                                               names=[p.name for p in _ck][:60]),
    environment=ENV_INFO)
save_json(DATA_INVENTORY, RESULTS / "data_inventory.json")
check("I1", "kiểm kê: số video / caption theo split khớp dữ liệu đã nạp; manifest split có sha1; cache có metadata mô hình",
      all(DATA_INVENTORY["loaded_counts_by_split"][s]["videos"] == len(SPLIT[s]) for s in DATA_INVENTORY["loaded_counts_by_split"] if s in SPLIT)
      and bool(SPLIT["sha1"]), {s: (v["videos"], v["captions"]) for s, v in DATA_INVENTORY["loaded_counts_by_split"].items()}, hard=False)
check("I2", "đồ thị đang chạy: không self-loop, không cạnh trùng (cùng nguồn, đích, loại); có thống kê bậc và node cô lập",
      GRAPH_INV["self_loops"] == 0 and GRAPH_INV["duplicate_edges"] == 0,
      {k_: GRAPH_INV[k_] for k_ in ("n_edges", "self_loops", "duplicate_edges", "isolated_non_video_nodes", "videos_without_tag")}, hard=False)

# ---------------- data_leakage_audit.md ----------------
_by = {c["id"]: c for c in CHECKS}
_LEAK = [("A5", "split theo video, không video nào ở hai split"), ("C7", "split MSVD tái lập được"),
         ("C1", "canary chỉ có ở caption test / tập ngoài KHÔNG lọt vào từ vựng, quan hệ, vị từ"), ("C1b", "đối chứng dương của canary"),
         ("C2", "xáo caption test / tập ngoài ⇒ mọi artifact học được GIỐNG HỆT"), ("C3", "hàng rào chặn mở test / dùng caption sai mục đích"),
         ("C8", "nhật ký: caption test / tập ngoài chỉ dùng cho encode / query_input / eval / report"),
         ("C9", "provenance: học chỉ từ train, chọn chỉ từ val"), ("D1", "mọi lựa chọn trước đóng băng, chỉ dựa trên val"),
         ("D4", "mọi lần chạm test / tập ngoài sau đóng băng"), ("A6", "tập ngoài rời các tập đã dùng"), ("QS1", "bộ truy vấn dựng trước khi chấm mô hình")]
_acc = CAPS.access_table()
L = ["# Audit rò rỉ dữ liệu — Level 4(b) (tự sinh)", "",
     f"{DATA_INVENTORY['created']} | chạy chính thức: {DATA_INVENTORY['official_run']} | tập ngoài: {EXT_STATUS}", "",
     "## Trạng thái từng benchmark", "", "| Tập | Đã nhìn trước | Trạng thái |", "|---|---|---|"]
L += [f"| {k_} | {v_.get('looks_before', '—')} | **{v_['status']}** |" for k_, v_ in BENCH.items()]
L += ["", "## Kiểm tra chống rò rỉ", "", "| id | nội dung | trạng thái | bằng chứng |", "|---|---|---|---|"]
L += [f"| {i} | {d} | {_by[i]['status'] if i in _by else 'NOT RUN'} | {(_by[i]['evidence'][:220] if i in _by else '').replace('|', '/')} |" for i, d in _LEAK]
L += ["", "## Nhật ký dùng caption (mục đích × split)", "", "| mục đích | split | số lần |", "|---|---|---|"]
L += [f"| {r.purpose} | {r.split} | {int(r.n)} |" for r in _acc.itertuples()]
L += ["", "## Chất lượng ngữ nghĩa của cạnh", "",
      f"- Kiểm chứng tự động (parser độc lập, SILVER — không phải precision): {AUTO_STATUS}",
      f"- Nhãn người chấm: {REL_QUALITY_STATUS}",
      "- ⇒ Chất lượng ngữ nghĩa của cạnh quan hệ **CHƯA được xác nhận bởi người**. Tỉ lệ đồng thuận giữa hai parser không phải precision thật.", "",
      "## Nguồn và giấy phép", ""] + [f"- **{v_['name']}** ({v_['paper']}) — `{v_['repo']}@{v_['revision'][:8]}` — {v_['license']} — vai trò: {v_['role']}"
                                     for v_ in DATA_SOURCES.values()]
(RESULTS / "data_leakage_audit.md").write_text("\n".join(L) + "\n")

# ---------------- audit_report.md: ma trận nghiệm thu (mục 12 của master instruction) ----------------
def _req_st(*ids):
    st, ev = _st(*ids)
    return st, ev


def _done(*gids):
    ks = [G_IDS.get(g_, g_) for g_ in gids]
    ok = all(EXPS.get(k_, {}).get("status") == "DONE" and (EXPS[k_].get("runs") or k_ == "G0_clip_only") for k_ in ks)   # CLIP-only không huấn luyện
    return ("PASS" if ok else "NOT RUN"), ", ".join(f"{k_}={EXPS.get(k_, {}).get('status', 'không có')}" for k_ in ks)


def _file(*names):
    ok = all(_nonempty(RESULTS / n) for n in names)
    return ("PASS" if ok else "FAIL"), ", ".join(names)


def _all(*pairs):
    sts = [p[0] for p in pairs]
    st = "FAIL" if "FAIL" in sts else ("NOT RUN" if "NOT RUN" in sts else "PASS")
    return st, " ; ".join(p[1] for p in pairs)


_n_seeds_ok = ("PASS" if len(CFG.seeds) >= 3 else "FAIL", f"{len(CFG.seeds)} seed {CFG.seeds}")
_conf_ok = ("PASS" if "confirm" in EVALS and EVALS["confirm"]["info"]["kind"] == "confirmatory" else "NOT RUN",
            f"tập ngoài: {EXT_STATUS}; kind={EVALS['confirm']['info']['kind'] if 'confirm' in EVALS else '—'}")
REQ = [
    ("A1", "Baseline CLIP đóng băng", *_all(_req_st("B0"), _done("G0_clip_only"))),
    ("A2", "Đồ thị từ nguồn hợp lệ (caption TRAIN + CLIP zero-shot trên frame), đúng schema", *_req_st("G2", "C1", "C9")),
    ("A3", "Ít nhất một GNN đã học (GraphSAGE có kiểu cạnh = R3; QC-RGAT = G3)", *_done("V3REF", "G3")),
    ("A4", "CLIP-only và CLIP + Graph so sánh trên cùng split, cùng ứng viên", *_all(_req_st("D2"), _done("G0_clip_only", "G3"))),
    ("A5", "Ablation quan hệ (G1, G7, G8 huấn luyện lại) + đồ thị ngẫu nhiên hợp lệ (G4, G5)", *_all(_done("G1", "G4", "G5", "G7", "G8"), _req_st("G6", "G7", "AB"))),
    ("A6", "R@1, R@5, R@10, MRR; ≥ 3 seed cho thí nghiệm chính", *_all(_n_seeds_ok, _req_st("D3"))),
    ("B1", "Schema node / cạnh rõ ràng", *_file("graph_schema_v3.json")),
    ("B2", "Nguồn gốc cạnh được ghi (bảng cạnh + provenance)", *_file("graph_edges_v3.csv", "provenance.csv")),
    ("B3", "Không rò rỉ test", *_req_st("A5", "C1", "C1b", "C2", "C3", "C8", "C9", "D1", "D4")),
    ("B4", "Audit cấu trúc (inventory cố định, schema, trọng số, self-loop / trùng / bậc)", *_req_st("G1", "G2", "G4", "I2")),
    ("B5", "Đối chứng ngẫu nhiên đúng thuộc tính bảo toàn đã công bố", *_req_st("G6", "G7")),
    ("B6", "Chất lượng ngữ nghĩa của cạnh được người xác minh", "NOT RUN",
     "chưa có nhãn người chấm ⇒ chưa xác nhận; chỉ có kiểm chứng tự động SILVER (spaCy)"),
    ("C1", "Pipeline text-to-video thật (giải mã video, lấy mẫu frame, gộp, gallery)", *_req_st("A3", "A4", "B1", "B3", "B6", "B7", "B8")),
    ("C2", "Encoder tiền huấn luyện đóng băng + lấy mẫu frame", *_req_st("B0", "B6")),
    ("C3", "Gallery, bộ truy vấn và metric xác định rõ (gồm bộ ≥ 20 truy vấn / nhóm)", *_req_st("B8", "QS1")),
    ("C4", "Xếp hạng lại bằng đồ thị + baseline, đánh giá trên tập CHƯA nhìn", *_all(_req_st("D3"), _conf_ok)),
    ("C5", "Giải thích lấy từ chính cơ chế xếp hạng", *_req_st("Q3", "X1", "X3", "X5")),
    ("C6", "Phân tích thành công / thất bại và giới hạn", *_req_st("K1")),
    ("D1", "Đường đi / quan hệ giải thích là thành phần thật của phép tính", *_req_st("Q3", "X3")),
    ("D2", "Validity, sparsity, fidelity (điểm + thứ hạng) trên ≥ 10 truy vấn", *_all(_req_st("X1", "X5"), _file("xai_rank_fidelity.csv"))),
    ("D3", "≥ 5 thành công + ≥ 5 thất bại được phân tích", *_req_st("K1")),
    ("D4", "Demo: nhập truy vấn → top-k + điểm + đường đi giải thích", *_req_st("DM1")),
    ("E1", "Notebook chạy hết (hoặc báo rõ giới hạn)", "PASS" if CFG.dataset == "msvd" and not CFG.dev_mode else "NOT RUN",
     "chạy chính thức tới ô này" if CFG.dataset == "msvd" and not CFG.dev_mode else f"dataset={CFG.dataset}, dev_mode={CFG.dev_mode}"),
    ("E2", "File môi trường + manifest", *_file("pip_freeze.txt", "env_versions.json", "experiment_manifest.json", "data_inventory.json")),
    ("E3", "Kết quả báo cáo khớp file nguồn", *_req_st("R1")),
]
REQ_DF = pd.DataFrame(REQ, columns=["id", "yêu cầu", "trạng thái", "bằng chứng"])
REQ_DF.to_csv(RESULTS / "requirements_status.csv", index=False)
md = ["# Audit nghiệm thu — Project 3 · Level 4(b) text-to-video (tự sinh)", "",
      f"{DATA_INVENTORY['created']} | chính thức: {DATA_INVENTORY['official_run']} | CLIP {CFG.clip_model} | seeds {CFG.seeds} | epochs {CFG.epochs} | "
      f"tập ngoài: {EXT_STATUS} | Kaggle={RUN_ON_KAGGLE}", "",
      "PASS ở đây là PASS của **kiểm thử / sản phẩm**, không phải kết luận khoa học. Kết luận khoa học nằm ở `level4b_conclusions.md`.", "",
      "| id | yêu cầu | trạng thái | bằng chứng |", "|---|---|---|---|"]
md += [f"| {a} | {b} | **{c}** | {str(d)[:240].replace('|', '/')} |" for a, b, c, d in REQ]
md += ["", "## Kết quả khoa học được báo cáo (FINDINGS)", "", "| id | câu hỏi | kết quả | diễn giải |", "|---|---|---|---|"]
md += [f"| {f['id']} | {f['question']} | {f['result'].replace('|', '/')} | {f['interpretation']} |" for f in FINDINGS]
md += ["", "## Thí nghiệm NOT RUN", ""] + ([f"- {n['stage']} {n.get('experiment_id', '')}: {n['reason']}" for n in NOT_RUN] or ["- (không có)"])
(RESULTS / "audit_report.md").write_text("\n".join(md) + "\n")
display(REQ_DF)

# ---------------- level4b_conclusions.md: 7 câu hỏi bắt buộc (mục 13) ----------------
_P = EVALS[CK]["part"]["name"] if CK else EVALS[EX]["part"]["name"]
_kind = EVALS[CK]["info"]["kind"] if CK else EVALS[EX]["info"]["kind"]


def _row(h):
    r = CONF_T[(CONF_T.hypothesis == h) & (CONF_T.part == _P)]
    return r.iloc[0] if len(r) and "diff" in r and pd.notna(r.iloc[0].get("diff")) else None


def _fmt(h):
    r = _row(h)
    if r is None:
        return "NOT RUN"
    return (f"ΔMRR {r['diff']:+.4f} (CI95 theo video, chưa hiệu chỉnh bội [{r.ci_lo:+.4f}, {r.ci_hi:+.4f}] ⇒ {_verdict(r.ci_lo, r.ci_hi)}; "
            f"giả thuyết một phía sau Holm: p {r.p_holm:.3f} ⇒ {'ỦNG HỘ' if r.supported else 'KHÔNG ủng hộ'})")


def _mrr(k_, part=_P, grp="all"):
    r = RES_ALL[(RES_ALL.experiment_id == k_) & (RES_ALL.part == part) & (RES_ALL.group == grp)]
    return f"{r.iloc[0]['MRR_mean']:.4f} ± {r.iloc[0]['MRR_std']:.4f}" if len(r) else "NOT RUN"


_fam = REL_DF[(REL_DF.section == "family_knockout") & (REL_DF.split == P_MAIN["name"])] if len(REL_DF) else REL_DF
Q = []
Q.append(("1. CLIP + Graph có tốt hơn CLIP-only không?",
          f"Trên {_P} ({_kind}): CLIP-only MRR {_mrr('G0_clip_only')}; QC-RGAT (G3) {_mrr(G_IDS.get('G3'))}; GraphSAGE có kiểu cạnh (R3) "
          f"{_mrr(G_IDS.get('V3REF'))}. H1 (G3 > CLIP): {_fmt('H1_graph_vs_clip')}."))
_gi = SIG_T[(SIG_T.comparison == "G3_vs_clip") & (SIG_T.part == _P) & (SIG_T.group == "indirect")] if len(SIG_T) else SIG_T
_qs = QS_PAIRED[(QS_PAIRED.part == _P) & (QS_PAIRED.group == "indirect") & (QS_PAIRED.a == G_IDS.get("G3"))] if len(QS_PAIRED) else QS_PAIRED
Q.append(("2. Đồ thị có giúp nhóm truy vấn gián tiếp không?",
          (f"Mọi caption nhóm gián tiếp (n={int(_gi.iloc[0].n)}): ΔMRR {_gi.iloc[0]['diff']:+.4f} [CI {_gi.iloc[0].ci_lo:+.4f}, {_gi.iloc[0].ci_hi:+.4f}]"
           if len(_gi) else "NOT RUN") + "; " +
          (f"bộ {int(_qs.iloc[0].n)} truy vấn gián tiếp đăng ký trước: ΔMRR {_qs.iloc[0]['diff']:+.4f} [CI {_qs.iloc[0].ci_lo:+.4f}, {_qs.iloc[0].ci_hi:+.4f}]"
           if len(_qs) else "bộ truy vấn: NOT RUN") + ". So sánh PHỤ (không Holm)."))
Q.append(("3. Những loại cạnh nào thực sự có giá trị?",
          f"Quan hệ vs chỉ-contains (H2): {_fmt('H2_relations_vs_contains_only')}. Họ vai agent_of/object_of (H8, huấn luyện lại không có họ): "
          f"{_fmt('H8_role_edges')}. Họ svo/prep (H9): {_fmt('H9_svo_prep_edges')}. Vị từ đúng cạnh (H4): {_fmt('H4_true_vs_shuffled_predicates')}. "
          f"Topology thật (H5): {_fmt('H5_true_vs_rewired_topology')}. Knockout lúc suy luận ({P_MAIN['name']}): "
          + ("; ".join(f"{r.item} {r.d_MRR_knockout:+.4f}" for r in _fam.itertuples()) if len(_fam) else "NOT RUN")
          + ". Một họ cạnh chỉ được coi là có giá trị khi H tương ứng được ủng hộ."))
Q.append(("4. Dùng đồ thị có làm lời giải thích đáng tin cậy hơn không?",
          f"Giải thích = số hạng thật của điểm (Σ = điểm: {XR_SUMM['decomposition_ok']:.2f}; hợp lệ {XR_SUMM['validity']:.2f}). Xoá giải thích đầy đủ: "
          f"tụt hạng {XR_SUMM['frac_rank_worse_expl']:.0%} truy vấn vs ngẫu nhiên {XR_SUMM['frac_rank_worse_rand']:.0%}; chỉ xoá quan hệ: hạng đổi "
          f"{XAI_SUMM['frac_rank_changed_expl']:.0%} (ngẫu nhiên {XAI_SUMM['frac_rank_changed_rand']:.0%}). Jaccard giải thích giữa seed "
          f"{XAI_SUMM['seed_sensitivity_mean_jaccard']:.2f}. ⇒ Giải thích trung thực với ĐIỂM theo cấu tạo (phân rã chính xác). Về QUYẾT ĐỊNH xếp hạng: "
          + ("xoá giải thích đầy đủ làm tụt hạng nhiều hơn xoá ngẫu nhiên, nên giải thích có chỉ ra phần đồ thị gây ra thứ hạng"
             if _better else "xoá giải thích đầy đủ KHÔNG mạnh hơn xoá ngẫu nhiên, nên không kết luận được giải thích phản ánh nguyên nhân thứ hạng")
          + ("; riêng phần QUAN HỆ hầu như không đổi thứ hạng" if XAI_SUMM["frac_rank_changed_expl"] <= XAI_SUMM["frac_rank_changed_rand"] + 0.05
             else "; phần quan hệ có đổi thứ hạng ở một số truy vấn")
          + f". Độ ổn định giữa seed thấp (Jaccard {XAI_SUMM['seed_sensitivity_mean_jaccard']:.2f}) ⇒ đường đi cụ thể phụ thuộc lần huấn luyện."))
Q.append(("5. Phần video tái sử dụng kiến trúc retrieval và graph explanation đến mức nào?",
          "Tái sử dụng nguyên: CLIP đóng băng (ảnh → frame), top-k theo cosine, điểm cuối α·z(CLIP) + (1−α)·z(đồ thị) chọn α trên val, "
          "schema node / cạnh (video ~ ảnh), GNN (GraphSAGE có kiểu cạnh, QC-RGAT), giải thích bằng phân rã điểm và xoá cạnh. "
          "Phần riêng của video: lấy mẫu 12 frame, gộp theo thời gian (chọn trên val), nhãn video = CLIP zero-shot trên frame, "
          f"quan hệ từ caption train cấp lớp. Bằng chứng chạy: {_st('A4', 'B6', 'B8')[1]}."))
_cnt = Counter(REQ_DF["trạng thái"])
Q.append(("6. Yêu cầu nào PASS, FAIL hay NOT RUN?",
          f"{dict(_cnt)}. FAIL: {', '.join(REQ_DF[REQ_DF['trạng thái'] == 'FAIL'].id) or '—'}; NOT RUN: "
          f"{', '.join(REQ_DF[REQ_DF['trạng thái'] == 'NOT RUN'].id) or '—'} (chi tiết: audit_report.md)."))
Q.append(("7. Level 4(b) được thực hiện đến đâu và giới hạn là gì?",
          f"Đã có pipeline text→video đầy đủ, xếp hạng lại bằng đồ thị, đối chứng, bộ truy vấn trực tiếp/gián tiếp, xAI theo thứ hạng, 5+5 ca, demo. "
          f"Tập xác nhận: {EXT_STATUS} ({EXT_REPORT.get('label', '—')}). Giới hạn: nhãn video là CLIP zero-shot (F1 "
          + ", ".join(f"{k_} {v_['F1']:.3f}" for k_, v_ in TAGGER_REPORT.items()) +
          "); quan hệ là thống kê cấp lớp từ caption train (video mới không có quan hệ riêng); parser heuristic; chất lượng ngữ nghĩa "
          "cạnh chưa được người xác minh; val chỉ 100 video; test MSVD đã nhìn 4 lần trước (thăm dò); fresh1k-B cùng miền với fresh1k đã ảnh "
          "hưởng tới thiết kế. PASS kiểm thử phần mềm không phải kết luận khoa học."))
CONCL = ["# Kết luận Level 4(b) — trả lời 7 câu hỏi bắt buộc (tự sinh từ số liệu của lần chạy này)", "",
         f"Tập chính: **{_P}** ({_kind}). Kết luận chung về quan hệ: {VERDICT}", ""]
for q_, a_ in Q:
    CONCL += [f"## {q_}", "", a_, ""]
(RESULTS / "level4b_conclusions.md").write_text("\n".join(CONCL))
print("\n".join(CONCL))

# ---------------- artifacts_manifest.json ----------------
_desc = {"eval_queries_level4b.csv": "bộ truy vấn trực tiếp / gián tiếp đăng ký trước", "eval_queries_metrics.csv": "metric trên bộ truy vấn",
         "xai_rank_fidelity.csv": "fidelity theo thứ hạng (giải thích đầy đủ)", "case_analysis.md": "5 thành công + 5 thất bại",
         "demo_outputs.json": "đầu ra demo", "data_inventory.json": "kiểm kê dữ liệu", "data_leakage_audit.md": "audit rò rỉ",
         "audit_report.md": "ma trận nghiệm thu", "level4b_conclusions.md": "7 câu kết luận", "graph_ablation_results.csv": "ablation (1 dòng / thí nghiệm)",
         "graph_experiment_results.csv": "metric mọi thí nghiệm × tập × nhóm", "graph_confirmatory_tests.csv": "họ so sánh chính (Holm)",
         "graph_paired_tests.csv": "so sánh phụ theo nhóm", "experiment_manifest.json": "cấu hình, seed, checkpoint mọi thí nghiệm",
         "split_manifest.json": "split tái lập", "graph_schema_v3.json": "schema đồ thị", "provenance.csv": "nguồn gốc artifact"}
ARTIFACTS = [dict(path=p.relative_to(CFG.out_dir).as_posix(), bytes=p.stat().st_size, sha1=file_sha1(p), description=_desc.get(p.name, ""))
             for p in sorted(RESULTS.rglob("*")) if p.is_file() and p.name != "artifacts_manifest.json"]
save_json(dict(created=time.strftime("%Y-%m-%d %H:%M:%S"), out_dir=str(CFG.out_dir), n_files=len(ARTIFACTS),
               not_packed=dict(thumbs=len(list((CFG.out_dir / "thumbs").glob("*.jpg"))), checkpoints=len(_ck)), files=ARTIFACTS),
          RESULTS / "artifacts_manifest.json")
print(f"\nartifacts_manifest.json: {len(ARTIFACTS)} file | yêu cầu: {dict(_cnt)} | FINDINGS: {len(FINDINGS)}")

In [109]:
%%script true
# v5: Phần 4 (MSVD → MSR-VTT) KHÔNG chạy lại ở bản này — kết quả xem ở version trước của notebook.
# ============================== GÓI KẾT QUẢ ĐỂ TẢI VỀ (không gồm dữ liệu) ==============================
mark_stage("pack", requires=("report",))
_z = pack_review("hoàn tất")
assert _z is not None, "Không tạo được file zip xem xét — xem cảnh báo phía trên."
with zipfile.ZipFile(_z) as _zf:
    _names = _zf.namelist()
    _man = json.loads(_zf.read(f"{_z.stem}/pack_manifest.json"))
print(f"📦 {_z} — {_z.stat().st_size / 2**20:.1f} MB, {len(_names)} file. Tải file này về (Kaggle: Output → {_z.name}).")
print("   Notebook đã chạy:", "có (executed_notebook.ipynb, đủ output tới cell trước)" if _man["executed_notebook"]
      else "KHÔNG tìm thấy (chạy ngoài Kaggle/papermill) — tải notebook riêng nếu cần")
print("   Không gói:", "; ".join(f"{k} {v['n_files']} file, {v['mb']} MB" if isinstance(v, dict) else k for k, v in _man["excluded"].items()))